In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [9]:
# ============================================================
# AMLC 2026 — KAGGLE CELL 1
# EXACT CHECKPOINT RESTORE + ENVIRONMENT VERIFICATION
# ============================================================

import os
import gc
import json
import psutil
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq


# ============================================================
# 1) EXACT PACKAGE ROOT
# ============================================================

WORKSPACE_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/"
    "amlc-2026-final-workspace/"
    "AMLC2026_KAGGLE_FINAL"
)

CHECKPOINT_ROOT = (
    WORKSPACE_ROOT /
    "checkpoint_after_cell58_FINAL_20260925"
)

DATASET_ROOT = (
    WORKSPACE_ROOT /
    "dataset"
)

TRAIN_ROOT = DATASET_ROOT / "train"
TEST_ROOT = DATASET_ROOT / "test"

STATE_ROOT = CHECKPOINT_ROOT / "state"
RUNTIME_ROOT = CHECKPOINT_ROOT / "runtime"
BLOCKING_ROOT = CHECKPOINT_ROOT / "blocking"
FULL_TRAIN_ROOT = CHECKPOINT_ROOT / "full_training"


# ============================================================
# 2) BASIC EXISTENCE CHECK
# ============================================================

print("=" * 72)
print("AMLC 2026 — KAGGLE CHECKPOINT RESTORE")
print("=" * 72)

required_dirs = {
    "WORKSPACE_ROOT": WORKSPACE_ROOT,
    "CHECKPOINT_ROOT": CHECKPOINT_ROOT,
    "DATASET_ROOT": DATASET_ROOT,
    "TRAIN_ROOT": TRAIN_ROOT,
    "TEST_ROOT": TEST_ROOT,
    "STATE_ROOT": STATE_ROOT,
    "RUNTIME_ROOT": RUNTIME_ROOT,
}

for name, path in required_dirs.items():
    print(
        f"{'OK   ' if path.exists() else 'MISS '}{name}: {path}"
    )

missing_dirs = [
    name
    for name, path in required_dirs.items()
    if not path.exists()
]

if missing_dirs:
    raise RuntimeError(
        "Missing required directories:\n" +
        "\n".join(missing_dirs)
    )


# ============================================================
# 3) KEY ARTIFACT PATHS
# ============================================================

CANDIDATE_PATH = (
    STATE_ROOT /
    "eval_candidates_address_rescue.parquet"
)

CANDIDATE_COUNTS_PATH = (
    STATE_ROOT /
    "eval_candidate_counts_address_rescue.parquet"
)

CANDIDATE_RESIDUAL_PATH = (
    STATE_ROOT /
    "eval_candidate_residual_address_rescue.parquet"
)

S1_LOOKUP_PATH = (
    RUNTIME_ROOT /
    "s1_pair_lookup.parquet"
)

S2_NAME_PATH = (
    RUNTIME_ROOT /
    "s2_name_lookup.parquet"
)

S2_ADDRESS_PATH = (
    RUNTIME_ROOT /
    "s2_address_lookup.parquet"
)

S3_NAME_PATH = (
    RUNTIME_ROOT /
    "s3_name_lookup.parquet"
)

S3_ADDRESS_PATH = (
    RUNTIME_ROOT /
    "s3_address_lookup.parquet"
)

GT_PATH = (
    FULL_TRAIN_ROOT /
    "gt.parquet"
)

GT_EDGES_PATH = (
    FULL_TRAIN_ROOT /
    "gt_edges.parquet"
)

CHECKPOINT_MANIFEST = (
    CHECKPOINT_ROOT /
    "CHECKPOINT_MANIFEST.json"
)


# ============================================================
# 4) VERIFY CRITICAL FILES
# ============================================================

critical_files = {
    "candidate_pairs": CANDIDATE_PATH,
    "candidate_counts": CANDIDATE_COUNTS_PATH,
    "candidate_residual": CANDIDATE_RESIDUAL_PATH,
    "s1_lookup": S1_LOOKUP_PATH,
    "s2_name_lookup": S2_NAME_PATH,
    "s2_address_lookup": S2_ADDRESS_PATH,
    "s3_name_lookup": S3_NAME_PATH,
    "s3_address_lookup": S3_ADDRESS_PATH,
    "gt": GT_PATH,
    "gt_edges": GT_EDGES_PATH,
    "manifest": CHECKPOINT_MANIFEST,
}

print("\nCritical artifacts:")

for name, path in critical_files.items():
    print(
        f"{'OK   ' if path.exists() else 'MISS '}{name}: {path.name}"
    )

missing_files = [
    name
    for name, path in critical_files.items()
    if not path.exists()
]

if missing_files:
    raise RuntimeError(
        "Missing critical checkpoint files:\n" +
        "\n".join(missing_files)
    )


# ============================================================
# 5) READ MANIFEST
# ============================================================

with open(
    CHECKPOINT_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    CHECKPOINT_INFO = json.load(f)

print("\nCheckpoint status:")
print(
    "  status:",
    CHECKPOINT_INFO.get("status")
)

print(
    "  stopping_point:",
    CHECKPOINT_INFO.get("stopping_point")
)

print(
    "  candidate_pairs:",
    f"{CHECKPOINT_INFO.get('candidate_pairs', 0):,}"
)


# ============================================================
# 6) VERIFY CANDIDATE PARQUET WITHOUT LOADING IT
# ============================================================

candidate_meta = pq.ParquetFile(
    str(CANDIDATE_PATH)
).metadata

candidate_rows = candidate_meta.num_rows

print("\nCandidate artifact:")
print(
    "  rows:",
    f"{candidate_rows:,}"
)

print(
    "  size:",
    round(
        CANDIDATE_PATH.stat().st_size /
        1024**2,
        2
    ),
    "MB"
)

assert candidate_rows == 22_302_012, (
    f"Unexpected candidate count: {candidate_rows:,}"
)


# ============================================================
# 7) READ ONLY SMALL / MANAGEABLE EVAL OBJECTS
# ============================================================

def read_parquet(path):
    return pl.read_parquet(path)


s1_eval = read_parquet(
    RUNTIME_ROOT /
    "s1_eval.parquet"
)

edge_eval = read_parquet(
    RUNTIME_ROOT /
    "edge_eval.parquet"
)

eval_gt = read_parquet(
    RUNTIME_ROOT /
    "eval_gt.parquet"
)

per_s1_recall = read_parquet(
    RUNTIME_ROOT /
    "per_s1_recall.parquet"
)

positive_pair_diagnostics = read_parquet(
    RUNTIME_ROOT /
    "positive_pair_diagnostics.parquet"
)

print("\nEvaluation state:")
print(
    "  s1_eval:",
    f"{s1_eval.height:,} rows"
)

print(
    "  edge_eval:",
    f"{edge_eval.height:,} rows"
)

print(
    "  eval_gt:",
    f"{eval_gt.height:,} rows"
)

print(
    "  per_s1_recall:",
    f"{per_s1_recall.height:,} rows"
)


# ============================================================
# 8) LOAD S1 LOOKUP
#
# Only 100k rows — safe.
# ============================================================

s1_lookup = pl.read_parquet(
    S1_LOOKUP_PATH
)

print(
    "\nS1 lookup:",
    s1_lookup.shape
)


# ============================================================
# 9) KEEP LARGE TABLES ON DISK
#
# These are PATHS, NOT in-memory DataFrames.
# Cell 59 will stream/join them in controlled chunks.
# ============================================================

S2_NAME_PATH = Path(S2_NAME_PATH)
S2_ADDRESS_PATH = Path(S2_ADDRESS_PATH)
S3_NAME_PATH = Path(S3_NAME_PATH)
S3_ADDRESS_PATH = Path(S3_ADDRESS_PATH)


# ============================================================
# 10) VERIFY RAW COMPETITION DATA
# ============================================================

TRAIN_FILES = sorted(
    p.name
    for p in TRAIN_ROOT.iterdir()
    if p.is_file()
)

TEST_FILES = sorted(
    p.name
    for p in TEST_ROOT.iterdir()
    if p.is_file()
)

print("\nTrain files:")
for x in TRAIN_FILES:
    print(" ", x)

print("\nTest files:")
for x in TEST_FILES:
    print(" ", x)

expected_train = {
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
}

expected_test = {
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv",
}

assert expected_train.issubset(
    set(TRAIN_FILES)
), "Train dataset files missing."

assert expected_test.issubset(
    set(TEST_FILES)
), "Test dataset files missing."


# ============================================================
# 11) FINAL RESOURCE REPORT
# ============================================================

mem = psutil.virtual_memory()

print("\n" + "=" * 72)
print("KAGGLE RESTORE SUCCESSFUL")
print("=" * 72)

print(
    "\nRAM:",
    round(mem.used / 1024**3, 2),
    "GB /",
    round(mem.total / 1024**3, 2),
    "GB"
)

print(
    "\nCandidate pairs:",
    f"{candidate_rows:,}"
)

print(
    "\nS1 eval:",
    f"{s1_eval.height:,}"
)

print(
    "\nEverything required for Cell 59 is present."
)

print(
    "\nIMPORTANT:"
)

print(
    "The 22.3M candidates are NOT loaded into RAM."
)

print(
    "The 5M+ S2/S3 lookup tables are NOT loaded into RAM."
)

print(
    "They will be streamed/queried during Cell 59."
)

print(
    "\nNEXT → KAGGLE CELL 2 = PAIRWISE BASELINE / CELL 59"
)

print("=" * 72)

AMLC 2026 — KAGGLE CHECKPOINT RESTORE
OK   WORKSPACE_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL
OK   CHECKPOINT_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925
OK   DATASET_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset
OK   TRAIN_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train
OK   TEST_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/test
OK   STATE_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/state
OK   RUNTIME_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/runtime

Critical artifacts:
OK   candidate_pairs: eval_candidates_address_res

In [55]:
# ============================================================
# AMLC 2026 — KAGGLE DEPENDENCY BOOTSTRAP
# Run BEFORE CELL 59
# ============================================================

import sys
import subprocess
import importlib.util

REQUIRED = {
    "rapidfuzz": "rapidfuzz==3.14.6",
    "lightgbm": "lightgbm==4.6.0",
    "duckdb": "duckdb==1.3.2",
    "polars": "polars==1.35.2",
    "pyarrow": "pyarrow",
    "joblib": "joblib",
    "psutil": "psutil",
}

missing = []

for module, package in REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        missing.append(package)

print("Missing packages:")
for x in missing:
    print("  ", x)

if missing:
    print("\nInstalling...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing,
    ])
else:
    print("\nAll required packages already installed.")

print("\nVerification:")

for module in REQUIRED:
    try:
        mod = __import__(module)
        version = getattr(mod, "__version__", "installed")
        print(f"  OK   {module}: {version}")
    except Exception as e:
        print(f"  FAIL {module}: {e}")

Missing packages:
   rapidfuzz==3.14.6

Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.1 MB/s eta 0:00:00

Verification:
  OK   rapidfuzz: 3.14.6
  OK   lightgbm: 4.6.0
  OK   duckdb: 1.3.2
  OK   polars: 1.35.2
  OK   pyarrow: 24.0.0
  OK   joblib: 1.5.3
  OK   psutil: 5.9.5


In [ ]:
# ============================================================
# REPAIR — restore candidate_recall_ceiling
# ============================================================

candidate_recall_ceiling = {
    "candidate_pairs": 22_302_012,
    "edge_recall_overall": 229_692 / 345_980,
    "edge_recall_s2": 0.673978,
    "edge_recall_s3": 0.654444,
    "full_set_recovery": 33_746 / 94_404,
    "eval_s1": 100_000,
    "eval_positive_edges": 345_980,
    "eval_full_positive_sets": 94_404,
}

print("candidate_recall_ceiling restored:")
for k, v in candidate_recall_ceiling.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6%}" if v <= 1 else f"  {k}: {v:,}")
    else:
        print(f"  {k}: {v:,}" if isinstance(v, int) else f"  {k}: {v}")

print("\n✅ Ready to rerun CELL 59 continuation.")

In [ ]:
# ==============================================================================
# AMLC 2026 — CELL 59 — SELF-CONTAINED PAIR-SAMPLE BRIDGE
#
# PURPOSE:
#   Recover enough state to continue into feature engineering.
#
# IMPORTANT:
#   This intentionally DOES NOT validate source == {"source2","source3"}.
#   The surviving Cell-58 parquet uses a different source encoding.
#
#   We preserve the original `source` column exactly as stored and derive
#   `source_is_s3` robustly.
# ==============================================================================

from pathlib import Path
import json
import time
import numpy as np
import polars as pl

print("=" * 78)
print("AMLC 2026 — CELL 59 — SELF-CONTAINED PAIR-SAMPLE BRIDGE")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------

MASTER_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL"
)

CANDIDATE_PATH = (
    MASTER_ROOT
    / "checkpoint_after_cell58_FINAL_20260925"
    / "state"
    / "eval_candidates_tier12_source_capped.parquet"
)

GT_PATH = (
    MASTER_ROOT
    / "dataset"
    / "train"
    / "train_ground_truth.tsv"
)

PAIR_SAMPLE_ROOT = Path(
    "/kaggle/working/AMLC2026/pair_sample_cell58"
)
PAIR_SAMPLE_ROOT.mkdir(parents=True, exist_ok=True)

BASELINE_PAIR_PATH = (
    PAIR_SAMPLE_ROOT / "baseline_candidate_pairs.parquet"
)

METADATA_PATH = (
    PAIR_SAMPLE_ROOT / "cell59_bridge_metadata.json"
)

TARGET_ROWS = 3_429_214
SEED = 2026

print(f"\nCandidate : {CANDIDATE_PATH}")
print(f"GT        : {GT_PATH}")
print(f"Output    : {BASELINE_PAIR_PATH}")

assert CANDIDATE_PATH.exists(), f"Candidate artifact missing: {CANDIDATE_PATH}"
assert GT_PATH.exists(), f"Ground truth missing: {GT_PATH}"

# ------------------------------------------------------------------------------
# 1. LOAD CANDIDATE POOL
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. LOAD SURVIVING CELL-58 CANDIDATE POOL")
print("-" * 78)

cand = pl.read_parquet(CANDIDATE_PATH)

print(f"Rows: {cand.height:,}")
print(f"Columns: {cand.columns}")

required = {
    "s1_entity_id",
    "candidate_entity_id",
    "source",
}

missing = required - set(cand.columns)

assert not missing, f"Missing columns: {sorted(missing)}"

cand = cand.with_columns([
    pl.col("s1_entity_id").cast(pl.Utf8),
    pl.col("candidate_entity_id").cast(pl.Utf8),
    pl.col("source").cast(pl.Utf8),
])

# ------------------------------------------------------------------------------
# 2. INSPECT ACTUAL SOURCE ENCODING — NO ASSERTION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. ACTUAL SOURCE VALUES")
print("-" * 78)

source_values = (
    cand
    .select("source")
    .unique()
    .sort("source")
)

print(source_values)

source_counts = (
    cand
    .group_by("source")
    .agg(pl.len().alias("rows"))
    .sort("source")
)

print("\nSource counts:")
print(source_counts)

# ------------------------------------------------------------------------------
# 3. ROBUST SOURCE NORMALIZATION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. BUILD ROBUST source_is_s3 FLAG")
print("-" * 78)

cand = cand.with_columns(
    pl.col("source")
    .str.strip_chars()
    .str.to_lowercase()
    .alias("_source_norm")
)

# Anything whose normalized source clearly denotes source 3 is S3.
#
# Handles forms such as:
#   source3
#   source_3
#   s3
#   s_3
#   3
#   S3
#
# Everything else is treated as non-S3 / S2 for this 2-source training pool.

cand = cand.with_columns(
    pl.when(
        pl.col("_source_norm").str.contains(r"(source|src|s)[_\- ]*3$")
        | (pl.col("_source_norm") == "3")
    )
    .then(1)
    .otherwise(0)
    .cast(pl.Int8)
    .alias("source_is_s3")
)

source_map_check = (
    cand
    .group_by(["source", "source_is_s3"])
    .agg(pl.len().alias("rows"))
    .sort(["source", "source_is_s3"])
)

print(source_map_check)

cand = cand.drop("_source_norm")

# ------------------------------------------------------------------------------
# 4. ADD MISSING BLOCKER FLAGS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. NORMALIZE BLOCKER COLUMNS")
print("-" * 78)

for col in [
    "block_exact",
    "block_first_last",
    "block_address_exact",
    "block_address_numeric",
]:
    if col not in cand.columns:
        print(f"Adding missing {col}=0")
        cand = cand.with_columns(
            pl.lit(0, dtype=pl.Int8).alias(col)
        )
    else:
        cand = cand.with_columns(
            pl.col(col).fill_null(0).cast(pl.Int8).alias(col)
        )

# ------------------------------------------------------------------------------
# 5. DEDUP CANDIDATE PAIRS
# ------------------------------------------------------------------------------

before = cand.height

cand = (
    cand
    .unique(
        subset=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        keep="first",
        maintain_order=False,
    )
)

after = cand.height

print(f"Before dedup: {before:,}")
print(f"After dedup : {after:,}")
print(f"Removed     : {before - after:,}")

# Surviving historical artifact is expected to be unique.
# Do not hard-fail if it isn't — continue.
if before != after:
    print("⚠️ Duplicate pair keys existed; first row retained.")

# ------------------------------------------------------------------------------
# 6. LOAD OFFICIAL GROUND TRUTH
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. LOAD OFFICIAL GROUND TRUTH")
print("-" * 78)

gt_raw = pl.read_csv(
    GT_PATH,
    separator="\t",
    has_header=True,
    infer_schema_length=10000,
)

print(f"GT rows: {gt_raw.height:,}")
print(f"GT columns: {gt_raw.columns}")

assert "source1_entity_id" in gt_raw.columns
assert "matched_entity_ids" in gt_raw.columns

# Explicitly align GT key naming with candidate artifact.
gt = (
    gt_raw
    .select([
        pl.col("source1_entity_id")
        .cast(pl.Utf8)
        .alias("s1_entity_id"),

        pl.col("matched_entity_ids")
        .cast(pl.Utf8)
        .alias("matched_entity_ids"),
    ])
)

# ------------------------------------------------------------------------------
# 7. EXPAND GT TO PAIR EDGES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. EXPAND GT EDGES")
print("-" * 78)

gt_edges = (
    gt
    .with_columns(
        pl.col("matched_entity_ids")
        .str.split(",")
        .alias("candidate_entity_id")
    )
    .explode("candidate_entity_id")
    .with_columns(
        pl.col("candidate_entity_id")
        .str.strip_chars()
        .cast(pl.Utf8)
    )
    .filter(
        pl.col("candidate_entity_id").is_not_null()
        & (pl.col("candidate_entity_id") != "")
    )
    .select([
        "s1_entity_id",
        "candidate_entity_id",
    ])
    .unique(
        subset=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        maintain_order=False,
    )
    .with_columns(
        pl.lit(1, dtype=pl.Int8).alias("is_positive")
    )
)

print(f"Expanded GT edges: {gt_edges.height:,}")

assert gt_edges.height == 7_638_365, (
    f"GT edge count mismatch: {gt_edges.height:,}"
)

print("✅ Official 7,638,365-edge GT recovered.")

# ------------------------------------------------------------------------------
# 8. LABEL CANDIDATE POOL
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. LABEL CANDIDATE POOL")
print("-" * 78)

labeled = (
    cand
    .join(
        gt_edges,
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="left",
    )
    .with_columns(
        pl.col("is_positive")
        .fill_null(0)
        .cast(pl.Int8)
    )
)

candidate_positive_count = (
    labeled
    .select(pl.col("is_positive").sum())
    .item()
)

candidate_negative_count = (
    labeled.height - candidate_positive_count
)

candidate_recall_ceiling = (
    candidate_positive_count / gt_edges.height
)

print(f"Candidate rows       : {labeled.height:,}")
print(f"Positive candidates  : {candidate_positive_count:,}")
print(f"Negative candidates  : {candidate_negative_count:,}")
print(f"Candidate recall cap : {candidate_recall_ceiling:.6%}")

# ------------------------------------------------------------------------------
# 9. CANDIDATE COUNT FEATURES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("8. COMPUTE CANDIDATE COUNT FEATURES")
print("-" * 78)

source_counts = (
    labeled
    .group_by([
        "s1_entity_id",
        "source",
    ])
    .agg(
        pl.len().alias("candidate_count_source")
    )
)

total_counts = (
    labeled
    .group_by("s1_entity_id")
    .agg(
        pl.len().alias("candidate_count_total")
    )
)

labeled = (
    labeled
    .join(
        source_counts,
        on=[
            "s1_entity_id",
            "source",
        ],
        how="left",
    )
    .join(
        total_counts,
        on="s1_entity_id",
        how="left",
    )
)

# ------------------------------------------------------------------------------
# 10. DETERMINISTIC NEGATIVE SAMPLING
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("9. BUILD 3,429,214-ROW TRAINING PAIR SAMPLE")
print("-" * 78)

assert candidate_positive_count <= TARGET_ROWS, (
    f"Candidate positives ({candidate_positive_count:,}) exceed "
    f"target ({TARGET_ROWS:,})"
)

negative_needed = TARGET_ROWS - candidate_positive_count

print(f"Target rows   : {TARGET_ROWS:,}")
print(f"Positives kept: {candidate_positive_count:,}")
print(f"Negatives req : {negative_needed:,}")

# Split positives / negatives.
positive_pairs = (
    labeled
    .filter(pl.col("is_positive") == 1)
)

negative_pairs = (
    labeled
    .filter(pl.col("is_positive") == 0)
)

assert negative_needed <= negative_pairs.height

# Stable hash independent of dataframe row order.
negative_pairs = (
    negative_pairs
    .with_columns(
        pl.concat_str(
            [
                pl.lit(str(SEED)),
                pl.col("source"),
                pl.col("s1_entity_id"),
                pl.col("candidate_entity_id"),
            ],
            separator="|",
        )
        .hash(seed=SEED)
        .alias("_sample_hash")
    )
    .sort("_sample_hash")
    .head(negative_needed)
    .drop("_sample_hash")
)

sampled = pl.concat(
    [
        positive_pairs,
        negative_pairs,
    ],
    how="vertical_relaxed",
)

print(f"Sample rows before final sort: {sampled.height:,}")

assert sampled.height == TARGET_ROWS

# ------------------------------------------------------------------------------
# 11. FINAL FEATURE-ENGINEERING INPUT COLUMNS
# ------------------------------------------------------------------------------

FINAL_COLUMNS = [
    "s1_entity_id",
    "candidate_entity_id",
    "source",
    "source_is_s3",
    "block_exact",
    "block_first_last",
    "block_address_exact",
    "block_address_numeric",
    "candidate_count_source",
    "candidate_count_total",
    "is_positive",
]

for col in FINAL_COLUMNS:
    assert col in sampled.columns, f"Missing final column: {col}"

sampled = sampled.select(FINAL_COLUMNS)

# Stable ordering.
sampled = sampled.sort(
    [
        "s1_entity_id",
        "source",
        "candidate_entity_id",
    ]
)

# ------------------------------------------------------------------------------
# 12. FINAL SANITY CHECKS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("10. FINAL SANITY CHECKS")
print("-" * 78)

assert sampled.height == TARGET_ROWS

dup_pairs = (
    sampled
    .group_by([
        "s1_entity_id",
        "candidate_entity_id",
    ])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print(f"Duplicate pairs: {dup_pairs:,}")

assert dup_pairs == 0

sample_positive_count = (
    sampled
    .select(pl.col("is_positive").sum())
    .item()
)

print(f"Sample positives: {sample_positive_count:,}")
print(f"Sample negatives: {sampled.height - sample_positive_count:,}")

# Every retained positive must actually be in GT.
bad_positive_count = (
    sampled
    .filter(pl.col("is_positive") == 1)
    .join(
        gt_edges.select([
            "s1_entity_id",
            "candidate_entity_id",
        ]),
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="anti",
    )
    .height
)

print(f"Bad positive labels: {bad_positive_count:,}")

assert bad_positive_count == 0

# ------------------------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("11. SAVE PAIR SAMPLE")
print("-" * 78)

if BASELINE_PAIR_PATH.exists():
    BASELINE_PAIR_PATH.unlink()

sampled.write_parquet(
    BASELINE_PAIR_PATH,
    compression="zstd",
    statistics=True,
)

print(f"Saved: {BASELINE_PAIR_PATH}")

# ------------------------------------------------------------------------------
# 14. RELOAD CHECK
# ------------------------------------------------------------------------------

check = pl.read_parquet(BASELINE_PAIR_PATH)

print(f"Reloaded rows: {check.height:,}")

assert check.height == TARGET_ROWS
assert check.columns == FINAL_COLUMNS

# ------------------------------------------------------------------------------
# 15. CREATE DOWNSTREAM GLOBALS
# ------------------------------------------------------------------------------

# These are deliberately exported so the next feature-engineering cell can
# consume this bridge without requiring the old Cell-58 Python state.

BASELINE_ROOT = Path(
    "/kaggle/working/AMLC2026/submission_01_baseline"
)
BASELINE_ROOT.mkdir(parents=True, exist_ok=True)

positive_edges = gt_edges.select([
    "s1_entity_id",
    "candidate_entity_id",
])

# Original validation convention:
# hash(S1 ID, seed=2026) % 10 == 0
#
# We construct this from the S1 universe rather than storing 200k+ Python
# objects unnecessarily.
all_s1 = (
    gt
    .select("s1_entity_id")
    .unique()
)

val_s1_df = (
    all_s1
    .with_columns(
        pl.col("s1_entity_id")
        .hash(seed=SEED)
        .mod(10)
        .alias("_fold")
    )
    .filter(pl.col("_fold") == 0)
    .select("s1_entity_id")
)

# A Python set is convenient for downstream membership checks.
val_s1 = set(
    val_s1_df
    .get_column("s1_entity_id")
    .to_list()
)

print(f"\nValidation S1 count: {len(val_s1):,}")

# ------------------------------------------------------------------------------
# 16. SAVE METADATA
# ------------------------------------------------------------------------------

metadata = {
    "cell": "59",
    "mode": "self-contained-pair-sample-bridge",
    "seed": SEED,

    "candidate_path": str(CANDIDATE_PATH),
    "gt_path": str(GT_PATH),
    "output_path": str(BASELINE_PAIR_PATH),

    "candidate_rows": int(cand.height),
    "gt_edges": int(gt_edges.height),

    "candidate_positive_count": int(candidate_positive_count),
    "candidate_negative_count": int(candidate_negative_count),

    "candidate_recall_ceiling": float(candidate_recall_ceiling),

    "target_sample_rows": int(TARGET_ROWS),
    "sample_positive_count": int(sample_positive_count),
    "sample_negative_count": int(
        TARGET_ROWS - sample_positive_count
    ),

    "validation_s1_count": int(len(val_s1)),

    "source_values": [
        str(x)
        for x in source_values.get_column("source").to_list()
    ],

    "note": (
        "Original Cell-58 candidate artifact preserved its own source "
        "encoding. No strict source2/source3 assertion is used."
    ),
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------------------------
# 17. FINAL
# ------------------------------------------------------------------------------

elapsed = (time.time() - T0) / 60

print("\n" + "=" * 78)
print("✅ CELL 59 BRIDGE COMPLETE")
print("=" * 78)

print(f"""
PAIR SAMPLE:
  {BASELINE_PAIR_PATH}

ROWS:
  {check.height:,}

POSITIVES:
  {sample_positive_count:,}

NEGATIVES:
  {check.height - sample_positive_count:,}

GT EDGES:
  {gt_edges.height:,}

CANDIDATE RECALL CEILING:
  {candidate_recall_ceiling:.6%}

SOURCE VALUES:
  {[str(x) for x in source_values.get_column("source").to_list()]}

VALIDATION S1:
  {len(val_s1):,}

RUNTIME:
  {elapsed:.2f} min

✅ No source-value assertion.
✅ Official GT joined.
✅ Exact 3,429,214 rows produced.
✅ Pair uniqueness verified.
✅ Positive-label audit passed.
✅ BASELINE_PAIR_PATH exported.
✅ positive_edges exported.
✅ val_s1 exported.
""")

In [ ]:
# ==============================================================================
# AMLC 2026 — CHECKPOINT AFTER CELL 59 BRIDGE
# Exact resumable state before CELL 61
# ==============================================================================

from pathlib import Path
import json
import hashlib
import shutil
import subprocess
import sys
import time

import polars as pl


print("=" * 78)
print("AMLC 2026 — CHECKPOINT AFTER CELL 59 BRIDGE")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# SOURCE STATE
# ------------------------------------------------------------------------------

MASTER_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL"
)

PAIR_SAMPLE_ROOT = Path(
    "/kaggle/working/AMLC2026/pair_sample_cell58"
)

BASELINE_PAIR_PATH = (
    PAIR_SAMPLE_ROOT / "baseline_candidate_pairs.parquet"
)

GT_PATH = (
    MASTER_ROOT
    / "dataset"
    / "train"
    / "train_ground_truth.tsv"
)

# ------------------------------------------------------------------------------
# CHECKPOINT LOCATION
# ------------------------------------------------------------------------------

CHECKPOINT_ROOT = Path(
    "/kaggle/working/AMLC2026/checkpoint_after_cell59_BRIDGE_20260925"
)

if CHECKPOINT_ROOT.exists():
    shutil.rmtree(CHECKPOINT_ROOT)

CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

STATE_ROOT = CHECKPOINT_ROOT / "state"
STATE_ROOT.mkdir(parents=True, exist_ok=True)

# Portable zip goes beside checkpoint.
CHECKPOINT_ZIP = Path(
    "/kaggle/working/AMLC2026/AMLC2026_AFTER_CELL59_BRIDGE_20260925.zip"
)

if CHECKPOINT_ZIP.exists():
    CHECKPOINT_ZIP.unlink()

print(f"\nCheckpoint root:\n{CHECKPOINT_ROOT}")

# ------------------------------------------------------------------------------
# 1. REQUIRED LIVE ARTIFACT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. VERIFY CURRENT CELL-59 STATE")
print("-" * 78)

assert BASELINE_PAIR_PATH.exists(), (
    f"Missing pair sample:\n{BASELINE_PAIR_PATH}"
)

pair_check = pl.read_parquet(BASELINE_PAIR_PATH)

print(f"Pair rows   : {pair_check.height:,}")
print(f"Pair columns: {pair_check.columns}")

assert pair_check.height == 3_429_214

required_pair_columns = [
    "s1_entity_id",
    "candidate_entity_id",
    "source",
    "source_is_s3",
    "block_exact",
    "block_first_last",
    "block_address_exact",
    "block_address_numeric",
    "candidate_count_source",
    "candidate_count_total",
    "is_positive",
]

assert pair_check.columns == required_pair_columns

# ------------------------------------------------------------------------------
# 2. RECOVER / VERIFY POSITIVE EDGES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. BUILD CHECKPOINT POSITIVE-EDGE TABLE")
print("-" * 78)

if "positive_edges" in globals():
    positive_edges_ckpt = positive_edges.clone()
else:
    positive_edges_ckpt = (
        pair_check
        .filter(pl.col("is_positive") == 1)
        .select([
            "s1_entity_id",
            "candidate_entity_id",
        ])
    )

print(f"Positive edges available in checkpoint: {positive_edges_ckpt.height:,}")

# IMPORTANT:
# The official full GT has 7,638,365 edges. The pair sample contains only
# candidate positives, so we preserve BOTH:
#   - full official GT edge table
#   - Cell-59 sampled positive edge table

GT_EDGES_PATH = STATE_ROOT / "gt_edges_full.parquet"

print("Re-expanding official GT for durable checkpoint...")

gt_raw = pl.read_csv(
    GT_PATH,
    separator="\t",
    has_header=True,
    infer_schema_length=10000,
)

gt_edges_full = (
    gt_raw
    .select([
        pl.col("source1_entity_id")
        .cast(pl.Utf8)
        .alias("s1_entity_id"),

        pl.col("matched_entity_ids")
        .cast(pl.Utf8)
        .alias("matched_entity_ids"),
    ])
    .with_columns(
        pl.col("matched_entity_ids")
        .str.split(",")
        .alias("candidate_entity_id")
    )
    .explode("candidate_entity_id")
    .with_columns(
        pl.col("candidate_entity_id")
        .str.strip_chars()
        .cast(pl.Utf8)
    )
    .filter(
        pl.col("candidate_entity_id").is_not_null()
        & (pl.col("candidate_entity_id") != "")
    )
    .select([
        "s1_entity_id",
        "candidate_entity_id",
    ])
    .unique(
        subset=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        maintain_order=False,
    )
)

print(f"Full GT edges: {gt_edges_full.height:,}")

assert gt_edges_full.height == 7_638_365

gt_edges_full.write_parquet(
    GT_EDGES_PATH,
    compression="zstd",
    statistics=True,
)

# Sample-positive table.
SAMPLE_POSITIVE_PATH = STATE_ROOT / "sample_positive_edges.parquet"

positive_edges_ckpt.write_parquet(
    SAMPLE_POSITIVE_PATH,
    compression="zstd",
    statistics=True,
)

# ------------------------------------------------------------------------------
# 3. COPY THE EXACT PAIR SAMPLE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. COPY EXACT 3,429,214-ROW PAIR SAMPLE")
print("-" * 78)

PAIR_SAMPLE_CKPT = (
    STATE_ROOT / "baseline_candidate_pairs.parquet"
)

shutil.copy2(
    BASELINE_PAIR_PATH,
    PAIR_SAMPLE_CKPT,
)

print(f"Saved:\n{PAIR_SAMPLE_CKPT}")

# ------------------------------------------------------------------------------
# 4. SAVE VALIDATION S1
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. SAVE VALIDATION S1")
print("-" * 78)

VAL_S1_PATH = STATE_ROOT / "val_s1.parquet"

if "val_s1" in globals():

    val_s1_list = sorted(
        str(x)
        for x in val_s1
    )

    val_s1_df = pl.DataFrame({
        "s1_entity_id": val_s1_list
    })

else:

    # Reconstruct from official GT S1 IDs using the same hash convention.
    all_s1 = (
        gt_edges_full
        .select("s1_entity_id")
        .unique()
    )

    val_s1_df = (
        all_s1
        .with_columns(
            pl.col("s1_entity_id")
            .hash(seed=2026)
            .mod(10)
            .alias("_fold")
        )
        .filter(pl.col("_fold") == 0)
        .select("s1_entity_id")
    )

val_s1_df = val_s1_df.unique().sort("s1_entity_id")

print(f"Validation S1 rows: {val_s1_df.height:,}")

val_s1_df.write_parquet(
    VAL_S1_PATH,
    compression="zstd",
)

# ------------------------------------------------------------------------------
# 5. SAVE EXACT STATE METADATA
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. SAVE STATE METADATA")
print("-" * 78)

candidate_recall_ceiling = (
    pair_check
    .filter(pl.col("is_positive") == 1)
    .height
    / 7_638_365.0
)

sample_positive_count = (
    pair_check
    .select(pl.col("is_positive").sum())
    .item()
)

sample_negative_count = (
    pair_check.height - sample_positive_count
)

source_values = sorted(
    str(x)
    for x in
    pair_check
    .select("source")
    .unique()
    .get_column("source")
    .to_list()
)

metadata = {
    "checkpoint_name":
        "AMLC2026_AFTER_CELL59_BRIDGE_20260925",

    "created_at":
        time.strftime("%Y-%m-%d %H:%M:%S"),

    "seed":
        2026,

    "target_pair_rows":
        3_429_214,

    "pair_rows":
        int(pair_check.height),

    "pair_positive_rows":
        int(sample_positive_count),

    "pair_negative_rows":
        int(sample_negative_count),

    "full_gt_edges":
        int(gt_edges_full.height),

    "sample_candidate_recall_ceiling":
        float(candidate_recall_ceiling),

    "validation_s1_rows":
        int(val_s1_df.height),

    "source_values":
        source_values,

    "master_root":
        str(MASTER_ROOT),

    "candidate_source_artifact":
        str(
            MASTER_ROOT
            / "checkpoint_after_cell58_FINAL_20260925"
            / "state"
            / "eval_candidates_tier12_source_capped.parquet"
        ),

    "ground_truth":
        str(GT_PATH),

    "pair_sample":
        "state/baseline_candidate_pairs.parquet",

    "full_gt_edges_file":
        "state/gt_edges_full.parquet",

    "sample_positive_edges_file":
        "state/sample_positive_edges.parquet",

    "validation_s1_file":
        "state/val_s1.parquet",

    "next_cell":
        "CELL 61",
}

METADATA_PATH = CHECKPOINT_ROOT / "CHECKPOINT_METADATA.json"

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

# ------------------------------------------------------------------------------
# 6. ENVIRONMENT MANIFEST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. SAVE ENVIRONMENT MANIFEST")
print("-" * 78)

ENV_PATH = CHECKPOINT_ROOT / "environment.txt"

try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        timeout=120,
    )

    ENV_PATH.write_text(result.stdout)

    print(f"Saved: {ENV_PATH}")

except Exception as e:
    ENV_PATH.write_text(
        f"pip freeze failed: {repr(e)}\n"
    )
    print(f"⚠️ Could not capture pip freeze: {e}")

# ------------------------------------------------------------------------------
# 7. SHA256 MANIFEST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. CREATE SHA256 MANIFEST")
print("-" * 78)

def sha256_file(path: Path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


manifest = {}

for path in CHECKPOINT_ROOT.rglob("*"):
    if path.is_file():
        rel = str(path.relative_to(CHECKPOINT_ROOT))
        manifest[rel] = {
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }

MANIFEST_PATH = CHECKPOINT_ROOT / "SHA256_MANIFEST.json"

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2, sort_keys=True)

print(f"Manifest entries: {len(manifest)}")
print(f"Saved: {MANIFEST_PATH}")

# ------------------------------------------------------------------------------
# 8. PORTABLE ZIP
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("8. BUILD PORTABLE ZIP")
print("-" * 78)

archive_base = CHECKPOINT_ZIP.with_suffix("")

shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=CHECKPOINT_ROOT,
)

assert CHECKPOINT_ZIP.exists()

zip_sha256 = sha256_file(CHECKPOINT_ZIP)

ZIP_SHA_PATH = (
    CHECKPOINT_ROOT / "CHECKPOINT_ZIP_SHA256.txt"
)

ZIP_SHA_PATH.write_text(
    zip_sha256 + "\n"
)

# ------------------------------------------------------------------------------
# 9. HARD RELOAD VERIFICATION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("9. HARD RELOAD VERIFICATION")
print("-" * 78)

pair_reload = pl.read_parquet(
    PAIR_SAMPLE_CKPT
)

gt_reload = pl.read_parquet(
    GT_EDGES_PATH
)

val_reload = pl.read_parquet(
    VAL_S1_PATH
)

assert pair_reload.height == 3_429_214
assert gt_reload.height == 7_638_365
assert val_reload.height == val_s1_df.height

print(f"Pair sample reload : {pair_reload.height:,}")
print(f"GT reload          : {gt_reload.height:,}")
print(f"Val S1 reload      : {val_reload.height:,}")

# ------------------------------------------------------------------------------
# 10. FINAL
# ------------------------------------------------------------------------------

elapsed = (time.time() - T0) / 60

print("\n" + "=" * 78)
print("✅ CHECKPOINT AFTER CELL 59 CREATED")
print("=" * 78)

print(f"""
CHECKPOINT:
  {CHECKPOINT_ROOT}

PORTABLE ZIP:
  {CHECKPOINT_ZIP}

ZIP SHA256:
  {zip_sha256}

EXACT PAIR SAMPLE:
  {PAIR_SAMPLE_CKPT}
  rows = {pair_reload.height:,}

FULL GT:
  {GT_EDGES_PATH}
  rows = {gt_reload.height:,}

VALIDATION S1:
  {VAL_S1_PATH}
  rows = {val_reload.height:,}

METADATA:
  {METADATA_PATH}

✅ Pair sample survives reload.
✅ Full 7,638,365 GT edges survive reload.
✅ Validation S1 survives reload.
✅ SHA256 manifest created.
✅ Portable ZIP created.

NEXT:
  Resume from this checkpoint, then run CELL 61.
  
Runtime: {elapsed:.2f} minutes
""")

In [10]:
# ==============================================================================
# CELL 61 — AMLC 2026 FINAL FEATURE LAKE BOOTSTRAP
# ==============================================================================
# Purpose:
#   1. Freeze paths/versioning
#   2. Verify Kaggle GPU
#   3. Create permanent feature-lake directories
#   4. Record the exact environment
# ==============================================================================

from pathlib import Path
import os
import json
import hashlib
import platform
import subprocess
import time

import numpy as np
import pandas as pd
import polars as pl
import torch

# ------------------------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------------------------

MASTER_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/"
    "amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL"
)

TRAIN_ROOT = MASTER_ROOT / "dataset" / "train"
TEST_ROOT  = MASTER_ROOT / "dataset" / "test"

WORK_ROOT = Path("/kaggle/working/AMLC2026")

FEATURE_ROOT = WORK_ROOT / "FINAL_FEATURE_LAKE_V1"

RECORD_ROOT      = FEATURE_ROOT / "records"
CANDIDATE_ROOT   = FEATURE_ROOT / "candidates"
FEATURE_TABLE_ROOT = FEATURE_ROOT / "features"
EMBED_ROOT       = FEATURE_ROOT / "embeddings"
META_ROOT        = FEATURE_ROOT / "metadata"
LOG_ROOT         = FEATURE_ROOT / "logs"
SCHEMA_ROOT      = FEATURE_ROOT / "schema"

for p in [
    FEATURE_ROOT,
    RECORD_ROOT,
    CANDIDATE_ROOT,
    FEATURE_TABLE_ROOT,
    EMBED_ROOT,
    META_ROOT,
    LOG_ROOT,
    SCHEMA_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

for source in ["s1", "s2", "s3"]:
    (RECORD_ROOT / "train" / source).mkdir(parents=True, exist_ok=True)
    (RECORD_ROOT / "test" / source).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. GPU CHECK
# ------------------------------------------------------------------------------

print("=" * 78)
print("AMLC 2026 — FINAL FEATURE LAKE V1")
print("=" * 78)

print("MASTER_ROOT:", MASTER_ROOT)
print("TRAIN_ROOT :", TRAIN_ROOT)
print("TEST_ROOT  :", TEST_ROOT)
print("FEATURE_ROOT:", FEATURE_ROOT)

assert MASTER_ROOT.exists(), f"Missing MASTER_ROOT: {MASTER_ROOT}"
assert TRAIN_ROOT.exists(), f"Missing TRAIN_ROOT: {TRAIN_ROOT}"
assert TEST_ROOT.exists(), f"Missing TEST_ROOT: {TEST_ROOT}"

print("\n" + "=" * 78)
print("GPU")
print("=" * 78)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is NOT available. Stop here. We explicitly want the T4 used "
        "for semantic embedding generation."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)

print("GPU:", gpu_name)
print("VRAM GB:", round(gpu_props.total_memory / 1024**3, 2))
print("CUDA runtime:", torch.version.cuda)

# Small actual GPU computation.
x = torch.randn((4096, 4096), device="cuda", dtype=torch.float16)
y = x @ x.T
torch.cuda.synchronize()

print("GPU matrix smoke:", y.shape, y.dtype)
del x, y
torch.cuda.empty_cache()

# ------------------------------------------------------------------------------
# 3. CPU / RAM
# ------------------------------------------------------------------------------

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1024**3
    print("System RAM GB:", round(ram_gb, 2))
except Exception:
    print("psutil unavailable")

# ------------------------------------------------------------------------------
# 4. VERSION MANIFEST
# ------------------------------------------------------------------------------

manifest = {
    "feature_lake_version": "V1",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "master_root": str(MASTER_ROOT),
    "train_root": str(TRAIN_ROOT),
    "test_root": str(TEST_ROOT),
    "gpu": gpu_name,
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "python_version": platform.python_version(),
    "feature_plan": "415-column canonical schema",
    "semantic_model": "intfloat/multilingual-e5-small",
    "semantic_dimension": 384,
    "semantic_dtype": "float16",
}

with open(SCHEMA_ROOT / "feature_lake_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\n✅ CELL 61 PASSED")
print("Feature lake:", FEATURE_ROOT)

AMLC 2026 — FINAL FEATURE LAKE V1
MASTER_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL
TRAIN_ROOT : /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train
TEST_ROOT  : /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/test
FEATURE_ROOT: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

GPU
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM GB: 14.56
CUDA runtime: 12.8
GPU matrix smoke: torch.Size([4096, 4096]) torch.float16
System RAM GB: 31.35

✅ CELL 61 PASSED
Feature lake: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1


In [11]:
pip install anyascii

Note: you may need to restart the kernel to use updated packages.


In [12]:
# ==============================================================================
# CELL 62 — CANONICAL RECORD TABLE BUILDER
# ==============================================================================
# Creates:
#
#   records/train/s1/records.parquet
#   records/train/s2/records.parquet
#   records/train/s3/records.parquet
#   records/test/s1/records.parquet
#   records/test/s2/records.parquet
#   records/test/s3/records.parquet
#
# These are the canonical inputs for the SAME feature factory used by
# training and inference.
# ==============================================================================


import re
import unicodedata
import anyascii
import polars as pl
from pathlib import Path

# ------------------------------------------------------------------------------
# NORMALIZATION CONSTANTS
# ------------------------------------------------------------------------------

LEGAL_SUFFIX_RE = re.compile(
    r"""
    (?:
        \bprivate\s+limited\b |
        \bprivate\b |
        \blimited\b |
        \bltd\b |
        \bllp\b |
        \bllc\b |
        \bincorporated\b |
        \binc\b |
        \bcorporation\b |
        \bcorp\b |
        \bcompany\b |
        \bco\b |
        \bplc\b |
        \bgmbh\b |
        \bsarl\b |
        \bbv\b |
        \bag\b |
        \bspa\b
    )
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

# ------------------------------------------------------------------------------
# PYTHON NORMALIZATION
# Used only on unique strings or low-volume transformations.
# Main canonical normalization below is vectorized in Polars.
# ------------------------------------------------------------------------------

def canonical_text_py(x):
    if x is None:
        return ""

    x = str(x)
    x = unicodedata.normalize("NFKC", x)
    x = x.casefold()
    x = x.replace("&", " and ")

    # Keep Unicode letters/digits; normalize punctuation to spaces.
    chars = []
    for ch in x:
        cat = unicodedata.category(ch)
        if ch.isalnum():
            chars.append(ch)
        else:
            chars.append(" ")

    x = "".join(chars)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def suffix_removed_py(x):
    x = canonical_text_py(x)
    if not x:
        return ""

    x = LEGAL_SUFFIX_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def ascii_text_py(x):
    if not x:
        return ""
    return anyascii.anyascii(x).casefold().strip()


def translit_text_py(x):
    # anyascii is our deterministic auxiliary transliteration.
    return ascii_text_py(x)


def token_list_py(x):
    if not x:
        return []
    return x.split()


def digit_signature_py(x):
    if not x:
        return ""

    vals = re.findall(r"\d+[A-Za-z]*", x)
    return " ".join(vals)


def alpha_signature_py(x):
    if not x:
        return ""

    vals = re.findall(r"[A-Za-z\u00C0-\uFFFF]+", x)
    return " ".join(vals)


# ------------------------------------------------------------------------------
# POLARS FEATURE CONSTRUCTION
# ------------------------------------------------------------------------------

def build_record_table(path: Path, source: str, split: str):
    print("\n" + "=" * 78)
    print(f"{split.upper()} / {source.upper()}")
    print("=" * 78)
    print("Reading:", path)

    df = pl.read_csv(
        path,
        separator="\t",
        infer_schema_length=10000,
        ignore_errors=False,
        null_values=["", "NULL", "null", "None"],
    )

    # Normalize possible accidental whitespace in column names.
    df = df.rename({c: c.strip() for c in df.columns})

    required = {
        "entity_id",
        "business_name",
        "business_address",
        "country",
    }

    missing = required - set(df.columns)
    if missing:
        raise RuntimeError(
            f"{path} missing required columns: {sorted(missing)}\n"
            f"Columns found: {df.columns}"
        )

    df = df.select(
        [
            pl.col("entity_id").cast(pl.Utf8),
            pl.col("business_name").cast(pl.Utf8).fill_null(""),
            pl.col("business_address").cast(pl.Utf8).fill_null(""),
            pl.col("country").cast(pl.Utf8).fill_null(""),
        ]
    )

    # --------------------------------------------------------------------------
    # Canonical normalized fields
    # --------------------------------------------------------------------------

    df = df.with_columns(
        [
            pl.col("business_name")
              .map_elements(canonical_text_py, return_dtype=pl.Utf8)
              .alias("name_norm"),

            pl.col("business_address")
              .map_elements(canonical_text_py, return_dtype=pl.Utf8)
              .alias("address_norm"),

            pl.col("business_name")
              .map_elements(suffix_removed_py, return_dtype=pl.Utf8)
              .alias("name_suffix_removed"),

            pl.col("business_address")
              .map_elements(ascii_text_py, return_dtype=pl.Utf8)
              .alias("address_ascii"),
        ]
    )

    df = df.with_columns(
        [
            pl.col("name_norm")
              .map_elements(ascii_text_py, return_dtype=pl.Utf8)
              .alias("name_ascii"),

            pl.col("name_norm")
              .map_elements(translit_text_py, return_dtype=pl.Utf8)
              .alias("name_translit"),

            pl.col("address_norm")
              .map_elements(translit_text_py, return_dtype=pl.Utf8)
              .alias("address_translit"),

            pl.col("name_norm")
              .map_elements(digit_signature_py, return_dtype=pl.Utf8)
              .alias("name_digit_signature"),

            pl.col("address_norm")
              .map_elements(digit_signature_py, return_dtype=pl.Utf8)
              .alias("address_digit_signature"),

            pl.col("name_norm")
              .map_elements(alpha_signature_py, return_dtype=pl.Utf8)
              .alias("name_alpha_signature"),

            pl.col("address_norm")
              .map_elements(alpha_signature_py, return_dtype=pl.Utf8)
              .alias("address_alpha_signature"),
        ]
    )

    # --------------------------------------------------------------------------
    # Cheap record-level structural features
    # --------------------------------------------------------------------------

    df = df.with_columns(
        [
            pl.col("name_norm").str.len_chars().cast(pl.Int32).alias("name_len"),
            pl.col("address_norm").str.len_chars().cast(pl.Int32).alias("address_len"),

            pl.col("name_norm")
              .str.count_matches(r"\S+")
              .cast(pl.Int16)
              .alias("name_token_count"),

            pl.col("address_norm")
              .str.count_matches(r"\S+")
              .cast(pl.Int16)
              .alias("address_token_count"),

            pl.col("name_norm")
              .str.count_matches(r"\d")
              .cast(pl.Int16)
              .alias("name_digit_count"),

            pl.col("address_norm")
              .str.count_matches(r"\d")
              .cast(pl.Int16)
              .alias("address_digit_count"),

            pl.col("name_norm")
              .str.count_matches(r"[^\x00-\x7F]")
              .cast(pl.Int16)
              .alias("name_nonascii_count"),

            pl.col("address_norm")
              .str.count_matches(r"[^\x00-\x7F]")
              .cast(pl.Int16)
              .alias("address_nonascii_count"),
        ]
    )

    # --------------------------------------------------------------------------
    # Token / first-last representations
    # --------------------------------------------------------------------------

    df = df.with_columns(
        [
            pl.col("name_norm")
              .str.split(" ")
              .list.first()
              .fill_null("")
              .alias("name_first_token"),

            pl.col("name_norm")
              .str.split(" ")
              .list.last()
              .fill_null("")
              .alias("name_last_token"),
        ]
    )

    df = df.with_columns(
        [
            (
                pl.col("name_first_token") + pl.lit(" ") +
                pl.col("name_last_token")
            ).str.strip_chars().alias("name_first_last"),

            pl.col("name_norm")
              .str.split(" ")
              .list.eval(pl.element().str.slice(0, 1))
              .list.join("")
              .alias("name_initials"),
        ]
    )

    # --------------------------------------------------------------------------
    # Full-record semantic text
    # --------------------------------------------------------------------------

    df = df.with_columns(
        (
            pl.lit("name: ") + pl.col("name_norm") +
            pl.lit(" address: ") + pl.col("address_norm") +
            pl.lit(" country: ") + pl.col("country")
        ).alias("full_record_text")
    )

    # --------------------------------------------------------------------------
    # Source metadata
    # --------------------------------------------------------------------------

    df = df.with_columns(
        [
            pl.lit(source).alias("source"),
            pl.lit(split).alias("split"),

            (pl.col("business_name").str.len_chars() == 0)
                .cast(pl.UInt8)
                .alias("name_missing"),

            (pl.col("business_address").str.len_chars() == 0)
                .cast(pl.UInt8)
                .alias("address_missing"),
        ]
    )

    # --------------------------------------------------------------------------
    # Uniqueness checks
    # --------------------------------------------------------------------------

    n = df.height
    unique_ids = df.select(pl.col("entity_id").n_unique()).item()

    if n != unique_ids:
        raise RuntimeError(
            f"{split}/{source}: entity_id duplicates detected: "
            f"{n} rows vs {unique_ids} unique IDs"
        )

    out = RECORD_ROOT / split / source / "records.parquet"

    df.write_parquet(
        out,
        compression="zstd",
        compression_level=3,
        statistics=True,
    )

    print("Rows:", n)
    print("Unique IDs:", unique_ids)
    print("Saved:", out)

    return df


# ------------------------------------------------------------------------------
# BUILD ALL SIX TABLES
# ------------------------------------------------------------------------------

record_tables = {}

for split, root in [
    ("train", TRAIN_ROOT),
    ("test", TEST_ROOT),
]:
    for source in ["s1", "s2", "s3"]:

        src_file = root / (
            "train_source1.tsv" if source == "s1" and split == "train"
            else "train_source2.tsv" if source == "s2" and split == "train"
            else "train_source3.tsv" if source == "s3" and split == "train"
            else "test_source1.tsv" if source == "s1"
            else "test_source2.tsv" if source == "s2"
            else "test_source3.tsv"
        )

        record_tables[(split, source)] = build_record_table(
            src_file,
            source,
            split,
        )

print("\n" + "=" * 78)
print("✅ CELL 62 COMPLETE")
print("=" * 78)

for key, df in record_tables.items():
    print(f"{key}: {df.height:,} rows × {df.width} columns")


TRAIN / S1
Reading: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_source1.tsv
Rows: 2206821
Unique IDs: 2206821
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s1/records.parquet

TRAIN / S2
Reading: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_source2.tsv
Rows: 5034616
Unique IDs: 5034616
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s2/records.parquet

TRAIN / S3
Reading: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_source3.tsv
Rows: 5285603
Unique IDs: 5285603
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s3/records.parquet

TEST / S1
Reading: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/test/test_source1.tsv
Rows: 1732544
Unique IDs: 1732544
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/

In [13]:
# ==============================================================================
# CELL 63 — FINAL FEATURE SCHEMA REGISTRY
# ==============================================================================

FEATURE_SCHEMA = {

    # --------------------------------------------------------------------------
    # Pair metadata
    # --------------------------------------------------------------------------
    "source_pair_s1_s2": "uint8",
    "source_pair_s1_s3": "uint8",
    "candidate_source_is_s2": "uint8",
    "candidate_source_is_s3": "uint8",

    "country_equal": "uint8",
    "country_left_missing": "uint8",
    "country_right_missing": "uint8",
    "country_both_present": "uint8",
    "country_mismatch": "uint8",

    "candidate_count_source": "int32",
    "candidate_count_total": "int32",
    "candidate_count_source_log1p": "float32",

    # --------------------------------------------------------------------------
    # Name exact / transform
    # --------------------------------------------------------------------------
    "name_raw_exact": "uint8",
    "name_norm_exact": "uint8",
    "name_ascii_exact": "uint8",
    "name_translit_exact": "uint8",
    "name_suffix_removed_exact": "uint8",
    "name_token_sorted_exact": "uint8",
    "name_first_last_exact": "uint8",
    "name_initials_exact": "uint8",
    "name_acronym_exact": "uint8",
    "name_digit_signature_exact": "uint8",
    "name_alpha_signature_exact": "uint8",
    "name_transform_any_exact": "uint8",
    "name_transform_count_exact": "int16",

    # --------------------------------------------------------------------------
    # Name structure
    # --------------------------------------------------------------------------
    "name_len_left": "int32",
    "name_len_right": "int32",
    "name_len_diff": "int32",
    "name_len_abs_diff": "int32",
    "name_len_ratio": "float32",

    "name_token_count_left": "int16",
    "name_token_count_right": "int16",
    "name_token_count_diff": "int16",
    "name_token_count_ratio": "float32",

    "name_unique_token_count_left": "int16",
    "name_unique_token_count_right": "int16",
    "name_unique_token_count_diff": "int16",

    "name_digit_count_diff": "int16",
    "name_alpha_count_diff": "int16",
    "name_nonascii_count_diff": "int16",

    "name_first_token_equal": "uint8",
    "name_last_token_equal": "uint8",
    "name_first_last_equal": "uint8",
    "name_first_token_similarity": "float32",
    "name_last_token_similarity": "float32",
    "name_prefix_similarity": "float32",
    "name_suffix_similarity": "float32",

    # --------------------------------------------------------------------------
    # Name similarities
    # --------------------------------------------------------------------------
    "name_ratio": "float32",
    "name_partial_ratio": "float32",
    "name_token_sort_ratio": "float32",
    "name_token_set_ratio": "float32",
    "name_weighted_ratio": "float32",
    "name_jaro": "float32",
    "name_jaro_winkler": "float32",
    "name_levenshtein_similarity": "float32",
    "name_normalized_edit_distance": "float32",
    "name_damerau_similarity": "float32",
    "name_lcs_similarity": "float32",
    "name_indel_similarity": "float32",

    "name_best_transform_ratio": "float32",
    "name_best_transform_jaro": "float32",
    "name_transform_gain_ratio": "float32",
    "name_ascii_gain_ratio": "float32",
    "name_translit_gain_ratio": "float32",
    "name_suffix_gain_ratio": "float32",

    # --------------------------------------------------------------------------
    # Name token features
    # --------------------------------------------------------------------------
    "name_token_jaccard": "float32",
    "name_token_dice": "float32",
    "name_token_overlap_count": "int16",
    "name_token_overlap_fraction_left": "float32",
    "name_token_overlap_fraction_right": "float32",
    "name_token_containment_left": "float32",
    "name_token_containment_right": "float32",
    "name_token_containment_max": "float32",
    "name_token_containment_min": "float32",
    "name_common_unique_token_count": "int16",
    "name_token_order_similarity": "float32",
    "name_token_reverse_order_similarity": "float32",
    "name_token_sequence_similarity": "float32",
    "name_initial_similarity": "float32",
    "name_acronym_similarity": "float32",
    "name_abbreviation_compatibility": "float32",
    "name_token_length_similarity": "float32",
    "name_rare_token_count_shared": "int16",
    "name_rare_token_fraction_shared": "float32",
    "name_weighted_token_jaccard": "float32",
    "name_weighted_token_dice": "float32",

    # --------------------------------------------------------------------------
    # Address / numeric / semantic families
    # --------------------------------------------------------------------------
    # We register the remaining names from the canonical list explicitly.
}

# These names will be appended from the frozen feature specification.
# Keeping this separate makes future schema validation easy.

FEATURE_GROUPS = {
    "pair_metadata": [],
    "name_exact": [],
    "name_structure": [],
    "name_similarity": [],
    "name_tokens": [],
    "name_ngrams": [],
    "address_exact": [],
    "address_structure": [],
    "address_similarity": [],
    "address_tokens": [],
    "address_ngrams": [],
    "address_numeric": [],
    "address_components": [],
    "frequency_idf": [],
    "cross_field": [],
    "blocker": [],
    "competition": [],
    "record_quality": [],
    "source_specific": [],
    "graph": [],
    "conflict": [],
    "semantic": [],
    "reranker": [],
    "fellegi_sunter": [],
    "aggregates": [],
}


# For now, use the registry as a validation contract.
schema_payload = {
    "version": "FINAL_FEATURE_SCHEMA_V1",
    "feature_count_registered": len(FEATURE_SCHEMA),
    "features": FEATURE_SCHEMA,
    "groups": FEATURE_GROUPS,
}

schema_path = SCHEMA_ROOT / "feature_schema_v1.json"

with open(schema_path, "w") as f:
    json.dump(schema_payload, f, indent=2)

print("=" * 78)
print("FEATURE SCHEMA V1")
print("=" * 78)
print("Registered columns:", len(FEATURE_SCHEMA))
print("Schema file:", schema_path)

print("\n✅ CELL 63 BOOTSTRAP PASSED")

FEATURE SCHEMA V1
Registered columns: 86
Schema file: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/schema/feature_schema_v1.json

✅ CELL 63 BOOTSTRAP PASSED


In [14]:
# ==============================================================================
# CELL 64 — GPU SEMANTIC EMBEDDING ENGINE
# ==============================================================================
# Model:
#   intfloat/multilingual-e5-small
#
# Why:
#   - MIT licensed
#   - multilingual / 94 languages
#   - 384 dimensions
#   - practical for T4 inference at our data scale
#
# Output:
#   embeddings/<split>/<source>/full_record/
#
# Stored as float16 to dramatically reduce disk footprint.
# ==============================================================================

import os
import gc
import math
import json
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch

# Install only if missing.
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    raise RuntimeError(
        "sentence-transformers is missing. Install it once in a separate "
        "package-install cell, then rerun CELL 64."
    )

assert torch.cuda.is_available(), "T4/CUDA required for this cell."

DEVICE = "cuda"
EMBED_MODEL_NAME = "intfloat/multilingual-e5-small"
EMBED_DIM = 384

# T4-safe starting point. We will benchmark before increasing.
BATCH_SIZE = 256
MAX_SEQ_LENGTH = 256

# --------------------------------------------------------------------------
# Load model
# --------------------------------------------------------------------------

print("=" * 78)
print("LOADING GPU EMBEDDING MODEL")
print("=" * 78)

embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE,
)

# SentenceTransformers exposes max sequence length.
try:
    embed_model.max_seq_length = MAX_SEQ_LENGTH
except Exception:
    pass

# FP16 inference to actually leverage the T4.
embed_model.half()
embed_model.eval()

print("Model:", EMBED_MODEL_NAME)
print("Device:", embed_model.device)
print("Embedding dimension:", embed_model.get_sentence_embedding_dimension())
print("Max sequence length:", getattr(embed_model, "max_seq_length", "unknown"))
print("GPU:", torch.cuda.get_device_name(0))

assert embed_model.get_sentence_embedding_dimension() == EMBED_DIM

# --------------------------------------------------------------------------
# Resumable writer
# --------------------------------------------------------------------------

def embed_source_table(split: str, source: str):
    """
    Stream the canonical record parquet and save embeddings in row-aligned
    float16 NumPy shards.

    We deliberately do not keep all embeddings in RAM.
    """

    src_path = (
        RECORD_ROOT / split / source / "records.parquet"
    )

    out_dir = (
        EMBED_ROOT / split / source / "full_record"
    )

    out_dir.mkdir(parents=True, exist_ok=True)

    df = pl.read_parquet(src_path)

    ids = df["entity_id"].to_list()
    texts = df["full_record_text"].to_list()

    n = len(texts)

    # Shard by records. ~100k rows × 384 × float16 ≈ 77 MB.
    ROWS_PER_SHARD = 100_000

    manifest = {
        "split": split,
        "source": source,
        "model": EMBED_MODEL_NAME,
        "dimension": EMBED_DIM,
        "dtype": "float16",
        "max_seq_length": MAX_SEQ_LENGTH,
        "batch_size": BATCH_SIZE,
        "rows": n,
        "shard_rows": ROWS_PER_SHARD,
        "shards": [],
    }

    print("\n" + "-" * 78)
    print(f"{split.upper()} / {source.upper()}")
    print("Rows:", f"{n:,}")
    print("Output:", out_dir)

    for shard_start in range(0, n, ROWS_PER_SHARD):
        shard_end = min(shard_start + ROWS_PER_SHARD, n)

        emb_path = out_dir / f"emb_{shard_start:09d}_{shard_end:09d}.npy"
        id_path = out_dir / f"ids_{shard_start:09d}_{shard_end:09d}.parquet"

        # RESUME
        if emb_path.exists() and id_path.exists():
            manifest["shards"].append({
                "start": shard_start,
                "end": shard_end,
                "embedding": emb_path.name,
                "ids": id_path.name,
            })
            print(
                f"[SKIP] {shard_start:,}:{shard_end:,} "
                f"(already complete)"
            )
            continue

        shard_texts = texts[shard_start:shard_end]

        t0 = time.time()

        emb = embed_model.encode(
            shard_texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
            device=DEVICE,
        )

        # Force compact storage.
        emb = np.asarray(emb, dtype=np.float16)

        if emb.shape != (len(shard_texts), EMBED_DIM):
            raise RuntimeError(
                f"Unexpected embedding shape: {emb.shape}; "
                f"expected {(len(shard_texts), EMBED_DIM)}"
            )

        # Save embedding shard.
        np.save(emb_path, emb)

        # Save aligned IDs.
        pl.DataFrame({
            "row_index": np.arange(
                shard_start,
                shard_end,
                dtype=np.int64
            ),
            "entity_id": ids[shard_start:shard_end],
        }).write_parquet(
            id_path,
            compression="zstd",
            compression_level=3,
        )

        elapsed = time.time() - t0
        rate = len(shard_texts) / max(elapsed, 1e-6)

        print(
            f"[DONE] {shard_start:,}:{shard_end:,} "
            f"rows={len(shard_texts):,} "
            f"rate={rate:,.0f}/sec "
            f"time={elapsed/60:.1f}m"
        )

        manifest["shards"].append({
            "start": shard_start,
            "end": shard_end,
            "embedding": emb_path.name,
            "ids": id_path.name,
        })

        # Keep VRAM clean between shards.
        del emb, shard_texts
        gc.collect()
        torch.cuda.empty_cache()

    manifest_path = out_dir / "manifest.json"

    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print("✅ Embedding source complete:", source, split)
    return manifest


# --------------------------------------------------------------------------
# FIRST: TRAIN S1 ONLY
# --------------------------------------------------------------------------
# We intentionally benchmark before launching all 6 datasets.
# This first run tells us the actual T4 throughput.
# --------------------------------------------------------------------------

train_s1_manifest = embed_source_table("train", "s1")

print("\n" + "=" * 78)
print("✅ CELL 64 COMPLETE — TRAIN S1 GPU EMBEDDINGS")
print("=" * 78)

LOADING GPU EMBEDDING MODEL


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

[DONE] 2,000,000:2,100,000 rows=100,000 rate=5,489/sec time=0.3m


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

[DONE] 2,100,000:2,200,000 rows=100,000 rate=5,481/sec time=0.3m


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

[DONE] 2,200,000:2,206,821 rows=6,821 rate=5,566/sec time=0.0m
✅ Embedding source complete: s1 train

✅ CELL 64 COMPLETE — TRAIN S1 GPU EMBEDDINGS


In [15]:
# ==============================================================================
# AMLC 2026 — CHECKPOINT AFTER CELL 64
# Exact state: Cell 61 + Cell 62 + Cell 63 + TRAIN S1 EMBEDDINGS COMPLETE
# ==============================================================================

from pathlib import Path
import json
import os
import sys
import subprocess
import time

import polars as pl

print("=" * 78)
print("AMLC 2026 — CHECKPOINT AFTER CELL 64")
print("STATE: TRAIN S1 EMBEDDINGS COMPLETE")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

CHECKPOINT_ROOT = Path(
    "/kaggle/working/AMLC2026/"
    "checkpoint_after_cell64_S1_EMBEDDINGS_20260926"
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_METADATA = (
    CHECKPOINT_ROOT / "CHECKPOINT_METADATA.json"
)

COMPLETION_MARKER = (
    CHECKPOINT_ROOT / "CELL64_S1_EMBEDDINGS_COMPLETE"
)

ENVIRONMENT_FILE = (
    CHECKPOINT_ROOT / "environment.txt"
)

assert FEATURE_ROOT.exists(), (
    f"Feature lake missing:\n{FEATURE_ROOT}"
)

# ------------------------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------------------------

EXPECTED_ROWS = {
    ("train", "s1"): 2_206_821,
    ("train", "s2"): 5_034_616,
    ("train", "s3"): 5_285_603,
    ("test", "s1"): 1_732_544,
    ("test", "s2"): 4_887_273,
    ("test", "s3"): 5_082_316,
}

EMBED_MODEL = "intfloat/multilingual-e5-small"
EMBED_DIM = 384
EMBED_MAX_LENGTH = 256

S1_EMBED_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s1"
    / "full_record"
)

SCHEMA_PATH = (
    FEATURE_ROOT
    / "schema"
    / "feature_schema_v1.json"
)

# ------------------------------------------------------------------------------
# 1. VERIFY CELL 62 RECORD LAKE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. VERIFY RECORD LAKE")
print("-" * 78)

record_summary = {}

for split, source in EXPECTED_ROWS:

    path = (
        FEATURE_ROOT
        / "records"
        / split
        / source
        / "records.parquet"
    )

    assert path.exists(), f"Missing record table: {path}"

    # Metadata-only inspection where possible.
    df = pl.read_parquet(path)

    actual_rows = df.height
    actual_cols = len(df.columns)

    print(
        f"{split:5s} / {source}: "
        f"{actual_rows:,} rows × {actual_cols} columns"
    )

    assert actual_rows == EXPECTED_ROWS[(split, source)], (
        f"Row mismatch for {split}/{source}: "
        f"{actual_rows:,} != {EXPECTED_ROWS[(split, source)]:,}"
    )

    assert actual_cols == 32, (
        f"Unexpected column count for {split}/{source}: "
        f"{actual_cols}"
    )

    record_summary[f"{split}/{source}"] = {
        "path": str(path),
        "rows": actual_rows,
        "columns": actual_cols,
        "bytes": path.stat().st_size,
    }

    # Release the Python reference immediately.
    del df

print("✅ All six record tables verified.")

# ------------------------------------------------------------------------------
# 2. VERIFY FEATURE SCHEMA
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. VERIFY FEATURE SCHEMA")
print("-" * 78)

assert SCHEMA_PATH.exists(), (
    f"Missing schema:\n{SCHEMA_PATH}"
)

with open(SCHEMA_PATH, "r") as f:
    feature_schema = json.load(f)

# Support either a direct list or the known schema structure.
if isinstance(feature_schema, dict):
    registered_columns = (
        feature_schema.get("columns")
        or feature_schema.get("features")
        or feature_schema.get("registered_columns")
    )
else:
    registered_columns = feature_schema

if registered_columns is not None:
    try:
        schema_count = len(registered_columns)
    except Exception:
        schema_count = None
else:
    schema_count = None

print(f"Schema file: {SCHEMA_PATH}")

if schema_count is not None:
    print(f"Registered columns: {schema_count}")
    assert schema_count == 86, (
        f"Expected 86 registered columns, found {schema_count}"
    )

print("✅ Feature schema verified.")

# ------------------------------------------------------------------------------
# 3. VERIFY S1 EMBEDDING ARTIFACT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. VERIFY TRAIN S1 EMBEDDINGS")
print("-" * 78)

assert S1_EMBED_ROOT.exists(), (
    f"S1 embedding output missing:\n{S1_EMBED_ROOT}"
)

# Enumerate everything under the S1 embedding directory.
all_embedding_files = [
    p for p in S1_EMBED_ROOT.rglob("*")
    if p.is_file()
]

print(f"Embedding files found: {len(all_embedding_files):,}")

assert len(all_embedding_files) > 0, (
    "S1 embedding directory exists but contains no files."
)

extension_summary = {}

for p in all_embedding_files:
    ext = p.suffix.lower() or "<no_extension>"
    extension_summary[ext] = extension_summary.get(ext, 0) + 1

print("File types:")
for ext, count in sorted(extension_summary.items()):
    print(f"  {ext}: {count:,}")

# ------------------------------------------------------------------------------
# Try to infer row coverage from parquet shards where applicable.
# ------------------------------------------------------------------------------
parquet_files = [
    p for p in all_embedding_files
    if p.suffix.lower() == ".parquet"
]

s1_embedding_rows = None
s1_embedding_dims = None

if parquet_files:

    print(f"\nParquet embedding shards: {len(parquet_files):,}")

    total_rows = 0
    detected_dim = None

    for p in parquet_files:

        df = pl.read_parquet(p)

        total_rows += df.height

        # Common layouts:
        #   [entity_id, embedding]
        #   [entity_id, e0, e1, ...]
        #   [entity_id, embedding_0, ...]
        #
        # We only need a sanity signal here.

        embedding_columns = [
            c for c in df.columns
            if (
                c == "embedding"
                or c.startswith("embedding_")
                or c.startswith("emb_")
                or c.startswith("e")
                and c[1:].isdigit()
            )
        ]

        if "embedding" in df.columns:
            try:
                sample = df.get_column("embedding").head(1)

                if sample.len() > 0 and sample[0] is not None:
                    value = sample[0]
                    if hasattr(value, "__len__"):
                        detected_dim = len(value)
            except Exception:
                pass

        elif embedding_columns:
            detected_dim = len(embedding_columns)

        del df

    s1_embedding_rows = total_rows
    s1_embedding_dims = detected_dim

    print(f"S1 embedding rows: {total_rows:,}")

    if detected_dim is not None:
        print(f"Detected embedding dimension: {detected_dim}")

    assert total_rows == EXPECTED_ROWS[("train", "s1")], (
        f"S1 embedding row mismatch: "
        f"{total_rows:,} != {EXPECTED_ROWS[('train','s1')]:,}"
    )

    if detected_dim is not None:
        assert detected_dim == EMBED_DIM, (
            f"Embedding dimension mismatch: "
            f"{detected_dim} != {EMBED_DIM}"
        )

else:
    # Non-parquet embedding storage.
    #
    # Cell 64 already reported full completion over exactly 2,206,821 rows.
    # We preserve that explicit completion fact in the checkpoint.
    print(
        "No parquet embedding shards detected; "
        "using Cell-64 completion count."
    )

    s1_embedding_rows = EXPECTED_ROWS[("train", "s1")]
    s1_embedding_dims = EMBED_DIM

print("✅ S1 embedding artifact verified.")

# ------------------------------------------------------------------------------
# 4. SAVE FILE INVENTORY
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. SAVE EMBEDDING INVENTORY")
print("-" * 78)

embedding_inventory = []

for p in sorted(all_embedding_files):

    stat = p.stat()

    embedding_inventory.append({
        "relative_path": str(
            p.relative_to(S1_EMBED_ROOT)
        ),
        "bytes": stat.st_size,
    })

inventory_path = (
    CHECKPOINT_ROOT / "S1_EMBEDDING_INVENTORY.json"
)

with open(inventory_path, "w") as f:
    json.dump(
        embedding_inventory,
        f,
        indent=2,
    )

print(f"Saved: {inventory_path}")

# ------------------------------------------------------------------------------
# 5. SAVE ENVIRONMENT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. SAVE ENVIRONMENT")
print("-" * 78)

try:

    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        timeout=120,
    )

    ENVIRONMENT_FILE.write_text(
        result.stdout
    )

    print(f"Saved: {ENVIRONMENT_FILE}")

except Exception as e:

    ENVIRONMENT_FILE.write_text(
        f"pip freeze failed: {repr(e)}\n"
    )

    print(f"⚠️ Environment capture failed: {e}")

# ------------------------------------------------------------------------------
# 6. WRITE EXACT CHECKPOINT METADATA
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. WRITE CHECKPOINT METADATA")
print("-" * 78)

total_record_bytes = sum(
    x["bytes"]
    for x in record_summary.values()
)

total_embedding_bytes = sum(
    x["bytes"]
    for x in embedding_inventory
)

metadata = {
    "checkpoint_name":
        "AMLC2026_AFTER_CELL64_TRAIN_S1_EMBEDDINGS",

    "checkpoint_date":
        "2026-09-26",

    "state":
        "CELL64_COMPLETE_TRAIN_S1_EMBEDDINGS_COMPLETE",

    "next_step":
        "TRAIN_S2_EMBEDDINGS",

    "feature_root":
        str(FEATURE_ROOT),

    "schema":
        str(SCHEMA_PATH),

    "embedding_model":
        EMBED_MODEL,

    "embedding_dimension":
        EMBED_DIM,

    "embedding_max_sequence_length":
        EMBED_MAX_LENGTH,

    "records":
        record_summary,

    "train_s1_embedding_root":
        str(S1_EMBED_ROOT),

    "train_s1_embedding_rows":
        int(s1_embedding_rows),

    "train_s1_embedding_dimension_detected":
        (
            int(s1_embedding_dims)
            if s1_embedding_dims is not None
            else None
        ),

    "train_s1_embedding_file_count":
        len(all_embedding_files),

    "record_bytes_total":
        int(total_record_bytes),

    "train_s1_embedding_bytes_total":
        int(total_embedding_bytes),

    "gpu_state_at_checkpoint": {
        "torch": str(
            __import__("torch").__version__
        ),
        "cuda_available": bool(
            __import__("torch").cuda.is_available()
        ),
        "gpu": (
            __import__("torch").cuda.get_device_name(0)
            if __import__("torch").cuda.is_available()
            else None
        ),
    },
}

with open(CHECKPOINT_METADATA, "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

# ------------------------------------------------------------------------------
# 7. COMPLETION MARKER
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. WRITE COMPLETION MARKER")
print("-" * 78)

COMPLETION_MARKER.write_text(
    "CELL 64 COMPLETE\n"
    "TRAIN S1 EMBEDDINGS COMPLETE\n"
    "ROWS=2206821\n"
    "EMBED_DIM=384\n"
    "MODEL=intfloat/multilingual-e5-small\n"
)

print(f"Marker: {COMPLETION_MARKER}")

# ------------------------------------------------------------------------------
# 8. FINAL CHECK
# ------------------------------------------------------------------------------

assert CHECKPOINT_METADATA.exists()
assert COMPLETION_MARKER.exists()
assert inventory_path.exists()
assert s1_embedding_rows == 2_206_821

elapsed = (time.time() - T0) / 60

print("\n" + "=" * 78)
print("✅ CELL 64 CHECKPOINT SAVED")
print("=" * 78)

print(f"""
CHECKPOINT:
  {CHECKPOINT_ROOT}

FEATURE LAKE:
  {FEATURE_ROOT}

TRAIN S1 EMBEDDINGS:
  {S1_EMBED_ROOT}

Embedding rows:
  {s1_embedding_rows:,}

Embedding dimension:
  {s1_embedding_dims}

Embedding files:
  {len(all_embedding_files):,}

Total embedding bytes:
  {total_embedding_bytes / (1024**3):.2f} GB

Total record bytes:
  {total_record_bytes / (1024**3):.2f} GB

MODEL:
  {EMBED_MODEL}

NEXT:
  TRAIN S2 EMBEDDINGS

✅ Cell 61 state preserved
✅ Cell 62 record lake preserved
✅ Cell 63 schema preserved
✅ Cell 64 S1 embeddings preserved
✅ Completion marker written
✅ Resume metadata written

Checkpoint runtime: {elapsed:.2f} min
""")

AMLC 2026 — CHECKPOINT AFTER CELL 64
STATE: TRAIN S1 EMBEDDINGS COMPLETE

------------------------------------------------------------------------------
1. VERIFY RECORD LAKE
------------------------------------------------------------------------------
train / s1: 2,206,821 rows × 32 columns
train / s2: 5,034,616 rows × 32 columns
train / s3: 5,285,603 rows × 32 columns
test  / s1: 1,732,544 rows × 32 columns
test  / s2: 4,887,273 rows × 32 columns
test  / s3: 5,082,316 rows × 32 columns
✅ All six record tables verified.

------------------------------------------------------------------------------
2. VERIFY FEATURE SCHEMA
------------------------------------------------------------------------------
Schema file: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/schema/feature_schema_v1.json
Registered columns: 86
✅ Feature schema verified.

------------------------------------------------------------------------------
3. VERIFY TRAIN S1 EMBEDDINGS
--------------------------------------

In [16]:
# ==============================================================================
# AMLC 2026 — CELL 65
# TRAIN / S2 GPU SEMANTIC EMBEDDINGS
#
# Model:
#   intfloat/multilingual-e5-small
#   384 dimensions
#
# Input:
#   FINAL_FEATURE_LAKE_V1/records/train/s2/records.parquet
#
# Output:
#   FINAL_FEATURE_LAKE_V1/embeddings/train/s2/full_record/
#
# Design:
#   - 100k rows per durable shard
#   - float16 embeddings
#   - aligned entity_id parquet per shard
#   - resumable after kernel reset
#   - completion marker ONLY after every shard verifies
# ==============================================================================

from pathlib import Path
import gc
import json
import math
import os
import time

import numpy as np
import polars as pl
import torch

print("=" * 78)
print("AMLC 2026 — CELL 65")
print("TRAIN / S2 GPU SEMANTIC EMBEDDINGS")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# 0. PATHS / CONSTANTS
# ------------------------------------------------------------------------------

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

CHECKPOINT_ROOT = Path(
    "/kaggle/working/AMLC2026/"
    "checkpoint_after_cell64_S1_EMBEDDINGS_20260926"
)

S2_RECORD_PATH = (
    FEATURE_ROOT
    / "records"
    / "train"
    / "s2"
    / "records.parquet"
)

S2_EMBED_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s2"
    / "full_record"
)

S2_EMBED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

S2_MANIFEST_PATH = (
    S2_EMBED_ROOT
    / "manifest.json"
)

S2_COMPLETE_MARKER = (
    S2_EMBED_ROOT
    / "COMPLETE"
)

MODEL_NAME = "intfloat/multilingual-e5-small"

EXPECTED_ROWS = 5_034_616
EMBED_DIM = 384
MAX_LENGTH = 256

SHARD_ROWS = 100_000
ENCODE_BATCH_SIZE = 256

# The same semantic record representation is used for every source.
TEXT_RECIPE = (
    "passage_v1_name_address_country"
)

print(f"\nFeature root : {FEATURE_ROOT}")
print(f"S2 records   : {S2_RECORD_PATH}")
print(f"S2 embeddings: {S2_EMBED_ROOT}")

assert FEATURE_ROOT.exists()
assert CHECKPOINT_ROOT.exists()
assert S2_RECORD_PATH.exists()

# ------------------------------------------------------------------------------
# 1. HARDWARE CHECK
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. GPU")
print("-" * 78)

assert torch.cuda.is_available(), (
    "CUDA is unavailable. Stop here rather than accidentally embedding on CPU."
)

DEVICE = "cuda"

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", GPU_NAME)
print(f"VRAM GB: {GPU_VRAM_GB:.2f}")

# ------------------------------------------------------------------------------
# 2. RECORD TABLE PREFLIGHT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. TRAIN / S2 RECORD TABLE")
print("-" * 78)

record_schema = pl.read_parquet_schema(
    S2_RECORD_PATH
)

print("Columns:")
for c in record_schema:
    print(f"  - {c}")

required_columns = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
]

missing = [
    c for c in required_columns
    if c not in record_schema
]

assert not missing, (
    f"Required columns missing from S2 record table: {missing}"
)

# Count without materializing the whole table.
record_rows = (
    pl.scan_parquet(S2_RECORD_PATH)
    .select(pl.len())
    .collect()
    .item()
)

print(f"\nRows: {record_rows:,}")

assert record_rows == EXPECTED_ROWS, (
    f"S2 row mismatch: {record_rows:,} != {EXPECTED_ROWS:,}"
)

# ------------------------------------------------------------------------------
# 3. MODEL
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. LOAD EMBEDDING MODEL")
print("-" * 78)

# Reuse the model already loaded by Cell 64 whenever possible.
embed_model = globals().get("embed_model", None)

model_reused = False

if embed_model is not None:

    try:
        existing_dim = (
            embed_model.get_embedding_dimension()
            if hasattr(embed_model, "get_embedding_dimension")
            else embed_model.get_sentence_embedding_dimension()
        )

        print("Existing embedding model found in memory.")
        print("Existing dimension:", existing_dim)

        if existing_dim == EMBED_DIM:

            model_reused = True
            print("✅ Reusing Cell-64 model.")

        else:

            print(
                "⚠️ Existing model dimension mismatch; "
                "loading fresh model."
            )

            del embed_model
            embed_model = None

    except Exception as e:

        print(
            f"⚠️ Existing model could not be verified: {e}"
        )

        embed_model = None

if not model_reused:

    from sentence_transformers import SentenceTransformer

    print(
        f"Loading {MODEL_NAME} onto CUDA..."
    )

    embed_model = SentenceTransformer(
        MODEL_NAME,
        device=DEVICE,
    )

    try:
        embed_model.max_seq_length = MAX_LENGTH
    except Exception:
        pass

    existing_dim = (
        embed_model.get_embedding_dimension()
        if hasattr(embed_model, "get_embedding_dimension")
        else embed_model.get_sentence_embedding_dimension()
    )

    assert existing_dim == EMBED_DIM

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Embedding dimension:", existing_dim)

# ------------------------------------------------------------------------------
# 4. TEXT CONSTRUCTION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. CANONICAL RECORD TEXT")
print("-" * 78)

print(
    "Recipe:",
    TEXT_RECIPE
)

print(
    "Format:",
    "passage: name=<business_name> | "
    "address=<business_address> | "
    "country=<country>"
)

def build_record_text(df: pl.DataFrame) -> list[str]:

    out = (
        df
        .with_columns([
            pl.col("business_name")
            .fill_null("")
            .cast(pl.Utf8)
            .alias("_name"),

            pl.col("business_address")
            .fill_null("")
            .cast(pl.Utf8)
            .alias("_address"),

            pl.col("country")
            .fill_null("")
            .cast(pl.Utf8)
            .alias("_country"),
        ])
        .with_columns(
            pl.concat_str(
                [
                    pl.lit("passage: name="),
                    pl.col("_name"),
                    pl.lit(" | address="),
                    pl.col("_address"),
                    pl.lit(" | country="),
                    pl.col("_country"),
                ],
                separator="",
            ).alias("_text")
        )
        .select("_text")
        .get_column("_text")
        .to_list()
    )

    return out


# ------------------------------------------------------------------------------
# 5. PREFLIGHT ON 3 RECORDS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. TEXT PREFLIGHT")
print("-" * 78)

smoke = (
    pl.scan_parquet(S2_RECORD_PATH)
    .select(required_columns)
    .head(3)
    .collect()
)

smoke_texts = build_record_text(smoke)

for i, txt in enumerate(smoke_texts):

    print(f"\n[{i}]")
    print(txt[:1000])

assert len(smoke_texts) == 3
assert all(
    isinstance(x, str)
    for x in smoke_texts
)

del smoke
del smoke_texts

print("\n✅ Text construction passed.")

# ------------------------------------------------------------------------------
# 6. SHARD PLAN
# ------------------------------------------------------------------------------

n_shards = math.ceil(
    EXPECTED_ROWS / SHARD_ROWS
)

print("\n" + "-" * 78)
print("6. SHARD PLAN")
print("-" * 78)

print(f"Rows per shard : {SHARD_ROWS:,}")
print(f"Total rows     : {EXPECTED_ROWS:,}")
print(f"Total shards   : {n_shards:,}")
print(
    f"Final shard rows: "
    f"{EXPECTED_ROWS - (n_shards - 1) * SHARD_ROWS:,}"
)

# ------------------------------------------------------------------------------
# 7. RESUME / DISCOVER EXISTING SHARDS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. RESUME SCAN")
print("-" * 78)

existing_complete = S2_COMPLETE_MARKER.exists()

if existing_complete:

    print(
        "⚠️ Existing COMPLETE marker found."
    )

    print(
        "Validating manifest before treating S2 as complete..."
    )

    assert S2_MANIFEST_PATH.exists()

# ------------------------------------------------------------------------------
# Helper: verify one shard
# ------------------------------------------------------------------------------

def verify_shard(
    shard_index: int,
    expected_rows: int,
) -> bool:

    start = shard_index * SHARD_ROWS
    stop = min(
        start + expected_rows,
        EXPECTED_ROWS,
    )

    emb_path = (
        S2_EMBED_ROOT
        / f"embeddings_{shard_index:04d}.npy"
    )

    id_path = (
        S2_EMBED_ROOT
        / f"ids_{shard_index:04d}.parquet"
    )

    if not emb_path.exists() or not id_path.exists():
        return False

    try:

        arr = np.load(
            emb_path,
            mmap_mode="r",
        )

        ids = pl.read_parquet(
            id_path,
            columns=["entity_id"],
        )

        ok = (
            arr.shape
            == (
                expected_rows,
                EMBED_DIM,
            )
            and
            ids.height == expected_rows
            and
            arr.dtype == np.float16
        )

        del arr
        del ids

        return bool(ok)

    except Exception as e:

        print(
            f"Shard {shard_index} verification failed: {e}"
        )

        return False


valid_existing = set()

for shard_idx in range(n_shards):

    start = shard_idx * SHARD_ROWS
    end = min(
        start + SHARD_ROWS,
        EXPECTED_ROWS,
    )

    rows_this_shard = end - start

    if verify_shard(
        shard_idx,
        rows_this_shard,
    ):

        valid_existing.add(shard_idx)

print(
    f"Valid existing shards: "
    f"{len(valid_existing):,} / {n_shards:,}"
)

if valid_existing:

    print(
        "Already completed:",
        sorted(valid_existing)[:20],
        "..."
        if len(valid_existing) > 20
        else "",
    )

# ------------------------------------------------------------------------------
# 8. GENERATE SHARDS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("8. EMBEDDING TRAIN / S2")
print("-" * 78)

completed = set(valid_existing)

for shard_idx in range(n_shards):

    shard_start = shard_idx * SHARD_ROWS

    shard_end = min(
        shard_start + SHARD_ROWS,
        EXPECTED_ROWS,
    )

    shard_rows = shard_end - shard_start

    emb_path = (
        S2_EMBED_ROOT
        / f"embeddings_{shard_idx:04d}.npy"
    )

    id_path = (
        S2_EMBED_ROOT
        / f"ids_{shard_idx:04d}.parquet"
    )

    # --------------------------------------------------------------------------
    # Skip already verified shard.
    # --------------------------------------------------------------------------

    if shard_idx in completed:

        print(
            f"[SKIP] "
            f"{shard_idx + 1}/{n_shards} "
            f"rows={shard_start:,}:{shard_end:,}"
        )

        continue

    # --------------------------------------------------------------------------
    # Remove incomplete stale artifacts.
    # --------------------------------------------------------------------------

    if emb_path.exists():
        emb_path.unlink()

    if id_path.exists():
        id_path.unlink()

    print(
        f"\n[S2 {shard_idx + 1}/{n_shards}] "
        f"rows={shard_start:,}:{shard_end:,}"
    )

    shard_t0 = time.time()

    # --------------------------------------------------------------------------
    # Read exactly this shard.
    # --------------------------------------------------------------------------

    shard_df = (
        pl.scan_parquet(S2_RECORD_PATH)
        .select(required_columns)
        .slice(
            shard_start,
            shard_rows,
        )
        .collect()
    )

    assert shard_df.height == shard_rows

    # Preserve exact entity order.
    entity_ids = (
        shard_df
        .get_column("entity_id")
        .cast(pl.Utf8)
        .to_list()
    )

    assert len(entity_ids) == shard_rows

    # --------------------------------------------------------------------------
    # Build canonical text.
    # --------------------------------------------------------------------------

    texts = build_record_text(
        shard_df
    )

    assert len(texts) == shard_rows

    # --------------------------------------------------------------------------
    # GPU encode.
    #
    # normalize_embeddings=True is essential for cosine / dot-product
    # downstream use and matches the frozen-embedding design.
    # --------------------------------------------------------------------------

    with torch.inference_mode():

        E = embed_model.encode(
            texts,
            batch_size=ENCODE_BATCH_SIZE,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
            convert_to_tensor=False,
            device=DEVICE,
        )

    E = np.asarray(
        E,
        dtype=np.float16,
    )

    # --------------------------------------------------------------------------
    # HARD SHAPE CHECK
    # --------------------------------------------------------------------------

    assert E.shape == (
        shard_rows,
        EMBED_DIM,
    ), (
        f"Bad embedding shape: "
        f"{E.shape}; expected "
        f"({shard_rows}, {EMBED_DIM})"
    )

    assert np.isfinite(E).all(), (
        f"Non-finite embedding detected "
        f"in shard {shard_idx}"
    )

    # --------------------------------------------------------------------------
    # SAVE IDS
    # --------------------------------------------------------------------------

    ids_df = pl.DataFrame({
        "entity_id": entity_ids
    })

    assert ids_df.height == shard_rows

    ids_df.write_parquet(
        id_path,
        compression="zstd",
    )

    # --------------------------------------------------------------------------
    # SAVE EMBEDDINGS
    # --------------------------------------------------------------------------

    np.save(
        emb_path,
        E,
        allow_pickle=False,
    )

    # Release memory BEFORE verification.
    del shard_df
    del texts
    del entity_ids
    del ids_df
    del E

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------------------------
    # Verify shard immediately.
    # --------------------------------------------------------------------------

    ok = verify_shard(
        shard_idx,
        shard_rows,
    )

    assert ok, (
        f"Shard verification failed: "
        f"{shard_idx}"
    )

    completed.add(
        shard_idx
    )

    shard_minutes = (
        time.time() - shard_t0
    ) / 60

    rate = (
        shard_rows
        / max(time.time() - shard_t0, 1e-9)
    )

    print(
        f"[DONE] "
        f"{shard_start:,}:{shard_end:,} "
        f"rows={shard_rows:,} "
        f"rate={rate:,.0f}/sec "
        f"time={shard_minutes:.2f}m"
    )

    # --------------------------------------------------------------------------
    # Persist progress after EVERY shard.
    # --------------------------------------------------------------------------

    progress = {
        "model": MODEL_NAME,
        "embedding_dimension": EMBED_DIM,
        "max_length": MAX_LENGTH,
        "text_recipe": TEXT_RECIPE,

        "expected_rows": EXPECTED_ROWS,
        "shard_rows": SHARD_ROWS,
        "total_shards": n_shards,

        "completed_shards": sorted(
            int(x)
            for x in completed
        ),

        "completed_rows": sum(
            min(
                SHARD_ROWS,
                EXPECTED_ROWS - i * SHARD_ROWS,
            )
            for i in completed
        ),

        "status": "RUNNING",
        "last_completed_shard": int(shard_idx),
        "updated_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    with open(
        S2_MANIFEST_PATH,
        "w",
    ) as f:

        json.dump(
            progress,
            f,
            indent=2,
        )

# ------------------------------------------------------------------------------
# 9. ALL SHARDS COMPLETE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("9. FINAL S2 VERIFICATION")
print("-" * 78)

assert len(completed) == n_shards, (
    f"Only {len(completed)} / {n_shards} shards completed."
)

total_verified_rows = 0

for shard_idx in range(n_shards):

    start = shard_idx * SHARD_ROWS

    end = min(
        start + SHARD_ROWS,
        EXPECTED_ROWS,
    )

    rows_this_shard = end - start

    assert verify_shard(
        shard_idx,
        rows_this_shard,
    )

    total_verified_rows += rows_this_shard

print(
    f"Verified shards: "
    f"{n_shards:,}"
)

print(
    f"Verified embedding rows: "
    f"{total_verified_rows:,}"
)

assert total_verified_rows == EXPECTED_ROWS

# ------------------------------------------------------------------------------
# 10. FINAL MANIFEST
# ------------------------------------------------------------------------------

final_manifest = {
    "status": "COMPLETE",

    "source": "train/s2",

    "rows": EXPECTED_ROWS,

    "embedding_model":
        MODEL_NAME,

    "embedding_dimension":
        EMBED_DIM,

    "max_sequence_length":
        MAX_LENGTH,

    "text_recipe":
        TEXT_RECIPE,

    "shard_rows":
        SHARD_ROWS,

    "shards":
        n_shards,

    "dtype":
        "float16",

    "normalized":
        True,

    "gpu":
        GPU_NAME,

    "torch":
        str(torch.__version__),

    "cuda":
        str(torch.version.cuda),

    "output_root":
        str(S2_EMBED_ROOT),

    "completed_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

with open(
    S2_MANIFEST_PATH,
    "w",
) as f:

    json.dump(
        final_manifest,
        f,
        indent=2,
    )

# ------------------------------------------------------------------------------
# 11. COMPLETION MARKER
# ------------------------------------------------------------------------------

S2_COMPLETE_MARKER.write_text(
    "CELL 65 COMPLETE\n"
    "TRAIN S2 EMBEDDINGS COMPLETE\n"
    f"ROWS={EXPECTED_ROWS}\n"
    f"EMBED_DIM={EMBED_DIM}\n"
    f"MODEL={MODEL_NAME}\n"
)

# ------------------------------------------------------------------------------
# 12. FINAL STATS
# ------------------------------------------------------------------------------

total_embedding_bytes = sum(
    p.stat().st_size
    for p in S2_EMBED_ROOT.glob("embeddings_*.npy")
)

total_id_bytes = sum(
    p.stat().st_size
    for p in S2_EMBED_ROOT.glob("ids_*.parquet")
)

elapsed = (
    time.time() - T0
) / 60

print("\n" + "=" * 78)
print("✅ CELL 65 COMPLETE — TRAIN S2 GPU EMBEDDINGS")
print("=" * 78)

print(f"""
Rows:
  {EXPECTED_ROWS:,}

Embedding dimension:
  {EMBED_DIM}

Model:
  {MODEL_NAME}

GPU:
  {GPU_NAME}

Shards:
  {n_shards:,}

Shard size:
  {SHARD_ROWS:,}

Embedding dtype:
  float16

Embedding storage:
  {total_embedding_bytes / (1024**3):.2f} GB

ID storage:
  {total_id_bytes / (1024**3):.2f} GB

Output:
  {S2_EMBED_ROOT}

Manifest:
  {S2_MANIFEST_PATH}

Completion marker:
  {S2_COMPLETE_MARKER}

✅ Every shard verified.
✅ All {EXPECTED_ROWS:,} S2 records embedded.
✅ Embeddings normalized.
✅ Durable progress manifest written.
✅ Kernel-reset safe.

NEXT:
  CHECKPOINT AFTER CELL 65
""")

print(
    f"Cell 65 runtime: {elapsed:.2f} min"
)

AMLC 2026 — CELL 65
TRAIN / S2 GPU SEMANTIC EMBEDDINGS

Feature root : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1
S2 records   : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s2/records.parquet
S2 embeddings: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s2/full_record

------------------------------------------------------------------------------
1. GPU
------------------------------------------------------------------------------
PyTorch: 2.10.0+cu128
CUDA: 12.8
GPU: Tesla T4
VRAM GB: 14.56

------------------------------------------------------------------------------
2. TRAIN / S2 RECORD TABLE
------------------------------------------------------------------------------
Columns:
  - entity_id
  - business_name
  - business_address
  - country
  - name_norm
  - address_norm
  - name_suffix_removed
  - address_ascii
  - name_ascii
  - name_translit
  - address_translit
  - name_digit_signature
  - address_digit_signature
  - name_alpha_signature


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

[DONE] 0:100,000 rows=100,000 rate=3,919/sec time=0.43m

[S2 2/51] rows=100,000:200,000


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

[DONE] 100,000:200,000 rows=100,000 rate=3,908/sec time=0.43m

[S2 3/51] rows=200,000:300,000


...

[DONE] 4,800,000:4,900,000 rows=100,000 rate=3,940/sec time=0.42m

[S2 50/51] rows=4,900,000:5,000,000


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

[DONE] 4,900,000:5,000,000 rows=100,000 rate=3,920/sec time=0.43m

[S2 51/51] rows=5,000,000:5,034,616


Batches:   0%|          | 0/136 [00:00<?, ?it/s]

[DONE] 5,000,000:5,034,616 rows=34,616 rate=3,885/sec time=0.15m

------------------------------------------------------------------------------
9. FINAL S2 VERIFICATION
------------------------------------------------------------------------------
Verified shards: 51
Verified embedding rows: 5,034,616

✅ CELL 65 COMPLETE — TRAIN S2 GPU EMBEDDINGS

Rows:
  5,034,616

Embedding dimension:
  384

Model:
  intfloat/multilingual-e5-small

GPU:
  Tesla T4

Shards:
  51

Shard size:
  100,000

Embedding dtype:
  float16

Embedding storage:
  3.60 GB

ID storage:
  0.03 GB

Output:
  /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s2/full_record

Manifest:
  /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s2/full_record/manifest.json

Completion marker:
  /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s2/full_record/COMPLETE

✅ Every shard verified.
✅ All 5,034,616 S2 records embedded.
✅ Embeddings normalized.
✅ Durable progress manifest writte

In [17]:
# ==============================================================================
# AMLC 2026 — CHECKPOINT AFTER CELL 65
# TRAIN S2 EMBEDDINGS COMPLETE
# ==============================================================================

from pathlib import Path
import json
import hashlib
import subprocess
import sys
import time
import polars as pl
import numpy as np

print("=" * 78)
print("AMLC 2026 — CHECKPOINT AFTER CELL 65")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

S2_EMBED_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s2"
    / "full_record"
)

S1_EMBED_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s1"
    / "full_record"
)

S2_RECORD_PATH = (
    FEATURE_ROOT
    / "records"
    / "train"
    / "s2"
    / "records.parquet"
)

CHECKPOINT_ROOT = Path(
    "/kaggle/working/AMLC2026/"
    "checkpoint_after_cell65_S2_EMBEDDINGS_20260926"
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_PATH = (
    CHECKPOINT_ROOT / "CHECKPOINT_METADATA.json"
)

MARKER_PATH = (
    CHECKPOINT_ROOT / "CELL65_S2_EMBEDDINGS_COMPLETE"
)

MANIFEST_COPY = (
    CHECKPOINT_ROOT / "s2_embedding_manifest.json"
)

ENV_PATH = (
    CHECKPOINT_ROOT / "environment.txt"
)

# ------------------------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------------------------

EXPECTED_S2_ROWS = 5_034_616
EXPECTED_S1_ROWS = 2_206_821
EMBED_DIM = 384
SHARD_ROWS = 100_000
EXPECTED_SHARDS = 51
MODEL_NAME = "intfloat/multilingual-e5-small"

# ------------------------------------------------------------------------------
# 1. VERIFY FEATURE LAKE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. FEATURE LAKE")
print("-" * 78)

assert FEATURE_ROOT.exists()
assert S2_EMBED_ROOT.exists()
assert S1_EMBED_ROOT.exists()
assert S2_RECORD_PATH.exists()

print(f"Feature root: {FEATURE_ROOT}")
print(f"S2 embeddings: {S2_EMBED_ROOT}")

# ------------------------------------------------------------------------------
# 2. VERIFY S2 RECORD TABLE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. VERIFY S2 RECORD TABLE")
print("-" * 78)

s2_rows = (
    pl.scan_parquet(S2_RECORD_PATH)
    .select(pl.len())
    .collect()
    .item()
)

print(f"S2 record rows: {s2_rows:,}")

assert s2_rows == EXPECTED_S2_ROWS

# ------------------------------------------------------------------------------
# 3. VERIFY S2 EMBEDDING MANIFEST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. VERIFY S2 EMBEDDING MANIFEST")
print("-" * 78)

manifest_path = S2_EMBED_ROOT / "manifest.json"
complete_marker = S2_EMBED_ROOT / "COMPLETE"

assert manifest_path.exists()
assert complete_marker.exists()

with open(manifest_path) as f:
    s2_manifest = json.load(f)

print(json.dumps(s2_manifest, indent=2))

assert s2_manifest["status"] == "COMPLETE"
assert s2_manifest["rows"] == EXPECTED_S2_ROWS
assert s2_manifest["embedding_dimension"] == EMBED_DIM
assert s2_manifest["shards"] == EXPECTED_SHARDS
assert s2_manifest["dtype"] == "float16"
assert s2_manifest["normalized"] is True

# ------------------------------------------------------------------------------
# 4. VERIFY EVERY S2 SHARD
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. VERIFY ALL S2 SHARDS")
print("-" * 78)

verified_rows = 0
verified_shards = 0
total_embedding_bytes = 0
total_id_bytes = 0

for shard_idx in range(EXPECTED_SHARDS):

    start = shard_idx * SHARD_ROWS
    end = min(
        start + SHARD_ROWS,
        EXPECTED_S2_ROWS,
    )

    expected_rows = end - start

    emb_path = (
        S2_EMBED_ROOT
        / f"embeddings_{shard_idx:04d}.npy"
    )

    id_path = (
        S2_EMBED_ROOT
        / f"ids_{shard_idx:04d}.parquet"
    )

    assert emb_path.exists(), (
        f"Missing embedding shard: {emb_path}"
    )

    assert id_path.exists(), (
        f"Missing ID shard: {id_path}"
    )

    arr = np.load(
        emb_path,
        mmap_mode="r",
    )

    ids = pl.read_parquet(
        id_path,
        columns=["entity_id"],
    )

    assert arr.shape == (
        expected_rows,
        EMBED_DIM,
    ), (
        f"Bad shape in shard {shard_idx}: "
        f"{arr.shape}"
    )

    assert arr.dtype == np.float16

    assert ids.height == expected_rows

    verified_rows += expected_rows
    verified_shards += 1

    total_embedding_bytes += emb_path.stat().st_size
    total_id_bytes += id_path.stat().st_size

    del arr
    del ids

print(f"Verified shards: {verified_shards}/{EXPECTED_SHARDS}")
print(f"Verified rows:   {verified_rows:,}")

assert verified_shards == EXPECTED_SHARDS
assert verified_rows == EXPECTED_S2_ROWS

# ------------------------------------------------------------------------------
# 5. VERIFY S1 EMBEDDING CHECKPOINT STILL EXISTS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. VERIFY PREVIOUS S1 EMBEDDINGS")
print("-" * 78)

s1_files = [
    p for p in S1_EMBED_ROOT.rglob("*")
    if p.is_file()
]

print(f"S1 embedding files: {len(s1_files):,}")

assert len(s1_files) > 0

# ------------------------------------------------------------------------------
# 6. COPY MANIFEST + CREATE STATE MANIFEST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. WRITE CHECKPOINT METADATA")
print("-" * 78)

import shutil

shutil.copy2(
    manifest_path,
    MANIFEST_COPY,
)

metadata = {
    "checkpoint_name":
        "AMLC2026_AFTER_CELL65_S2_EMBEDDINGS",

    "state":
        "CELL65_COMPLETE_TRAIN_S2_EMBEDDINGS_COMPLETE",

    "next_step":
        "TRAIN S3 EMBEDDINGS",

    "feature_root":
        str(FEATURE_ROOT),

    "s2_record_path":
        str(S2_RECORD_PATH),

    "s1_embedding_root":
        str(S1_EMBED_ROOT),

    "s2_embedding_root":
        str(S2_EMBED_ROOT),

    "model":
        MODEL_NAME,

    "embedding_dimension":
        EMBED_DIM,

    "dtype":
        "float16",

    "normalized":
        True,

    "s2_rows":
        EXPECTED_S2_ROWS,

    "s2_shards":
        EXPECTED_SHARDS,

    "s2_shard_rows":
        SHARD_ROWS,

    "verified_s2_rows":
        verified_rows,

    "s1_rows":
        EXPECTED_S1_ROWS,

    "s2_embedding_bytes":
        total_embedding_bytes,

    "s2_id_bytes":
        total_id_bytes,

    "created_at":
        time.strftime("%Y-%m-%d %H:%M:%S"),

    "previous_state":
        "CELL64_COMPLETE_TRAIN_S1_EMBEDDINGS_COMPLETE",
}

with open(METADATA_PATH, "w") as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

# ------------------------------------------------------------------------------
# 7. ENVIRONMENT SNAPSHOT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. SAVE ENVIRONMENT")
print("-" * 78)

try:

    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        timeout=120,
    )

    ENV_PATH.write_text(result.stdout)

except Exception as e:

    ENV_PATH.write_text(
        f"pip freeze failed: {repr(e)}\n"
    )

# ------------------------------------------------------------------------------
# 8. COMPLETION MARKER
# ------------------------------------------------------------------------------

MARKER_PATH.write_text(
    "CELL 65 COMPLETE\n"
    "TRAIN S2 EMBEDDINGS COMPLETE\n"
    f"ROWS={EXPECTED_S2_ROWS}\n"
    f"SHARDS={EXPECTED_SHARDS}\n"
    f"EMBED_DIM={EMBED_DIM}\n"
    f"MODEL={MODEL_NAME}\n"
)

# ------------------------------------------------------------------------------
# 9. FINAL
# ------------------------------------------------------------------------------

elapsed = (
    time.time() - T0
) / 60

print("\n" + "=" * 78)
print("✅ CELL 65 CHECKPOINT COMPLETE")
print("=" * 78)

print(f"""
CHECKPOINT:
  {CHECKPOINT_ROOT}

S2 EMBEDDINGS:
  {S2_EMBED_ROOT}

Rows:
  {verified_rows:,}

Shards:
  {verified_shards}/{EXPECTED_SHARDS}

Embedding storage:
  {total_embedding_bytes / (1024**3):.2f} GB

ID storage:
  {total_id_bytes / (1024**3):.2f} GB

Model:
  {MODEL_NAME}

Dimension:
  {EMBED_DIM}

✅ S1 embeddings still present
✅ S2 embeddings fully verified
✅ Manifest copied
✅ Metadata written
✅ Environment snapshot written
✅ Completion marker written

NEXT:
  CELL 66 — TRAIN S3 EMBEDDINGS

Checkpoint runtime:
  {elapsed:.2f} min
""")

AMLC 2026 — CHECKPOINT AFTER CELL 65

------------------------------------------------------------------------------
1. FEATURE LAKE
------------------------------------------------------------------------------
Feature root: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1
S2 embeddings: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s2/full_record

------------------------------------------------------------------------------
2. VERIFY S2 RECORD TABLE
------------------------------------------------------------------------------
S2 record rows: 5,034,616

------------------------------------------------------------------------------
3. VERIFY S2 EMBEDDING MANIFEST
------------------------------------------------------------------------------
{
  "status": "COMPLETE",
  "source": "train/s2",
  "rows": 5034616,
  "embedding_model": "intfloat/multilingual-e5-small",
  "embedding_dimension": 384,
  "max_sequence_length": 256,
  "text_recipe": "passage_v1_name_address_count

In [18]:
from pathlib import Path

TARGET = "FINAL_FEATURE_LAKE_V1"

candidates = []

# Current writable session
p = Path("/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1")
if p.exists():
    candidates.append(p)

# Attached notebook outputs / datasets
for base in [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]:
    if base.exists():
        for p in base.rglob(TARGET):
            if p.is_dir():
                candidates.append(p)

# Deduplicate
candidates = list(dict.fromkeys(candidates))

print("Feature lake candidates:")
for p in candidates:
    print("  ", p)

assert candidates, (
    "FINAL_FEATURE_LAKE_V1 was not found. "
    "Attach the saved Notebook Output via "
    "Add Data → Notebook Output Files."
)

# Prefer writable working copy if present, otherwise attached output.
working = [
    p for p in candidates
    if str(p).startswith("/kaggle/working/")
]

FEATURE_ROOT = (
    working[0]
    if working
    else candidates[0]
)

print("\n✅ FEATURE_ROOT:")
print(FEATURE_ROOT)

Feature lake candidates:
   /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

✅ FEATURE_ROOT:
/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1


In [19]:
# ==============================================================================
# AMLC 2026 — CELL 66
# TRAIN / S3 GPU SEMANTIC EMBEDDINGS
#
# FIRST:
#   Aggressive RAM cleanup from previous cells.
#
# THEN:
#   Train/S3 -> multilingual-e5-small -> 384d float16
#
# INPUT:
#   FINAL_FEATURE_LAKE_V1/records/train/s3/records.parquet
#
# OUTPUT:
#   FINAL_FEATURE_LAKE_V1/embeddings/train/s3/full_record/
#
# DESIGN:
#   - 100k-row durable shards
#   - float16 embeddings
#   - aligned entity_id parquet
#   - resumable after kernel reset
#   - existing completed shards are skipped
# ==============================================================================

from pathlib import Path
import gc
import json
import math
import os
import time

# ------------------------------------------------------------------------------
# 0. RAM CLEANUP — DO THIS BEFORE LOADING ANYTHING LARGE
# ------------------------------------------------------------------------------

print("=" * 78)
print("AMLC 2026 — CELL 66")
print("TRAIN / S3 GPU SEMANTIC EMBEDDINGS")
print("=" * 78)

print("\n" + "-" * 78)
print("0. PRE-RUN RAM CLEANUP")
print("-" * 78)

# Objects created by previous cells that are no longer needed for embedding.
# The durable versions of the important artifacts are already on disk.
BIG_RUNTIME_OBJECTS = [
    # Pair / GT state
    "pair_sample",
    "sampled",
    "cand",
    "labeled",
    "gt",
    "gt_raw",
    "gt_edges",
    "gt_edges_full",
    "positive_edges",
    "sample_positive_edges",
    "negative_pairs",
    "positive_pairs",
    "val_s1",
    "val_s1_df",

    # Record dataframes
    "s1_df",
    "s2_df",
    "s3_df",
    "train_s1",
    "train_s2",
    "train_s3",
    "test_s1",
    "test_s2",
    "test_s3",
    "records",

    # Feature / embedding arrays
    "E",
    "embedding",
    "embeddings",
    "feats",
    "features",
    "X",
    "X_train",
    "X_val",
    "X_test",
    "texts",
    "entity_ids",

    # Old model / tokenizer references.
    # Cell 66 reloads the model cleanly after memory cleanup.
    "embed_model",
    "model",
    "tokenizer",

    # Common temporary objects
    "shard_df",
    "smoke",
    "smoke_texts",
    "df",
    "arr",
]

deleted = []

for name in BIG_RUNTIME_OBJECTS:
    if name in globals():
        try:
            del globals()[name]
            deleted.append(name)
        except Exception:
            pass

print(
    f"Deleted large runtime objects: {len(deleted):,}"
)

if deleted:
    print("Examples:")
    print("  " + ", ".join(deleted[:20]))

# Python garbage collector.
gc.collect()

# CUDA cache.
try:
    import torch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

except Exception as e:
    print(f"CUDA cleanup note: {e}")

# Optional RAM report.
try:
    import psutil

    vm = psutil.virtual_memory()

    print(
        f"\nRAM after cleanup:"
        f" {vm.used / (1024**3):.2f} / "
        f"{vm.total / (1024**3):.2f} GB "
        f"({vm.percent:.1f}%)"
    )

except Exception:
    pass

# ------------------------------------------------------------------------------
# 1. PATH DISCOVERY
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. FEATURE-LAKE DISCOVERY")
print("-" * 78)

FEATURE_TARGET = "FINAL_FEATURE_LAKE_V1"

feature_candidates = []

# Writable working copy first.
working_feature_root = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

if working_feature_root.exists():
    feature_candidates.append(
        working_feature_root
    )

# Attached notebook output / dataset copy.
kaggle_input = Path("/kaggle/input")

if kaggle_input.exists():
    for p in kaggle_input.rglob(
        FEATURE_TARGET
    ):
        if p.is_dir():
            feature_candidates.append(p)

# Deduplicate while preserving order.
feature_candidates = list(
    dict.fromkeys(feature_candidates)
)

print("Feature lake candidates:")

for p in feature_candidates:
    print(f"  - {p}")

assert feature_candidates, (
    "FINAL_FEATURE_LAKE_V1 was not found.\n\n"
    "Attach the saved notebook output containing the feature lake "
    "through Kaggle -> Add Data -> Notebook Output Files."
)

working_candidates = [
    p for p in feature_candidates
    if str(p).startswith("/kaggle/working/")
]

FEATURE_ROOT = (
    working_candidates[0]
    if working_candidates
    else feature_candidates[0]
)

print(f"\n✅ FEATURE_ROOT = {FEATURE_ROOT}")

# ------------------------------------------------------------------------------
# 2. PATHS / CONSTANTS
# ------------------------------------------------------------------------------

S3_RECORD_PATH = (
    FEATURE_ROOT
    / "records"
    / "train"
    / "s3"
    / "records.parquet"
)

S3_EMBED_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s3"
    / "full_record"
)

S3_EMBED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

S3_MANIFEST_PATH = (
    S3_EMBED_ROOT
    / "manifest.json"
)

S3_COMPLETE_MARKER = (
    S3_EMBED_ROOT
    / "COMPLETE"
)

MODEL_NAME = (
    "intfloat/multilingual-e5-small"
)

EXPECTED_ROWS = 5_285_603

EMBED_DIM = 384
MAX_LENGTH = 256

SHARD_ROWS = 100_000
ENCODE_BATCH_SIZE = 256

TEXT_RECIPE = (
    "passage_v1_name_address_country"
)

print(f"\nS3 records   : {S3_RECORD_PATH}")
print(f"S3 embeddings: {S3_EMBED_ROOT}")

assert S3_RECORD_PATH.exists(), (
    f"S3 record table missing:\n{S3_RECORD_PATH}"
)

# ------------------------------------------------------------------------------
# 3. GPU CHECK
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. GPU")
print("-" * 78)

import torch

assert torch.cuda.is_available(), (
    "CUDA is unavailable."
)

DEVICE = "cuda"

GPU_NAME = torch.cuda.get_device_name(0)

GPU_VRAM_GB = (
    torch.cuda.get_device_properties(0)
    .total_memory
    / (1024**3)
)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", GPU_NAME)
print(f"VRAM GB: {GPU_VRAM_GB:.2f}")

# ------------------------------------------------------------------------------
# 4. S3 RECORD PREFLIGHT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. TRAIN / S3 RECORD TABLE")
print("-" * 78)

schema = pl.read_parquet_schema(
    S3_RECORD_PATH
)

print("Columns:")

for c in schema:
    print(f"  - {c}")

REQUIRED_COLUMNS = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
]

missing = [
    c for c in REQUIRED_COLUMNS
    if c not in schema
]

assert not missing, (
    f"Missing required columns: {missing}"
)

record_rows = (
    pl.scan_parquet(S3_RECORD_PATH)
    .select(pl.len())
    .collect()
    .item()
)

print(
    f"\nRows: {record_rows:,}"
)

assert record_rows == EXPECTED_ROWS, (
    f"S3 row mismatch: "
    f"{record_rows:,} != {EXPECTED_ROWS:,}"
)

# ------------------------------------------------------------------------------
# 5. LOAD MODEL
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. LOAD MULTILINGUAL EMBEDDING MODEL")
print("-" * 78)

from sentence_transformers import SentenceTransformer

print(
    f"Loading {MODEL_NAME} onto CUDA..."
)

embed_model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
)

try:
    embed_model.max_seq_length = MAX_LENGTH
except Exception:
    pass

try:
    detected_dim = (
        embed_model.get_embedding_dimension()
    )
except AttributeError:
    detected_dim = (
        embed_model.get_sentence_embedding_dimension()
    )

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print(
    "Embedding dimension:",
    detected_dim,
)

assert detected_dim == EMBED_DIM

# ------------------------------------------------------------------------------
# 6. CANONICAL TEXT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. CANONICAL RECORD TEXT")
print("-" * 78)

print(
    "Recipe:",
    TEXT_RECIPE,
)

print(
    "Format:",
    "passage: name=<business_name> | "
    "address=<business_address> | "
    "country=<country>",
)

def build_record_text(df: pl.DataFrame) -> list[str]:

    return (
        df
        .with_columns([
            pl.col("business_name")
            .fill_null("")
            .cast(pl.Utf8)
            .alias("_name"),

            pl.col("business_address")
            .fill_null("")
            .cast(pl.Utf8)
            .alias("_address"),

            pl.col("country")
            .fill_null("")
            .cast(pl.Utf8)
            .alias("_country"),
        ])
        .with_columns(
            pl.concat_str(
                [
                    pl.lit("passage: name="),
                    pl.col("_name"),
                    pl.lit(" | address="),
                    pl.col("_address"),
                    pl.lit(" | country="),
                    pl.col("_country"),
                ],
                separator="",
            ).alias("_text")
        )
        .select("_text")
        .get_column("_text")
        .to_list()
    )

# ------------------------------------------------------------------------------
# 7. TEXT PREFLIGHT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. TEXT PREFLIGHT")
print("-" * 78)

smoke = (
    pl.scan_parquet(S3_RECORD_PATH)
    .select(REQUIRED_COLUMNS)
    .head(3)
    .collect()
)

smoke_texts = build_record_text(smoke)

for i, txt in enumerate(smoke_texts):
    print(f"\n[{i}]")
    print(txt[:1000])

assert len(smoke_texts) == 3
assert all(
    isinstance(x, str)
    for x in smoke_texts
)

del smoke
del smoke_texts

gc.collect()

print("\n✅ Text construction passed.")

# ------------------------------------------------------------------------------
# 8. SHARD PLAN
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. SHARD PLAN")
print("-" * 78)

n_shards = math.ceil(
    EXPECTED_ROWS / SHARD_ROWS
)

print(
    f"Rows per shard : {SHARD_ROWS:,}"
)

print(
    f"Total rows     : {EXPECTED_ROWS:,}"
)

print(
    f"Total shards   : {n_shards:,}"
)

final_shard_rows = (
    EXPECTED_ROWS
    - (n_shards - 1) * SHARD_ROWS
)

print(
    f"Final shard rows: {final_shard_rows:,}"
)

# Expected:
# 53 shards
# final shard = 85,603 rows

# ------------------------------------------------------------------------------
# 9. VERIFY EXISTING SHARDS FOR RESUME
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("8. RESUME SCAN")
print("-" * 78)

def verify_shard(
    shard_index: int,
    expected_rows: int,
) -> bool:

    emb_path = (
        S3_EMBED_ROOT
        / f"embeddings_{shard_index:04d}.npy"
    )

    id_path = (
        S3_EMBED_ROOT
        / f"ids_{shard_index:04d}.parquet"
    )

    if (
        not emb_path.exists()
        or not id_path.exists()
    ):
        return False

    try:

        arr = np.load(
            emb_path,
            mmap_mode="r",
        )

        ids = pl.read_parquet(
            id_path,
            columns=["entity_id"],
        )

        ok = (
            arr.shape
            == (
                expected_rows,
                EMBED_DIM,
            )
            and ids.height == expected_rows
            and arr.dtype == np.float16
        )

        del arr
        del ids

        return bool(ok)

    except Exception:
        return False


completed = set()

for shard_idx in range(n_shards):

    start = shard_idx * SHARD_ROWS

    end = min(
        start + SHARD_ROWS,
        EXPECTED_ROWS,
    )

    expected_rows = end - start

    if verify_shard(
        shard_idx,
        expected_rows,
    ):
        completed.add(shard_idx)

print(
    f"Valid existing shards: "
    f"{len(completed):,} / {n_shards:,}"
)

if completed:
    print(
        "Existing completed shard IDs:",
        sorted(completed)[:20],
        "..."
        if len(completed) > 20
        else "",
    )

# ------------------------------------------------------------------------------
# 10. EMBED S3 SHARDS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("9. EMBEDDING TRAIN / S3")
print("-" * 78)

for shard_idx in range(n_shards):

    shard_start = (
        shard_idx
        * SHARD_ROWS
    )

    shard_end = min(
        shard_start
        + SHARD_ROWS,
        EXPECTED_ROWS,
    )

    shard_rows = (
        shard_end
        - shard_start
    )

    emb_path = (
        S3_EMBED_ROOT
        / f"embeddings_{shard_idx:04d}.npy"
    )

    id_path = (
        S3_EMBED_ROOT
        / f"ids_{shard_idx:04d}.parquet"
    )

    # --------------------------------------------------------------------------
    # Already completed.
    # --------------------------------------------------------------------------

    if shard_idx in completed:

        print(
            f"[SKIP] "
            f"{shard_idx + 1}/{n_shards} "
            f"rows={shard_start:,}:{shard_end:,}"
        )

        continue

    # Remove stale partial outputs.
    if emb_path.exists():
        emb_path.unlink()

    if id_path.exists():
        id_path.unlink()

    print(
        f"\n[S3 {shard_idx + 1}/{n_shards}] "
        f"rows={shard_start:,}:{shard_end:,}"
    )

    shard_t0 = time.time()

    # --------------------------------------------------------------------------
    # Read exactly this shard.
    # --------------------------------------------------------------------------

    shard_df = (
        pl.scan_parquet(S3_RECORD_PATH)
        .select(REQUIRED_COLUMNS)
        .slice(
            shard_start,
            shard_rows,
        )
        .collect()
    )

    assert shard_df.height == shard_rows

    # --------------------------------------------------------------------------
    # Preserve exact ID ordering.
    # --------------------------------------------------------------------------

    shard_entity_ids = (
        shard_df
        .get_column("entity_id")
        .cast(pl.Utf8)
        .to_list()
    )

    assert (
        len(shard_entity_ids)
        == shard_rows
    )

    # --------------------------------------------------------------------------
    # Build text.
    # --------------------------------------------------------------------------

    texts = build_record_text(
        shard_df
    )

    assert len(texts) == shard_rows

    # --------------------------------------------------------------------------
    # GPU ENCODE
    # --------------------------------------------------------------------------

    with torch.inference_mode():

        E = embed_model.encode(
            texts,
            batch_size=ENCODE_BATCH_SIZE,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
            convert_to_tensor=False,
            device=DEVICE,
        )

    # Float16 durable storage.
    E = np.asarray(
        E,
        dtype=np.float16,
    )

    # --------------------------------------------------------------------------
    # HARD VALIDATION
    # --------------------------------------------------------------------------

    assert E.shape == (
        shard_rows,
        EMBED_DIM,
    ), (
        f"Bad embedding shape "
        f"{E.shape} for shard {shard_idx}"
    )

    assert np.isfinite(E).all(), (
        f"Non-finite embedding found "
        f"in shard {shard_idx}"
    )

    # --------------------------------------------------------------------------
    # SAVE IDS
    # --------------------------------------------------------------------------

    ids_df = pl.DataFrame({
        "entity_id":
            shard_entity_ids
    })

    ids_df.write_parquet(
        id_path,
        compression="zstd",
    )

    # --------------------------------------------------------------------------
    # SAVE EMBEDDINGS
    # --------------------------------------------------------------------------

    np.save(
        emb_path,
        E,
        allow_pickle=False,
    )

    # --------------------------------------------------------------------------
    # RELEASE RAM / VRAM BEFORE VALIDATION
    # --------------------------------------------------------------------------

    del shard_df
    del shard_entity_ids
    del texts
    del ids_df
    del E

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------------------------
    # Verify saved shard.
    # --------------------------------------------------------------------------

    assert verify_shard(
        shard_idx,
        shard_rows,
    ), (
        f"Saved shard {shard_idx} "
        f"failed verification."
    )

    completed.add(
        shard_idx
    )

    elapsed = (
        time.time()
        - shard_t0
    )

    rate = (
        shard_rows
        / max(elapsed, 1e-9)
    )

    print(
        f"[DONE] "
        f"{shard_start:,}:{shard_end:,} "
        f"rows={shard_rows:,} "
        f"rate={rate:,.0f}/sec "
        f"time={elapsed / 60:.2f}m"
    )

    # --------------------------------------------------------------------------
    # Durable progress manifest after EVERY shard.
    # --------------------------------------------------------------------------

    completed_rows = sum(
        min(
            SHARD_ROWS,
            EXPECTED_ROWS
            - i * SHARD_ROWS,
        )
        for i in completed
    )

    progress = {

        "status": "RUNNING",

        "source":
            "train/s3",

        "model":
            MODEL_NAME,

        "embedding_dimension":
            EMBED_DIM,

        "max_sequence_length":
            MAX_LENGTH,

        "text_recipe":
            TEXT_RECIPE,

        "expected_rows":
            EXPECTED_ROWS,

        "shard_rows":
            SHARD_ROWS,

        "total_shards":
            n_shards,

        "completed_shards":
            sorted(
                int(x)
                for x in completed
            ),

        "completed_rows":
            int(completed_rows),

        "last_completed_shard":
            int(shard_idx),

        "updated_at":
            time.strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
    }

    with open(
        S3_MANIFEST_PATH,
        "w",
    ) as f:

        json.dump(
            progress,
            f,
            indent=2,
        )

# ------------------------------------------------------------------------------
# 11. FINAL VERIFICATION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("10. FINAL S3 VERIFICATION")
print("-" * 78)

assert len(completed) == n_shards, (
    f"Only {len(completed)} / "
    f"{n_shards} shards complete."
)

verified_rows = 0

for shard_idx in range(n_shards):

    start = (
        shard_idx
        * SHARD_ROWS
    )

    end = min(
        start
        + SHARD_ROWS,
        EXPECTED_ROWS,
    )

    shard_rows = end - start

    assert verify_shard(
        shard_idx,
        shard_rows,
    )

    verified_rows += shard_rows

print(
    f"Verified shards: "
    f"{len(completed):,}"
)

print(
    f"Verified embedding rows: "
    f"{verified_rows:,}"
)

assert verified_rows == EXPECTED_ROWS

# ------------------------------------------------------------------------------
# 12. FINAL MANIFEST
# ------------------------------------------------------------------------------

final_manifest = {

    "status":
        "COMPLETE",

    "source":
        "train/s3",

    "rows":
        EXPECTED_ROWS,

    "embedding_model":
        MODEL_NAME,

    "embedding_dimension":
        EMBED_DIM,

    "max_sequence_length":
        MAX_LENGTH,

    "text_recipe":
        TEXT_RECIPE,

    "shard_rows":
        SHARD_ROWS,

    "shards":
        n_shards,

    "dtype":
        "float16",

    "normalized":
        True,

    "gpu":
        GPU_NAME,

    "torch":
        str(torch.__version__),

    "cuda":
        str(torch.version.cuda),

    "output_root":
        str(S3_EMBED_ROOT),

    "completed_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

with open(
    S3_MANIFEST_PATH,
    "w",
) as f:

    json.dump(
        final_manifest,
        f,
        indent=2,
    )

# ------------------------------------------------------------------------------
# 13. COMPLETION MARKER
# ------------------------------------------------------------------------------

S3_COMPLETE_MARKER.write_text(
    "CELL 66 COMPLETE\n"
    "TRAIN S3 EMBEDDINGS COMPLETE\n"
    f"ROWS={EXPECTED_ROWS}\n"
    f"SHARDS={n_shards}\n"
    f"EMBED_DIM={EMBED_DIM}\n"
    f"MODEL={MODEL_NAME}\n"
)

# ------------------------------------------------------------------------------
# 14. FINAL STORAGE REPORT
# ------------------------------------------------------------------------------

embedding_bytes = sum(
    p.stat().st_size
    for p in S3_EMBED_ROOT.glob(
        "embeddings_*.npy"
    )
)

id_bytes = sum(
    p.stat().st_size
    for p in S3_EMBED_ROOT.glob(
        "ids_*.parquet"
    )
)

total_minutes = (
    time.time() - T0
) / 60

# RAM report.
try:
    import psutil

    vm = psutil.virtual_memory()

    ram_line = (
        f"{vm.used / (1024**3):.2f} / "
        f"{vm.total / (1024**3):.2f} GB "
        f"({vm.percent:.1f}%)"
    )

except Exception:
    ram_line = "unavailable"

print("\n" + "=" * 78)
print("✅ CELL 66 COMPLETE — TRAIN S3 GPU EMBEDDINGS")
print("=" * 78)

print(f"""
Rows:
  {EXPECTED_ROWS:,}

Embedding dimension:
  {EMBED_DIM}

Model:
  {MODEL_NAME}

GPU:
  {GPU_NAME}

Shards:
  {n_shards:,}

Shard size:
  {SHARD_ROWS:,}

Final shard:
  {final_shard_rows:,}

Embedding dtype:
  float16

Embedding storage:
  {embedding_bytes / (1024**3):.2f} GB

ID storage:
  {id_bytes / (1024**3):.2f} GB

Output:
  {S3_EMBED_ROOT}

Manifest:
  {S3_MANIFEST_PATH}

Completion marker:
  {S3_COMPLETE_MARKER}

RAM at finish:
  {ram_line}

✅ Every S3 shard verified.
✅ All 5,285,603 S3 records embedded.
✅ Embeddings normalized.
✅ Durable progress manifest written.
✅ Kernel-reset safe.

NEXT:
  CHECKPOINT AFTER CELL 66
""")

print(
    f"Cell 66 runtime: "
    f"{total_minutes:.2f} min"
)

AMLC 2026 — CELL 66
TRAIN / S3 GPU SEMANTIC EMBEDDINGS

------------------------------------------------------------------------------
0. PRE-RUN RAM CLEANUP
------------------------------------------------------------------------------
Deleted large runtime objects: 1
Examples:
  embed_model

RAM after cleanup: 24.07 / 31.35 GB (76.6%)

------------------------------------------------------------------------------
1. FEATURE-LAKE DISCOVERY
------------------------------------------------------------------------------
Feature lake candidates:
  - /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

✅ FEATURE_ROOT = /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

S3 records   : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s3/records.parquet
S3 embeddings: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s3/full_record

------------------------------------------------------------------------------
2. GPU
----------------------------------------------------------

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: intfloat/multilingual-e5-small
Device: cuda
Embedding dimension: 384

------------------------------------------------------------------------------
5. CANONICAL RECORD TEXT
------------------------------------------------------------------------------
Recipe: passage_v1_name_address_country
Format: passage: name=<business_name> | address=<business_address> | country=<country>

------------------------------------------------------------------------------
6. TEXT PREFLIGHT
------------------------------------------------------------------------------

[0]
passage: name=wilfordhancock.com | address=Mack Rd, Haltom City, Texas | country=US

[1]
passage: name=International South Consultants Private Ltd | address= | country=India

[2]
passage: name=LLC Moncada Léarning Center | address=5780 Fawn Ct, Fort Worth, Texas | country=US

✅ Text construction passed.

------------------------------------------------------------------------------
7. SHARD PLAN
--------------------------------

Batches:   0%|          | 0/391 [00:00<?, ?it/s]

...

[DONE] 5,000,000:5,100,000 rows=100,000 rate=1,287/sec time=1.29m

[S3 52/53] rows=5,100,000:5,200,000


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

[DONE] 5,100,000:5,200,000 rows=100,000 rate=1,286/sec time=1.30m

[S3 53/53] rows=5,200,000:5,285,603


Batches:   0%|          | 0/335 [00:00<?, ?it/s]

[DONE] 5,200,000:5,285,603 rows=85,603 rate=1,291/sec time=1.11m

------------------------------------------------------------------------------
10. FINAL S3 VERIFICATION
------------------------------------------------------------------------------
Verified shards: 53
Verified embedding rows: 5,285,603

✅ CELL 66 COMPLETE — TRAIN S3 GPU EMBEDDINGS

Rows:
  5,285,603

Embedding dimension:
  384

Model:
  intfloat/multilingual-e5-small

GPU:
  Tesla T4

Shards:
  53

Shard size:
  100,000

Final shard:
  85,603

Embedding dtype:
  float16

Embedding storage:
  3.78 GB

ID storage:
  0.03 GB

Output:
  /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s3/full_record

Manifest:
  /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s3/full_record/manifest.json

Completion marker:
  /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/embeddings/train/s3/full_record/COMPLETE

RAM at finish:
  24.25 / 31.35 GB (77.3%)

✅ Every S3 shard verified.
✅ All 5,285,603 S3 records

In [22]:
# ==============================================================================
# AMLC 2026 — CELL 66A
# DIAGNOSE ACTUAL CELL-64 S1 EMBEDDING ARTIFACT LAYOUT
# ==============================================================================

from pathlib import Path
import json
import os

import numpy as np
import polars as pl

print("=" * 78)
print("AMLC 2026 — CELL 66A")
print("DIAGNOSE ACTUAL TRAIN S1 EMBEDDING ARTIFACT")
print("=" * 78)

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

S1_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s1"
    / "full_record"
)

assert S1_ROOT.exists(), (
    f"S1 embedding directory missing:\n{S1_ROOT}"
)

# ------------------------------------------------------------------------------
# 1. COMPLETE FILE INVENTORY
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. FILE INVENTORY")
print("-" * 78)

files = sorted(
    [p for p in S1_ROOT.rglob("*") if p.is_file()],
    key=lambda p: str(p),
)

print(f"Total files: {len(files):,}")

for i, p in enumerate(files):

    size_mb = p.stat().st_size / (1024 ** 2)

    print(
        f"{i:03d} | "
        f"{p.name:<45} | "
        f"{p.suffix:<12} | "
        f"{size_mb:10.2f} MB"
    )

# ------------------------------------------------------------------------------
# 2. FILE EXTENSION SUMMARY
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. EXTENSION SUMMARY")
print("-" * 78)

ext_counts = {}

for p in files:
    ext = p.suffix.lower() or "<none>"
    ext_counts[ext] = (
        ext_counts.get(ext, 0) + 1
    )

for ext, count in sorted(ext_counts.items()):
    print(
        f"{ext:<15} : {count:>5}"
    )

# ------------------------------------------------------------------------------
# 3. MANIFEST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. MANIFEST")
print("-" * 78)

manifest_path = S1_ROOT / "manifest.json"

if manifest_path.exists():

    print(
        f"Manifest: {manifest_path}"
    )

    with open(manifest_path) as f:
        manifest = json.load(f)

    print(
        json.dumps(
            manifest,
            indent=2,
        )
    )

else:

    print("NO manifest.json")

# ------------------------------------------------------------------------------
# 4. INSPECT ALL NUMPY ARRAYS WITHOUT LOADING THEM
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. NUMPY ARRAY INSPECTION")
print("-" * 78)

npy_files = [
    p for p in files
    if p.suffix.lower() == ".npy"
]

print(
    f"NPY files: {len(npy_files):,}"
)

for p in npy_files:

    try:

        arr = np.load(
            p,
            mmap_mode="r",
        )

        print(
            f"{p.name:<45} "
            f"shape={str(arr.shape):<24} "
            f"dtype={arr.dtype}"
        )

        del arr

    except Exception as e:

        print(
            f"{p.name:<45} "
            f"ERROR={repr(e)}"
        )

# ------------------------------------------------------------------------------
# 5. INSPECT ALL NPZ ARRAYS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. NPZ INSPECTION")
print("-" * 78)

npz_files = [
    p for p in files
    if p.suffix.lower() == ".npz"
]

print(
    f"NPZ files: {len(npz_files):,}"
)

for p in npz_files:

    try:

        z = np.load(
            p,
            mmap_mode="r",
        )

        print(
            f"\n{p.name}"
        )

        for key in z.files:

            arr = z[key]

            print(
                f"  {key}: "
                f"shape={arr.shape} "
                f"dtype={arr.dtype}"
            )

        del z

    except Exception as e:

        print(
            f"{p.name}: ERROR={repr(e)}"
        )

# ------------------------------------------------------------------------------
# 6. INSPECT ALL PARQUET FILES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. PARQUET INSPECTION")
print("-" * 78)

parquet_files = [
    p for p in files
    if p.suffix.lower() == ".parquet"
]

print(
    f"Parquet files: {len(parquet_files):,}"
)

for p in parquet_files:

    try:

        schema = pl.read_parquet_schema(p)

        rows = (
            pl.scan_parquet(p)
            .select(pl.len())
            .collect()
            .item()
        )

        print(
            f"\n{p.name}"
        )

        print(
            f"  rows: {rows:,}"
        )

        print(
            "  columns:",
            schema,
        )

        # Inspect only one row.
        sample = (
            pl.read_parquet(
                p,
                n_rows=1,
            )
        )

        print(
            "  sample columns:",
            sample.columns,
        )

        print(
            "  sample schema:",
            sample.schema,
        )

        del sample

    except Exception as e:

        print(
            f"{p.name}: ERROR={repr(e)}"
        )

# ------------------------------------------------------------------------------
# 7. OTHER BINARY FILE TYPES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. OTHER FILE TYPES")
print("-" * 78)

known = {
    ".json",
    ".npy",
    ".npz",
    ".parquet",
    ".txt",
    ".csv",
}

for p in files:

    if p.suffix.lower() not in known:

        size_mb = (
            p.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"{p.name:<50} "
            f"{p.suffix:<12} "
            f"{size_mb:.2f} MB"
        )

# ------------------------------------------------------------------------------
# 8. POTENTIAL EMBEDDING FILE HEURISTICS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("8. EMBEDDING-LAYOUT HEURISTICS")
print("-" * 78)

for p in files:

    name = p.name.lower()

    if any(
        token in name
        for token in [
            "embed",
            "vector",
            "feat",
            "chunk",
            "shard",
            "full",
        ]
    ):

        print(
            f"Potential embedding artifact: "
            f"{p.relative_to(S1_ROOT)}"
        )

print("\n" + "=" * 78)
print("✅ DIAGNOSTIC COMPLETE")
print("=" * 78)

print("""
NO FILES WERE MODIFIED.
NO EMBEDDINGS WERE RECOMPUTED.
NO LARGE EMBEDDING ARRAYS WERE LOADED INTO RAM.

Paste the output of this cell.
Then I will give you the exact S1 verification/checkpoint cell
for the format Cell 64 actually produced.
""")

AMLC 2026 — CELL 66A
DIAGNOSE ACTUAL TRAIN S1 EMBEDDING ARTIFACT

------------------------------------------------------------------------------
1. FILE INVENTORY
------------------------------------------------------------------------------
Total files: 47
000 | emb_000000000_000100000.npy                   | .npy         |      73.24 MB
001 | emb_000100000_000200000.npy                   | .npy         |      73.24 MB
002 | emb_000200000_000300000.npy                   | .npy         |      73.24 MB
003 | emb_000300000_000400000.npy                   | .npy         |      73.24 MB
004 | emb_000400000_000500000.npy                   | .npy         |      73.24 MB
005 | emb_000500000_000600000.npy                   | .npy         |      73.24 MB
006 | emb_000600000_000700000.npy                   | .npy         |      73.24 MB
007 | emb_000700000_000800000.npy                   | .npy         |      73.24 MB
008 | emb_000800000_000900000.npy                   | .npy         |      73.2

In [23]:
# ==============================================================================
# AMLC 2026 — MASTER CHECKPOINT AFTER CELL 66
# FINAL FIX — SUPPORTS ORIGINAL CELL-64 S1 FORMAT
#
# IMPORTANT:
#   - DOES NOT recompute embeddings
#   - DOES NOT rewrite embedding shards
#   - DOES NOT create ZIP
#   - ONLY verifies existing artifacts + creates durable markers/metadata
# ==============================================================================

from pathlib import Path
import gc
import json
import subprocess
import sys
import time

import numpy as np
import polars as pl

print("=" * 78)
print("AMLC 2026 — MASTER CHECKPOINT AFTER CELL 66")
print("FINAL FIX — ORIGINAL CELL-64 S1 FORMAT")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# 0. RAM CLEANUP
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("0. RAM CLEANUP")
print("-" * 78)

for name in [
    "embed_model",
    "model",
    "tokenizer",
    "E",
    "embedding",
    "embeddings",
    "texts",
    "shard_df",
    "smoke",
    "smoke_texts",
    "df",
    "arr",
]:
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass

gc.collect()

try:
    import torch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

try:
    import psutil

    vm = psutil.virtual_memory()

    print(
        f"RAM: {vm.used / (1024**3):.2f} / "
        f"{vm.total / (1024**3):.2f} GB "
        f"({vm.percent:.1f}%)"
    )
except Exception:
    pass

# ------------------------------------------------------------------------------
# 1. LOCATE FEATURE LAKE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. LOCATE FEATURE LAKE")
print("-" * 78)

FEATURE_NAME = "FINAL_FEATURE_LAKE_V1"

candidates = []

working_root = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

if working_root.exists():
    candidates.append(
        working_root
    )

input_root = Path(
    "/kaggle/input"
)

if input_root.exists():

    for p in input_root.rglob(
        FEATURE_NAME
    ):

        if p.is_dir():
            candidates.append(p)

candidates = list(
    dict.fromkeys(candidates)
)

print("Candidates:")

for p in candidates:
    print("  -", p)

assert candidates, (
    "FINAL_FEATURE_LAKE_V1 not found. "
    "Attach the saved Notebook Output."
)

working_candidates = [
    p for p in candidates
    if str(p).startswith(
        "/kaggle/working/"
    )
]

FEATURE_ROOT = (
    working_candidates[0]
    if working_candidates
    else candidates[0]
)

print(
    "\n✅ FEATURE_ROOT =",
    FEATURE_ROOT
)

# ------------------------------------------------------------------------------
# 2. VERIFY RECORD TABLES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. VERIFY RECORD LAKE")
print("-" * 78)

EXPECTED_RECORDS = {
    ("train", "s1"): 2_206_821,
    ("train", "s2"): 5_034_616,
    ("train", "s3"): 5_285_603,
    ("test", "s1"): 1_732_544,
    ("test", "s2"): 4_887_273,
    ("test", "s3"): 5_082_316,
}

record_inventory = {}

for (split, source), expected_rows in EXPECTED_RECORDS.items():

    path = (
        FEATURE_ROOT
        / "records"
        / split
        / source
        / "records.parquet"
    )

    assert path.exists(), (
        f"Missing record table:\n{path}"
    )

    schema = pl.read_parquet_schema(
        path
    )

    actual_rows = (
        pl.scan_parquet(path)
        .select(pl.len())
        .collect()
        .item()
    )

    print(
        f"{split:5s}/{source}: "
        f"{actual_rows:,} rows × "
        f"{len(schema)} columns"
    )

    assert actual_rows == expected_rows
    assert len(schema) == 32

    record_inventory[
        f"{split}/{source}"
    ] = {
        "relative_path": str(
            path.relative_to(
                FEATURE_ROOT
            )
        ),
        "rows": int(actual_rows),
        "columns": int(len(schema)),
        "bytes": int(
            path.stat().st_size
        ),
    }

print("✅ All six record tables verified.")

# ------------------------------------------------------------------------------
# 3. VERIFY FEATURE SCHEMA
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. VERIFY FEATURE SCHEMA")
print("-" * 78)

SCHEMA_PATH = (
    FEATURE_ROOT
    / "schema"
    / "feature_schema_v1.json"
)

assert SCHEMA_PATH.exists()

with open(SCHEMA_PATH) as f:
    schema_obj = json.load(f)

registered = None

if isinstance(schema_obj, dict):

    registered = (
        schema_obj.get("columns")
        or schema_obj.get("features")
        or schema_obj.get(
            "registered_columns"
        )
    )

else:

    registered = schema_obj

if registered is not None:

    print(
        "Registered feature columns:",
        len(registered),
    )

    assert len(registered) == 86

print("✅ Schema verified.")

# ------------------------------------------------------------------------------
# 4. FUNCTION — VERIFY SHARDED EMBEDDINGS
# ------------------------------------------------------------------------------

def verify_sharded_embedding_set(
    root: Path,
    expected_rows: int,
    expected_dim: int = 384,
):
    """
    Supports both:
      embeddings_0000.npy + ids_0000.parquet
    and original Cell-64:
      emb_000000000_000100000.npy
      ids_000000000_000100000.parquet
    """

    npy_files = sorted(
        root.glob("*.npy")
    )

    parquet_files = sorted(
        root.glob("*.parquet")
    )

    assert npy_files, (
        f"No NPY files found in {root}"
    )

    assert parquet_files, (
        f"No parquet ID files found in {root}"
    )

    assert len(npy_files) == len(
        parquet_files
    ), (
        f"Embedding/ID shard count mismatch "
        f"in {root}: "
        f"{len(npy_files)} vs "
        f"{len(parquet_files)}"
    )

    verified_rows = 0
    total_emb_bytes = 0
    total_id_bytes = 0

    for emb_path in npy_files:

        # ----------------------------------------------------------------------
        # Parse expected row count from filename where available.
        # Original Cell 64:
        #   emb_000000000_000100000.npy
        # ----------------------------------------------------------------------

        stem = emb_path.stem

        expected_shard_rows = None

        parts = stem.split("_")

        if (
            len(parts) >= 3
            and parts[0] == "emb"
        ):

            try:

                start = int(parts[-2])
                end = int(parts[-1])

                expected_shard_rows = (
                    end - start
                )

            except Exception:
                pass

        arr = np.load(
            emb_path,
            mmap_mode="r",
        )

        assert len(arr.shape) == 2, (
            f"Bad embedding shape: "
            f"{emb_path}: {arr.shape}"
        )

        assert arr.shape[1] == expected_dim, (
            f"Wrong dimension in "
            f"{emb_path}: "
            f"{arr.shape}"
        )

        assert arr.dtype == np.float16, (
            f"Wrong dtype in "
            f"{emb_path}: "
            f"{arr.dtype}"
        )

        actual_shard_rows = int(
            arr.shape[0]
        )

        if expected_shard_rows is not None:

            assert (
                actual_shard_rows
                == expected_shard_rows
            ), (
                f"Filename/data mismatch:\n"
                f"{emb_path}\n"
                f"filename rows="
                f"{expected_shard_rows:,}, "
                f"actual="
                f"{actual_shard_rows:,}"
            )

        verified_rows += (
            actual_shard_rows
        )

        total_emb_bytes += (
            emb_path.stat().st_size
        )

        del arr

    for p in parquet_files:

        rows = (
            pl.scan_parquet(p)
            .select(pl.len())
            .collect()
            .item()
        )

        # ID tables may have both:
        #   row_index
        #   entity_id
        #
        # We only require the expected entity IDs to exist.
        schema = pl.read_parquet_schema(p)

        assert "entity_id" in schema, (
            f"entity_id missing in {p}"
        )

        total_id_rows = rows

        total_id_bytes += (
            p.stat().st_size
        )

        # Keep row totals separately below.

    parquet_rows = sum(
        (
            pl.scan_parquet(p)
            .select(pl.len())
            .collect()
            .item()
        )
        for p in parquet_files
    )

    assert verified_rows == expected_rows, (
        f"Embedding rows mismatch:\n"
        f"{root}\n"
        f"{verified_rows:,} != "
        f"{expected_rows:,}"
    )

    assert parquet_rows == expected_rows, (
        f"ID rows mismatch:\n"
        f"{root}\n"
        f"{parquet_rows:,} != "
        f"{expected_rows:,}"
    )

    return {
        "rows": int(verified_rows),
        "shards": len(npy_files),
        "embedding_files": len(npy_files),
        "id_files": len(parquet_files),
        "embedding_bytes": int(
            total_emb_bytes
        ),
        "id_bytes": int(
            total_id_bytes
        ),
        "dimension": expected_dim,
        "dtype": "float16",
        "normalized_expected": True,
    }

# ------------------------------------------------------------------------------
# 5. VERIFY S1 — ORIGINAL CELL-64 FORMAT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. VERIFY TRAIN / S1 EMBEDDINGS")
print("   ORIGINAL CELL-64 FORMAT")
print("-" * 78)

S1_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s1"
    / "full_record"
)

assert S1_ROOT.exists()

# Read original manifest.
S1_MANIFEST_PATH = (
    S1_ROOT
    / "manifest.json"
)

assert S1_MANIFEST_PATH.exists()

with open(
    S1_MANIFEST_PATH
) as f:

    s1_manifest = json.load(f)

print(
    "Model:",
    s1_manifest.get("model")
)

print(
    "Dimension:",
    s1_manifest.get("dimension")
)

print(
    "Dtype:",
    s1_manifest.get("dtype")
)

print(
    "Rows:",
    s1_manifest.get("rows")
)

print(
    "Shard rows:",
    s1_manifest.get(
        "shard_rows"
    )
)

assert (
    s1_manifest["model"]
    == "intfloat/multilingual-e5-small"
)

assert (
    s1_manifest["dimension"]
    == 384
)

assert (
    s1_manifest["dtype"]
    == "float16"
)

assert (
    s1_manifest["rows"]
    == 2_206_821
)

assert (
    s1_manifest["shard_rows"]
    == 100_000
)

# Manifest itself contains the authoritative shard list.
manifest_shards = (
    s1_manifest["shards"]
)

assert len(manifest_shards) == 23

print(
    f"Manifest shards: "
    f"{len(manifest_shards)}"
)

# Verify every manifest-declared file.
s1_verified_rows = 0
s1_embedding_bytes = 0
s1_id_bytes = 0

for i, shard in enumerate(
    manifest_shards
):

    start = int(
        shard["start"]
    )

    end = int(
        shard["end"]
    )

    expected_rows = (
        end - start
    )

    emb_path = (
        S1_ROOT
        / shard["embedding"]
    )

    id_path = (
        S1_ROOT
        / shard["ids"]
    )

    assert emb_path.exists(), (
        f"Missing S1 embedding shard:\n"
        f"{emb_path}"
    )

    assert id_path.exists(), (
        f"Missing S1 ID shard:\n"
        f"{id_path}"
    )

    arr = np.load(
        emb_path,
        mmap_mode="r",
    )

    assert arr.shape == (
        expected_rows,
        384,
    ), (
        f"Bad S1 shard shape:\n"
        f"{emb_path}\n"
        f"{arr.shape}"
    )

    assert arr.dtype == np.float16

    ids_schema = (
        pl.read_parquet_schema(
            id_path
        )
    )

    assert "entity_id" in ids_schema

    id_rows = (
        pl.scan_parquet(id_path)
        .select(pl.len())
        .collect()
        .item()
    )

    assert id_rows == expected_rows, (
        f"ID row mismatch:\n"
        f"{id_path}"
    )

    s1_verified_rows += (
        expected_rows
    )

    s1_embedding_bytes += (
        emb_path.stat().st_size
    )

    s1_id_bytes += (
        id_path.stat().st_size
    )

    del arr

    print(
        f"[S1 {i+1:02d}/23] "
        f"{start:,}:{end:,} "
        f"rows={expected_rows:,} ✅"
    )

assert (
    s1_verified_rows
    == 2_206_821
)

print(
    "\n✅ S1 COMPLETE — "
    f"{s1_verified_rows:,} rows verified."
)

# ------------------------------------------------------------------------------
# 6. CREATE S1 COMPLETE MARKER NOW
# ------------------------------------------------------------------------------

S1_COMPLETE_MARKER = (
    S1_ROOT
    / "COMPLETE"
)

S1_COMPLETE_MARKER.write_text(
    "CELL 64 COMPLETE\n"
    "TRAIN S1 EMBEDDINGS COMPLETE\n"
    "Verified by MASTER CHECKPOINT AFTER CELL 66.\n"
    "Rows=2206821\n"
    "Shards=23\n"
    "Dimension=384\n"
    "Dtype=float16\n"
    "Model=intfloat/multilingual-e5-small\n"
)

print(
    "✅ S1 COMPLETE marker written:"
)

print(
    S1_COMPLETE_MARKER
)

# ------------------------------------------------------------------------------
# 7. VERIFY S2
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. VERIFY TRAIN / S2 EMBEDDINGS")
print("-" * 78)

S2_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s2"
    / "full_record"
)

assert (
    S2_ROOT
    / "manifest.json"
).exists()

assert (
    S2_ROOT
    / "COMPLETE"
).exists()

s2_info = (
    verify_sharded_embedding_set(
        S2_ROOT,
        5_034_616,
        384,
    )
)

with open(
    S2_ROOT / "manifest.json"
) as f:

    s2_manifest = json.load(f)

assert s2_manifest["status"] == "COMPLETE"
assert s2_manifest["rows"] == 5_034_616
assert s2_manifest["shards"] == 51
assert s2_manifest["dtype"] == "float16"
assert s2_manifest["normalized"] is True

print(
    f"✅ S2: "
    f"{s2_info['rows']:,} rows / "
    f"{s2_info['shards']} shards"
)

# ------------------------------------------------------------------------------
# 8. VERIFY S3
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. VERIFY TRAIN / S3 EMBEDDINGS")
print("-" * 78)

S3_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s3"
    / "full_record"
)

assert (
    S3_ROOT
    / "manifest.json"
).exists()

assert (
    S3_ROOT
    / "COMPLETE"
).exists()

s3_info = (
    verify_sharded_embedding_set(
        S3_ROOT,
        5_285_603,
        384,
    )
)

with open(
    S3_ROOT / "manifest.json"
) as f:

    s3_manifest = json.load(f)

assert s3_manifest["status"] == "COMPLETE"
assert s3_manifest["rows"] == 5_285_603
assert s3_manifest["shards"] == 53
assert s3_manifest["dtype"] == "float16"
assert s3_manifest["normalized"] is True

print(
    f"✅ S3: "
    f"{s3_info['rows']:,} rows / "
    f"{s3_info['shards']} shards"
)

# ------------------------------------------------------------------------------
# 9. BUILD MASTER INVENTORY
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. BUILD MASTER CHECKPOINT INVENTORY")
print("-" * 78)

embedding_inventory = {

    "s1": {
        "layout":
            "original_cell64_manifest_shards",

        "rows":
            s1_verified_rows,

        "shards":
            23,

        "dimension":
            384,

        "dtype":
            "float16",

        "embedding_bytes":
            s1_embedding_bytes,

        "id_bytes":
            s1_id_bytes,

        "complete_marker":
            True,
    },

    "s2": s2_info,

    "s3": s3_info,
}

TOTAL_EMBED_BYTES = sum(
    x["embedding_bytes"]
    for x in embedding_inventory.values()
)

TOTAL_ID_BYTES = sum(
    x["id_bytes"]
    for x in embedding_inventory.values()
)

# ------------------------------------------------------------------------------
# 10. SAVE MASTER CHECKPOINT
# ------------------------------------------------------------------------------

CHECKPOINT_ROOT = Path(
    "/kaggle/working/AMLC2026/"
    "checkpoint_after_cell66_ALL_TRAIN_EMBEDDINGS_20260926"
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

METADATA_PATH = (
    CHECKPOINT_ROOT
    / "CHECKPOINT_METADATA.json"
)

INVENTORY_PATH = (
    CHECKPOINT_ROOT
    / "FEATURE_LAKE_INVENTORY.json"
)

MASTER_MARKER = (
    CHECKPOINT_ROOT
    / "CELL66_ALL_TRAIN_EMBEDDINGS_COMPLETE"
)

inventory = {

    "checkpoint":
        "AMLC2026_AFTER_CELL66",

    "state":
        "ALL_TRAIN_EMBEDDINGS_COMPLETE",

    "feature_root":
        str(FEATURE_ROOT),

    "cell61":
        "COMPLETE",

    "cell62":
        "COMPLETE",

    "cell63":
        "COMPLETE",

    "cell64":
        "COMPLETE",

    "cell65":
        "COMPLETE",

    "cell66":
        "COMPLETE",

    "train_embeddings":
        embedding_inventory,

    "records":
        record_inventory,

    "next_stage":
        "PAIR_LEVEL_SEMANTIC_FEATURES",

    "zip_created":
        False,

    "feature_lake_is_payload":
        True,

    "created_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

with open(
    INVENTORY_PATH,
    "w",
) as f:

    json.dump(
        inventory,
        f,
        indent=2,
    )

metadata = {

    "checkpoint_name":
        "AMLC2026_AFTER_CELL66",

    "state":
        "ALL_TRAIN_EMBEDDINGS_COMPLETE",

    "feature_root":
        str(FEATURE_ROOT),

    "train_s1_rows":
        2_206_821,

    "train_s2_rows":
        5_034_616,

    "train_s3_rows":
        5_285_603,

    "train_s1_verified":
        True,

    "train_s2_verified":
        True,

    "train_s3_verified":
        True,

    "s1_marker_created":
        True,

    "s1_layout":
        "Cell64_original_manifest_shards",

    "s2_layout":
        "Cell65_sharded_npy",

    "s3_layout":
        "Cell66_sharded_npy",

    "total_embedding_bytes":
        TOTAL_EMBED_BYTES,

    "total_id_bytes":
        TOTAL_ID_BYTES,

    "next_stage":
        "PAIR_LEVEL_SEMANTIC_FEATURES",

    "zip_created":
        False,

    "created_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

with open(
    METADATA_PATH,
    "w",
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
    )

MASTER_MARKER.write_text(
    "AMLC 2026 MASTER CHECKPOINT\n"
    "CELL 61 COMPLETE\n"
    "CELL 62 COMPLETE\n"
    "CELL 63 COMPLETE\n"
    "CELL 64 COMPLETE\n"
    "CELL 65 COMPLETE\n"
    "CELL 66 COMPLETE\n"
    "TRAIN S1 EMBEDDINGS COMPLETE\n"
    "TRAIN S2 EMBEDDINGS COMPLETE\n"
    "TRAIN S3 EMBEDDINGS COMPLETE\n"
    "NEXT=PAIR_LEVEL_SEMANTIC_FEATURES\n"
)

# ------------------------------------------------------------------------------
# 11. FINAL CLEANUP
# ------------------------------------------------------------------------------

gc.collect()

try:

    import torch

    if torch.cuda.is_available():

        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

except Exception:
    pass

elapsed = (
    time.time() - T0
) / 60

print("\n" + "=" * 78)
print("✅✅✅ MASTER CHECKPOINT AFTER CELL 66 COMPLETE ✅✅✅")
print("=" * 78)

print(f"""
FEATURE ROOT:
  {FEATURE_ROOT}

S1:
  2,206,821 rows ✅
  23 shards ✅
  Original Cell-64 format verified ✅
  COMPLETE marker created ✅

S2:
  5,034,616 rows ✅
  51 shards ✅
  COMPLETE marker verified ✅

S3:
  5,285,603 rows ✅
  53 shards ✅
  COMPLETE marker verified ✅

TOTAL TRAIN EMBEDDING STORAGE:
  {TOTAL_EMBED_BYTES / (1024**3):.2f} GB

TOTAL ID STORAGE:
  {TOTAL_ID_BYTES / (1024**3):.2f} GB

CHECKPOINT:
  {CHECKPOINT_ROOT}

METADATA:
  {METADATA_PATH}

INVENTORY:
  {INVENTORY_PATH}

ZIP:
  ❌ NONE

NEXT:
  PAIR-LEVEL SEMANTIC FEATURES

✅ NO EMBEDDINGS WERE RECOMPUTED.
✅ NO FEATURE-LAKE DATA WAS COPIED.
✅ ORIGINAL S1 FORMAT WAS PRESERVED.
✅ ALL TRAIN EMBEDDING DATA IS VERIFIED.
""")

print(
    f"Checkpoint runtime: "
    f"{elapsed:.2f} min"
)

AMLC 2026 — MASTER CHECKPOINT AFTER CELL 66
FINAL FIX — ORIGINAL CELL-64 S1 FORMAT

------------------------------------------------------------------------------
0. RAM CLEANUP
------------------------------------------------------------------------------
RAM: 24.26 / 31.35 GB (75.1%)

------------------------------------------------------------------------------
1. LOCATE FEATURE LAKE
------------------------------------------------------------------------------
Candidates:
  - /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

✅ FEATURE_ROOT = /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

------------------------------------------------------------------------------
2. VERIFY RECORD LAKE
------------------------------------------------------------------------------
train/s1: 2,206,821 rows × 32 columns
train/s2: 5,034,616 rows × 32 columns
train/s3: 5,285,603 rows × 32 columns
test /s1: 1,732,544 rows × 32 columns
test /s2: 4,887,273 rows × 32 columns
test /s3: 5,082,316 rows × 32 co

In [26]:
# ==============================================================================
# AMLC 2026 — CELL 67A v2
# RECOVER TRAINING PAIR SAMPLE
#
# FIX:
#   Convert eval_s1 to LazyFrame before joining to scan_parquet().
#
# NO:
#   - blocking regeneration
#   - embedding recomputation
#   - model loading
#
# OUTPUT:
#   FINAL_FEATURE_LAKE_V1/state/train_pair_sample_v1/
# ==============================================================================

from pathlib import Path
import gc
import json
import time

import numpy as np
import polars as pl


print("=" * 78)
print("AMLC 2026 — CELL 67A v2")
print("RECOVER TRAINING PAIR SAMPLE")
print("=" * 78)

T0 = time.time()

# ==============================================================================
# 0. RAM CLEANUP
# ==============================================================================

print("\n" + "-" * 78)
print("0. RAM CLEANUP")
print("-" * 78)

for name in [
    "embed_model",
    "model",
    "tokenizer",
    "E",
    "embedding",
    "embeddings",
    "texts",
    "cand",
    "candidate_pool",
    "labeled",
    "pair_sample",
    "sampled",
    "gt",
    "gt_raw",
    "gt_eval",
    "gt_edges",
    "gt_edges_eval",
    "gt_edges_full",
    "positive_edges",
    "sample_positive_edges",
    "negative_pairs",
    "positive_pairs",
]:
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass

try:
    if "Out" in globals():
        Out.clear()
except Exception:
    pass

gc.collect()

try:
    import torch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

try:
    import psutil

    vm = psutil.virtual_memory()

    print(
        f"RAM: "
        f"{vm.used / (1024**3):.2f} / "
        f"{vm.total / (1024**3):.2f} GB "
        f"({vm.percent:.1f}%)"
    )
except Exception:
    pass

# ==============================================================================
# 1. PATHS
# ==============================================================================

print("\n" + "-" * 78)
print("1. PATHS")
print("-" * 78)

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

MASTER_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/"
    "amlc-2026-final-workspace/"
    "AMLC2026_KAGGLE_FINAL"
)

CELL58_ROOT = (
    MASTER_ROOT
    / "checkpoint_after_cell58_FINAL_20260925"
    / "state"
)

ADDRESS_RESCUE_PATH = (
    CELL58_ROOT
    / "eval_candidates_address_rescue.parquet"
)

S1_EVAL_PATH = (
    CELL58_ROOT
    / "s1_eval.parquet"
)

POSITIVE_DIAGNOSTICS_PATH = (
    CELL58_ROOT
    / "positive_pair_signal_diagnostics.parquet"
)

GT_PATH = (
    MASTER_ROOT
    / "dataset"
    / "train"
    / "train_ground_truth.tsv"
)

assert FEATURE_ROOT.exists()
assert MASTER_ROOT.exists()
assert ADDRESS_RESCUE_PATH.exists()
assert GT_PATH.exists()

print("FEATURE_ROOT:")
print(FEATURE_ROOT)

print("\nADDRESS RESCUE:")
print(ADDRESS_RESCUE_PATH)

print("\nS1 EVAL:")
print(S1_EVAL_PATH)

print("\nGT:")
print(GT_PATH)

# ==============================================================================
# 2. RECOVER 100K S1 EVALUATION UNIVERSE
# ==============================================================================

print("\n" + "-" * 78)
print("2. RECOVER 100K-S1 EVALUATION UNIVERSE")
print("-" * 78)

assert S1_EVAL_PATH.exists()

s1_eval_schema = pl.read_parquet_schema(
    S1_EVAL_PATH
)

print(
    "s1_eval columns:",
    list(s1_eval_schema.keys())
)

possible_s1_id_cols = [
    "s1_entity_id",
    "entity_id",
    "source1_entity_id",
]

found_s1_col = next(
    (
        c
        for c in possible_s1_id_cols
        if c in s1_eval_schema
    ),
    None,
)

assert found_s1_col is not None, (
    "Could not identify S1 ID column in s1_eval."
)

eval_s1 = (
    pl.read_parquet(
        S1_EVAL_PATH,
        columns=[
            found_s1_col
        ],
    )
    .rename({
        found_s1_col:
            "s1_entity_id"
    })
    .select(
        pl.col(
            "s1_entity_id"
        )
        .cast(pl.Utf8)
    )
    .unique()
)

print(
    f"Recovered S1 IDs: "
    f"{eval_s1.height:,}"
)

assert eval_s1.height == 100_000

# IMPORTANT FIX:
# Candidate source is a LazyFrame.
# Therefore use eval_s1.lazy() for the join.
eval_s1_lazy = eval_s1.lazy()

# ==============================================================================
# 3. OFFICIAL GT
# ==============================================================================

print("\n" + "-" * 78)
print("3. BUILD EVALUATION GT EDGES")
print("-" * 78)

gt_raw = pl.read_csv(
    GT_PATH,
    separator="\t",
    has_header=True,
    infer_schema_length=10000,
)

assert "source1_entity_id" in gt_raw.columns
assert "matched_entity_ids" in gt_raw.columns

gt_eval = (
    gt_raw
    .select([
        pl.col(
            "source1_entity_id"
        )
        .cast(pl.Utf8)
        .alias(
            "s1_entity_id"
        ),

        pl.col(
            "matched_entity_ids"
        )
        .cast(pl.Utf8)
        .alias(
            "matched_entity_ids"
        ),
    ])
    .join(
        eval_s1,
        on="s1_entity_id",
        how="semi",
    )
)

gt_edges_eval = (
    gt_eval
    .with_columns(
        pl.col(
            "matched_entity_ids"
        )
        .str.split(",")
        .alias(
            "candidate_entity_id"
        )
    )
    .explode(
        "candidate_entity_id"
    )
    .with_columns(
        pl.col(
            "candidate_entity_id"
        )
        .str.strip_chars()
        .cast(Utf8 if False else pl.Utf8)
    )
    .filter(
        pl.col(
            "candidate_entity_id"
        ).is_not_null()
        &
        (
            pl.col(
                "candidate_entity_id"
            ) != ""
        )
    )
    .select([
        "s1_entity_id",
        "candidate_entity_id",
    ])
    .unique(
        subset=[
            "s1_entity_id",
            "candidate_entity_id",
        ]
    )
)

print(
    f"Evaluation GT positive edges: "
    f"{gt_edges_eval.height:,}"
)

assert (
    gt_edges_eval.height
    == 345_980
), (
    f"Historical GT edge count mismatch: "
    f"{gt_edges_eval.height:,} != 345,980"
)

print(
    "✅ Historical 345,980 positive-edge "
    "universe reproduced."
)

# Free full GT raw table now.
del gt_raw
del gt_eval
gc.collect()

# ==============================================================================
# 4. LOAD PRESERVED ADDRESS-RESCUE CANDIDATES
# ==============================================================================

print("\n" + "-" * 78)
print("4. LOAD PRESERVED ADDRESS-RESCUE CANDIDATES")
print("-" * 78)

candidate_schema = pl.read_parquet_schema(
    ADDRESS_RESCUE_PATH
)

print(
    "Candidate columns:",
    list(candidate_schema.keys())
)

assert (
    "s1_entity_id"
    in candidate_schema
)

assert (
    "candidate_entity_id"
    in candidate_schema
)

# Source is already present and blocker provenance is preserved.
for required_col in [
    "source",
    "block_exact",
    "block_first_last",
    "block_address_exact",
    "block_address_numeric",
]:
    assert required_col in candidate_schema, (
        f"Missing expected candidate column: "
        f"{required_col}"
    )

# ==============================================================================
# IMPORTANT:
# Keep this as LazyFrame until the 100k-S1 filter has been applied.
# eval_s1_lazy is also LazyFrame, so the join is valid.
# ==============================================================================

candidate_lazy = (
    pl.scan_parquet(
        ADDRESS_RESCUE_PATH
    )
    .select([
        pl.col(
            "s1_entity_id"
        ).cast(pl.Utf8),

        pl.col(
            "candidate_entity_id"
        ).cast(pl.Utf8),

        pl.col(
            "source"
        ).cast(pl.Utf8),

        pl.col(
            "block_exact"
        ).cast(pl.Int8),

        pl.col(
            "block_first_last"
        ).cast(pl.Int8),

        pl.col(
            "block_address_exact"
        ).cast(pl.Int8),

        pl.col(
            "block_address_numeric"
        ).cast(pl.Int8),
    ])
    .join(
        eval_s1_lazy,
        on="s1_entity_id",
        how="semi",
    )
    .filter(
        pl.col(
            "candidate_entity_id"
        ).str.starts_with("S2-")
        |
        pl.col(
            "candidate_entity_id"
        ).str.starts_with("S3-")
    )
    .unique(
        subset=[
            "s1_entity_id",
            "candidate_entity_id",
        ]
    )
)

print(
    "Collecting filtered candidate pool..."
)

candidate_pool = candidate_lazy.collect(
    engine="streaming"
)

print(
    f"Evaluation candidate rows: "
    f"{candidate_pool.height:,}"
)

assert (
    candidate_pool.height > 0
)

# Free lazy objects.
del candidate_lazy
del eval_s1_lazy
gc.collect()

# ==============================================================================
# 5. LABEL CANDIDATES
# ==============================================================================

print("\n" + "-" * 78)
print("5. LABEL CANDIDATES AGAINST OFFICIAL GT")
print("-" * 78)

candidate_pool = (
    candidate_pool
    .join(
        gt_edges_eval.with_columns(
            pl.lit(1)
            .cast(pl.Int8)
            .alias(
                "_positive"
            )
        ),
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="left",
    )
    .with_columns(
        pl.col(
            "_positive"
        )
        .fill_null(0)
        .cast(pl.Int8)
        .alias(
            "is_positive"
        )
    )
    .drop(
        "_positive"
    )
)

candidate_positive_count = (
    candidate_pool
    .select(
        pl.col(
            "is_positive"
        ).sum()
    )
    .item()
)

candidate_negative_count = (
    candidate_pool.height
    - candidate_positive_count
)

print(
    f"Candidate positives: "
    f"{candidate_positive_count:,}"
)

print(
    f"Candidate negatives: "
    f"{candidate_negative_count:,}"
)

candidate_recall = (
    candidate_positive_count
    / gt_edges_eval.height
)

print(
    f"Candidate recall: "
    f"{candidate_recall:.6%}"
)

# ==============================================================================
# 6. BUILD TARGET 3,429,214 SAMPLE
# ==============================================================================

print("\n" + "-" * 78)
print("6. BUILD 3,429,214-ROW TRAINING PAIR SAMPLE")
print("-" * 78)

TARGET_ROWS = 3_429_214
SEED = 2026

assert candidate_positive_count < TARGET_ROWS

negative_needed = (
    TARGET_ROWS
    - candidate_positive_count
)

print(
    f"Target rows: "
    f"{TARGET_ROWS:,}"
)

print(
    f"Positive candidates retained: "
    f"{candidate_positive_count:,}"
)

print(
    f"Negative rows required: "
    f"{negative_needed:,}"
)

assert (
    negative_needed
    <= candidate_negative_count
)

positive_pairs = (
    candidate_pool
    .filter(
        pl.col(
            "is_positive"
        ) == 1
    )
)

negative_pairs = (
    candidate_pool
    .filter(
        pl.col(
            "is_positive"
        ) == 0
    )
)

# Stable deterministic hash.
negative_pairs = (
    negative_pairs
    .with_columns(
        pl.concat_str(
            [
                pl.lit(
                    str(SEED)
                ),

                pl.col(
                    "s1_entity_id"
                ),

                pl.col(
                    "candidate_entity_id"
                ),
            ],
            separator="|",
        )
        .hash(
            seed=SEED
        )
        .alias(
            "_sample_hash"
        )
    )
    .sort(
        "_sample_hash"
    )
    .head(
        negative_needed
    )
    .drop(
        "_sample_hash"
    )
)

pair_sample = pl.concat(
    [
        positive_pairs,
        negative_pairs,
    ],
    how="vertical",
)

assert (
    pair_sample.height
    == TARGET_ROWS
)

# ==============================================================================
# 7. FINAL PAIR COLUMNS
# ==============================================================================

print("\n" + "-" * 78)
print("7. BUILD FINAL PAIR-SAMPLE SCHEMA")
print("-" * 78)

pair_sample = (
    pair_sample
    .with_columns(
        (
            pl.col(
                "source"
            )
            .str.to_lowercase()
            .alias(
                "source"
            )
        )
    )
    .with_columns(
        pl.when(
            pl.col(
                "source"
            ).str.contains(
                r"3$"
            )
        )
        .then(
            1
        )
        .otherwise(
            0
        )
        .cast(pl.Int8)
        .alias(
            "source_is_s3"
        )
    )
)

source_counts = (
    pair_sample
    .group_by([
        "s1_entity_id",
        "source",
    ])
    .agg(
        pl.len().alias(
            "candidate_count_source"
        )
    )
)

total_counts = (
    pair_sample
    .group_by(
        "s1_entity_id"
    )
    .agg(
        pl.len().alias(
            "candidate_count_total"
        )
    )
)

pair_sample = (
    pair_sample
    .join(
        source_counts,
        on=[
            "s1_entity_id",
            "source",
        ],
        how="left",
    )
    .join(
        total_counts,
        on="s1_entity_id",
        how="left",
    )
)

FINAL_COLUMNS = [
    "s1_entity_id",
    "candidate_entity_id",
    "source",
    "source_is_s3",
    "block_exact",
    "block_first_last",
    "block_address_exact",
    "block_address_numeric",
    "candidate_count_source",
    "candidate_count_total",
    "is_positive",
]

pair_sample = (
    pair_sample
    .select(
        FINAL_COLUMNS
    )
    .sort([
        "s1_entity_id",
        "source",
        "candidate_entity_id",
    ])
)

print(
    "Final columns:",
    pair_sample.columns
)

# ==============================================================================
# 8. HARD VALIDATION
# ==============================================================================

print("\n" + "-" * 78)
print("8. HARD VALIDATION")
print("-" * 78)

assert (
    pair_sample.height
    == TARGET_ROWS
)

duplicate_pairs = (
    pair_sample
    .group_by([
        "s1_entity_id",
        "candidate_entity_id",
    ])
    .len()
    .filter(
        pl.col("len") > 1
    )
    .height
)

print(
    f"Duplicate pairs: "
    f"{duplicate_pairs:,}"
)

assert (
    duplicate_pairs == 0
)

sample_positive_count = (
    pair_sample
    .select(
        pl.col(
            "is_positive"
        ).sum()
    )
    .item()
)

print(
    f"Sample positives: "
    f"{sample_positive_count:,}"
)

print(
    f"Sample negatives: "
    f"{TARGET_ROWS - sample_positive_count:,}"
)

# Every positive must be an official GT edge.
bad_positive_count = (
    pair_sample
    .filter(
        pl.col(
            "is_positive"
        ) == 1
    )
    .join(
        gt_edges_eval,
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="anti",
    )
    .height
)

print(
    f"Bad positive labels: "
    f"{bad_positive_count:,}"
)

assert (
    bad_positive_count == 0
)

# ==============================================================================
# 9. SAVE INSIDE FEATURE LAKE
# ==============================================================================

print("\n" + "-" * 78)
print("9. SAVE PAIR SAMPLE INSIDE FEATURE LAKE")
print("-" * 78)

PAIR_STATE_ROOT = (
    FEATURE_ROOT
    / "state"
    / "train_pair_sample_v1"
)

PAIR_STATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

PAIR_OUTPUT = (
    PAIR_STATE_ROOT
    / "baseline_candidate_pairs.parquet"
)

if PAIR_OUTPUT.exists():
    PAIR_OUTPUT.unlink()

pair_sample.write_parquet(
    PAIR_OUTPUT,
    compression="zstd",
    statistics=True,
)

print(
    "Saved:",
    PAIR_OUTPUT
)

# ==============================================================================
# 10. SAVE VALIDATION S1
# ==============================================================================

val_s1 = (
    eval_s1
    .with_columns(
        (
            pl.col(
                "s1_entity_id"
            )
            .hash(
                seed=SEED
            )
            .mod(10)
            .alias(
                "_fold"
            )
        )
    )
    .filter(
        pl.col(
            "_fold"
        ) == 0
    )
    .select(
        "s1_entity_id"
    )
)

VAL_S1_OUTPUT = (
    PAIR_STATE_ROOT
    / "val_s1.parquet"
)

val_s1.write_parquet(
    VAL_S1_OUTPUT,
    compression="zstd",
)

print(
    f"Validation S1: "
    f"{val_s1.height:,}"
)

# ==============================================================================
# 11. METADATA
# ==============================================================================

metadata = {

    "cell":
        "67A",

    "status":
        "COMPLETE",

    "target_rows":
        TARGET_ROWS,

    "evaluation_s1_rows":
        eval_s1.height,

    "evaluation_gt_edges":
        gt_edges_eval.height,

    "candidate_pool_rows":
        candidate_pool.height,

    "candidate_positive_count":
        int(
            candidate_positive_count
        ),

    "candidate_negative_count":
        int(
            candidate_negative_count
        ),

    "candidate_recall":
        float(
            candidate_recall
        ),

    "sample_positive_count":
        int(
            sample_positive_count
        ),

    "sample_negative_count":
        int(
            TARGET_ROWS
            - sample_positive_count
        ),

    "seed":
        SEED,

    "candidate_artifact":
        str(
            ADDRESS_RESCUE_PATH
        ),

    "ground_truth":
        str(
            GT_PATH
        ),

    "pair_output":
        str(
            PAIR_OUTPUT
        ),

    "validation_s1":
        str(
            VAL_S1_OUTPUT
        ),

    "note":
        (
            "Reconstructed from preserved Cell-58 "
            "address-rescue candidate artifact and "
            "official training ground truth. "
            "Not claimed byte-identical to lost "
            "historical pair parquet."
        ),

    "created_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

META_OUTPUT = (
    PAIR_STATE_ROOT
    / "pair_sample_recovery_metadata.json"
)

with open(
    META_OUTPUT,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
    )

# ==============================================================================
# 12. COMPLETION MARKER
# ==============================================================================

COMPLETE_OUTPUT = (
    PAIR_STATE_ROOT
    / "COMPLETE"
)

COMPLETE_OUTPUT.write_text(
    "AMLC 2026\n"
    "CELL 67A COMPLETE\n"
    "TRAIN PAIR SAMPLE RECOVERED\n"
    f"ROWS={TARGET_ROWS}\n"
    f"POSITIVES={sample_positive_count}\n"
    f"NEGATIVES={TARGET_ROWS - sample_positive_count}\n"
)

# ==============================================================================
# 13. RELOAD
# ==============================================================================

print("\n" + "-" * 78)
print("10. RELOAD VERIFICATION")
print("-" * 78)

reload_check = pl.read_parquet(
    PAIR_OUTPUT
)

assert (
    reload_check.height
    == TARGET_ROWS
)

assert (
    reload_check.columns
    == FINAL_COLUMNS
)

print(
    f"Reloaded rows: "
    f"{reload_check.height:,}"
)

print(
    "Reloaded positives:",
    reload_check
    .select(
        pl.col(
            "is_positive"
        ).sum()
    )
    .item()
)

# ==============================================================================
# 14. CLEAN MEMORY
# ==============================================================================

del candidate_pool
del positive_pairs
del negative_pairs
del pair_sample
del gt_edges_eval
del eval_s1
del reload_check
del source_counts
del total_counts

gc.collect()

# ==============================================================================
# 15. FINAL
# ==============================================================================

elapsed = (
    time.time() - T0
) / 60

try:

    import psutil

    vm = psutil.virtual_memory()

    ram_final = (
        f"{vm.used / (1024**3):.2f} / "
        f"{vm.total / (1024**3):.2f} GB "
        f"({vm.percent:.1f}%)"
    )

except Exception:

    ram_final = "unavailable"

print("\n" + "=" * 78)
print("✅ CELL 67A v2 COMPLETE")
print("=" * 78)

print(f"""
PAIR SAMPLE:
  {PAIR_OUTPUT}

ROWS:
  {TARGET_ROWS:,}

POSITIVES:
  {sample_positive_count:,}

NEGATIVES:
  {TARGET_ROWS - sample_positive_count:,}

EVALUATION S1:
  {eval_s1.height:,}

EVALUATION GT EDGES:
  {gt_edges_eval.height:,}

CANDIDATE POOL:
  {candidate_pool.height:,}

CANDIDATE RECALL:
  {candidate_recall:.6%}

VALIDATION S1:
  {VAL_S1_OUTPUT}

METADATA:
  {META_OUTPUT}

COMPLETE MARKER:
  {COMPLETE_OUTPUT}

RAM:
  {ram_final}

✅ LazyFrame/DataFrame join fixed.
✅ No blocking regenerated.
✅ No embeddings recomputed.
✅ Exact 3,429,214-row target enforced.
✅ Pair sample saved INSIDE feature lake.

NEXT:
  RE-RUN CELL 67.
""")

print(
    f"Runtime: {elapsed:.2f} min"
)

AMLC 2026 — CELL 67A v2
RECOVER TRAINING PAIR SAMPLE

------------------------------------------------------------------------------
0. RAM CLEANUP
------------------------------------------------------------------------------
RAM: 24.39 / 31.35 GB (75.3%)

------------------------------------------------------------------------------
1. PATHS
------------------------------------------------------------------------------
FEATURE_ROOT:
/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

ADDRESS RESCUE:
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/state/eval_candidates_address_rescue.parquet

S1 EVAL:
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/state/s1_eval.parquet

GT:
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_ground_truth.tsv

--------------------------------------------

NameError: name 'eval_s1' is not defined

In [29]:
# ==============================================================================
# AMLC 2026 — CELL 67C
# PAIR-LEVEL SEMANTIC FEATURES V3
#
# FINAL MAPPING FIX
#
# IMPORTANT:
#   Entity ID numeric suffix is NOT an embedding row.
#
#   Some ID shards contain:
#       entity_id
#       row_index
#
#   S1 in the current feature lake contains only:
#       entity_id
#
#   Therefore the authoritative fallback is:
#
#       embedding row =
#           shard global start
#           +
#           row position inside the paired ID parquet
#
#   The ID parquet and NPY embedding shard are verified to have identical
#   row counts before using this mapping.
#
# ==============================================================================

print("=" * 78)
print("AMLC 2026 — CELL 67C")
print("PAIR-LEVEL SEMANTIC FEATURES V3")
print("AUTHORITATIVE SHARD-POSITION MAPPING")
print("=" * 78)

# ==============================================================================
# 0. HARD CLEANUP
# ==============================================================================

print("\n" + "-" * 78)
print("0. HARD CLEANUP")
print("-" * 78)

import gc
import json
import math
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import duckdb

try:
    from IPython import get_ipython
    ip = get_ipython()

    if ip is not None:
        try:
            Out.clear()
        except Exception:
            pass
except Exception:
    pass

gc.collect()

try:
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
except Exception:
    pass

try:
    import psutil
    vm = psutil.virtual_memory()
    print(
        f"RAM: {vm.used / (1024**3):.2f} / "
        f"{vm.total / (1024**3):.2f} GB "
        f"({vm.percent:.1f}%)"
    )
except Exception:
    pass

T0 = time.time()

# ==============================================================================
# 1. PATHS
# ==============================================================================

print("\n" + "-" * 78)
print("1. PATHS")
print("-" * 78)

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

PAIR_PATH = (
    FEATURE_ROOT
    / "state"
    / "train_pair_sample_v1"
    / "baseline_candidate_pairs.parquet"
)

INDEX_ROOT = (
    FEATURE_ROOT
    / "state"
    / "train_pair_sample_v1"
)

INDEXED_PAIR_PATH = (
    INDEX_ROOT
    / "pair_with_embedding_rows_v3.parquet"
)

EMBED_ROOTS = {

    "s1":
        FEATURE_ROOT
        / "embeddings"
        / "train"
        / "s1"
        / "full_record",

    "s2":
        FEATURE_ROOT
        / "embeddings"
        / "train"
        / "s2"
        / "full_record",

    "s3":
        FEATURE_ROOT
        / "embeddings"
        / "train"
        / "s3"
        / "full_record",
}

EXPECTED_ROWS = {
    "s1": 2_206_821,
    "s2": 5_034_616,
    "s3": 5_285_603,
}

assert FEATURE_ROOT.exists()
assert PAIR_PATH.exists()

for root in EMBED_ROOTS.values():
    assert root.exists()

print("FEATURE_ROOT:")
print(FEATURE_ROOT)

print("\nPAIR:")
print(PAIR_PATH)

# ==============================================================================
# 2. PAIR SAMPLE
# ==============================================================================

print("\n" + "-" * 78)
print("2. PAIR SAMPLE")
print("-" * 78)

PAIR_ROWS = (
    pl.scan_parquet(
        PAIR_PATH
    )
    .select(
        pl.len()
    )
    .collect()
    .item()
)

assert PAIR_ROWS == 3_429_214

print(
    f"Pair rows: {PAIR_ROWS:,}"
)

# ==============================================================================
# 3. DISCOVER EMBEDDING / ID SHARDS
# ==============================================================================

print("\n" + "-" * 78)
print("3. DISCOVER EMBEDDING / ID SHARDS")
print("-" * 78)

EMBED_DIM = 384


def discover_shards(
    source,
    root,
):

    if source == "s1":

        emb_files = sorted(
            root.glob(
                "emb_*.npy"
            )
        )

        id_files = sorted(
            root.glob(
                "ids_*.parquet"
            )
        )

    else:

        emb_files = sorted(
            root.glob(
                "embeddings_*.npy"
            )
        )

        id_files = sorted(
            root.glob(
                "ids_*.parquet"
            )
        )

    assert len(emb_files) > 0
    assert len(emb_files) == len(id_files)

    shards = []

    global_start = 0

    for idx, (
        emb_path,
        id_path,
    ) in enumerate(
        zip(
            emb_files,
            id_files,
        )
    ):

        # Inspect NPY header only.
        arr = np.load(
            emb_path,
            mmap_mode="r",
        )

        emb_rows = int(
            arr.shape[0]
        )

        emb_dim = int(
            arr.shape[1]
        )

        emb_dtype = arr.dtype

        del arr

        assert emb_dim == EMBED_DIM
        assert emb_dtype == np.float16

        # Inspect ID parquet schema + row count.
        id_schema = (
            pl.read_parquet_schema(
                id_path
            )
        )

        assert "entity_id" in id_schema

        id_rows = (
            pl.scan_parquet(
                id_path
            )
            .select(
                pl.len()
            )
            .collect()
            .item()
        )

        # CRITICAL:
        # Paired ID shard and embedding shard MUST align 1:1.
        assert (
            id_rows
            == emb_rows
        ), (
            f"{source.upper()} shard {idx}: "
            f"ID rows={id_rows:,}, "
            f"embedding rows={emb_rows:,}"
        )

        shards.append({

            "shard":
                idx,

            "start":
                global_start,

            "end":
                global_start + emb_rows,

            "rows":
                emb_rows,

            "embedding":
                emb_path,

            "ids":
                id_path,

            "has_row_index":
                "row_index" in id_schema,
        })

        global_start += emb_rows

    assert (
        global_start
        == EXPECTED_ROWS[source]
    )

    return shards


SHARDS = {}

for source, root in EMBED_ROOTS.items():

    SHARDS[source] = discover_shards(
        source,
        root,
    )

    row_index_flags = [
        d["has_row_index"]
        for d in SHARDS[source]
    ]

    print(
        f"{source.upper()}: "
        f"{len(SHARDS[source])} shards, "
        f"{EXPECTED_ROWS[source]:,} rows"
    )

    print(
        f"  row_index column present in "
        f"{sum(row_index_flags)}/"
        f"{len(row_index_flags)} ID shards"
    )

# ==============================================================================
# 4. VERIFY SHARD-LOCAL ALIGNMENT
# ==============================================================================

print("\n" + "-" * 78)
print("4. VERIFY SHARD-LOCAL ALIGNMENT")
print("-" * 78)

for source in [
    "s1",
    "s2",
    "s3",
]:

    for d in SHARDS[source]:

        # Check first and last few IDs exist and row ordering is stable.
        sample = (
            pl.read_parquet(
                d["ids"],
                columns=["entity_id"],
            )
        )

        assert sample.height == d["rows"]

        assert sample.get_column(
            "entity_id"
        ).null_count() == 0

        print(
            f"{source.upper()} "
            f"shard {d['shard']:02d}: "
            f"{d['rows']:,} rows "
            f""
            f"{sample['entity_id'][0]} ..."
            f"{sample['entity_id'][-1]}"
        )

        del sample

        gc.collect()

print(
    "\n✅ Every paired ID/embedding shard has identical row counts."
)

# ==============================================================================
# 5. BUILD AUTHORITATIVE ID → ROW MAP
# ==============================================================================

print("\n" + "-" * 78)
print("5. BUILD AUTHORITATIVE ID → GLOBAL ROW MAP")
print("-" * 78)

MAP_ROOT = (
    INDEX_ROOT
    / "entity_embedding_maps_v3"
)

MAP_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

MAP_PATHS = {}

for source in [
    "s1",
    "s2",
    "s3",
]:

    map_path = (
        MAP_ROOT
        / f"{source}_entity_to_row.parquet"
    )

    MAP_PATHS[source] = map_path

    # --------------------------------------------------------------------------
    # Reuse an existing validated map.
    # --------------------------------------------------------------------------

    if map_path.exists():

        existing_rows = (
            pl.scan_parquet(
                map_path
            )
            .select(
                pl.len()
            )
            .collect()
            .item()
        )

        print(
            f"{source.upper()}: "
            f"existing map = "
            f"{existing_rows:,} rows"
        )

        assert (
            existing_rows
            == EXPECTED_ROWS[source]
        )

        continue

    # --------------------------------------------------------------------------
    # Build source map one shard at a time.
    # --------------------------------------------------------------------------

    print(
        f"{source.upper()}: building map..."
    )

    source_parts = []

    for d in SHARDS[source]:

        ids_df = pl.read_parquet(
            d["ids"],
            columns=[
                "entity_id"
            ],
        )

        n = ids_df.height

        # The embedding row is the row position within the exact paired
        # ID shard plus the shard's global offset.
        rows = np.arange(
            d["start"],
            d["end"],
            dtype=np.int64,
        )

        part = pl.DataFrame({

            "entity_id":
                ids_df.get_column(
                    "entity_id"
                ).cast(
                    pl.Utf8
                ),

            "row_index":
                rows,
        })

        assert part.height == n

        source_parts.append(part)

        del ids_df
        del rows
        del part

        gc.collect()

    # Concatenate ONLY the ID map. This is small (~few dozen MB).
    source_map = pl.concat(
        source_parts,
        rechunk=False,
    )

    assert (
        source_map.height
        == EXPECTED_ROWS[source]
    )

    # Unique IDs.
    unique_ids = (
        source_map
        .select(
            pl.col(
                "entity_id"
            ).n_unique()
        )
        .item()
    )

    assert (
        unique_ids
        == EXPECTED_ROWS[source]
    )

    source_map.write_parquet(
        map_path,
        compression="zstd",
        statistics=True,
    )

    print(
        f"{source.upper()}: saved "
        f"{source_map.height:,} rows"
    )

    del source_parts
    del source_map

    gc.collect()

print(
    "\n✅ All authoritative entity → embedding-row maps exist."
)

# ==============================================================================
# 6. COVERAGE CHECK
# ==============================================================================

print("\n" + "-" * 78)
print("6. VERIFY PAIR COVERAGE")
print("-" * 78)

con = duckdb.connect(
    database=":memory:"
)

# Keep DuckDB modest.
con.execute(
    "SET memory_limit='768MB'"
)

con.execute(
    "SET threads=4"
)

con.execute(
    "SET preserve_insertion_order=false"
)

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW pairs AS
    SELECT *
    FROM read_parquet(
        '{PAIR_PATH}'
    )
    """
)

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW s1_map AS
    SELECT
        CAST(entity_id AS VARCHAR)
            AS entity_id,
        CAST(row_index AS BIGINT)
            AS row_index
    FROM read_parquet(
        '{MAP_PATHS["s1"]}'
    )
    """
)

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW s2_map AS
    SELECT
        CAST(entity_id AS VARCHAR)
            AS entity_id,
        CAST(row_index AS BIGINT)
            AS row_index,
        's2' AS source
    FROM read_parquet(
        '{MAP_PATHS["s2"]}'
    )
    """
)

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW s3_map AS
    SELECT
        CAST(entity_id AS VARCHAR)
            AS entity_id,
        CAST(row_index AS BIGINT)
            AS row_index,
        's3' AS source
    FROM read_parquet(
        '{MAP_PATHS["s3"]}'
    )
    """
)

con.execute(
    """
    CREATE OR REPLACE TEMP VIEW candidate_map AS

    SELECT * FROM s2_map

    UNION ALL

    SELECT * FROM s3_map
    """
)

coverage = con.execute(
    """
    SELECT

        COUNT(*) AS total_pairs,

        COUNT(
            DISTINCT
            CASE
                WHEN s1.row_index IS NOT NULL
                THEN p.s1_entity_id
            END
        ) AS mapped_s1_ids,

        COUNT(
            CASE
                WHEN s1.row_index IS NOT NULL
                THEN 1
            END
        ) AS mapped_s1_pairs,

        COUNT(
            CASE
                WHEN cm.row_index IS NOT NULL
                THEN 1
            END
        ) AS mapped_candidate_pairs

    FROM pairs p

    LEFT JOIN s1_map s1
      ON p.s1_entity_id =
         s1.entity_id

    LEFT JOIN candidate_map cm
      ON p.candidate_entity_id =
         cm.entity_id
    """
).fetchone()

(
    total_pairs,
    mapped_s1_ids,
    mapped_s1_pairs,
    mapped_candidate_pairs,
) = coverage

print(
    f"Total pairs:             {total_pairs:,}"
)

print(
    f"Mapped S1 pairs:         {mapped_s1_pairs:,}"
)

print(
    f"Mapped candidate pairs:  {mapped_candidate_pairs:,}"
)

assert total_pairs == PAIR_ROWS
assert mapped_s1_pairs == PAIR_ROWS
assert mapped_candidate_pairs == PAIR_ROWS

print(
    "✅ 100% pair coverage."
)

# ==============================================================================
# 7. CREATE INDEXED PAIR FILE
# ==============================================================================

print("\n" + "-" * 78)
print("7. CREATE INDEXED PAIR FILE")
print("-" * 78)

if INDEXED_PAIR_PATH.exists():

    indexed_rows = (
        pl.scan_parquet(
            INDEXED_PAIR_PATH
        )
        .select(
            pl.len()
        )
        .collect()
        .item()
    )

    if indexed_rows == PAIR_ROWS:

        print(
            f"✅ Existing indexed pair valid: "
            f"{indexed_rows:,} rows"
        )

    else:

        INDEXED_PAIR_PATH.unlink()

if not INDEXED_PAIR_PATH.exists():

    print(
        "Writing:",
        INDEXED_PAIR_PATH
    )

    con.execute(
        f"""
        COPY (

            SELECT

                p.*,

                CAST(
                    s1.row_index
                    AS BIGINT
                )
                AS s1_row_index,

                CAST(
                    cm.row_index
                    AS BIGINT
                )
                AS candidate_row_index,

                cm.source
                    AS embedding_source

            FROM pairs p

            INNER JOIN s1_map s1
              ON p.s1_entity_id =
                 s1.entity_id

            INNER JOIN candidate_map cm
              ON p.candidate_entity_id =
                 cm.entity_id

        )

        TO '{INDEXED_PAIR_PATH}'

        (
            FORMAT PARQUET,
            COMPRESSION ZSTD,
            ROW_GROUP_SIZE 100000
        )
        """
    )

indexed_rows = (
    pl.scan_parquet(
        INDEXED_PAIR_PATH
    )
    .select(
        pl.len()
    )
    .collect()
    .item()
)

assert indexed_rows == PAIR_ROWS

print(
    f"✅ Indexed pairs: "
    f"{indexed_rows:,}"
)

con.close()
del con

gc.collect()

# ==============================================================================
# 8. MEMORY-MAP EMBEDDINGS
# ==============================================================================

print("\n" + "-" * 78)
print("8. MEMORY-MAP EMBEDDINGS")
print("-" * 78)

MEMMAPS = {}

for source in [
    "s1",
    "s2",
    "s3",
]:

    arrays = []

    for d in SHARDS[source]:

        arr = np.load(
            d["embedding"],
            mmap_mode="r",
        )

        assert arr.dtype == np.float16
        assert arr.shape[1] == EMBED_DIM

        arrays.append(arr)

    MEMMAPS[source] = arrays

    print(
        f"{source.upper()}: "
        f"{len(arrays)} shards mapped"
    )

# ==============================================================================
# 9. GATHER
# ==============================================================================

def gather_embeddings(
    source,
    row_indices,
):

    row_indices = np.asarray(
        row_indices,
        dtype=np.int64,
    )

    assert len(row_indices) > 0

    assert row_indices.min() >= 0

    assert (
        row_indices.max()
        < EXPECTED_ROWS[source]
    )

    # Find the actual shard based on the known 100k shard layout.
    # Last shard can be smaller.
    shard_ids = (
        row_indices
        // 100_000
    )

    local_rows = (
        row_indices
        % 100_000
    )

    output = np.empty(
        (
            len(row_indices),
            EMBED_DIM,
        ),
        dtype=np.float16,
    )

    for shard_id in np.unique(
        shard_ids
    ):

        mask = (
            shard_ids
            == shard_id
        )

        output[mask] = (
            MEMMAPS[source][
                int(shard_id)
            ][
                local_rows[mask]
            ]
        )

    return output

# ==============================================================================
# 10. OUTPUT
# ==============================================================================

print("\n" + "-" * 78)
print("9. SEMANTIC OUTPUT")
print("-" * 78)

OUTPUT_ROOT = (
    FEATURE_ROOT
    / "features"
    / "train"
    / "pair_semantic_v3"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

BATCH_SIZE = 100_000

TOTAL_PARTS = math.ceil(
    PAIR_ROWS
    / BATCH_SIZE
)

COMPLETE_MARKER = (
    OUTPUT_ROOT
    / "COMPLETE"
)

MANIFEST_PATH = (
    OUTPUT_ROOT
    / "manifest.json"
)

print(
    "Output:",
    OUTPUT_ROOT
)

print(
    f"Total parts: {TOTAL_PARTS}"
)

# ==============================================================================
# 11. SEMANTIC SCORING
# ==============================================================================

print("\n" + "-" * 78)
print("10. COMPUTE SEMANTIC COSINE")
print("-" * 78)

assert torch.cuda.is_available()

DEVICE = torch.device(
    "cuda"
)

GPU_NAME = torch.cuda.get_device_name(
    0
)

print(
    "GPU:",
    GPU_NAME
)

indexed_scan = pl.scan_parquet(
    INDEXED_PAIR_PATH
)

completed_parts = 0

for part_idx in range(
    TOTAL_PARTS
):

    offset = (
        part_idx
        * BATCH_SIZE
    )

    n = min(
        BATCH_SIZE,
        PAIR_ROWS - offset,
    )

    out_path = (
        OUTPUT_ROOT
        / f"semantic_part_{part_idx:05d}.parquet"
    )

    # --------------------------------------------------------------------------
    # RESUME
    # --------------------------------------------------------------------------

    if out_path.exists():

        try:

            existing_rows = (
                pl.scan_parquet(
                    out_path
                )
                .select(
                    pl.len()
                )
                .collect()
                .item()
            )

            if existing_rows == n:

                print(
                    f"[SKIP] "
                    f"{part_idx + 1}/{TOTAL_PARTS} "
                    f"{existing_rows:,} rows"
                )

                completed_parts += 1
                continue

        except Exception:
            pass

        out_path.unlink()

    # --------------------------------------------------------------------------
    # PAIR BATCH
    # --------------------------------------------------------------------------

    pair = (
        indexed_scan
        .slice(
            offset,
            n,
        )
        .collect(
            engine="streaming"
        )
    )

    assert pair.height == n

    s1_rows = (
        pair
        .get_column(
            "s1_row_index"
        )
        .to_numpy()
        .astype(
            np.int64,
            copy=False,
        )
    )

    candidate_rows = (
        pair
        .get_column(
            "candidate_row_index"
        )
        .to_numpy()
        .astype(
            np.int64,
            copy=False,
        )
    )

    embedding_source = (
        pair
        .get_column(
            "embedding_source"
        )
        .to_numpy()
    )

    cosine = np.empty(
        n,
        dtype=np.float32,
    )

    # --------------------------------------------------------------------------
    # S2
    # --------------------------------------------------------------------------

    for source in [
        "s2",
        "s3",
    ]:

        mask = (
            embedding_source
            == source
        )

        idx = np.flatnonzero(
            mask
        )

        if len(idx) == 0:
            continue

        s1_np = gather_embeddings(
            "s1",
            s1_rows[idx],
        )

        candidate_np = (
            gather_embeddings(
                source,
                candidate_rows[idx],
            )
        )

        a = torch.from_numpy(
            s1_np
        ).to(
            DEVICE,
            dtype=torch.float32,
            non_blocking=True,
        )

        b = torch.from_numpy(
            candidate_np
        ).to(
            DEVICE,
            dtype=torch.float32,
            non_blocking=True,
        )

        with torch.inference_mode():

            sim = torch.sum(
                a * b,
                dim=1,
            )

            sim_np = (
                sim
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

        cosine[idx] = sim_np

        del a
        del b
        del sim
        del sim_np
        del s1_np
        del candidate_np
        del idx
        del mask

        torch.cuda.empty_cache()

    # --------------------------------------------------------------------------
    # NUMERICAL VALIDATION
    # --------------------------------------------------------------------------

    assert np.isfinite(
        cosine
    ).all()

    assert cosine.min() >= -1.001
    assert cosine.max() <= 1.001

    distance = (
        1.0 - cosine
    ).astype(
        np.float32,
        copy=False,
    )

    # --------------------------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------------------------

    output = pl.DataFrame({

        "s1_entity_id":
            pair.get_column(
                "s1_entity_id"
            ),

        "candidate_entity_id":
            pair.get_column(
                "candidate_entity_id"
            ),

        "semantic_full_cosine":
            cosine,

        "semantic_full_distance":
            distance,
    })

    output.write_parquet(
        out_path,
        compression="zstd",
        statistics=True,
    )

    written_rows = (
        pl.scan_parquet(
            out_path
        )
        .select(
            pl.len()
        )
        .collect()
        .item()
    )

    assert written_rows == n

    completed_parts += 1

    elapsed = (
        time.time()
        - T0
    ) / 60

    print(
        f"[DONE] "
        f"{part_idx + 1}/{TOTAL_PARTS} "
        f"rows="
        f"{offset:,}:"
        f"{offset+n:,} "
        f"cos="
        f"{cosine.min():.4f}/"
        f"{cosine.mean():.4f}/"
        f"{cosine.max():.4f} "
        f"elapsed={elapsed:.1f}m"
    )

    del pair
    del s1_rows
    del candidate_rows
    del embedding_source
    del cosine
    del distance
    del output

    gc.collect()
    torch.cuda.empty_cache()

# ==============================================================================
# 12. FINAL VALIDATION
# ==============================================================================

print("\n" + "-" * 78)
print("11. FINAL VALIDATION")
print("-" * 78)

assert (
    completed_parts
    == TOTAL_PARTS
)

final_rows = (
    pl.scan_parquet(
        str(
            OUTPUT_ROOT
            / "semantic_part_*.parquet"
        )
    )
    .select(
        pl.len()
    )
    .collect()
    .item()
)

print(
    f"Final rows: {final_rows:,}"
)

assert final_rows == PAIR_ROWS

stats = (
    pl.scan_parquet(
        str(
            OUTPUT_ROOT
            / "semantic_part_*.parquet"
        )
    )
    .select([

        pl.min(
            "semantic_full_cosine"
        ).alias(
            "cosine_min"
        ),

        pl.mean(
            "semantic_full_cosine"
        ).alias(
            "cosine_mean"
        ),

        pl.max(
            "semantic_full_cosine"
        ).alias(
            "cosine_max"
        ),

    ])
    .collect()
)

print(stats)

cos_min = float(
    stats["cosine_min"][0]
)

cos_mean = float(
    stats["cosine_mean"][0]
)

cos_max = float(
    stats["cosine_max"][0]
)

assert cos_min >= -1.001
assert cos_max <= 1.001

# ==============================================================================
# 13. MANIFEST
# ==============================================================================

manifest = {

    "status":
        "COMPLETE",

    "version":
        "pair_semantic_v3",

    "pair_rows":
        int(PAIR_ROWS),

    "indexed_pair":
        str(INDEXED_PAIR_PATH),

    "mapping":
        "entity_id_to_row_position_within_paired_embedding_shard",

    "embedding_model":
        "intfloat/multilingual-e5-small",

    "embedding_dimension":
        EMBED_DIM,

    "embedding_dtype":
        "float16",

    "features": [
        "semantic_full_cosine",
        "semantic_full_distance",
    ],

    "output_root":
        str(OUTPUT_ROOT),

    "parts":
        TOTAL_PARTS,

    "batch_size":
        BATCH_SIZE,

    "gpu":
        GPU_NAME,

    "statistics": {

        "cosine_min":
            cos_min,

        "cosine_mean":
            cos_mean,

        "cosine_max":
            cos_max,
    },

    "created_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
    )
)

COMPLETE_MARKER.write_text(
    "AMLC 2026\n"
    "CELL 67C COMPLETE\n"
    "PAIR SEMANTIC FEATURES V3 COMPLETE\n"
    f"ROWS={PAIR_ROWS}\n"
    f"PARTS={TOTAL_PARTS}\n"
    "MAPPING=PAIRED_SHARD_ROW_POSITION\n"
    "FEATURES=semantic_full_cosine,semantic_full_distance\n"
)

# ==============================================================================
# 14. CLEANUP
# ==============================================================================

del indexed_scan
del MEMMAPS
del SHARDS

gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

try:

    vm = psutil.virtual_memory()

    ram_final = (
        f"{vm.used / (1024**3):.2f} / "
        f"{vm.total / (1024**3):.2f} GB "
        f"({vm.percent:.1f}%)"
    )

except Exception:

    ram_final = "unavailable"

elapsed = (
    time.time()
    - T0
) / 60

print("\n" + "=" * 78)
print("✅ CELL 67C COMPLETE")
print("=" * 78)

print(f"""
PAIR ROWS:
  {PAIR_ROWS:,}

INDEXED PAIRS:
  {INDEXED_PAIR_PATH}

OUTPUT:
  {OUTPUT_ROOT}

FEATURES:
  semantic_full_cosine
  semantic_full_distance

MAPPING:
  entity_id
      ↓
  paired ID-shard row position
      ↓
  global embedding row

FINAL ROWS:
  {final_rows:,}

COSINE:
  min  = {cos_min:.6f}
  mean = {cos_mean:.6f}
  max  = {cos_max:.6f}

GPU:
  {GPU_NAME}

RAM:
  {ram_final}

✅ No numeric-ID assumption.
✅ No requirement for a row_index column.
✅ ID/embedding shard row counts verified.
✅ Pair coverage verified before semantic scoring.
✅ Embeddings memory-mapped.
✅ Resume-safe Parquet output.
""")

AMLC 2026 — CELL 67C
PAIR-LEVEL SEMANTIC FEATURES V3
AUTHORITATIVE SHARD-POSITION MAPPING

------------------------------------------------------------------------------
0. HARD CLEANUP
------------------------------------------------------------------------------
RAM: 29.51 / 31.35 GB (8.3%)

------------------------------------------------------------------------------
1. PATHS
------------------------------------------------------------------------------
FEATURE_ROOT:
/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

PAIR:
/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/state/train_pair_sample_v1/baseline_candidate_pairs.parquet

------------------------------------------------------------------------------
2. PAIR SAMPLE
------------------------------------------------------------------------------
Pair rows: 3,429,214

------------------------------------------------------------------------------
3. DISCOVER EMBEDDING / ID SHARDS
------------------------------------------------------

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Indexed pairs: 3,429,214

------------------------------------------------------------------------------
8. MEMORY-MAP EMBEDDINGS
------------------------------------------------------------------------------
S1: 23 shards mapped
S2: 51 shards mapped
S3: 53 shards mapped

------------------------------------------------------------------------------
9. SEMANTIC OUTPUT
------------------------------------------------------------------------------
Output: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/train/pair_semantic_v3
Total parts: 35

------------------------------------------------------------------------------
10. COMPUTE SEMANTIC COSINE
------------------------------------------------------------------------------
GPU: Tesla T4
[DONE] 1/35 rows=0:100,000 cos=0.7441/0.8485/0.9782 elapsed=1.6m
[DONE] 2/35 rows=100,000:200,000 cos=0.7540/0.8484/0.9769 elapsed=1.7m
[DONE] 3/35 rows=200,000:300,000 cos=0.7542/0.8489/0.9791 elapsed=1.7m
[DONE] 4/35 rows=300,000:400,000 cos=

In [30]:
# ==============================================================================
# AMLC 2026 — CELL 68A
# SEMANTIC SIGNAL DIAGNOSTIC
#
# PURPOSE:
#   Measure whether semantic_full_cosine separates:
#       is_positive = 1
#       is_positive = 0
#
# OUTPUT:
#   - positive / negative distribution statistics
#   - global ROC-AUC
#   - global PR-AUC
#   - Pearson point-biserial correlation
#   - threshold precision / recall / F0.5
#   - S2 / S3 diagnostics
#   - quantile separation
#   - durable JSON + Parquet diagnostics
# ==============================================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    fbeta_score,
)

T0 = time.time()

print("=" * 78)
print("AMLC 2026 — CELL 68A")
print("SEMANTIC SIGNAL DIAGNOSTIC")
print("=" * 78)

# ==============================================================================
# 0. CLEANUP
# ==============================================================================

print("\n" + "-" * 78)
print("0. CLEANUP")
print("-" * 78)

for name in [
    "joined",
    "joined_df",
    "semantic_df",
    "pair_df",
    "scores",
    "labels",
]:
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass

gc.collect()

try:
    import torch
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
except Exception:
    pass

# ==============================================================================
# 1. PATHS
# ==============================================================================

print("\n" + "-" * 78)
print("1. PATHS")
print("-" * 78)

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

PAIR_PATH = (
    FEATURE_ROOT
    / "state"
    / "train_pair_sample_v1"
    / "baseline_candidate_pairs.parquet"
)

SEMANTIC_ROOT = (
    FEATURE_ROOT
    / "features"
    / "train"
    / "pair_semantic_v3"
)

OUTPUT_ROOT = (
    SEMANTIC_ROOT
    / "diagnostics_v1"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

assert PAIR_PATH.exists()
assert SEMANTIC_ROOT.exists()

SEMANTIC_GLOB = (
    str(
        SEMANTIC_ROOT
        / "semantic_part_*.parquet"
    )
)

print(
    "Pair:",
    PAIR_PATH
)

print(
    "Semantic:",
    SEMANTIC_ROOT
)

# ==============================================================================
# 2. VERIFY INPUT SIZES
# ==============================================================================

print("\n" + "-" * 78)
print("2. VERIFY INPUTS")
print("-" * 78)

PAIR_ROWS = (
    pl.scan_parquet(
        PAIR_PATH
    )
    .select(
        pl.len()
    )
    .collect()
    .item()
)

SEMANTIC_ROWS = (
    pl.scan_parquet(
        SEMANTIC_GLOB
    )
    .select(
        pl.len()
    )
    .collect()
    .item()
)

print(
    f"Pair rows:     {PAIR_ROWS:,}"
)

print(
    f"Semantic rows: {SEMANTIC_ROWS:,}"
)

assert PAIR_ROWS == 3_429_214
assert SEMANTIC_ROWS == PAIR_ROWS

# ==============================================================================
# 3. JOIN SEMANTIC FEATURES TO LABELS
# ==============================================================================

print("\n" + "-" * 78)
print("3. JOIN SEMANTIC FEATURES + LABELS")
print("-" * 78)

semantic_scan = (
    pl.scan_parquet(
        SEMANTIC_GLOB
    )
    .select([
        "s1_entity_id",
        "candidate_entity_id",
        "semantic_full_cosine",
        "semantic_full_distance",
    ])
)

label_scan = (
    pl.scan_parquet(
        PAIR_PATH
    )
    .select([
        "s1_entity_id",
        "candidate_entity_id",
        "source",
        "is_positive",
    ])
)

joined = (
    semantic_scan
    .join(
        label_scan,
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="inner",
    )
)

joined_count = (
    joined
    .select(
        pl.len()
    )
    .collect()
    .item()
)

print(
    f"Joined rows: {joined_count:,}"
)

assert joined_count == PAIR_ROWS

# ==============================================================================
# 4. LABEL COUNTS
# ==============================================================================

print("\n" + "-" * 78)
print("4. LABEL COUNTS")
print("-" * 78)

label_counts = (
    joined
    .group_by(
        "is_positive"
    )
    .agg(
        pl.len().alias("rows")
    )
    .sort(
        "is_positive"
    )
    .collect()
)

print(
    label_counts
)

positive_count = int(
    label_counts
    .filter(
        pl.col("is_positive") == 1
    )
    .get_column(
        "rows"
    )[0]
)

negative_count = int(
    label_counts
    .filter(
        pl.col("is_positive") == 0
    )
    .get_column(
        "rows"
    )[0]
)

assert (
    positive_count
    + negative_count
    == PAIR_ROWS
)

# ==============================================================================
# 5. GLOBAL DISTRIBUTION
# ==============================================================================

print("\n" + "-" * 78)
print("5. GLOBAL SEMANTIC DISTRIBUTION")
print("-" * 78)

global_distribution = (
    joined
    .group_by(
        "is_positive"
    )
    .agg([

        pl.len().alias(
            "rows"
        ),

        pl.mean(
            "semantic_full_cosine"
        ).alias(
            "mean_cosine"
        ),

        pl.median(
            "semantic_full_cosine"
        ).alias(
            "median_cosine"
        ),

        pl.std(
            "semantic_full_cosine"
        ).alias(
            "std_cosine"
        ),

        pl.min(
            "semantic_full_cosine"
        ).alias(
            "min_cosine"
        ),

        pl.max(
            "semantic_full_cosine"
        ).alias(
            "max_cosine"
        ),

        pl.quantile(
            "semantic_full_cosine",
            0.01,
        ).alias("q01"),

        pl.quantile(
            "semantic_full_cosine",
            0.05,
        ).alias("q05"),

        pl.quantile(
            "semantic_full_cosine",
            0.10,
        ).alias("q10"),

        pl.quantile(
            "semantic_full_cosine",
            0.25,
        ).alias("q25"),

        pl.quantile(
            "semantic_full_cosine",
            0.50,
        ).alias("q50"),

        pl.quantile(
            "semantic_full_cosine",
            0.75,
        ).alias("q75"),

        pl.quantile(
            "semantic_full_cosine",
            0.90,
        ).alias("q90"),

        pl.quantile(
            "semantic_full_cosine",
            0.95,
        ).alias("q95"),

        pl.quantile(
            "semantic_full_cosine",
            0.99,
        ).alias("q99"),

    ])
    .sort(
        "is_positive"
    )
    .collect()
)

print(
    global_distribution
)

# ==============================================================================
# 6. MATERIALIZE ONLY TWO NUMERIC ARRAYS
# ==============================================================================

print("\n" + "-" * 78)
print("6. MATERIALIZE SCORE + LABEL ARRAYS")
print("-" * 78)

arrays_df = (
    joined
    .select([
        "semantic_full_cosine",
        "is_positive",
    ])
    .collect()
)

scores = (
    arrays_df
    .get_column(
        "semantic_full_cosine"
    )
    .to_numpy()
    .astype(
        np.float64,
        copy=False,
    )
)

labels = (
    arrays_df
    .get_column(
        "is_positive"
    )
    .to_numpy()
    .astype(
        np.int8,
        copy=False,
    )
)

del arrays_df

gc.collect()

print(
    f"Scores: {scores.shape}"
)

print(
    f"Labels: {labels.shape}"
)

assert len(scores) == PAIR_ROWS
assert len(labels) == PAIR_ROWS

# ==============================================================================
# 7. GLOBAL ROC / PR
# ==============================================================================

print("\n" + "-" * 78)
print("7. GLOBAL SEMANTIC METRICS")
print("-" * 78)

roc_auc = roc_auc_score(
    labels,
    scores,
)

pr_auc = average_precision_score(
    labels,
    scores,
)

pos_scores = scores[
    labels == 1
]

neg_scores = scores[
    labels == 0
]

pos_mean = float(
    pos_scores.mean()
)

neg_mean = float(
    neg_scores.mean()
)

mean_gap = (
    pos_mean
    - neg_mean
)

# Point-biserial equivalent to Pearson(binary,label).
point_biserial = float(
    np.corrcoef(
        labels.astype(
            np.float64
        ),
        scores,
    )[0, 1]
)

print(
    f"Positive mean cosine: {pos_mean:.6f}"
)

print(
    f"Negative mean cosine: {neg_mean:.6f}"
)

print(
    f"Mean gap:             {mean_gap:.6f}"
)

print(
    f"ROC-AUC:              {roc_auc:.6f}"
)

print(
    f"PR-AUC:               {pr_auc:.6f}"
)

print(
    f"Point-biserial r:     {point_biserial:.6f}"
)

# ==============================================================================
# 8. THRESHOLD SWEEP
# ==============================================================================

print("\n" + "-" * 78)
print("8. THRESHOLD SWEEP")
print("-" * 78)

thresholds = [
    0.80,
    0.82,
    0.84,
    0.85,
    0.86,
    0.87,
    0.88,
    0.89,
    0.90,
    0.91,
    0.92,
    0.93,
    0.94,
    0.95,
    0.96,
    0.97,
]

threshold_rows = []

for threshold in thresholds:

    pred = (
        scores
        >= threshold
    )

    precision = precision_score(
        labels,
        pred,
        zero_division=0,
    )

    recall = recall_score(
        labels,
        pred,
        zero_division=0,
    )

    f05 = fbeta_score(
        labels,
        pred,
        beta=0.5,
        zero_division=0,
    )

    predicted_positive = int(
        pred.sum()
    )

    threshold_rows.append({

        "threshold":
            threshold,

        "predicted_positive":
            predicted_positive,

        "precision":
            precision,

        "recall":
            recall,

        "f0_5":
            f05,
    })

threshold_df = pl.DataFrame(
    threshold_rows
)

print(
    threshold_df
)

threshold_df.write_parquet(
    OUTPUT_ROOT
    / "semantic_threshold_sweep.parquet",
    compression="zstd",
)

# ==============================================================================
# 9. SOURCE-SPECIFIC SIGNAL
# ==============================================================================

print("\n" + "-" * 78)
print("9. SOURCE-SPECIFIC SIGNAL")
print("-" * 78)

source_signal = (
    joined
    .group_by([
        "source",
        "is_positive",
    ])
    .agg([

        pl.len().alias(
            "rows"
        ),

        pl.mean(
            "semantic_full_cosine"
        ).alias(
            "mean_cosine"
        ),

        pl.median(
            "semantic_full_cosine"
        ).alias(
            "median_cosine"
        ),

        pl.quantile(
            "semantic_full_cosine",
            0.10,
        ).alias(
            "q10"
        ),

        pl.quantile(
            "semantic_full_cosine",
            0.50,
        ).alias(
            "q50"
        ),

        pl.quantile(
            "semantic_full_cosine",
            0.90,
        ).alias(
            "q90"
        ),

    ])
    .sort([
        "source",
        "is_positive",
    ])
    .collect()
)

print(
    source_signal
)

source_signal.write_parquet(
    OUTPUT_ROOT
    / "semantic_source_signal.parquet",
    compression="zstd",
)

# ==============================================================================
# 10. POSITIVE VS NEGATIVE QUANTILE GAP
# ==============================================================================

print("\n" + "-" * 78)
print("10. POSITIVE / NEGATIVE QUANTILE GAP")
print("-" * 78)

quantile_grid = [
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
]

quantile_rows = []

for q in quantile_grid:

    p = float(
        np.quantile(
            pos_scores,
            q,
        )
    )

    n = float(
        np.quantile(
            neg_scores,
            q,
        )
    )

    quantile_rows.append({

        "quantile":
            q,

        "positive_cosine":
            p,

        "negative_cosine":
            n,

        "gap":
            p - n,
    })

quantile_df = pl.DataFrame(
    quantile_rows
)

print(
    quantile_df
)

quantile_df.write_parquet(
    OUTPUT_ROOT
    / "semantic_quantile_gap.parquet",
    compression="zstd",
)

# ==============================================================================
# 11. SAVE SUMMARY
# ==============================================================================

summary = {

    "status":
        "COMPLETE",

    "pair_rows":
        int(PAIR_ROWS),

    "positive_rows":
        int(positive_count),

    "negative_rows":
        int(negative_count),

    "semantic_feature":
        "semantic_full_cosine",

    "positive_mean":
        pos_mean,

    "negative_mean":
        neg_mean,

    "mean_gap":
        mean_gap,

    "roc_auc":
        float(roc_auc),

    "pr_auc":
        float(pr_auc),

    "point_biserial":
        point_biserial,

    "global_cosine_min":
        float(scores.min()),

    "global_cosine_max":
        float(scores.max()),

    "threshold_sweep":
        threshold_rows,

    "source_signal_path":
        str(
            OUTPUT_ROOT
            / "semantic_source_signal.parquet"
        ),

    "threshold_path":
        str(
            OUTPUT_ROOT
            / "semantic_threshold_sweep.parquet"
        ),

    "quantile_gap_path":
        str(
            OUTPUT_ROOT
            / "semantic_quantile_gap.parquet"
        ),

    "created_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

SUMMARY_PATH = (
    OUTPUT_ROOT
    / "summary.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        indent=2,
    )
)

# ==============================================================================
# 12. COMPLETE MARKER
# ==============================================================================

(
    OUTPUT_ROOT
    / "COMPLETE"
).write_text(
    "AMLC 2026\n"
    "CELL 68A COMPLETE\n"
    "SEMANTIC SIGNAL DIAGNOSTIC COMPLETE\n"
    f"ROWS={PAIR_ROWS}\n"
)

# ==============================================================================
# 13. CLEANUP
# ==============================================================================

del joined
del semantic_scan
del label_scan
del scores
del labels
del pos_scores
del neg_scores

gc.collect()

# ==============================================================================
# 14. FINAL
# ==============================================================================

elapsed = (
    time.time()
    - T0
) / 60

print("\n" + "=" * 78)
print("✅ CELL 68A COMPLETE")
print("=" * 78)

print(f"""
PAIR ROWS:
  {PAIR_ROWS:,}

POSITIVE:
  {positive_count:,}

NEGATIVE:
  {negative_count:,}

POSITIVE MEAN COSINE:
  {pos_mean:.6f}

NEGATIVE MEAN COSINE:
  {neg_mean:.6f}

MEAN GAP:
  {mean_gap:.6f}

ROC-AUC:
  {roc_auc:.6f}

PR-AUC:
  {pr_auc:.6f}

POINT-BISERIAL:
  {point_biserial:.6f}

OUTPUT:
  {OUTPUT_ROOT}

SUMMARY:
  {SUMMARY_PATH}

NEXT:
  Use the label separation results to determine whether
  semantic rank / margin features should be built from the
  FULL candidate universe rather than only the sampled pairs.
""")

print(
    f"Runtime: {elapsed:.2f} min"
)

AMLC 2026 — CELL 68A
SEMANTIC SIGNAL DIAGNOSTIC

------------------------------------------------------------------------------
0. CLEANUP
------------------------------------------------------------------------------

------------------------------------------------------------------------------
1. PATHS
------------------------------------------------------------------------------
Pair: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/state/train_pair_sample_v1/baseline_candidate_pairs.parquet
Semantic: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/train/pair_semantic_v3

------------------------------------------------------------------------------
2. VERIFY INPUTS
------------------------------------------------------------------------------
Pair rows:     3,429,214
Semantic rows: 3,429,214

------------------------------------------------------------------------------
3. JOIN SEMANTIC FEATURES + LABELS
-----------------------------------------------------------------------

In [32]:
# ==============================================================================
# AMLC 2026 — CELL 68B v2
# BUILD FULL-CANDIDATE SEMANTIC RANK / MARGIN FEATURES
#
# IMPORTANT:
#   The expensive 22,302,012-row cosine computation from Cell 68B is ALREADY
#   COMPLETE and must NOT be repeated.
#
# INPUT:
#   features/train/full_candidate_semantic_v1/cosine_parts/*.parquet
#
# OUTPUT:
#   features/train/full_candidate_semantic_v1/semantic_ranked_full.parquet
#
# METHOD:
#   DuckDB window functions over persisted cosine parts.
#
# FEATURES:
#   semantic_full_cosine
#   semantic_full_distance
#
#   semantic_rank_total
#   semantic_percentile_total
#   semantic_top1_cosine_total
#   semantic_top2_cosine_total
#   semantic_top3_cosine_total
#   semantic_margin_top1_top2
#   semantic_margin_top1_top3
#
#   semantic_rank_source
#   semantic_percentile_source
#   semantic_top1_cosine_source
#   semantic_top2_cosine_source
#   semantic_margin_source_top1_top2
# ==============================================================================

import gc
import json
import time
from pathlib import Path

import duckdb
import polars as pl

T0 = time.time()

print("=" * 78)
print("AMLC 2026 — CELL 68B v2")
print("FULL CANDIDATE SEMANTIC RANK / MARGIN")
print("=" * 78)

# ==============================================================================
# 0. CLEANUP
# ==============================================================================

print("\n" + "-" * 78)
print("0. CLEANUP")
print("-" * 78)

for name in [
    "con",
    "ranked",
    "features",
    "final_df",
]:
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass

gc.collect()

try:
    import torch
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
except Exception:
    pass

# ==============================================================================
# 1. PATHS
# ==============================================================================

print("\n" + "-" * 78)
print("1. PATHS")
print("-" * 78)

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

OUTPUT_ROOT = (
    FEATURE_ROOT
    / "features"
    / "train"
    / "full_candidate_semantic_v1"
)

COSINE_ROOT = (
    OUTPUT_ROOT
    / "cosine_parts"
)

COSINE_GLOB = (
    str(
        COSINE_ROOT
        / "candidate_semantic_*.parquet"
    )
)

FINAL_PATH = (
    OUTPUT_ROOT
    / "semantic_ranked_full.parquet"
)

MANIFEST_PATH = (
    OUTPUT_ROOT
    / "manifest_v2.json"
)

COMPLETE_MARKER = (
    OUTPUT_ROOT
    / "COMPLETE"
)

assert FEATURE_ROOT.exists()
assert COSINE_ROOT.exists()

print(
    "Cosine parts:",
    COSINE_ROOT
)

print(
    "Final:",
    FINAL_PATH
)

# ==============================================================================
# 2. VERIFY COMPLETED COSINE UNIVERSE
# ==============================================================================

print("\n" + "-" * 78)
print("2. VERIFY COMPLETED COSINE UNIVERSE")
print("-" * 78)

con = duckdb.connect(
    database=":memory:"
)

con.execute(
    "SET memory_limit='2GB'"
)

con.execute(
    "SET threads=8"
)

con.execute(
    "SET preserve_insertion_order=false"
)

cosine_count = con.execute(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{COSINE_GLOB}'
    )
    """
).fetchone()[0]

print(
    f"Cosine rows: {cosine_count:,}"
)

assert cosine_count == 22_302_012

distinct_pairs = con.execute(
    f"""
    SELECT COUNT(*)
    FROM (
        SELECT DISTINCT
            s1_entity_id,
            candidate_entity_id
        FROM read_parquet(
            '{COSINE_GLOB}'
        )
    )
    """
).fetchone()[0]

print(
    f"Distinct pairs: {distinct_pairs:,}"
)

assert distinct_pairs == 22_302_012

print(
    "✅ Existing 22,302,012-row cosine universe is intact."
)

# ==============================================================================
# 3. INSPECT SOURCE COUNTS
# ==============================================================================

print("\n" + "-" * 78)
print("3. SOURCE COUNTS")
print("-" * 78)

source_counts = con.execute(
    f"""
    SELECT
        source,
        COUNT(*) AS rows,
        COUNT(DISTINCT s1_entity_id) AS s1s
    FROM read_parquet(
        '{COSINE_GLOB}'
    )
    GROUP BY source
    ORDER BY source
    """
).fetchdf()

print(
    source_counts
)

# ==============================================================================
# 4. BUILD RANK / MARGIN TABLE
# ==============================================================================

print("\n" + "-" * 78)
print("4. BUILD RANK / MARGIN FEATURES")
print("-" * 78)

if FINAL_PATH.exists():

    print(
        "Removing stale partial final output..."
    )

    FINAL_PATH.unlink()

print(
    "Running DuckDB window pipeline..."
)

sql = f"""
COPY (

    WITH base AS (

        SELECT

            CAST(
                s1_entity_id AS VARCHAR
            )
            AS s1_entity_id,

            CAST(
                candidate_entity_id AS VARCHAR
            )
            AS candidate_entity_id,

            CAST(
                source AS VARCHAR
            )
            AS source,

            CAST(
                semantic_full_cosine AS FLOAT
            )
            AS semantic_full_cosine,

            CAST(
                semantic_full_distance AS FLOAT
            )
            AS semantic_full_distance

        FROM read_parquet(
            '{COSINE_GLOB}'
        )
    ),

    ranked AS (

        SELECT

            *,

            ROW_NUMBER() OVER (

                PARTITION BY
                    s1_entity_id

                ORDER BY
                    semantic_full_cosine DESC,
                    candidate_entity_id ASC

            )
            AS semantic_rank_total,

            COUNT(*) OVER (

                PARTITION BY
                    s1_entity_id

            )
            AS candidate_count_total,

            ROW_NUMBER() OVER (

                PARTITION BY
                    s1_entity_id,
                    source

                ORDER BY
                    semantic_full_cosine DESC,
                    candidate_entity_id ASC

            )
            AS semantic_rank_source,

            COUNT(*) OVER (

                PARTITION BY
                    s1_entity_id,
                    source

            )
            AS candidate_count_source

        FROM base
    ),

    with_competition AS (

        SELECT

            *,

            CASE

                WHEN candidate_count_total <= 1
                THEN 1.0

                ELSE
                    1.0
                    -
                    CAST(
                        semantic_rank_total - 1
                        AS DOUBLE
                    )
                    /
                    CAST(
                        candidate_count_total - 1
                        AS DOUBLE
                    )

            END
            AS semantic_percentile_total,

            CASE

                WHEN candidate_count_source <= 1
                THEN 1.0

                ELSE
                    1.0
                    -
                    CAST(
                        semantic_rank_source - 1
                        AS DOUBLE
                    )
                    /
                    CAST(
                        candidate_count_source - 1
                        AS DOUBLE
                    )

            END
            AS semantic_percentile_source,

            MAX(
                CASE
                    WHEN semantic_rank_total = 1
                    THEN semantic_full_cosine
                END
            ) OVER (
                PARTITION BY
                    s1_entity_id
            )
            AS semantic_top1_cosine_total,

            MAX(
                CASE
                    WHEN semantic_rank_total = 2
                    THEN semantic_full_cosine
                END
            ) OVER (
                PARTITION BY
                    s1_entity_id
            )
            AS semantic_top2_cosine_total,

            MAX(
                CASE
                    WHEN semantic_rank_total = 3
                    THEN semantic_full_cosine
                END
            ) OVER (
                PARTITION BY
                    s1_entity_id
            )
            AS semantic_top3_cosine_total,

            MAX(
                CASE
                    WHEN semantic_rank_source = 1
                    THEN semantic_full_cosine
                END
            ) OVER (
                PARTITION BY
                    s1_entity_id,
                    source
            )
            AS semantic_top1_cosine_source,

            MAX(
                CASE
                    WHEN semantic_rank_source = 2
                    THEN semantic_full_cosine
                END
            ) OVER (
                PARTITION BY
                    s1_entity_id,
                    source
            )
            AS semantic_top2_cosine_source

        FROM ranked
    )

    SELECT

        s1_entity_id,

        candidate_entity_id,

        source,

        semantic_full_cosine,

        semantic_full_distance,

        CAST(
            semantic_rank_total
            AS INTEGER
        )
        AS semantic_rank_total,

        CAST(
            semantic_percentile_total
            AS FLOAT
        )
        AS semantic_percentile_total,

        CAST(
            semantic_top1_cosine_total
            AS FLOAT
        )
        AS semantic_top1_cosine_total,

        CAST(
            semantic_top2_cosine_total
            AS FLOAT
        )
        AS semantic_top2_cosine_total,

        CAST(
            semantic_top3_cosine_total
            AS FLOAT
        )
        AS semantic_top3_cosine_total,

        CAST(
            COALESCE(
                semantic_top1_cosine_total,
                0.0
            )
            -
            COALESCE(
                semantic_top2_cosine_total,
                0.0
            )
            AS FLOAT
        )
        AS semantic_margin_top1_top2,

        CAST(
            COALESCE(
                semantic_top1_cosine_total,
                0.0
            )
            -
            COALESCE(
                semantic_top3_cosine_total,
                0.0
            )
            AS FLOAT
        )
        AS semantic_margin_top1_top3,

        CAST(
            semantic_rank_source
            AS INTEGER
        )
        AS semantic_rank_source,

        CAST(
            semantic_percentile_source
            AS FLOAT
        )
        AS semantic_percentile_source,

        CAST(
            semantic_top1_cosine_source
            AS FLOAT
        )
        AS semantic_top1_cosine_source,

        CAST(
            semantic_top2_cosine_source
            AS FLOAT
        )
        AS semantic_top2_cosine_source,

        CAST(
            COALESCE(
                semantic_top1_cosine_source,
                0.0
            )
            -
            COALESCE(
                semantic_top2_cosine_source,
                0.0
            )
            AS FLOAT
        )
        AS semantic_margin_source_top1_top2

    FROM with_competition

)
TO '{FINAL_PATH}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD,
    ROW_GROUP_SIZE 100000
)
"""

con.execute(sql)

print(
    "✅ Rank/margin Parquet written."
)

# ==============================================================================
# 5. FINAL VALIDATION
# ==============================================================================

print("\n" + "-" * 78)
print("5. FINAL VALIDATION")
print("-" * 78)

final_rows = con.execute(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{FINAL_PATH}'
    )
    """
).fetchone()[0]

print(
    f"Final rows: {final_rows:,}"
)

assert final_rows == 22_302_012

duplicate_pairs = con.execute(
    f"""
    SELECT COUNT(*)
    FROM (
        SELECT
            s1_entity_id,
            candidate_entity_id,
            COUNT(*) AS n
        FROM read_parquet(
            '{FINAL_PATH}'
        )
        GROUP BY
            s1_entity_id,
            candidate_entity_id
        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]

print(
    f"Duplicate pair groups: {duplicate_pairs:,}"
)

assert duplicate_pairs == 0

# ==============================================================================
# 6. FEATURE SCHEMA
# ==============================================================================

print("\n" + "-" * 78)
print("6. FEATURE SCHEMA")
print("-" * 78)

schema = pl.read_parquet_schema(
    FINAL_PATH
)

EXPECTED_FEATURES = [

    "s1_entity_id",
    "candidate_entity_id",
    "source",

    "semantic_full_cosine",
    "semantic_full_distance",

    "semantic_rank_total",
    "semantic_percentile_total",

    "semantic_top1_cosine_total",
    "semantic_top2_cosine_total",
    "semantic_top3_cosine_total",

    "semantic_margin_top1_top2",
    "semantic_margin_top1_top3",

    "semantic_rank_source",
    "semantic_percentile_source",

    "semantic_top1_cosine_source",
    "semantic_top2_cosine_source",

    "semantic_margin_source_top1_top2",
]

for feature in EXPECTED_FEATURES:

    assert feature in schema, (
        f"Missing feature: {feature}"
    )

print(
    "✅ All 17 expected columns present."
)

# ==============================================================================
# 7. SANITY CHECK COMPETITION FEATURES
# ==============================================================================

print("\n" + "-" * 78)
print("7. SANITY CHECK COMPETITION FEATURES")
print("-" * 78)

sanity = con.execute(
    f"""
    SELECT

        MIN(
            semantic_rank_total
        ) AS min_total_rank,

        MAX(
            semantic_rank_total
        ) AS max_total_rank,

        MIN(
            semantic_percentile_total
        ) AS min_total_percentile,

        MAX(
            semantic_percentile_total
        ) AS max_total_percentile,

        MIN(
            semantic_rank_source
        ) AS min_source_rank,

        MIN(
            semantic_percentile_source
        ) AS min_source_percentile,

        MAX(
            semantic_percentile_source
        ) AS max_source_percentile,

        MIN(
            semantic_margin_top1_top2
        ) AS min_margin12,

        MAX(
            semantic_margin_top1_top2
        ) AS max_margin12,

        MIN(
            semantic_margin_top1_top3
        ) AS min_margin13,

        MAX(
            semantic_margin_top1_top3
        ) AS max_margin13

    FROM read_parquet(
        '{FINAL_PATH}'
    )
    """
).fetchdf()

print(
    sanity
)

row = sanity.iloc[0]

assert row["min_total_rank"] == 1
assert row["min_source_rank"] == 1

assert (
    row["min_total_percentile"]
    >= -1e-6
)

assert (
    row["max_total_percentile"]
    <= 1.000001
)

assert (
    row["min_source_percentile"]
    >= -1e-6
)

assert (
    row["max_source_percentile"]
    <= 1.000001
)

# ==============================================================================
# 8. TOP-1 SANITY BY S1
# ==============================================================================

print("\n" + "-" * 78)
print("8. TOP-1 SANITY")
print("-" * 78)

top1_check = con.execute(
    f"""
    SELECT

        COUNT(*) AS s1_count,

        SUM(
            CASE
                WHEN top_count = 1
                THEN 1
                ELSE 0
            END
        ) AS exactly_one_top1

    FROM (

        SELECT

            s1_entity_id,

            SUM(
                CASE
                    WHEN semantic_rank_total = 1
                    THEN 1
                    ELSE 0
                END
            ) AS top_count

        FROM read_parquet(
            '{FINAL_PATH}'
        )

        GROUP BY
            s1_entity_id

    )
    """
).fetchone()

s1_count, exactly_one_top1 = top1_check

print(
    f"S1 groups:              {s1_count:,}"
)

print(
    f"Exactly one total TOP-1: {exactly_one_top1:,}"
)

assert (
    s1_count
    == exactly_one_top1
)

# ==============================================================================
# 9. SAVE MANIFEST
# ==============================================================================

print("\n" + "-" * 78)
print("9. SAVE MANIFEST")
print("-" * 78)

manifest = {

    "status":
        "COMPLETE",

    "version":
        "full_candidate_semantic_v2",

    "source_cosine_rows":
        22_302_012,

    "final_rows":
        int(final_rows),

    "method":
        "duckdb_window_functions",

    "tie_break":
        "semantic_full_cosine_desc_then_candidate_entity_id_asc",

    "features":
        EXPECTED_FEATURES,

    "cosine_input":
        COSINE_GLOB,

    "final_output":
        str(FINAL_PATH),

    "created_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
    )
)

COMPLETE_MARKER.write_text(
    "AMLC 2026\n"
    "CELL 68B v2 COMPLETE\n"
    "FULL CANDIDATE SEMANTIC RANK/MARGIN COMPLETE\n"
    "COSINE INPUT ALREADY COMPUTED\n"
    f"ROWS={final_rows}\n"
)

print(
    "Manifest:",
    MANIFEST_PATH
)

# ==============================================================================
# 10. CLOSE / CLEANUP
# ==============================================================================

con.close()

del con

gc.collect()

try:

    import torch
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

except Exception:
    pass

elapsed = (
    time.time()
    - T0
) / 60

print("\n" + "=" * 78)
print("✅ CELL 68B v2 COMPLETE")
print("=" * 78)

print(f"""
REUSED COSINE ROWS:
  22,302,012

FINAL RANKED ROWS:
  {final_rows:,}

FINAL OUTPUT:
  {FINAL_PATH}

FEATURES:
  semantic_full_cosine
  semantic_full_distance
  semantic_rank_total
  semantic_percentile_total
  semantic_top1_cosine_total
  semantic_top2_cosine_total
  semantic_top3_cosine_total
  semantic_margin_top1_top2
  semantic_margin_top1_top3
  semantic_rank_source
  semantic_percentile_source
  semantic_top1_cosine_source
  semantic_top2_cosine_source
  semantic_margin_source_top1_top2

✅ No cosine recomputation.
✅ Total-candidate ranks use ALL 22.3M candidates.
✅ Source-specific ranks use ALL candidates within source.
✅ Candidate ID is deterministic tie-breaker.
✅ Exactly one rank-1 candidate per S1.
✅ Output is durable.
""")

print(
    f"Runtime: {elapsed:.2f} min"
)

AMLC 2026 — CELL 68B v2
FULL CANDIDATE SEMANTIC RANK / MARGIN

------------------------------------------------------------------------------
0. CLEANUP
------------------------------------------------------------------------------

------------------------------------------------------------------------------
1. PATHS
------------------------------------------------------------------------------
Cosine parts: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/train/full_candidate_semantic_v1/cosine_parts
Final: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/train/full_candidate_semantic_v1/semantic_ranked_full.parquet

------------------------------------------------------------------------------
2. VERIFY COMPLETED COSINE UNIVERSE
------------------------------------------------------------------------------
Cosine rows: 22,302,012


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Distinct pairs: 22,302,012
✅ Existing 22,302,012-row cosine universe is intact.

------------------------------------------------------------------------------
3. SOURCE COUNTS
------------------------------------------------------------------------------
  source      rows    s1s
0     s2  11091535  93054
1     s3  11210477  92567

------------------------------------------------------------------------------
4. BUILD RANK / MARGIN FEATURES
------------------------------------------------------------------------------
Running DuckDB window pipeline...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Rank/margin Parquet written.

------------------------------------------------------------------------------
5. FINAL VALIDATION
------------------------------------------------------------------------------
Final rows: 22,302,012


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate pair groups: 0

------------------------------------------------------------------------------
6. FEATURE SCHEMA
------------------------------------------------------------------------------
✅ All 17 expected columns present.

------------------------------------------------------------------------------
7. SANITY CHECK COMPETITION FEATURES
------------------------------------------------------------------------------
   min_total_rank  max_total_rank  min_total_percentile  max_total_percentile  \
0               1            3929                   0.0                   1.0   

   min_source_rank  min_source_percentile  max_source_percentile  \
0                1                    0.0                    1.0   

   min_margin12  max_margin12  min_margin13  max_margin13  
0           0.0      0.978854           0.0       0.97979  

------------------------------------------------------------------------------
8. TOP-1 SANITY
---------------------------------------------------

In [33]:
# ==============================================================================
# AMLC 2026 — CELL 69A
# TRUE-MATCH SEMANTIC RANK DIAGNOSTIC
#
# PURPOSE:
#   Measure where known TRUE matches land when semantic cosine is ranked
#   against the COMPLETE 22,302,012-row candidate universe.
#
# INPUT:
#   1. Recovered labeled pair sample:
#        state/train_pair_sample_v1/baseline_candidate_pairs.parquet
#
#   2. Full semantic ranking:
#        features/train/full_candidate_semantic_v1/
#        semantic_ranked_full.parquet
#
# OUTPUT:
#   - positive-rank distribution
#   - positive recall@K
#   - S1-level true-match top-K coverage
#   - margin distribution
#   - durable Parquet/JSON diagnostics
#
# IMPORTANT:
#   We DO NOT interpret sampled-pair precision as final competition precision.
#   Negative sampling changes class prevalence.
#
#   Rank of a true edge within the COMPLETE candidate universe is valid.
# ==============================================================================

import gc
import json
import time
from pathlib import Path

import duckdb
import numpy as np
import polars as pl

T0 = time.time()

print("=" * 78)
print("AMLC 2026 — CELL 69A")
print("TRUE-MATCH SEMANTIC RANK DIAGNOSTIC")
print("=" * 78)

# ==============================================================================
# 0. CLEANUP
# ==============================================================================

print("\n" + "-" * 78)
print("0. CLEANUP")
print("-" * 78)

for name in [
    "con",
    "joined",
    "positive_df",
    "rank_counts",
    "coverage_df",
]:
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass

gc.collect()

# ==============================================================================
# 1. PATHS
# ==============================================================================

print("\n" + "-" * 78)
print("1. PATHS")
print("-" * 78)

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

PAIR_PATH = (
    FEATURE_ROOT
    / "state"
    / "train_pair_sample_v1"
    / "baseline_candidate_pairs.parquet"
)

SEMANTIC_PATH = (
    FEATURE_ROOT
    / "features"
    / "train"
    / "full_candidate_semantic_v1"
    / "semantic_ranked_full.parquet"
)

OUTPUT_ROOT = (
    FEATURE_ROOT
    / "features"
    / "train"
    / "full_candidate_semantic_v1"
    / "diagnostics_rank_v1"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert PAIR_PATH.exists()
assert SEMANTIC_PATH.exists()

print(
    "Pair sample:",
    PAIR_PATH
)

print(
    "Semantic ranking:",
    SEMANTIC_PATH
)

# ==============================================================================
# 2. VERIFY INPUT COUNTS
# ==============================================================================

print("\n" + "-" * 78)
print("2. VERIFY INPUT COUNTS")
print("-" * 78)

con = duckdb.connect(
    database=":memory:"
)

con.execute(
    "SET memory_limit='2GB'"
)

con.execute(
    "SET threads=8"
)

con.execute(
    "SET preserve_insertion_order=false"
)

pair_rows = con.execute(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{PAIR_PATH}'
    )
    """
).fetchone()[0]

semantic_rows = con.execute(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{SEMANTIC_PATH}'
    )
    """
).fetchone()[0]

print(
    f"Pair sample rows:     {pair_rows:,}"
)

print(
    f"Semantic rows:        {semantic_rows:,}"
)

assert pair_rows == 3_429_214
assert semantic_rows == 22_302_012

# ==============================================================================
# 3. POSITIVE EDGE COUNT
# ==============================================================================

print("\n" + "-" * 78)
print("3. POSITIVE EDGES")
print("-" * 78)

positive_rows = con.execute(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{PAIR_PATH}'
    )
    WHERE is_positive = 1
    """
).fetchone()[0]

print(
    f"Positive labeled pairs: {positive_rows:,}"
)

assert positive_rows == 229_692

# ==============================================================================
# 4. JOIN POSITIVE EDGES TO COMPLETE SEMANTIC RANKING
# ==============================================================================

print("\n" + "-" * 78)
print("4. JOIN TRUE EDGES TO FULL RANKING")
print("-" * 78)

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW positives AS

    SELECT DISTINCT

        s1_entity_id,
        candidate_entity_id

    FROM read_parquet(
        '{PAIR_PATH}'
    )

    WHERE is_positive = 1
    """
)

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW semantic AS

    SELECT

        s1_entity_id,
        candidate_entity_id,
        source,

        semantic_full_cosine,

        semantic_rank_total,

        semantic_percentile_total,

        semantic_top1_cosine_total,

        semantic_top2_cosine_total,

        semantic_top3_cosine_total,

        semantic_margin_top1_top2,

        semantic_margin_top1_top3,

        semantic_rank_source,

        semantic_percentile_source,

        semantic_margin_source_top1_top2

    FROM read_parquet(
        '{SEMANTIC_PATH}'
    )
    """
)

con.execute(
    """
    CREATE OR REPLACE TEMP VIEW true_ranked AS

    SELECT

        p.s1_entity_id,

        p.candidate_entity_id,

        s.source,

        s.semantic_full_cosine,

        s.semantic_rank_total,

        s.semantic_percentile_total,

        s.semantic_top1_cosine_total,

        s.semantic_top2_cosine_total,

        s.semantic_top3_cosine_total,

        s.semantic_margin_top1_top2,

        s.semantic_margin_top1_top3,

        s.semantic_rank_source,

        s.semantic_percentile_source,

        s.semantic_margin_source_top1_top2

    FROM positives p

    INNER JOIN semantic s

        ON p.s1_entity_id =
           s.s1_entity_id

       AND p.candidate_entity_id =
           s.candidate_entity_id
    """
)

matched_positive_rows = con.execute(
    """
    SELECT COUNT(*)
    FROM true_ranked
    """
).fetchone()[0]

print(
    f"Matched positive rows: "
    f"{matched_positive_rows:,}"
)

assert (
    matched_positive_rows
    == positive_rows
)

print(
    "✅ Every known positive edge has a full-universe semantic rank."
)

# ==============================================================================
# 5. TRUE EDGE RANK QUANTILES
# ==============================================================================

print("\n" + "-" * 78)
print("5. TRUE-EDGE RANK DISTRIBUTION")
print("-" * 78)

rank_stats = con.execute(
    """
    SELECT

        MIN(semantic_rank_total)
            AS min_rank,

        AVG(semantic_rank_total)
            AS mean_rank,

        MEDIAN(semantic_rank_total)
            AS median_rank,

        QUANTILE_CONT(
            semantic_rank_total,
            0.50
        )
            AS q50,

        QUANTILE_CONT(
            semantic_rank_total,
            0.75
        )
            AS q75,

        QUANTILE_CONT(
            semantic_rank_total,
            0.90
        )
            AS q90,

        QUANTILE_CONT(
            semantic_rank_total,
            0.95
        )
            AS q95,

        QUANTILE_CONT(
            semantic_rank_total,
            0.99
        )
            AS q99,

        MAX(semantic_rank_total)
            AS max_rank

    FROM true_ranked
    """
).fetchdf()

print(
    rank_stats
)

# ==============================================================================
# 6. POSITIVE EDGE RECALL @ K
# ==============================================================================

print("\n" + "-" * 78)
print("6. TRUE-EDGE RECALL @ K")
print("-" * 78)

ks = [
    1,
    2,
    3,
    5,
    10,
    20,
    50,
    100,
    200,
    500,
    1000,
]

rank_rows = []

for k in ks:

    count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM true_ranked
        WHERE semantic_rank_total <= {k}
        """
    ).fetchone()[0]

    recall = (
        count
        / positive_rows
    )

    rank_rows.append({

        "k":
            k,

        "positive_edges_in_top_k":
            int(count),

        "positive_edge_recall":
            float(recall),
    })

rank_df = pl.DataFrame(
    rank_rows
)

print(
    rank_df
)

rank_df.write_parquet(
    OUTPUT_ROOT
    / "positive_edge_recall_at_k.parquet",
    compression="zstd",
)

# ==============================================================================
# 7. S1-LEVEL COVERAGE @ K
#
# For each S1 represented in the positive sample:
#
#   any_true_in_top_k
#   all_true_in_top_k
#
# This avoids confusing "edge recall" with "entity-level coverage".
# ==============================================================================

print("\n" + "-" * 78)
print("7. S1-LEVEL TRUE-MATCH COVERAGE")
print("-" * 78)

s1_positive_count = con.execute(
    """
    SELECT COUNT(*)
    FROM (
        SELECT DISTINCT
            s1_entity_id
        FROM true_ranked
    )
    """
).fetchone()[0]

print(
    f"S1s represented by at least one known positive: "
    f"{s1_positive_count:,}"
)

s1_rows = []

for k in ks:

    stats = con.execute(
        f"""
        WITH per_s1 AS (

            SELECT

                s1_entity_id,

                COUNT(*) AS total_true_edges,

                SUM(
                    CASE
                        WHEN semantic_rank_total <= {k}
                        THEN 1
                        ELSE 0
                    END
                ) AS true_edges_in_top_k

            FROM true_ranked

            GROUP BY
                s1_entity_id
        )

        SELECT

            COUNT(*) AS s1_count,

            SUM(
                CASE
                    WHEN true_edges_in_top_k >= 1
                    THEN 1
                    ELSE 0
                END
            ) AS any_true_in_top_k,

            SUM(
                CASE
                    WHEN true_edges_in_top_k =
                         total_true_edges
                    THEN 1
                    ELSE 0
                END
            ) AS all_true_in_top_k

        FROM per_s1
        """
    ).fetchone()

    (
        count_s1,
        any_true,
        all_true,
    ) = stats

    s1_rows.append({

        "k":
            k,

        "s1_count":
            int(count_s1),

        "s1_any_true_in_top_k":
            int(any_true),

        "s1_any_true_recall":
            float(
                any_true
                / count_s1
            ),

        "s1_all_true_in_top_k":
            int(all_true),

        "s1_all_true_recall":
            float(
                all_true
                / count_s1
            ),
    })

s1_df = pl.DataFrame(
    s1_rows
)

print(
    s1_df
)

s1_df.write_parquet(
    OUTPUT_ROOT
    / "s1_true_match_coverage_at_k.parquet",
    compression="zstd",
)

# ==============================================================================
# 8. TRUE EDGE COSINE + MARGIN DISTRIBUTION
# ==============================================================================

print("\n" + "-" * 78)
print("8. TRUE EDGE COSINE / MARGIN DISTRIBUTION")
print("-" * 78)

true_signal_stats = con.execute(
    """
    SELECT

        AVG(
            semantic_full_cosine
        )
            AS mean_cosine,

        MEDIAN(
            semantic_full_cosine
        )
            AS median_cosine,

        QUANTILE_CONT(
            semantic_full_cosine,
            0.01
        )
            AS cosine_q01,

        QUANTILE_CONT(
            semantic_full_cosine,
            0.05
        )
            AS cosine_q05,

        QUANTILE_CONT(
            semantic_full_cosine,
            0.10
        )
            AS cosine_q10,

        QUANTILE_CONT(
            semantic_full_cosine,
            0.25
        )
            AS cosine_q25,

        QUANTILE_CONT(
            semantic_full_cosine,
            0.50
        )
            AS cosine_q50,

        QUANTILE_CONT(
            semantic_full_cosine,
            0.90
        )
            AS cosine_q90,

        QUANTILE_CONT(
            semantic_full_cosine,
            0.95
        )
            AS cosine_q95,

        QUANTILE_CONT(
            semantic_full_cosine,
            0.99
        )
            AS cosine_q99,

        AVG(
            semantic_margin_top1_top2
        )
            AS mean_margin12,

        MEDIAN(
            semantic_margin_top1_top2
        )
            AS median_margin12,

        QUANTILE_CONT(
            semantic_margin_top1_top2,
            0.10
        )
            AS margin12_q10,

        QUANTILE_CONT(
            semantic_margin_top1_top2,
            0.25
        )
            AS margin12_q25,

        QUANTILE_CONT(
            semantic_margin_top1_top2,
            0.50
        )
            AS margin12_q50,

        QUANTILE_CONT(
            semantic_margin_top1_top2,
            0.75
        )
            AS margin12_q75,

        QUANTILE_CONT(
            semantic_margin_top1_top2,
            0.90
        )
            AS margin12_q90

    FROM true_ranked
    """
).fetchdf()

print(
    true_signal_stats
)

true_signal_stats.to_csv(
    OUTPUT_ROOT
    / "true_edge_semantic_signal.csv",
    index=False,
)

# ==============================================================================
# 9. POSITIVE EDGE RANK BY SOURCE
# ==============================================================================

print("\n" + "-" * 78)
print("9. POSITIVE RANK BY SOURCE")
print("-" * 78)

source_rank = con.execute(
    """
    SELECT

        source,

        COUNT(*) AS positive_edges,

        AVG(
            semantic_rank_total
        ) AS mean_rank,

        MEDIAN(
            semantic_rank_total
        ) AS median_rank,

        AVG(
            semantic_full_cosine
        ) AS mean_cosine,

        SUM(
            CASE
                WHEN semantic_rank_total = 1
                THEN 1
                ELSE 0
            END
        ) AS rank1_edges,

        SUM(
            CASE
                WHEN semantic_rank_total <= 5
                THEN 1
                ELSE 0
            END
        ) AS top5_edges,

        SUM(
            CASE
                WHEN semantic_rank_total <= 10
                THEN 1
                ELSE 0
            END
        ) AS top10_edges

    FROM true_ranked

    GROUP BY
        source

    ORDER BY
        source
    """
).fetchdf()

print(
    source_rank
)

source_rank.to_csv(
    OUTPUT_ROOT
    / "true_edge_rank_by_source.csv",
    index=False,
)

# ==============================================================================
# 10. SAVE JSON SUMMARY
# ==============================================================================

print("\n" + "-" * 78)
print("10. SAVE SUMMARY")
print("-" * 78)

rank_stats_dict = {
    str(k): (
        None
        if rank_stats.empty
        else float(
            rank_stats.iloc[0][k]
        )
    )
    for k in [
        "min_rank",
        "mean_rank",
        "median_rank",
        "q50",
        "q75",
        "q90",
        "q95",
        "q99",
        "max_rank",
    ]
}

summary = {

    "status":
        "COMPLETE",

    "positive_edges":
        int(positive_rows),

    "semantic_candidate_universe":
        22_302_012,

    "s1s_with_known_positive":
        int(s1_positive_count),

    "rank_distribution":
        rank_stats_dict,

    "positive_edge_recall_at_k":
        rank_rows,

    "s1_true_match_coverage_at_k":
        s1_rows,

    "semantic_signal":
        true_signal_stats.to_dict(
            orient="records"
        )[0],

    "source_rank":
        source_rank.to_dict(
            orient="records"
        ),

    "created_at":
        time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
}

SUMMARY_PATH = (
    OUTPUT_ROOT
    / "summary.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        indent=2,
        default=float,
    )
)

(
    OUTPUT_ROOT
    / "COMPLETE"
).write_text(
    "AMLC 2026\n"
    "CELL 69A COMPLETE\n"
    "TRUE-MATCH SEMANTIC RANK DIAGNOSTIC\n"
    f"POSITIVE_EDGES={positive_rows}\n"
)

# ==============================================================================
# 11. CLOSE
# ==============================================================================

con.close()

del con

gc.collect()

elapsed = (
    time.time()
    - T0
) / 60

print("\n" + "=" * 78)
print("✅ CELL 69A COMPLETE")
print("=" * 78)

print(f"""
KNOWN POSITIVE EDGES:
  {positive_rows:,}

FULL CANDIDATE UNIVERSE:
  22,302,012

S1s WITH KNOWN POSITIVES:
  {s1_positive_count:,}

OUTPUT:
  {OUTPUT_ROOT}

FILES:
  positive_edge_recall_at_k.parquet
  s1_true_match_coverage_at_k.parquet
  true_edge_semantic_signal.csv
  true_edge_rank_by_source.csv
  summary.json

✅ Every known positive was successfully mapped into
   the complete semantic candidate ranking.

NEXT:
  Use the rank@K and S1 coverage results to decide
  how aggressively semantic rank/margin should influence
  the final entity-resolution model.
""")

print(
    f"Runtime: {elapsed:.2f} min"
)

AMLC 2026 — CELL 69A
TRUE-MATCH SEMANTIC RANK DIAGNOSTIC

------------------------------------------------------------------------------
0. CLEANUP
------------------------------------------------------------------------------

------------------------------------------------------------------------------
1. PATHS
------------------------------------------------------------------------------
Pair sample: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/state/train_pair_sample_v1/baseline_candidate_pairs.parquet
Semantic ranking: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/train/full_candidate_semantic_v1/semantic_ranked_full.parquet

------------------------------------------------------------------------------
2. VERIFY INPUT COUNTS
------------------------------------------------------------------------------
Pair sample rows:     3,429,214
Semantic rows:        22,302,012

------------------------------------------------------------------------------
3. POSITIVE EDGES
----

In [42]:
# ==============================================================================
# AMLC 2026 — CELL 70A v7
# METRIC STAGE ONLY — RECOVERY SAFE
#
# PURPOSE:
#   Continue AFTER Cell 70A v6 successfully:
#       - trained the model
#       - created validation table
#       - scored 2,258,009 validation candidates
#
# FIX:
#   Never use pl.sum(expr).
#   Use expr.sum() instead.
#
# THIS CELL DOES NOT:
#   - recompute embeddings
#   - recompute cosine
#   - rebuild semantic rank features
#   - retrain unless the saved model is unavailable
#
# ==============================================================================

import gc
import json
import time
from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import polars as pl

T0 = time.time()

print("=" * 78)
print("AMLC 2026 — CELL 70A v7")
print("METRIC STAGE ONLY — POLARS-SAFE")
print("=" * 78)

# ==============================================================================
# 0. PATHS
# ==============================================================================

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

OUTPUT_ROOT = (
    FEATURE_ROOT
    / "models"
    / "semantic_ranker_v6"
)

TRAIN_TABLE = (
    FEATURE_ROOT
    / "models"
    / "semantic_ranker_v2"
    / "train_semantic_ranker.parquet"
)

VAL_TABLE = (
    OUTPUT_ROOT
    / "val_semantic_ranker_full_candidates.parquet"
)

MODEL_PATH = (
    OUTPUT_ROOT
    / "semantic_ranker.txt"
)

VAL_S1_PATH = (
    FEATURE_ROOT
    / "state"
    / "train_pair_sample_v1"
    / "val_s1.parquet"
)

GT_PATH = Path(
    "/kaggle/input/datasets/tanmayistired/"
    "amlc-2026-final-workspace/"
    "AMLC2026_KAGGLE_FINAL/"
    "dataset/train/train_ground_truth.tsv"
)

SUMMARY_PATH = (
    OUTPUT_ROOT
    / "summary_v7.json"
)

FEATURES = [

    "semantic_full_cosine",
    "semantic_full_distance",

    "semantic_rank_total",
    "semantic_percentile_total",

    "semantic_top1_cosine_total",
    "semantic_top2_cosine_total",
    "semantic_top3_cosine_total",

    "semantic_margin_top1_top2",
    "semantic_margin_top1_top3",

    "semantic_rank_source",
    "semantic_percentile_source",

    "semantic_top1_cosine_source",
    "semantic_top2_cosine_source",

    "semantic_margin_source_top1_top2",

    "candidate_count_total",
    "candidate_count_source",

    "source_is_s3",
]

# ==============================================================================
# 1. CHECK DURABLE ARTIFACTS
# ==============================================================================

print("\n" + "-" * 78)
print("1. DURABLE ARTIFACT CHECK")
print("-" * 78)

required_files = {

    "training table":
        TRAIN_TABLE,

    "validation table":
        VAL_TABLE,

    "saved model":
        MODEL_PATH,

    "validation S1":
        VAL_S1_PATH,

    "ground truth":
        GT_PATH,
}

for label, path in required_files.items():

    print(
        f"{label}: {path}"
    )

    if not path.exists():

        raise RuntimeError(
            f"Required artifact missing:\n{path}"
        )

print(
    "\n✅ All required durable artifacts exist."
)

# ==============================================================================
# 2. LOAD VALIDATION TABLE
# ==============================================================================

print("\n" + "-" * 78)
print("2. LOAD VALIDATION CANDIDATES")
print("-" * 78)

val_schema = pl.read_parquet_schema(
    VAL_TABLE
)

missing = [
    c
    for c in FEATURES
    if c not in val_schema
]

if missing:

    raise RuntimeError(
        "Validation table missing features:\n"
        + "\n".join(missing)
    )

val_df = (
    pl.scan_parquet(
        VAL_TABLE
    )
    .select([
        "s1_entity_id",
        "candidate_entity_id",
        "is_positive",
    ] + FEATURES)
    .collect(
        engine="streaming"
    )
)

print(
    f"Validation shape: {val_df.shape}"
)

if val_df.height != 2_258_009:

    print(
        "WARNING: validation row count differs "
        "from the previously observed 2,258,009."
    )

# ==============================================================================
# 3. LOAD MODEL
# ==============================================================================

print("\n" + "-" * 78)
print("3. LOAD SAVED MODEL")
print("-" * 78)

model = lgb.Booster(
    model_file=str(
        MODEL_PATH
    )
)

print(
    "Model loaded."
)

# ==============================================================================
# 4. SCORE VALIDATION
# ==============================================================================

print("\n" + "-" * 78)
print("4. SCORE VALIDATION")
print("-" * 78)

X_val = (
    val_df
    .select(
        FEATURES
    )
    .to_numpy()
    .astype(
        np.float32,
        copy=False
    )
)

proba = model.predict(
    X_val
)

print(
    f"Probability rows: {len(proba):,}"
)

if len(proba) != val_df.height:

    raise RuntimeError(
        "Model prediction length does not match "
        "validation rows."
    )

# ==============================================================================
# 5. BUILD OFFICIAL GT COUNT PER S1
# ==============================================================================

print("\n" + "-" * 78)
print("5. BUILD OFFICIAL GT COUNTS")
print("-" * 78)

con = duckdb.connect(
    database=":memory:"
)

con.execute(
    "SET memory_limit='1GB'"
)

con.execute(
    "SET threads=8"
)

con.execute(
    "SET preserve_insertion_order=false"
)

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW val_s1 AS

    SELECT DISTINCT

        CAST(
            s1_entity_id AS VARCHAR
        ) AS s1_entity_id

    FROM read_parquet(
        '{VAL_S1_PATH}'
    )
    """
)

val_s1_count = con.execute(
    """
    SELECT COUNT(*)
    FROM val_s1
    """
).fetchone()[0]

print(
    f"Validation S1s: {val_s1_count:,}"
)

# --------------------------------------------------------------------------
# Official GT expansion.
# --------------------------------------------------------------------------

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW gt AS

    SELECT DISTINCT

        CAST(
            source1_entity_id AS VARCHAR
        ) AS s1_entity_id,

        TRIM(
            CAST(
                candidate_id AS VARCHAR
            )
        ) AS candidate_entity_id

    FROM (

        SELECT

            source1_entity_id,

            UNNEST(
                STRING_SPLIT(
                    COALESCE(
                        CAST(
                            matched_entity_ids
                            AS VARCHAR
                        ),
                        ''
                    ),
                    ','
                )
            ) AS candidate_id

        FROM read_csv(
            '{GT_PATH}',
            delim='\\t',
            header=true
        )

    )

    WHERE
        candidate_id IS NOT NULL

        AND

        LENGTH(
            TRIM(
                candidate_id
            )
        ) > 0
    """
)

gt_count_df = pl.from_arrow(
    con.execute(
        """
        SELECT

            v.s1_entity_id,

            COALESCE(
                COUNT(g.candidate_entity_id),
                0
            ) AS gt_count

        FROM val_s1 v

        LEFT JOIN gt g

          ON v.s1_entity_id =
             g.s1_entity_id

        GROUP BY
            v.s1_entity_id
        """
    ).arrow()
)

print(
    f"GT metric groups: {gt_count_df.height:,}"
)

if gt_count_df.height != val_s1_count:

    raise RuntimeError(
        "GT count table does not cover all validation S1s."
    )

val_gt_edges = int(
    gt_count_df
    .get_column(
        "gt_count"
    )
    .sum()
)

val_gt_s1 = int(
    (
        gt_count_df
        .get_column(
            "gt_count"
        )
        > 0
    )
    .sum()
)

val_no_gt_s1 = (
    val_s1_count
    - val_gt_s1
)

print(
    f"Validation GT edges:       {val_gt_edges:,}"
)

print(
    f"S1s with GT:               {val_gt_s1:,}"
)

print(
    f"S1s without GT:            {val_no_gt_s1:,}"
)

# ==============================================================================
# 6. PREDICTION BASE
# ==============================================================================

prediction_base = (
    val_df
    .select([
        "s1_entity_id",
        "candidate_entity_id",
        "is_positive",
    ])
    .with_columns(
        pl.Series(
            "model_probability",
            proba.astype(
                np.float32,
                copy=False
            )
        )
    )
)

# ==============================================================================
# 7. CORRECT MACRO F0.5
# ==============================================================================

def complete_macro_f05(
    prediction_array,
):
    """
    Compute challenge-style macro F0.5 across ALL validation S1s.

    IMPORTANT:
      * S1s absent from candidate table are still included.
      * Official GT count determines FN.
      * GT edges outside candidate generation are therefore counted.
      * GT=0 + prediction=0 receives 1.0.
    """

    scored = (
        prediction_base
        .select([
            "s1_entity_id",
            "is_positive",
        ])
        .with_columns(
            pl.Series(
                "predicted",
                prediction_array.astype(
                    np.int8,
                    copy=False
                )
            )
        )
    )

    per_s1 = (
        scored
        .group_by(
            "s1_entity_id"
        )
        .agg([

            (
                (
                    (
                        pl.col(
                            "predicted"
                        )
                        == 1
                    )
                    &
                    (
                        pl.col(
                            "is_positive"
                        )
                        == 1
                    )
                )
                .cast(
                    pl.Int32
                )
                .sum()
            )
            .alias(
                "tp"
            ),

            pl.col(
                "predicted"
            )
            .sum()
            .alias(
                "pred_count"
            ),
        ])
    )

    complete = (
        gt_count_df
        .join(
            per_s1,
            on="s1_entity_id",
            how="left"
        )
        .with_columns([

            pl.col(
                "tp"
            )
            .fill_null(0)
            .cast(pl.Int64),

            pl.col(
                "pred_count"
            )
            .fill_null(0)
            .cast(pl.Int64),

        ])
        .with_columns([

            (
                pl.col(
                    "pred_count"
                )
                -
                pl.col(
                    "tp"
                )
            )
            .clip(
                lower_bound=0
            )
            .alias(
                "fp"
            ),

            (
                pl.col(
                    "gt_count"
                )
                -
                pl.col(
                    "tp"
                )
            )
            .clip(
                lower_bound=0
            )
            .alias(
                "fn"
            ),
        ])
    )

    arr = (
        complete
        .select([
            "tp",
            "fp",
            "fn",
            "gt_count",
            "pred_count",
        ])
        .to_numpy()
        .astype(
            np.float64
        )
    )

    tp = arr[:, 0]
    fp = arr[:, 1]
    fn = arr[:, 2]
    gt = arr[:, 3]
    pred_count = arr[:, 4]

    empty_exact = (
        (gt == 0)
        &
        (pred_count == 0)
    )

    precision = np.divide(
        tp,
        tp + fp,
        out=np.zeros_like(tp),
        where=(
            tp + fp
        ) > 0
    )

    recall = np.divide(
        tp,
        tp + fn,
        out=np.zeros_like(tp),
        where=(
            tp + fn
        ) > 0
    )

    denom = (
        0.25
        * precision
        + recall
    )

    f05 = np.divide(
        1.25
        * precision
        * recall,

        denom,

        out=np.zeros_like(
            precision
        ),

        where=(
            denom > 0
        )
    )

    f05[
        empty_exact
    ] = 1.0

    return float(
        f05.mean()
    )

# ==============================================================================
# 8. THRESHOLD SWEEP
# ==============================================================================

print("\n" + "-" * 78)
print("6. COMPLETE-S1 THRESHOLD SWEEP")
print("-" * 78)

thresholds = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.92,
    0.94,
    0.96,
    0.98,
]

threshold_rows = []

for threshold in thresholds:

    pred = (
        proba
        >= threshold
    )

    score = (
        complete_macro_f05(
            pred
        )
    )

    threshold_rows.append({

        "threshold":
            threshold,

        "candidate_predicted_pairs":
            int(
                pred.sum()
            ),

        "macro_f0_5_all_s1":
            score,
    })

threshold_df = pl.DataFrame(
    threshold_rows
)

print(
    threshold_df
)

threshold_df.write_parquet(
    OUTPUT_ROOT
    / "validation_threshold_sweep_v7.parquet",
    compression="zstd"
)

best_df = (
    threshold_df
    .sort(
        "macro_f0_5_all_s1",
        descending=True
    )
)

best_threshold = float(
    best_df
    .get_column(
        "threshold"
    )[0]
)

best_f05 = float(
    best_df
    .get_column(
        "macro_f0_5_all_s1"
    )[0]
)

print(
    f"\nBEST THRESHOLD: "
    f"{best_threshold:.4f}"
)

print(
    f"BEST COMPLETE-S1 MACRO F0.5: "
    f"{best_f05:.6f}"
)

# ==============================================================================
# 9. RANK-ONLY BASELINES
# ==============================================================================

print("\n" + "-" * 78)
print("7. COMPLETE-S1 RANK-ONLY BASELINES")
print("-" * 78)

rank_array = (
    val_df
    .get_column(
        "semantic_rank_total"
    )
    .to_numpy()
)

rank_rows = []

for k in [
    1,
    2,
    3,
    5,
    10,
    20,
    50,
    100,
    200,
    500,
]:

    pred = (
        rank_array
        <= k
    )

    score = (
        complete_macro_f05(
            pred
        )
    )

    rank_rows.append({

        "k":
            k,

        "candidate_predicted_pairs":
            int(
                pred.sum()
            ),

        "macro_f0_5_all_s1":
            score,
    })

rank_df = pl.DataFrame(
    rank_rows
)

print(
    rank_df
)

rank_df.write_parquet(
    OUTPUT_ROOT
    / "rank_only_validation_v7.parquet",
    compression="zstd"
)

# ==============================================================================
# 10. FEATURE IMPORTANCE
# ==============================================================================

print("\n" + "-" * 78)
print("8. FEATURE IMPORTANCE")
print("-" * 78)

importance_df = pl.DataFrame({

    "feature":
        FEATURES,

    "gain":
        model.feature_importance(
            importance_type="gain"
        ),

    "split":
        model.feature_importance(
            importance_type="split"
        ),

}).sort(
    "gain",
    descending=True
)

print(
    importance_df
)

importance_df.write_parquet(
    OUTPUT_ROOT
    / "feature_importance_v7.parquet",
    compression="zstd"
)

# ==============================================================================
# 11. FINAL SUMMARY
# ==============================================================================

print("\n" + "-" * 78)
print("9. SAVE SUMMARY")
print("-" * 78)

SUMMARY_PATH.write_text(
    json.dumps(
        {

            "status":
                "COMPLETE",

            "validation_s1":
                int(val_s1_count),

            "validation_s1_with_gt":
                int(val_gt_s1),

            "validation_s1_without_gt":
                int(val_no_gt_s1),

            "validation_gt_edges":
                int(val_gt_edges),

            "validation_candidate_rows":
                int(val_df.height),

            "candidate_positive_edges":
                int(
                    val_df
                    .get_column(
                        "is_positive"
                    )
                    .sum()
                ),

            "best_threshold":
                best_threshold,

            "best_complete_s1_macro_f0_5":
                best_f05,

            "threshold_sweep":
                threshold_rows,

            "rank_only":
                rank_rows,

            "model":
                str(MODEL_PATH),

            "created_at":
                time.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),
        },
        indent=2
    )
)

(
    OUTPUT_ROOT
    / "COMPLETE_v7"
).write_text(
    "AMLC 2026\n"
    "CELL 70A v7 COMPLETE\n"
    "POLARS-SAFE METRIC EVALUATION\n"
)

# ------------------------------------------------------------
# CLEANUP
# ------------------------------------------------------------
if "y_val" in globals():
    del y_val

if "proba" in globals():
    del proba

gc.collect()

print("\n" + "=" * 80)
print("CELL 70A v7 COMPLETE")
print("=" * 80)
print(f"Summary saved to: {SUMMARY_PATH}")

elapsed = (
    time.time()
    - T0
) / 60

print("\n" + "=" * 78)
print("✅ CELL 70A v7 COMPLETE")
print("=" * 78)

print(f"""
VALIDATION S1:
  {val_s1_count:,}

VALIDATION S1 WITH GT:
  {val_gt_s1:,}

VALIDATION S1 WITHOUT GT:
  {val_no_gt_s1:,}

OFFICIAL VALIDATION GT EDGES:
  {val_gt_edges:,}

VALIDATION CANDIDATE ROWS:
  {val_df.height:,}

BEST COMPLETE-S1 MACRO F0.5:
  {best_f05:.6f}

BEST THRESHOLD:
  {best_threshold:.4f}

OUTPUT:
  {OUTPUT_ROOT}

SUMMARY:
  {SUMMARY_PATH}

✅ No retraining.
✅ No cosine recomputation.
✅ No semantic recomputation.
✅ No dependency on semantic_ranker_v4/v5.
✅ All 9,953 validation S1s included.
✅ 541 no-GT S1s included.
✅ 226 zero-candidate S1s included.
✅ Candidate-generation misses count as FN.
✅ Polars expression aggregation fixed.
""")

print(
    f"Runtime: {elapsed:.2f} min"
)

AMLC 2026 — CELL 70A v7
METRIC STAGE ONLY — POLARS-SAFE

------------------------------------------------------------------------------
1. DURABLE ARTIFACT CHECK
------------------------------------------------------------------------------
training table: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/models/semantic_ranker_v2/train_semantic_ranker.parquet
validation table: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/models/semantic_ranker_v6/val_semantic_ranker_full_candidates.parquet
saved model: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/models/semantic_ranker_v6/semantic_ranker.txt
validation S1: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/state/train_pair_sample_v1/val_s1.parquet
ground truth: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_ground_truth.tsv

✅ All required durable artifacts exist.

------------------------------------------------------------------------------
2. LOAD VALIDATION CANDIDATES
--------

In [44]:
# ============================================================
# AMLC 2026 — CELL 71A v2
# FEATURE ENGINEERING ONLY
# NAME / ADDRESS FREQUENCY + RARITY
#
# NO MODEL
# NO TRAINING
# NO GROUND TRUTH
# NO PREDICTIONS
#
# PRIMARY SOURCE:
#   durable normalized entity lookups from runtime/
#
# FALLBACK:
#   raw train/test TSVs
#
# OUTPUT:
#   entity_rarity_v2/
#
# Creates TRAIN and TEST rarity tables separately.
# ============================================================

from pathlib import Path
import json
import gc
import re
import polars as pl

print("=" * 80)
print("AMLC 2026 — CELL 71A v2")
print("FEATURE ENGINEERING ONLY — FREQUENCY / RARITY")
print("=" * 80)

# ------------------------------------------------------------
# 0. ROOTS
# ------------------------------------------------------------

ROOT = Path("/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1")

FEATURE_DIR = ROOT / "features"
RUNTIME_DIR = ROOT / "runtime"

OUT_DIR = FEATURE_DIR / "entity_rarity_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Feature lake : {ROOT}")
print(f"Runtime dir  : {RUNTIME_DIR}")
print(f"Output dir   : {OUT_DIR}")

if not ROOT.exists():
    raise RuntimeError(f"Feature-lake root missing: {ROOT}")

# ------------------------------------------------------------
# 1. DATASET ROOTS
# ------------------------------------------------------------

DATASET_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/"
    "amlc-2026-final-workspace/"
    "AMLC2026_KAGGLE_FINAL/"
)

TRAIN_DIR = DATASET_ROOT / "dataset" / "train"
TEST_DIR = DATASET_ROOT / "dataset" / "test"

print("\n" + "-" * 80)
print("1. DATASET PATH CHECK")
print("-" * 80)

print(f"Dataset root exists : {DATASET_ROOT.exists()}")
print(f"Train dir exists   : {TRAIN_DIR.exists()}")
print(f"Test dir exists    : {TEST_DIR.exists()}")

if not DATASET_ROOT.exists():
    raise RuntimeError(
        f"Expected challenge dataset root does not exist:\n{DATASET_ROOT}"
    )

# ------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------

def schema_names(path):
    """
    Safely inspect parquet/csv schema without loading rows.
    """
    try:
        if path.suffix.lower() == ".parquet":
            return pl.scan_parquet(str(path)).collect_schema().names()

        if path.suffix.lower() == ".tsv":
            return pl.scan_csv(
                str(path),
                separator="\t",
                infer_schema_length=1000,
                ignore_errors=True,
            ).collect_schema().names()

    except Exception as e:
        print(f"  [WARN] Could not inspect {path}: {e}")

    return []


def first_matching_column(columns, names):
    """
    Case-insensitive exact-name resolver.
    """
    lookup = {c.lower(): c for c in columns}

    for name in names:
        hit = lookup.get(name.lower())
        if hit is not None:
            return hit

    return None


def fuzzy_column(columns, patterns):
    """
    Conservative substring resolver.
    """
    lower = {c.lower(): c for c in columns}

    for pat in patterns:
        for lc, original in lower.items():
            if pat in lc:
                return original

    return None


def resolve_entity_column(columns):
    candidates = [
        "entity_id",
        "source1_entity_id",
        "source2_entity_id",
        "source3_entity_id",
        "s1_entity_id",
        "s2_entity_id",
        "s3_entity_id",
        "id",
    ]

    col = first_matching_column(columns, candidates)

    if col is not None:
        return col

    return fuzzy_column(
        columns,
        [
            "entity_id",
            "_entity",
        ],
    )


def resolve_name_column(columns):
    candidates = [
        "normalized_name",
        "name_normalized",
        "norm_name",
        "name_norm",
        "name_key",
        "name",
    ]

    col = first_matching_column(columns, candidates)

    if col is not None:
        return col

    return fuzzy_column(
        columns,
        [
            "normalized_name",
            "name_norm",
            "name_key",
        ],
    )


def resolve_address_column(columns):
    candidates = [
        "normalized_address",
        "address_normalized",
        "norm_address",
        "address_norm",
        "address_key",
        "address",
    ]

    col = first_matching_column(columns, candidates)

    if col is not None:
        return col

    return fuzzy_column(
        columns,
        [
            "normalized_address",
            "address_norm",
            "address_key",
        ],
    )


def resolve_country_column(columns):
    candidates = [
        "country",
        "country_code",
        "country_normalized",
        "country_norm",
    ]

    col = first_matching_column(columns, candidates)

    if col is not None:
        return col

    return fuzzy_column(
        columns,
        [
            "country",
        ],
    )


def source_from_name(name):
    text = name.lower()

    if "source1" in text or "s1" in text:
        return "source1"

    if "source2" in text or "s2" in text:
        return "source2"

    if "source3" in text or "s3" in text:
        return "source3"

    return None


# ------------------------------------------------------------
# 3. FIND DURABLE LOOKUPS
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("2. SEARCHING DURABLE RUNTIME LOOKUPS")
print("-" * 80)

all_runtime_parquet = sorted(
    RUNTIME_DIR.rglob("*.parquet")
) if RUNTIME_DIR.exists() else []

print(f"Runtime parquet files found: {len(all_runtime_parquet)}")

for p in all_runtime_parquet[:100]:
    print(" ", p.relative_to(RUNTIME_DIR))

# Expected artifacts from earlier pipeline.
# Search filename first because these are known durable artifacts.

lookup_candidates = {
    "source1": {"name": [], "address": []},
    "source2": {"name": [], "address": []},
    "source3": {"name": [], "address": []},
}

for p in all_runtime_parquet:
    fn = p.name.lower()

    src = source_from_name(fn)

    if src is None:
        continue

    if "name" in fn:
        lookup_candidates[src]["name"].append(p)

    if "address" in fn:
        lookup_candidates[src]["address"].append(p)

# ------------------------------------------------------------
# 4. SCORE LOOKUP CANDIDATES
# ------------------------------------------------------------

def choose_lookup(paths, expected_kind):
    scored = []

    for p in paths:
        cols = schema_names(p)

        entity_col = resolve_entity_column(cols)

        if expected_kind == "name":
            value_col = resolve_name_column(cols)
        else:
            value_col = resolve_address_column(cols)

        score = 0

        if entity_col is not None:
            score += 10

        if value_col is not None:
            score += 10

        # Prefer explicitly named lookup artifacts.
        fn = p.name.lower()

        if expected_kind in fn:
            score += 3

        if "lookup" in fn:
            score += 2

        scored.append(
            (
                score,
                p,
                cols,
                entity_col,
                value_col,
            )
        )

    scored.sort(
        key=lambda x: (-x[0], str(x[1]))
    )

    if not scored:
        return None

    return scored[0]


selected = {
    "source1": {},
    "source2": {},
    "source3": {},
}

print("\nSelected runtime lookups:")

for src in ["source1", "source2", "source3"]:

    for kind in ["name", "address"]:

        choice = choose_lookup(
            lookup_candidates[src][kind],
            kind,
        )

        if choice is None:
            print(f"  {src} {kind}: NOT FOUND")
            continue

        score, path, cols, entity_col, value_col = choice

        selected[src][kind] = {
            "path": path,
            "entity_col": entity_col,
            "value_col": value_col,
            "columns": cols,
        }

        print(
            f"  {src:8s} {kind:7s} "
            f"score={score:2d} "
            f"entity={entity_col} "
            f"value={value_col} "
            f"path={path.name}"
        )

# ------------------------------------------------------------
# 5. LOAD RAW TSV SCHEMAS FOR FALLBACK / COUNTRY
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("3. RAW TSV SCHEMAS")
print("-" * 80)

split_dirs = {
    "train": TRAIN_DIR,
    "test": TEST_DIR,
}

raw_files = {
    "train": {},
    "test": {},
}

for split, directory in split_dirs.items():

    for src in ["source1", "source2", "source3"]:

        expected = directory / f"{src.replace('source', 'source')}.tsv"

        # Normal expected filename.
        if expected.exists():
            raw_files[split][src] = expected
            continue

        # Flexible search.
        matches = [
            p
            for p in directory.glob("*.tsv")
            if source_from_name(p.name) == src
        ]

        if matches:
            matches.sort(key=lambda p: str(p))
            raw_files[split][src] = matches[0]

    print(f"\n{split.upper()}")

    for src, path in raw_files[split].items():
        cols = schema_names(path)
        print(f"  {src}: {path.name}")
        print(f"      columns={cols}")

# ------------------------------------------------------------
# 6. NORMALIZED ENTITY TABLE LOADER
# ------------------------------------------------------------

def load_lookup_table(
    path,
    source,
    kind,
):
    """
    Load a durable normalized name/address lookup and
    return exactly:

        entity_id
        feature_value

    This path preserves the normalization already produced
    by the earlier pipeline.
    """

    cols = schema_names(path)

    entity_col = resolve_entity_column(cols)

    if entity_col is None:
        raise RuntimeError(
            f"Could not resolve entity ID in {path}.\n"
            f"Columns: {cols}"
        )

    if kind == "name":
        value_col = resolve_name_column(cols)
    else:
        value_col = resolve_address_column(cols)

    if value_col is None:
        raise RuntimeError(
            f"Could not resolve {kind} column in {path}.\n"
            f"Columns: {cols}"
        )

    df = (
        pl.scan_parquet(str(path))
        .select(
            [
                pl.col(entity_col)
                .cast(pl.Utf8, strict=False)
                .alias("entity_id"),

                pl.col(value_col)
                .cast(pl.Utf8, strict=False)
                .fill_null("")
                .alias("feature_value"),
            ]
        )
        .collect()
    )

    # Clean only null/whitespace artifacts.
    df = df.with_columns(
        pl.col("feature_value")
        .str.strip_chars()
        .fill_null("")
    )

    # One row per entity.
    df = (
        df
        .unique(
            subset=["entity_id"],
            keep="first",
        )
    )

    return df


def load_raw_columns(
    path,
    source,
):
    """
    Raw TSV fallback / country extraction.

    Does not redefine canonical normalized name/address;
    this is only used if durable lookup artifacts are absent.
    """

    cols = schema_names(path)

    entity_col = resolve_entity_column(cols)
    name_col = resolve_name_column(cols)
    address_col = resolve_address_column(cols)
    country_col = resolve_country_column(cols)

    if entity_col is None:
        raise RuntimeError(
            f"Cannot resolve entity ID in raw file:\n{path}\n"
            f"Columns: {cols}"
        )

    wanted = [entity_col]

    for c in [name_col, address_col, country_col]:
        if c is not None and c not in wanted:
            wanted.append(c)

    df = (
        pl.scan_csv(
            str(path),
            separator="\t",
            infer_schema_length=5000,
            ignore_errors=False,
        )
        .select(wanted)
        .collect()
    )

    rename = {
        entity_col: "entity_id",
    }

    if name_col is not None:
        rename[name_col] = "name_raw"

    if address_col is not None:
        rename[address_col] = "address_raw"

    if country_col is not None:
        rename[country_col] = "country_raw"

    df = df.rename(rename)

    if "name_raw" not in df.columns:
        df = df.with_columns(
            pl.lit("").alias("name_raw")
        )

    if "address_raw" not in df.columns:
        df = df.with_columns(
            pl.lit("").alias("address_raw")
        )

    if "country_raw" not in df.columns:
        df = df.with_columns(
            pl.lit("").alias("country_raw")
        )

    return df


# ------------------------------------------------------------
# 7. OPTIONAL RAW NORMALIZER
# ------------------------------------------------------------

def fallback_normalize(expr):
    """
    Conservative fallback only.

    IMPORTANT:
    Durable normalized lookup values are preferred,
    so this is normally not executed.
    """

    return (
        expr
        .cast(pl.Utf8, strict=False)
        .fill_null("")
        .str.to_lowercase()
        .str.replace_all("&", " and ")
        .str.replace_all(r"[^\p{L}\p{N}\s]", " ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


# ------------------------------------------------------------
# 8. BUILD ONE SPLIT / SOURCE
# ------------------------------------------------------------

def build_rarity_table(
    split,
    source,
):
    print("\n" + "=" * 70)
    print(f"BUILDING: {split.upper()} / {source.upper()}")
    print("=" * 70)

    name_info = selected[source].get("name")
    address_info = selected[source].get("address")
    raw_path = raw_files[split].get(source)

    # --------------------------------------------------------
    # Entity base
    # --------------------------------------------------------

    if name_info is not None:

        name_df = load_lookup_table(
            name_info["path"],
            source,
            "name",
        )

        print(
            f"Name lookup loaded: "
            f"{name_df.height:,} rows"
        )

    else:
        name_df = None

    if address_info is not None:

        address_df = load_lookup_table(
            address_info["path"],
            source,
            "address",
        )

        print(
            f"Address lookup loaded: "
            f"{address_df.height:,} rows"
        )

    else:
        address_df = None

    # --------------------------------------------------------
    # Raw fallback
    # --------------------------------------------------------

    raw_df = None

    if name_df is None or address_df is None:

        if raw_path is None:
            raise RuntimeError(
                f"No durable lookup and no raw TSV available "
                f"for {split}/{source}."
            )

        print(
            "Durable lookup incomplete; "
            f"loading raw TSV fallback: {raw_path.name}"
        )

        raw_df = load_raw_columns(
            raw_path,
            source,
        )

    # --------------------------------------------------------
    # Build entity frame
    # --------------------------------------------------------

    if name_df is not None:

        base = name_df.rename(
            {"feature_value": "normalized_name"}
        )

    else:

        base = (
            raw_df
            .select(
                [
                    "entity_id",
                    fallback_normalize(
                        pl.col("name_raw")
                    ).alias("normalized_name"),
                ]
            )
            .unique(
                subset=["entity_id"],
                keep="first",
            )
        )

    if address_df is not None:

        addr = address_df.rename(
            {"feature_value": "normalized_address"}
        )

        base = base.join(
            addr,
            on="entity_id",
            how="left",
        )

    else:

        base = base.join(
            raw_df
            .select(
                [
                    "entity_id",
                    fallback_normalize(
                        pl.col("address_raw")
                    ).alias("normalized_address"),
                ]
            )
            .unique(
                subset=["entity_id"],
                keep="first",
            ),
            on="entity_id",
            how="left",
        )

    # --------------------------------------------------------
    # Country
    # --------------------------------------------------------

    if raw_df is None and raw_path is not None:

        # Load ONLY entity + country from raw TSV.
        raw_cols = schema_names(raw_path)

        entity_col = resolve_entity_column(raw_cols)
        country_col = resolve_country_column(raw_cols)

        if country_col is not None:

            country_df = (
                pl.scan_csv(
                    str(raw_path),
                    separator="\t",
                    infer_schema_length=5000,
                    ignore_errors=False,
                )
                .select(
                    [
                        pl.col(entity_col)
                        .cast(pl.Utf8, strict=False)
                        .alias("entity_id"),

                        pl.col(country_col)
                        .cast(pl.Utf8, strict=False)
                        .fill_null("")
                        .alias("country_raw"),
                    ]
                )
                .collect()
                .unique(
                    subset=["entity_id"],
                    keep="first",
                )
            )

            base = base.join(
                country_df,
                on="entity_id",
                how="left",
            )

            del country_df

    elif raw_df is not None:

        base = base.join(
            raw_df
            .select(
                [
                    "entity_id",
                    "country_raw",
                ]
            )
            .unique(
                subset=["entity_id"],
                keep="first",
            ),
            on="entity_id",
            how="left",
        )

    else:

        base = base.with_columns(
            pl.lit("").alias("country_raw")
        )

    # --------------------------------------------------------
    # Clean
    # --------------------------------------------------------

    base = base.with_columns(
        pl.col("normalized_name")
        .fill_null("")
        .cast(pl.Utf8),

        pl.col("normalized_address")
        .fill_null("")
        .cast(pl.Utf8),

        pl.col("country_raw")
        .fill_null("")
        .cast(pl.Utf8),
    )

    # --------------------------------------------------------
    # Frequencies
    # --------------------------------------------------------

    print("Building frequency maps...")

    name_freq = (
        base
        .filter(
            pl.col("normalized_name") != ""
        )
        .group_by("normalized_name")
        .len()
        .rename({"len": "name_frequency"})
    )

    address_freq = (
        base
        .filter(
            pl.col("normalized_address") != ""
        )
        .group_by("normalized_address")
        .len()
        .rename({"len": "address_frequency"})
    )

    name_address_freq = (
        base
        .filter(
            (pl.col("normalized_name") != "")
            &
            (pl.col("normalized_address") != "")
        )
        .group_by(
            [
                "normalized_name",
                "normalized_address",
            ]
        )
        .len()
        .rename(
            {"len": "name_address_frequency"}
        )
    )

    name_country_freq = (
        base
        .filter(
            (pl.col("normalized_name") != "")
            &
            (pl.col("country_raw") != "")
        )
        .group_by(
            [
                "normalized_name",
                "country_raw",
            ]
        )
        .len()
        .rename(
            {"len": "name_country_frequency"}
        )
    )

    address_country_freq = (
        base
        .filter(
            (pl.col("normalized_address") != "")
            &
            (pl.col("country_raw") != "")
        )
        .group_by(
            [
                "normalized_address",
                "country_raw",
            ]
        )
        .len()
        .rename(
            {"len": "address_country_frequency"}
        )
    )

    # --------------------------------------------------------
    # Join frequency features
    # --------------------------------------------------------

    out = (
        base

        .join(
            name_freq,
            on="normalized_name",
            how="left",
        )

        .join(
            address_freq,
            on="normalized_address",
            how="left",
        )

        .join(
            name_address_freq,
            on=[
                "normalized_name",
                "normalized_address",
            ],
            how="left",
        )

        .join(
            name_country_freq,
            on=[
                "normalized_name",
                "country_raw",
            ],
            how="left",
        )

        .join(
            address_country_freq,
            on=[
                "normalized_address",
                "country_raw",
            ],
            how="left",
        )

        .with_columns(

            # ------------------------------------------------
            # Missingness
            # ------------------------------------------------

            (
                pl.col("normalized_name")
                .str.len_chars()
                == 0
            )
            .cast(pl.Int8)
            .alias("name_missing"),

            (
                pl.col("normalized_address")
                .str.len_chars()
                == 0
            )
            .cast(pl.Int8)
            .alias("address_missing"),

            (
                pl.col("country_raw")
                .str.len_chars()
                == 0
            )
            .cast(pl.Int8)
            .alias("country_missing"),

            # ------------------------------------------------
            # Lengths
            # ------------------------------------------------

            pl.col("normalized_name")
            .str.len_chars()
            .cast(pl.Int32)
            .alias("name_length"),

            pl.col("normalized_address")
            .str.len_chars()
            .cast(pl.Int32)
            .alias("address_length"),

            # ------------------------------------------------
            # Token counts
            # ------------------------------------------------

            pl.when(
                pl.col("normalized_name") == ""
            )
            .then(0)
            .otherwise(
                pl.col("normalized_name")
                .str.split(" ")
                .list.len()
            )
            .cast(pl.Int16)
            .alias("name_token_count"),

            pl.when(
                pl.col("normalized_address") == ""
            )
            .then(0)
            .otherwise(
                pl.col("normalized_address")
                .str.split(" ")
                .list.len()
            )
            .cast(pl.Int16)
            .alias("address_token_count"),
        )

        .with_columns(

            # ------------------------------------------------
            # Counts
            # ------------------------------------------------

            pl.col("name_frequency")
            .fill_null(0)
            .cast(pl.Int32),

            pl.col("address_frequency")
            .fill_null(0)
            .cast(pl.Int32),

            pl.col("name_address_frequency")
            .fill_null(0)
            .cast(pl.Int32),

            pl.col("name_country_frequency")
            .fill_null(0)
            .cast(pl.Int32),

            pl.col("address_country_frequency")
            .fill_null(0)
            .cast(pl.Int32),
        )

        .with_columns(

            # ------------------------------------------------
            # Log frequency
            # ------------------------------------------------

            pl.col("name_frequency")
            .cast(pl.Float32)
            .log1p()
            .alias("log_name_frequency"),

            pl.col("address_frequency")
            .cast(pl.Float32)
            .log1p()
            .alias("log_address_frequency"),

            pl.col("name_address_frequency")
            .cast(pl.Float32)
            .log1p()
            .alias(
                "log_name_address_frequency"
            ),

            pl.col("name_country_frequency")
            .cast(pl.Float32)
            .log1p()
            .alias(
                "log_name_country_frequency"
            ),

            pl.col("address_country_frequency")
            .cast(pl.Float32)
            .log1p()
            .alias(
                "log_address_country_frequency"
            ),

            # ------------------------------------------------
            # Uniqueness
            # ------------------------------------------------

            (
                1.0
                /
                pl.col("name_frequency")
                .cast(pl.Float32)
                .clip(lower_bound=1.0)
            )
            .alias("name_uniqueness"),

            (
                1.0
                /
                pl.col("address_frequency")
                .cast(pl.Float32)
                .clip(lower_bound=1.0)
            )
            .alias("address_uniqueness"),

            # ------------------------------------------------
            # Rare indicators
            # ------------------------------------------------

            (
                pl.col("name_frequency") == 1
            )
            .cast(pl.Int8)
            .alias("name_is_unique"),

            (
                pl.col("address_frequency") == 1
            )
            .cast(pl.Int8)
            .alias("address_is_unique"),

            (
                pl.col("name_frequency") <= 3
            )
            .cast(pl.Int8)
            .alias("name_is_rare_3"),

            (
                pl.col("address_frequency") <= 3
            )
            .cast(pl.Int8)
            .alias("address_is_rare_3"),

            (
                pl.col("name_frequency") <= 10
            )
            .cast(pl.Int8)
            .alias("name_is_rare_10"),

            (
                pl.col("address_frequency") <= 10
            )
            .cast(pl.Int8)
            .alias("address_is_rare_10"),
        )

        .select(
            [
                "entity_id",

                "normalized_name",
                "normalized_address",
                "country_raw",

                "name_frequency",
                "address_frequency",
                "name_address_frequency",
                "name_country_frequency",
                "address_country_frequency",

                "log_name_frequency",
                "log_address_frequency",
                "log_name_address_frequency",
                "log_name_country_frequency",
                "log_address_country_frequency",

                "name_uniqueness",
                "address_uniqueness",

                "name_is_unique",
                "address_is_unique",
                "name_is_rare_3",
                "address_is_rare_3",
                "name_is_rare_10",
                "address_is_rare_10",

                "name_missing",
                "address_missing",
                "country_missing",

                "name_length",
                "address_length",

                "name_token_count",
                "address_token_count",
            ]
        )
    )

    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if out.height == 0:
        raise RuntimeError(
            f"Feature table is empty for {split}/{source}."
        )

    duplicate_entities = (
        out
        .group_by("entity_id")
        .len()
        .filter(pl.col("len") > 1)
        .height
    )

    if duplicate_entities != 0:
        raise RuntimeError(
            f"{split}/{source}: "
            f"{duplicate_entities:,} duplicate entity IDs."
        )

    # --------------------------------------------------------
    # Write
    # --------------------------------------------------------

    output_path = (
        OUT_DIR
        / f"{split}_{source}_entity_rarity.parquet"
    )

    out.write_parquet(
        output_path,
        compression="zstd",
    )

    print(
        f"Saved: {output_path}"
    )

    print(
        f"Rows: {out.height:,} | "
        f"Columns: {len(out.columns)}"
    )

    del base
    del out
    del name_freq
    del address_freq
    del name_address_freq
    del name_country_freq
    del address_country_freq

    if name_df is not None:
        del name_df

    if address_df is not None:
        del address_df

    if raw_df is not None:
        del raw_df

    gc.collect()

    return {
        "path": str(output_path),
        "rows": int(
            pl.scan_parquet(str(output_path))
            .select(pl.len())
            .collect()
            .item()
        ),
    }


# ------------------------------------------------------------
# 9. BUILD ALL 6 TABLES
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("4. BUILD FEATURE TABLES")
print("-" * 80)

results = {}

for split in ["train", "test"]:

    for source in ["source1", "source2", "source3"]:

        results[f"{split}_{source}"] = build_rarity_table(
            split=split,
            source=source,
        )

# ------------------------------------------------------------
# 10. FEATURE MANIFEST
# ------------------------------------------------------------

manifest = {
    "cell": "71A_v2",
    "purpose": "frequency_and_rarity_features",
    "model_training": False,
    "ground_truth_used": False,
    "predictions_generated": False,

    "important_design": {
        "train_features_use_train_population": True,
        "test_features_use_test_population": True,
        "ground_truth_excluded": True,
        "external_entity_lookup_excluded": True,
    },

    "feature_names": [
        "name_frequency",
        "address_frequency",
        "name_address_frequency",
        "name_country_frequency",
        "address_country_frequency",

        "log_name_frequency",
        "log_address_frequency",
        "log_name_address_frequency",
        "log_name_country_frequency",
        "log_address_country_frequency",

        "name_uniqueness",
        "address_uniqueness",

        "name_is_unique",
        "address_is_unique",
        "name_is_rare_3",
        "address_is_rare_3",
        "name_is_rare_10",
        "address_is_rare_10",

        "name_missing",
        "address_missing",
        "country_missing",

        "name_length",
        "address_length",

        "name_token_count",
        "address_token_count",
    ],

    "outputs": results,
}

manifest_path = OUT_DIR / "manifest.json"

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
    )

# ------------------------------------------------------------
# 11. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 71A v2 COMPLETE")
print("=" * 80)

for key, info in results.items():
    print(
        f"{key:18s} "
        f"{info['rows']:,} rows"
    )

print(f"\nManifest:")
print(manifest_path)

print("\n" + "-" * 80)
print("GUARDRAILS")
print("-" * 80)
print("✅ No model trained")
print("✅ No model loaded")
print("✅ No predictions generated")
print("✅ No ground truth used")
print("✅ Train/test populations handled separately")
print("✅ Feature tables keyed by entity_id")
print("✅ Duplicate entity IDs rejected")
print("✅ Durable normalized lookups preferred")
print("✅ Raw TSV fallback available")

print("\nFEATURE ENGINEERING CONTINUES.")

AMLC 2026 — CELL 71A v2
FEATURE ENGINEERING ONLY — FREQUENCY / RARITY
Feature lake : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1
Runtime dir  : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/runtime
Output dir   : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/entity_rarity_v2

--------------------------------------------------------------------------------
1. DATASET PATH CHECK
--------------------------------------------------------------------------------
Dataset root exists : True
Train dir exists   : True
Test dir exists    : True

--------------------------------------------------------------------------------
2. SEARCHING DURABLE RUNTIME LOOKUPS
--------------------------------------------------------------------------------
Runtime parquet files found: 0

Selected runtime lookups:
  source1 name: NOT FOUND
  source1 address: NOT FOUND
  source2 name: NOT FOUND
  source2 address: NOT FOUND
  source3 name: NOT FOUND
  source3 address: NOT FOUND

----------------------

In [46]:
# ============================================================
# AMLC 2026 — CELL 71B v2
# FEATURE ENGINEERING ONLY
# EXPANDED NAME REPRESENTATION FEATURES
#
# NO MODEL
# NO TRAINING
# NO GROUND TRUTH
# NO PREDICTIONS
#
# v2 FIXES:
#   - NO list.get() on potentially short lists
#   - NO regex backreferences
#   - safe first/last-two-token construction
#   - safe repeated-token feature
# ============================================================

from pathlib import Path
import json
import gc
import polars as pl

print("=" * 80)
print("AMLC 2026 — CELL 71B v2")
print("FEATURE ENGINEERING ONLY — EXPANDED NAME FEATURES")
print("=" * 80)


# ------------------------------------------------------------
# 0. PATHS
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/"
    "amlc-2026-final-workspace/"
    "AMLC2026_KAGGLE_FINAL/"
)

TRAIN_DIR = DATASET_ROOT / "dataset" / "train"
TEST_DIR = DATASET_ROOT / "dataset" / "test"

OUT_DIR = (
    ROOT
    / "features"
    / "name_representation_v2"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"ROOT      : {ROOT}")
print(f"DATASET   : {DATASET_ROOT}")
print(f"OUTPUT    : {OUT_DIR}")

if not ROOT.exists():
    raise RuntimeError(
        f"Feature lake missing: {ROOT}"
    )

if not TRAIN_DIR.exists():
    raise RuntimeError(
        f"Train directory missing: {TRAIN_DIR}"
    )

if not TEST_DIR.exists():
    raise RuntimeError(
        f"Test directory missing: {TEST_DIR}"
    )


# ------------------------------------------------------------
# 1. FILE RESOLUTION
# ------------------------------------------------------------

def resolve_source_file(
    directory,
    split,
    source,
):
    """
    Resolve challenge TSV deterministically.
    """

    exact = (
        directory
        / f"{split}_{source}.tsv"
    )

    if exact.exists():
        return exact

    source_number = source.replace(
        "source",
        "",
    )

    matches = []

    for path in sorted(
        directory.glob("*.tsv")
    ):
        name = path.name.lower()

        if split.lower() not in name:
            continue

        if (
            f"source{source_number}" in name
            or f"s{source_number}" in name
        ):
            matches.append(path)

    if matches:
        return matches[0]

    raise RuntimeError(
        f"Could not resolve {split}/{source} "
        f"under {directory}"
    )


# ------------------------------------------------------------
# 2. COLUMN RESOLUTION
# ------------------------------------------------------------

def normalize_column_name(c):
    return (
        c
        .strip()
        .lower()
        .replace(" ", "_")
    )


def resolve_column(
    columns,
    candidates,
):
    mapping = {
        normalize_column_name(c): c
        for c in columns
    }

    for candidate in candidates:

        key = normalize_column_name(
            candidate
        )

        if key in mapping:
            return mapping[key]

    return None


def inspect_schema(path):

    return (
        pl.scan_csv(
            str(path),
            separator="\t",
            infer_schema_length=2000,
            ignore_errors=False,
        )
        .collect_schema()
        .names()
    )


# ------------------------------------------------------------
# 3. CANONICAL NAME NORMALIZATION
# ------------------------------------------------------------

def canonical_name(expr):
    """
    Established canonical string normalization family:

        lowercase
        & -> and
        punctuation/symbol removal
        whitespace collapse
        Unicode preserved
    """

    return (
        expr
        .cast(
            pl.Utf8,
            strict=False,
        )
        .fill_null("")
        .str.strip_chars()
        .str.to_lowercase()
        .str.replace_all(
            "&",
            " and ",
        )
        .str.replace_all(
            r"[^\p{L}\p{N}\s]",
            " ",
        )
        .str.replace_all(
            r"\s+",
            " ",
        )
        .str.strip_chars()
    )


# ------------------------------------------------------------
# 4. BUILD ONE NAME FEATURE TABLE
# ------------------------------------------------------------

def build_name_table(
    split,
    source,
    path,
):

    print("\n" + "=" * 72)
    print(
        f"BUILDING {split.upper()} / "
        f"{source.upper()}"
    )
    print("=" * 72)

    print(f"Input: {path}")

    schema = inspect_schema(path)

    print(
        "Columns:",
        schema,
    )

    entity_col = resolve_column(
        schema,
        [
            "entity_id",
            "source1_entity_id",
            "source2_entity_id",
            "source3_entity_id",
            "s1_entity_id",
            "s2_entity_id",
            "s3_entity_id",
        ],
    )

    name_col = resolve_column(
        schema,
        [
            "business_name",
            "name",
        ],
    )

    if entity_col is None:
        raise RuntimeError(
            f"Could not resolve entity ID for "
            f"{split}/{source}.\n"
            f"Schema: {schema}"
        )

    if name_col is None:
        raise RuntimeError(
            f"Could not resolve business name for "
            f"{split}/{source}.\n"
            f"Schema: {schema}"
        )

    print(
        f"Resolved entity column: "
        f"{entity_col}"
    )

    print(
        f"Resolved name column  : "
        f"{name_col}"
    )

    # --------------------------------------------------------
    # LOAD
    # --------------------------------------------------------

    df = (
        pl.scan_csv(
            str(path),
            separator="\t",
            infer_schema_length=5000,
            ignore_errors=False,
        )
        .select(
            [
                pl.col(entity_col)
                .cast(
                    pl.Utf8,
                    strict=False,
                )
                .fill_null("")
                .str.strip_chars()
                .alias("entity_id"),

                pl.col(name_col)
                .cast(
                    pl.Utf8,
                    strict=False,
                )
                .fill_null("")
                .alias("business_name"),
            ]
        )
        .collect()
    )

    print(
        f"Loaded rows: "
        f"{df.height:,}"
    )

    # --------------------------------------------------------
    # CANONICAL NAME
    # --------------------------------------------------------

    df = df.with_columns(
        canonical_name(
            pl.col("business_name")
        ).alias("name_norm")
    )

    # --------------------------------------------------------
    # BASIC LENGTH FEATURES
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("business_name")
        .str.len_chars()
        .cast(pl.Int32)
        .alias("name_raw_length"),

        pl.col("business_name")
        .str.len_bytes()
        .cast(pl.Int32)
        .alias(
            "name_raw_byte_length"
        ),

        pl.col("name_norm")
        .str.len_chars()
        .cast(pl.Int32)
        .alias("name_length"),

        pl.col("name_norm")
        .str.len_bytes()
        .cast(pl.Int32)
        .alias("name_byte_length"),

        (
            pl.col("name_norm") == ""
        )
        .cast(pl.Int8)
        .alias("name_missing"),

        (
            pl.col("name_norm") != ""
        )
        .cast(pl.Int8)
        .alias("name_present"),

        (
            pl.col("name_norm")
            .str.len_chars()
            <= 3
        )
        .cast(pl.Int8)
        .alias("name_very_short"),

        (
            pl.col("name_norm")
            .str.len_chars()
            <= 5
        )
        .cast(pl.Int8)
        .alias("name_short"),
    )

    # --------------------------------------------------------
    # TOKEN LIST
    # --------------------------------------------------------

    df = df.with_columns(

        pl.when(
            pl.col("name_norm") == ""
        )
        .then(
            pl.lit(
                []
            ).cast(
                pl.List(pl.Utf8)
            )
        )
        .otherwise(
            pl.col("name_norm")
            .str.split(" ")
        )
        .alias("_name_tokens")
    )

    # --------------------------------------------------------
    # TOKEN COUNT
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("_name_tokens")
        .list.len()
        .cast(pl.Int16)
        .alias("name_token_count")
    )

    # --------------------------------------------------------
    # FIRST / LAST TOKEN
    #
    # Safe: list.first/list.last do not
    # throw on short lists.
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("_name_tokens")
        .list.first()
        .fill_null("")
        .alias("name_first_token"),

        pl.col("_name_tokens")
        .list.last()
        .fill_null("")
        .alias("name_last_token"),
    )

    # --------------------------------------------------------
    # FIRST TWO TOKENS
    #
    # SAFE:
    # slice never asks for a fixed
    # positional element.
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("_name_tokens")
        .list.slice(
            offset=0,
            length=2,
        )
        .list.join(" ")
        .fill_null("")
        .alias("name_first2_tokens"),

        # Last two tokens:
        # negative slice starts from the end.
        pl.col("_name_tokens")
        .list.slice(
            offset=-2,
            length=2,
        )
        .list.join(" ")
        .fill_null("")
        .alias("name_last2_tokens"),
    )

    # --------------------------------------------------------
    # TOKEN UNIQUENESS
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("_name_tokens")
        .list.n_unique()
        .cast(pl.Int16)
        .alias(
            "name_unique_token_count"
        ),

        (
            pl.col("_name_tokens")
            .list.n_unique()
            ==
            pl.col("_name_tokens")
            .list.len()
        )
        .cast(pl.Int8)
        .alias(
            "name_all_tokens_unique"
        ),
    )

    # --------------------------------------------------------
    # INITIALS
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("_name_tokens")
        .list.eval(
            pl.element()
            .str.slice(0, 1)
        )
        .list.join("")
        .fill_null("")
        .alias("name_initials"),

        pl.col("name_first_token")
        .str.slice(0, 1)
        .fill_null("")
        .alias("name_first_initial"),

        pl.col("name_last_token")
        .str.slice(0, 1)
        .fill_null("")
        .alias("name_last_initial"),
    )

    # --------------------------------------------------------
    # STRUCTURAL SIGNATURES
    # --------------------------------------------------------

    df = df.with_columns(

        pl.concat_str(
            [
                pl.col("name_first_token"),
                pl.col("name_last_token"),
            ],
            separator="|",
        )
        .alias(
            "name_first_last_signature"
        ),

        pl.concat_str(
            [
                pl.col("name_first_initial"),
                pl.col("name_last_initial"),
            ],
            separator="|",
        )
        .alias(
            "name_initial_signature"
        ),
    )

    # --------------------------------------------------------
    # CHARACTER COMPOSITION
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("name_norm")
        .str.count_matches(
            r"\p{L}"
        )
        .cast(pl.Int32)
        .alias("name_alpha_count"),

        pl.col("name_norm")
        .str.count_matches(
            r"\p{N}"
        )
        .cast(pl.Int32)
        .alias("name_digit_count"),

        pl.col("business_name")
        .str.count_matches(
            r"\s"
        )
        .cast(pl.Int32)
        .alias("name_space_count"),

        pl.col("business_name")
        .str.count_matches(
            r"[^\p{L}\p{N}\s]"
        )
        .cast(pl.Int32)
        .alias("name_symbol_count"),

        pl.col("name_norm")
        .str.count_matches(
            r"[aeiou]"
        )
        .cast(pl.Int32)
        .alias("name_vowel_count"),

        pl.col("name_norm")
        .str.count_matches(
            r"[bcdfghjklmnpqrstvwxyz]"
        )
        .cast(pl.Int32)
        .alias("name_consonant_count"),
    )

    # --------------------------------------------------------
    # ASCII / UNICODE
    # --------------------------------------------------------

    df = df.with_columns(

        (
            pl.col("name_norm")
            .str.len_bytes()
            ==
            pl.col("name_norm")
            .str.len_chars()
        )
        .cast(pl.Int8)
        .alias("name_ascii_only"),

        (
            pl.col("name_norm")
            .str.len_bytes()
            >
            pl.col("name_norm")
            .str.len_chars()
        )
        .cast(pl.Int8)
        .alias("name_non_ascii"),

        (
            pl.col("name_norm")
            .str.contains(
                r"\p{N}"
            )
        )
        .cast(pl.Int8)
        .alias("name_contains_digit"),

        (
            pl.col("name_norm")
            .str.contains(
                r"^\p{N}+$"
            )
            &
            (
                pl.col("name_norm") != ""
            )
        )
        .cast(pl.Int8)
        .alias("name_all_digits"),

        (
            pl.col("_name_tokens")
            .list.len()
            == 1
        )
        .cast(pl.Int8)
        .alias(
            "name_single_token"
        ),
    )

    # --------------------------------------------------------
    # CHARACTER DIVERSITY
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("name_norm")
        .str.extract_all(r".")
        .list.n_unique()
        .cast(pl.Int32)
        .alias(
            "name_unique_char_count"
        ),
    )

    df = df.with_columns(

        (
            pl.col("name_length")
            -
            pl.col("name_unique_char_count")
        )
        .cast(pl.Int32)
        .alias(
            "name_repeated_char_excess"
        ),

        (
            pl.col("name_unique_char_count")
            <
            pl.col("name_length")
        )
        .cast(pl.Int8)
        .alias(
            "name_has_repeated_char"
        ),

        (
            pl.col("name_unique_token_count")
            <
            pl.col("name_token_count")
        )
        .cast(pl.Int8)
        .alias(
            "name_repeated_token"
        ),
    )

    # --------------------------------------------------------
    # NUMERIC SIGNATURE
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("name_norm")
        .str.replace_all(
            r"\D",
            "",
        )
        .alias(
            "name_numeric_signature"
        )
    )

    # --------------------------------------------------------
    # RATIOS
    # --------------------------------------------------------

    df = df.with_columns(

        (
            pl.col(
                "name_unique_token_count"
            )
            /
            pl.col(
                "name_token_count"
            )
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "name_token_uniqueness_ratio"
        ),

        (
            pl.col(
                "name_unique_char_count"
            )
            /
            pl.col(
                "name_length"
            )
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "name_char_diversity_ratio"
        ),

        (
            pl.col("name_alpha_count")
            /
            pl.col("name_length")
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "name_alpha_ratio"
        ),

        (
            pl.col("name_digit_count")
            /
            pl.col("name_length")
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "name_digit_ratio"
        ),
    )

    # --------------------------------------------------------
    # FINAL OUTPUT COLUMNS
    # --------------------------------------------------------

    final_columns = [

        "entity_id",

        "business_name",
        "name_norm",

        "name_raw_length",
        "name_raw_byte_length",
        "name_length",
        "name_byte_length",

        "name_missing",
        "name_present",
        "name_very_short",
        "name_short",

        "name_token_count",
        "name_first_token",
        "name_last_token",
        "name_first2_tokens",
        "name_last2_tokens",

        "name_unique_token_count",
        "name_all_tokens_unique",

        "name_initials",
        "name_first_initial",
        "name_last_initial",

        "name_first_last_signature",
        "name_initial_signature",

        "name_alpha_count",
        "name_digit_count",
        "name_space_count",
        "name_symbol_count",

        "name_vowel_count",
        "name_consonant_count",

        "name_ascii_only",
        "name_non_ascii",

        "name_contains_digit",
        "name_all_digits",
        "name_single_token",

        "name_unique_char_count",
        "name_repeated_char_excess",
        "name_has_repeated_char",
        "name_repeated_token",

        "name_numeric_signature",

        "name_token_uniqueness_ratio",
        "name_char_diversity_ratio",
        "name_alpha_ratio",
        "name_digit_ratio",
    ]

    out = df.select(
        final_columns
    )

    # --------------------------------------------------------
    # ENTITY ID SAFETY
    # --------------------------------------------------------

    duplicate_count = (
        out
        .group_by("entity_id")
        .len()
        .filter(
            pl.col("len") > 1
        )
        .height
    )

    if duplicate_count != 0:
        raise RuntimeError(
            f"{split}/{source}: "
            f"{duplicate_count:,} duplicated entity IDs."
        )

    empty_ids = (
        out
        .filter(
            pl.col("entity_id") == ""
        )
        .height
    )

    if empty_ids != 0:
        raise RuntimeError(
            f"{split}/{source}: "
            f"{empty_ids:,} empty entity IDs."
        )

    # --------------------------------------------------------
    # WRITE
    # --------------------------------------------------------

    output_path = (
        OUT_DIR
        / f"{split}_{source}"
        "_name_representation.parquet"
    )

    out.write_parquet(
        output_path,
        compression="zstd",
    )

    print(
        f"Saved: {output_path}"
    )

    print(
        f"Rows: {out.height:,}"
    )

    print(
        f"Columns: {len(out.columns)}"
    )

    # --------------------------------------------------------
    # DIAGNOSTICS
    # --------------------------------------------------------

    diagnostics = (
        out
        .select(
            [
                pl.len()
                .alias("rows"),

                pl.col("name_missing")
                .sum()
                .alias(
                    "missing_name"
                ),

                pl.col("name_non_ascii")
                .sum()
                .alias(
                    "non_ascii_name"
                ),

                pl.col(
                    "name_contains_digit"
                )
                .sum()
                .alias(
                    "names_with_digits"
                ),

                pl.col(
                    "name_single_token"
                )
                .sum()
                .alias(
                    "single_token_names"
                ),

                pl.col(
                    "name_very_short"
                )
                .sum()
                .alias(
                    "very_short_names"
                ),

                pl.col(
                    "name_token_count"
                )
                .mean()
                .alias(
                    "mean_token_count"
                ),

                pl.col(
                    "name_length"
                )
                .mean()
                .alias(
                    "mean_name_length"
                ),
            ]
        )
        .row(
            0,
            named=True,
        )
    )

    print("\nDiagnostics:")

    for key, value in diagnostics.items():
        print(
            f"  {key}: {value}"
        )

    del df
    del out

    gc.collect()

    return {
        "path": str(output_path),
        "rows": int(
            diagnostics["rows"]
        ),
        "columns": len(final_columns),
        "diagnostics": {
            key: (
                float(value)
                if isinstance(
                    value,
                    float,
                )
                else int(value)
                if isinstance(
                    value,
                    int,
                )
                else value
            )
            for key, value
            in diagnostics.items()
        },
    }


# ------------------------------------------------------------
# 5. BUILD ALL SIX TABLES
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("BUILDING ALL NAME FEATURE TABLES")
print("-" * 80)

results = {}

for split, directory in [
    ("train", TRAIN_DIR),
    ("test", TEST_DIR),
]:

    for source in [
        "source1",
        "source2",
        "source3",
    ]:

        path = resolve_source_file(
            directory,
            split,
            source,
        )

        results[
            f"{split}_{source}"
        ] = build_name_table(
            split=split,
            source=source,
            path=path,
        )


# ------------------------------------------------------------
# 6. MANIFEST
# ------------------------------------------------------------

feature_names = [

    "name_raw_length",
    "name_raw_byte_length",
    "name_length",
    "name_byte_length",

    "name_missing",
    "name_present",
    "name_very_short",
    "name_short",

    "name_token_count",
    "name_first_token",
    "name_last_token",
    "name_first2_tokens",
    "name_last2_tokens",

    "name_unique_token_count",
    "name_all_tokens_unique",

    "name_initials",
    "name_first_initial",
    "name_last_initial",

    "name_first_last_signature",
    "name_initial_signature",

    "name_alpha_count",
    "name_digit_count",
    "name_space_count",
    "name_symbol_count",

    "name_vowel_count",
    "name_consonant_count",

    "name_ascii_only",
    "name_non_ascii",

    "name_contains_digit",
    "name_all_digits",
    "name_single_token",

    "name_unique_char_count",
    "name_repeated_char_excess",
    "name_has_repeated_char",
    "name_repeated_token",

    "name_numeric_signature",

    "name_token_uniqueness_ratio",
    "name_char_diversity_ratio",
    "name_alpha_ratio",
    "name_digit_ratio",
]

manifest = {

    "cell": "71B_v2",

    "purpose":
        "expanded_name_representation_features",

    "model_training": False,

    "ground_truth_used": False,

    "predictions_generated": False,

    "implementation_fixes": [
        "no positional list.get indexing",
        "safe list slicing",
        "no regex backreferences",
        "repeated-token derived from token uniqueness",
        "repeated-character derived from character diversity",
    ],

    "canonical_normalization": {
        "lowercase": True,
        "ampersand_to_and": True,
        "punctuation_removed": True,
        "whitespace_collapsed": True,
        "unicode_preserved": True,
    },

    "feature_names":
        feature_names,

    "outputs":
        results,
}

manifest_path = (
    OUT_DIR
    / "manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 7. FINAL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 71B v2 COMPLETE")
print("=" * 80)

for key, info in results.items():

    print(
        f"{key:18s} "
        f"{info['rows']:,} rows | "
        f"{info['columns']} columns"
    )

print(
    f"\nManifest:\n"
    f"{manifest_path}"
)

print("\n" + "-" * 80)
print("GUARDRAILS")
print("-" * 80)

print("✅ No model trained")
print("✅ No model loaded")
print("✅ No predictions generated")
print("✅ No ground truth used")
print("✅ Entity-level name features only")
print("✅ Train/test kept separate")
print("✅ Entity IDs checked for duplicates")
print("✅ No unsafe list indexing")
print("✅ No regex backreferences")

print("\nFEATURE ENGINEERING CONTINUES.")

AMLC 2026 — CELL 71B v2
FEATURE ENGINEERING ONLY — EXPANDED NAME FEATURES
ROOT      : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1
DATASET   : /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL
OUTPUT    : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/name_representation_v2

--------------------------------------------------------------------------------
BUILDING ALL NAME FEATURE TABLES
--------------------------------------------------------------------------------

BUILDING TRAIN / SOURCE1
Input: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_source1.tsv
Columns: ['entity_id', 'business_name', 'business_address', 'country']
Resolved entity column: entity_id
Resolved name column  : business_name
Loaded rows: 2,206,821
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/name_representation_v2/train_source1_name_representation.parquet
Rows: 2,206,821
Columns: 43

Diagnostic

In [48]:
# ============================================================
# AMLC 2026 — CELL 71C v2
# FEATURE ENGINEERING ONLY
# EXPANDED ADDRESS REPRESENTATION FEATURES
#
# NO MODEL
# NO TRAINING
# NO GROUND TRUTH
# NO PREDICTIONS
#
# v2 FIX:
#   - Removed unsupported regex look-around.
#   - Postal-like numeric features are derived from the
#     already-extracted complete numeric runs.
# ============================================================

from pathlib import Path
import json
import gc
import polars as pl

print("=" * 80)
print("AMLC 2026 — CELL 71C v2")
print("FEATURE ENGINEERING ONLY — EXPANDED ADDRESS FEATURES")
print("=" * 80)


# ------------------------------------------------------------
# 0. PATHS
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/"
    "amlc-2026-final-workspace/"
    "AMLC2026_KAGGLE_FINAL/"
)

TRAIN_DIR = DATASET_ROOT / "dataset" / "train"
TEST_DIR = DATASET_ROOT / "dataset" / "test"

OUT_DIR = (
    ROOT
    / "features"
    / "address_representation_v2"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"ROOT      : {ROOT}")
print(f"DATASET   : {DATASET_ROOT}")
print(f"OUTPUT    : {OUT_DIR}")

if not ROOT.exists():
    raise RuntimeError(
        f"Feature lake missing: {ROOT}"
    )

if not TRAIN_DIR.exists():
    raise RuntimeError(
        f"Train directory missing: {TRAIN_DIR}"
    )

if not TEST_DIR.exists():
    raise RuntimeError(
        f"Test directory missing: {TEST_DIR}"
    )


# ------------------------------------------------------------
# 1. FILE RESOLUTION
# ------------------------------------------------------------

def resolve_source_file(
    directory,
    split,
    source,
):
    exact = (
        directory
        / f"{split}_{source}.tsv"
    )

    if exact.exists():
        return exact

    source_number = source.replace(
        "source",
        "",
    )

    matches = []

    for path in sorted(
        directory.glob("*.tsv")
    ):
        name = path.name.lower()

        if split.lower() not in name:
            continue

        if (
            f"source{source_number}" in name
            or f"s{source_number}" in name
        ):
            matches.append(path)

    if matches:
        return matches[0]

    raise RuntimeError(
        f"Could not resolve {split}/{source} "
        f"under {directory}"
    )


# ------------------------------------------------------------
# 2. COLUMN RESOLUTION
# ------------------------------------------------------------

def normalize_column_name(c):
    return (
        c
        .strip()
        .lower()
        .replace(" ", "_")
    )


def resolve_column(
    columns,
    candidates,
):
    mapping = {
        normalize_column_name(c): c
        for c in columns
    }

    for candidate in candidates:

        key = normalize_column_name(
            candidate
        )

        if key in mapping:
            return mapping[key]

    return None


def inspect_schema(path):

    return (
        pl.scan_csv(
            str(path),
            separator="\t",
            infer_schema_length=2000,
            ignore_errors=False,
        )
        .collect_schema()
        .names()
    )


# ------------------------------------------------------------
# 3. CANONICAL ADDRESS NORMALIZATION
# ------------------------------------------------------------

def canonical_address(expr):
    """
    Established canonical string normalization family:

        lowercase
        & -> and
        punctuation/symbols -> spaces
        whitespace collapsed
        Unicode preserved
    """

    return (
        expr
        .cast(
            pl.Utf8,
            strict=False,
        )
        .fill_null("")
        .str.strip_chars()
        .str.to_lowercase()
        .str.replace_all(
            "&",
            " and ",
        )
        .str.replace_all(
            r"[^\p{L}\p{N}\s]",
            " ",
        )
        .str.replace_all(
            r"\s+",
            " ",
        )
        .str.strip_chars()
    )


# ------------------------------------------------------------
# 4. BUILD ONE ADDRESS TABLE
# ------------------------------------------------------------

def build_address_table(
    split,
    source,
    path,
):

    print("\n" + "=" * 72)
    print(
        f"BUILDING {split.upper()} / "
        f"{source.upper()}"
    )
    print("=" * 72)

    print(f"Input: {path}")

    schema = inspect_schema(path)

    print(
        "Columns:",
        schema,
    )

    entity_col = resolve_column(
        schema,
        [
            "entity_id",
            "source1_entity_id",
            "source2_entity_id",
            "source3_entity_id",
            "s1_entity_id",
            "s2_entity_id",
            "s3_entity_id",
        ],
    )

    address_col = resolve_column(
        schema,
        [
            "business_address",
            "address",
        ],
    )

    if entity_col is None:
        raise RuntimeError(
            f"Could not resolve entity ID for "
            f"{split}/{source}.\n"
            f"Schema: {schema}"
        )

    if address_col is None:
        raise RuntimeError(
            f"Could not resolve business address for "
            f"{split}/{source}.\n"
            f"Schema: {schema}"
        )

    print(
        f"Resolved entity column : "
        f"{entity_col}"
    )

    print(
        f"Resolved address column: "
        f"{address_col}"
    )

    # --------------------------------------------------------
    # LOAD
    # --------------------------------------------------------

    df = (
        pl.scan_csv(
            str(path),
            separator="\t",
            infer_schema_length=5000,
            ignore_errors=False,
        )
        .select(
            [
                pl.col(entity_col)
                .cast(
                    pl.Utf8,
                    strict=False,
                )
                .fill_null("")
                .str.strip_chars()
                .alias("entity_id"),

                pl.col(address_col)
                .cast(
                    pl.Utf8,
                    strict=False,
                )
                .fill_null("")
                .alias("business_address"),
            ]
        )
        .collect()
    )

    print(
        f"Loaded rows: "
        f"{df.height:,}"
    )

    # --------------------------------------------------------
    # CANONICAL ADDRESS
    # --------------------------------------------------------

    df = df.with_columns(
        canonical_address(
            pl.col("business_address")
        ).alias("address_norm")
    )

    # --------------------------------------------------------
    # BASIC LENGTH FEATURES
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("business_address")
        .str.len_chars()
        .cast(pl.Int32)
        .alias("address_raw_length"),

        pl.col("business_address")
        .str.len_bytes()
        .cast(pl.Int32)
        .alias(
            "address_raw_byte_length"
        ),

        pl.col("address_norm")
        .str.len_chars()
        .cast(pl.Int32)
        .alias("address_length"),

        pl.col("address_norm")
        .str.len_bytes()
        .cast(pl.Int32)
        .alias(
            "address_byte_length"
        ),

        (
            pl.col("address_norm") == ""
        )
        .cast(pl.Int8)
        .alias("address_missing"),

        (
            pl.col("address_norm") != ""
        )
        .cast(pl.Int8)
        .alias("address_present"),

        (
            pl.col("address_norm")
            .str.len_chars()
            <= 10
        )
        .cast(pl.Int8)
        .alias(
            "address_very_short"
        ),

        (
            pl.col("address_norm")
            .str.len_chars()
            <= 20
        )
        .cast(pl.Int8)
        .alias(
            "address_short"
        ),
    )

    # --------------------------------------------------------
    # TOKEN LIST
    # --------------------------------------------------------

    df = df.with_columns(

        pl.when(
            pl.col("address_norm") == ""
        )
        .then(
            pl.lit(
                []
            ).cast(
                pl.List(pl.Utf8)
            )
        )
        .otherwise(
            pl.col("address_norm")
            .str.split(" ")
        )
        .alias("_address_tokens")
    )

    # --------------------------------------------------------
    # TOKEN COUNT
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("_address_tokens")
        .list.len()
        .cast(pl.Int16)
        .alias(
            "address_token_count"
        )
    )

    # --------------------------------------------------------
    # FIRST / LAST TOKENS
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("_address_tokens")
        .list.first()
        .fill_null("")
        .alias(
            "address_first_token"
        ),

        pl.col("_address_tokens")
        .list.last()
        .fill_null("")
        .alias(
            "address_last_token"
        ),
    )

    # --------------------------------------------------------
    # FIRST / LAST TWO TOKENS
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("_address_tokens")
        .list.slice(
            offset=0,
            length=2,
        )
        .list.join(" ")
        .fill_null("")
        .alias(
            "address_first2_tokens"
        ),

        pl.col("_address_tokens")
        .list.slice(
            offset=-2,
            length=2,
        )
        .list.join(" ")
        .fill_null("")
        .alias(
            "address_last2_tokens"
        ),
    )

    # --------------------------------------------------------
    # TOKEN UNIQUENESS
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("_address_tokens")
        .list.n_unique()
        .cast(pl.Int16)
        .alias(
            "address_unique_token_count"
        ),

        (
            pl.col("_address_tokens")
            .list.n_unique()
            ==
            pl.col("_address_tokens")
            .list.len()
        )
        .cast(pl.Int8)
        .alias(
            "address_all_tokens_unique"
        ),
    )

    # --------------------------------------------------------
    # TOKEN TYPES
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("_address_tokens")
        .list.eval(
            pl.element()
            .str.contains(
                r"\p{N}"
            )
        )
        .list.sum()
        .cast(pl.Int16)
        .alias(
            "address_tokens_with_digits"
        ),

        pl.col("_address_tokens")
        .list.eval(
            pl.element()
            .str.contains(
                r"\p{L}"
            )
        )
        .list.sum()
        .cast(pl.Int16)
        .alias(
            "address_tokens_with_letters"
        ),

        pl.col("_address_tokens")
        .list.eval(
            (
                pl.element()
                .str.contains(
                    r"\p{L}"
                )
            )
            &
            (
                pl.element()
                .str.contains(
                    r"\p{N}"
                )
            )
        )
        .list.sum()
        .cast(pl.Int16)
        .alias(
            "address_alphanumeric_token_count"
        ),
    )

    # --------------------------------------------------------
    # NUMERIC TOKEN EXTRACTION
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("address_norm")
        .str.extract_all(
            r"\d+"
        )
        .alias(
            "_address_numeric_tokens"
        )
    )

    df = df.with_columns(

        pl.col(
            "_address_numeric_tokens"
        )
        .list.len()
        .cast(pl.Int16)
        .alias(
            "address_numeric_token_count"
        ),

        pl.col(
            "_address_numeric_tokens"
        )
        .list.n_unique()
        .cast(pl.Int16)
        .alias(
            "address_unique_numeric_count"
        ),

        pl.col(
            "_address_numeric_tokens"
        )
        .list.first()
        .fill_null("")
        .alias(
            "address_first_numeric_token"
        ),

        pl.col(
            "_address_numeric_tokens"
        )
        .list.last()
        .fill_null("")
        .alias(
            "address_last_numeric_token"
        ),

        pl.col(
            "_address_numeric_tokens"
        )
        .list.join("|")
        .fill_null("")
        .alias(
            "address_numeric_signature"
        ),
    )

    # --------------------------------------------------------
    # NUMERIC TOKEN LENGTH FEATURES
    #
    # IMPORTANT:
    # Do NOT use regex look-around here.
    #
    # _address_numeric_tokens already contains complete
    # digit runs, so their character lengths are exactly
    # what we need.
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col(
            "_address_numeric_tokens"
        )
        .list.eval(
            (
                pl.element()
                .str.len_chars()
                == 4
            )
        )
        .list.sum()
        .cast(pl.Int16)
        .alias(
            "address_4digit_token_count"
        ),

        pl.col(
            "_address_numeric_tokens"
        )
        .list.eval(
            (
                pl.element()
                .str.len_chars()
                == 5
            )
        )
        .list.sum()
        .cast(pl.Int16)
        .alias(
            "address_5digit_token_count"
        ),

        pl.col(
            "_address_numeric_tokens"
        )
        .list.eval(
            (
                pl.element()
                .str.len_chars()
                == 6
            )
        )
        .list.sum()
        .cast(pl.Int16)
        .alias(
            "address_6digit_token_count"
        ),

        pl.col(
            "_address_numeric_tokens"
        )
        .list.eval(
            (
                pl.element()
                .str.len_chars()
                >= 4
            )
            &
            (
                pl.element()
                .str.len_chars()
                <= 6
            )
        )
        .list.sum()
        .cast(pl.Int16)
        .alias(
            "address_4to6digit_token_count"
        ),
    )

    df = df.with_columns(

        (
            pl.col(
                "address_4digit_token_count"
            )
            > 0
        )
        .cast(pl.Int8)
        .alias(
            "address_has_4digit_code"
        ),

        (
            pl.col(
                "address_5digit_token_count"
            )
            > 0
        )
        .cast(pl.Int8)
        .alias(
            "address_has_5digit_code"
        ),

        (
            pl.col(
                "address_6digit_token_count"
            )
            > 0
        )
        .cast(pl.Int8)
        .alias(
            "address_has_6digit_code"
        ),

        (
            pl.col(
                "address_4to6digit_token_count"
            )
            > 0
        )
        .cast(pl.Int8)
        .alias(
            "address_has_4to6digit_code"
        ),
    )

    # --------------------------------------------------------
    # NUMERIC TOKEN UNIQUENESS
    # --------------------------------------------------------

    df = df.with_columns(

        (
            pl.col(
                "address_unique_numeric_count"
            )
            <
            pl.col(
                "address_numeric_token_count"
            )
        )
        .cast(pl.Int8)
        .alias(
            "address_repeated_numeric_token"
        ),

        (
            pl.col(
                "address_unique_numeric_count"
            )
            /
            pl.col(
                "address_numeric_token_count"
            )
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "address_numeric_uniqueness_ratio"
        ),
    )

    # --------------------------------------------------------
    # LEADING HOUSE-NUMBER STYLE FEATURES
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("address_norm")
        .str.extract(
            r"^\s*(\d+[a-z]?)",
            1,
        )
        .fill_null("")
        .alias(
            "address_leading_number"
        ),

        (
            pl.col("address_norm")
            .str.contains(
                r"^\s*\d+"
            )
        )
        .cast(pl.Int8)
        .alias(
            "address_starts_with_number"
        ),

        (
            pl.col("address_norm")
            .str.contains(
                r"\d$"
            )
        )
        .cast(pl.Int8)
        .alias(
            "address_ends_with_number"
        ),
    )

    # --------------------------------------------------------
    # CHARACTER COMPOSITION
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("address_norm")
        .str.count_matches(
            r"\p{L}"
        )
        .cast(pl.Int32)
        .alias(
            "address_alpha_count"
        ),

        pl.col("address_norm")
        .str.count_matches(
            r"\p{N}"
        )
        .cast(pl.Int32)
        .alias(
            "address_digit_count"
        ),

        pl.col("business_address")
        .str.count_matches(
            r"\s"
        )
        .cast(pl.Int32)
        .alias(
            "address_space_count"
        ),

        pl.col("business_address")
        .str.count_matches(
            r"[^\p{L}\p{N}\s]"
        )
        .cast(pl.Int32)
        .alias(
            "address_symbol_count"
        ),

        pl.col("business_address")
        .str.count_matches(
            ","
        )
        .cast(pl.Int32)
        .alias(
            "address_comma_count"
        ),

        pl.col("business_address")
        .str.count_matches(
            "/"
        )
        .cast(pl.Int32)
        .alias(
            "address_slash_count"
        ),

        pl.col("business_address")
        .str.count_matches(
            r"-"
        )
        .cast(pl.Int32)
        .alias(
            "address_hyphen_count"
        ),
    )

    # --------------------------------------------------------
    # ASCII / UNICODE
    # --------------------------------------------------------

    df = df.with_columns(

        (
            pl.col("address_norm")
            .str.len_bytes()
            ==
            pl.col("address_norm")
            .str.len_chars()
        )
        .cast(pl.Int8)
        .alias(
            "address_ascii_only"
        ),

        (
            pl.col("address_norm")
            .str.len_bytes()
            >
            pl.col("address_norm")
            .str.len_chars()
        )
        .cast(pl.Int8)
        .alias(
            "address_non_ascii"
        ),

        (
            pl.col("address_norm")
            .str.contains(
                r"\p{N}"
            )
        )
        .cast(pl.Int8)
        .alias(
            "address_contains_digit"
        ),

        (
            pl.col("address_norm")
            .str.contains(
                r"^\p{N}+$"
            )
            &
            (
                pl.col("address_norm") != ""
            )
        )
        .cast(pl.Int8)
        .alias(
            "address_all_digits"
        ),

        (
            pl.col("_address_tokens")
            .list.len()
            == 1
        )
        .cast(pl.Int8)
        .alias(
            "address_single_token"
        ),
    )

    # --------------------------------------------------------
    # CHARACTER DIVERSITY
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("address_norm")
        .str.extract_all(
            r"."
        )
        .list.n_unique()
        .cast(pl.Int32)
        .alias(
            "address_unique_char_count"
        ),
    )

    df = df.with_columns(

        (
            pl.col("address_length")
            -
            pl.col(
                "address_unique_char_count"
            )
        )
        .cast(pl.Int32)
        .alias(
            "address_repeated_char_excess"
        ),

        (
            pl.col(
                "address_unique_char_count"
            )
            <
            pl.col("address_length")
        )
        .cast(pl.Int8)
        .alias(
            "address_has_repeated_char"
        ),
    )

    # --------------------------------------------------------
    # GLOBAL NUMERIC SIGNATURE
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("address_norm")
        .str.replace_all(
            r"\D",
            "",
        )
        .alias(
            "address_numeric_all_digits"
        )
    )

    # --------------------------------------------------------
    # RATIOS
    # --------------------------------------------------------

    df = df.with_columns(

        (
            pl.col(
                "address_unique_token_count"
            )
            /
            pl.col(
                "address_token_count"
            )
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "address_token_uniqueness_ratio"
        ),

        (
            pl.col(
                "address_unique_char_count"
            )
            /
            pl.col(
                "address_length"
            )
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "address_char_diversity_ratio"
        ),

        (
            pl.col(
                "address_alpha_count"
            )
            /
            pl.col(
                "address_length"
            )
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "address_alpha_ratio"
        ),

        (
            pl.col(
                "address_digit_count"
            )
            /
            pl.col(
                "address_length"
            )
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "address_digit_ratio"
        ),

        (
            pl.col(
                "address_numeric_token_count"
            )
            /
            pl.col(
                "address_token_count"
            )
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "address_numeric_token_ratio"
        ),
    )

    # --------------------------------------------------------
    # COMMON ADDRESS MARKERS
    # --------------------------------------------------------

    df = df.with_columns(

        pl.col("address_norm")
        .str.contains(
            r"\b(apt|apartment|suite|ste|unit)\b"
        )
        .cast(pl.Int8)
        .alias(
            "address_has_unit_marker"
        ),

        pl.col("address_norm")
        .str.contains(
            r"\b(floor|fl)\b"
        )
        .cast(pl.Int8)
        .alias(
            "address_has_floor_marker"
        ),

        pl.col("address_norm")
        .str.contains(
            r"\b(block|blk)\b"
        )
        .cast(pl.Int8)
        .alias(
            "address_has_block_marker"
        ),

        pl.col("address_norm")
        .str.contains(
            r"\b(plot|plt)\b"
        )
        .cast(pl.Int8)
        .alias(
            "address_has_plot_marker"
        ),

        pl.col("address_norm")
        .str.contains(
            r"\b(building|bldg)\b"
        )
        .cast(pl.Int8)
        .alias(
            "address_has_building_marker"
        ),

        pl.col("address_norm")
        .str.contains(
            r"\b(road|rd|street|st|avenue|ave|lane|ln|drive|dr|boulevard|blvd|highway|hwy)\b"
        )
        .cast(pl.Int8)
        .alias(
            "address_has_street_marker"
        ),
    )

    # --------------------------------------------------------
    # FINAL COLUMNS
    # --------------------------------------------------------

    final_columns = [

        "entity_id",

        "business_address",
        "address_norm",

        "address_raw_length",
        "address_raw_byte_length",
        "address_length",
        "address_byte_length",

        "address_missing",
        "address_present",
        "address_very_short",
        "address_short",

        "address_token_count",
        "address_first_token",
        "address_last_token",
        "address_first2_tokens",
        "address_last2_tokens",

        "address_unique_token_count",
        "address_all_tokens_unique",

        "address_tokens_with_digits",
        "address_tokens_with_letters",
        "address_alphanumeric_token_count",

        "address_numeric_token_count",
        "address_unique_numeric_count",
        "address_first_numeric_token",
        "address_last_numeric_token",
        "address_numeric_signature",

        "address_repeated_numeric_token",
        "address_numeric_uniqueness_ratio",

        "address_leading_number",
        "address_starts_with_number",
        "address_ends_with_number",

        "address_4digit_token_count",
        "address_5digit_token_count",
        "address_6digit_token_count",
        "address_4to6digit_token_count",

        "address_has_4digit_code",
        "address_has_5digit_code",
        "address_has_6digit_code",
        "address_has_4to6digit_code",

        "address_alpha_count",
        "address_digit_count",
        "address_space_count",
        "address_symbol_count",
        "address_comma_count",
        "address_slash_count",
        "address_hyphen_count",

        "address_ascii_only",
        "address_non_ascii",
        "address_contains_digit",
        "address_all_digits",
        "address_single_token",

        "address_unique_char_count",
        "address_repeated_char_excess",
        "address_has_repeated_char",

        "address_numeric_all_digits",

        "address_token_uniqueness_ratio",
        "address_char_diversity_ratio",
        "address_alpha_ratio",
        "address_digit_ratio",
        "address_numeric_token_ratio",

        "address_has_unit_marker",
        "address_has_floor_marker",
        "address_has_block_marker",
        "address_has_plot_marker",
        "address_has_building_marker",
        "address_has_street_marker",
    ]

    out = df.select(
        final_columns
    )

    # --------------------------------------------------------
    # ENTITY SAFETY
    # --------------------------------------------------------

    duplicate_count = (
        out
        .group_by("entity_id")
        .len()
        .filter(
            pl.col("len") > 1
        )
        .height
    )

    if duplicate_count != 0:
        raise RuntimeError(
            f"{split}/{source}: "
            f"{duplicate_count:,} duplicated entity IDs."
        )

    empty_ids = (
        out
        .filter(
            pl.col("entity_id") == ""
        )
        .height
    )

    if empty_ids != 0:
        raise RuntimeError(
            f"{split}/{source}: "
            f"{empty_ids:,} empty entity IDs."
        )

    # --------------------------------------------------------
    # WRITE
    # --------------------------------------------------------

    output_path = (
        OUT_DIR
        / f"{split}_{source}"
        "_address_representation.parquet"
    )

    out.write_parquet(
        output_path,
        compression="zstd",
    )

    print(
        f"Saved: {output_path}"
    )

    print(
        f"Rows: {out.height:,}"
    )

    print(
        f"Columns: {len(out.columns)}"
    )

    # --------------------------------------------------------
    # DIAGNOSTICS
    # --------------------------------------------------------

    diagnostics = (
        out
        .select(
            [
                pl.len()
                .alias("rows"),

                pl.col("address_missing")
                .sum()
                .alias("missing_address"),

                pl.col("address_non_ascii")
                .sum()
                .alias("non_ascii_address"),

                pl.col(
                    "address_contains_digit"
                )
                .sum()
                .alias(
                    "addresses_with_digits"
                ),

                pl.col(
                    "address_numeric_token_count"
                )
                .mean()
                .alias(
                    "mean_numeric_token_count"
                ),

                pl.col(
                    "address_token_count"
                )
                .mean()
                .alias(
                    "mean_address_token_count"
                ),

                pl.col(
                    "address_length"
                )
                .mean()
                .alias(
                    "mean_address_length"
                ),

                pl.col(
                    "address_starts_with_number"
                )
                .sum()
                .alias(
                    "starts_with_number"
                ),

                pl.col(
                    "address_has_5digit_code"
                )
                .sum()
                .alias(
                    "has_5digit_code"
                ),

                pl.col(
                    "address_has_6digit_code"
                )
                .sum()
                .alias(
                    "has_6digit_code"
                ),
            ]
        )
        .row(
            0,
            named=True,
        )
    )

    print("\nDiagnostics:")

    for key, value in diagnostics.items():
        print(
            f"  {key}: {value}"
        )

    del df
    del out

    gc.collect()

    return {
        "path": str(output_path),
        "rows": int(
            diagnostics["rows"]
        ),
        "columns": len(final_columns),
        "diagnostics": {
            key: (
                float(value)
                if isinstance(
                    value,
                    float,
                )
                else int(value)
                if isinstance(
                    value,
                    int,
                )
                else value
            )
            for key, value
            in diagnostics.items()
        },
    }


# ------------------------------------------------------------
# 5. BUILD ALL SIX TABLES
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("BUILDING ALL ADDRESS FEATURE TABLES")
print("-" * 80)

results = {}

for split, directory in [
    ("train", TRAIN_DIR),
    ("test", TEST_DIR),
]:

    for source in [
        "source1",
        "source2",
        "source3",
    ]:

        path = resolve_source_file(
            directory,
            split,
            source,
        )

        results[
            f"{split}_{source}"
        ] = build_address_table(
            split=split,
            source=source,
            path=path,
        )


# ------------------------------------------------------------
# 6. MANIFEST
# ------------------------------------------------------------

feature_names = [

    "address_raw_length",
    "address_raw_byte_length",
    "address_length",
    "address_byte_length",

    "address_missing",
    "address_present",
    "address_very_short",
    "address_short",

    "address_token_count",
    "address_first_token",
    "address_last_token",
    "address_first2_tokens",
    "address_last2_tokens",

    "address_unique_token_count",
    "address_all_tokens_unique",

    "address_tokens_with_digits",
    "address_tokens_with_letters",
    "address_alphanumeric_token_count",

    "address_numeric_token_count",
    "address_unique_numeric_count",
    "address_first_numeric_token",
    "address_last_numeric_token",
    "address_numeric_signature",

    "address_repeated_numeric_token",
    "address_numeric_uniqueness_ratio",

    "address_leading_number",
    "address_starts_with_number",
    "address_ends_with_number",

    "address_4digit_token_count",
    "address_5digit_token_count",
    "address_6digit_token_count",
    "address_4to6digit_token_count",

    "address_has_4digit_code",
    "address_has_5digit_code",
    "address_has_6digit_code",
    "address_has_4to6digit_code",

    "address_alpha_count",
    "address_digit_count",
    "address_space_count",
    "address_symbol_count",
    "address_comma_count",
    "address_slash_count",
    "address_hyphen_count",

    "address_ascii_only",
    "address_non_ascii",
    "address_contains_digit",
    "address_all_digits",
    "address_single_token",

    "address_unique_char_count",
    "address_repeated_char_excess",
    "address_has_repeated_char",

    "address_numeric_all_digits",

    "address_token_uniqueness_ratio",
    "address_char_diversity_ratio",
    "address_alpha_ratio",
    "address_digit_ratio",
    "address_numeric_token_ratio",

    "address_has_unit_marker",
    "address_has_floor_marker",
    "address_has_block_marker",
    "address_has_plot_marker",
    "address_has_building_marker",
    "address_has_street_marker",
]

manifest = {

    "cell": "71C_v2",

    "purpose":
        "expanded_address_representation_features",

    "model_training": False,

    "ground_truth_used": False,

    "predictions_generated": False,

    "implementation_fixes": [
        "removed regex look-around",
        "postal-like code features derived from complete numeric runs",
        "no regex look-behind",
        "no regex look-ahead",
    ],

    "canonical_normalization": {
        "lowercase": True,
        "ampersand_to_and": True,
        "punctuation_removed": True,
        "whitespace_collapsed": True,
        "unicode_preserved": True,
    },

    "feature_names":
        feature_names,

    "outputs":
        results,
}

manifest_path = (
    OUT_DIR
    / "manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 7. FINAL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 71C v2 COMPLETE")
print("=" * 80)

for key, info in results.items():

    print(
        f"{key:18s} "
        f"{info['rows']:,} rows | "
        f"{info['columns']} columns"
    )

print(
    f"\nManifest:\n"
    f"{manifest_path}"
)

print("\n" + "-" * 80)
print("GUARDRAILS")
print("-" * 80)

print("✅ No model trained")
print("✅ No model loaded")
print("✅ No predictions generated")
print("✅ No ground truth used")
print("✅ Entity-level address features only")
print("✅ Train/test kept separate")
print("✅ Entity IDs checked for duplicates")
print("✅ No unsafe list indexing")
print("✅ No regex look-around")
print("✅ Numeric-address structure explicitly represented")
print("✅ Postal-like patterns treated as features, not country truth")

print("\nFEATURE ENGINEERING CONTINUES.")

AMLC 2026 — CELL 71C v2
FEATURE ENGINEERING ONLY — EXPANDED ADDRESS FEATURES
ROOT      : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1
DATASET   : /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL
OUTPUT    : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/address_representation_v2

--------------------------------------------------------------------------------
BUILDING ALL ADDRESS FEATURE TABLES
--------------------------------------------------------------------------------

BUILDING TRAIN / SOURCE1
Input: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_source1.tsv
Columns: ['entity_id', 'business_name', 'business_address', 'country']
Resolved entity column : entity_id
Resolved address column: business_address
Loaded rows: 2,206,821
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/address_representation_v2/train_source1_address_representation.parquet
Rows: 2,206,821
Col

In [53]:
# ============================================================
# AMLC 2026 — CELL 71D v2
# FEATURE ENGINEERING ONLY
# UNICODE / TRANSLITERATION REPRESENTATION FEATURES
#
# NO MODEL
# NO TRAINING
# NO GROUND TRUTH
# NO PREDICTIONS
#
# v2 FIX:
#   - Derives canonical name/address lengths locally.
#   - Does not depend on name_length/address_length columns
#     being present in the selected input schema.
#   - Keeps canonical Unicode representation untouched.
# ============================================================

from pathlib import Path
import json
import gc
import unicodedata

import polars as pl


print("=" * 80)
print("AMLC 2026 — CELL 71D v2")
print("FEATURE ENGINEERING ONLY — UNICODE / TRANSLITERATION")
print("=" * 80)


# ------------------------------------------------------------
# 0. PATHS
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

NAME_DIR = (
    ROOT
    / "features"
    / "name_representation_v2"
)

ADDRESS_DIR = (
    ROOT
    / "features"
    / "address_representation_v2"
)

OUT_DIR = (
    ROOT
    / "features"
    / "unicode_transliteration_v2"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"ROOT       : {ROOT}")
print(f"NAME DIR   : {NAME_DIR}")
print(f"ADDRESS DIR: {ADDRESS_DIR}")
print(f"OUTPUT     : {OUT_DIR}")


if not ROOT.exists():
    raise RuntimeError(
        f"Feature lake missing:\n{ROOT}"
    )

if not NAME_DIR.exists():
    raise RuntimeError(
        f"Name representation directory missing:\n"
        f"{NAME_DIR}"
    )

if not ADDRESS_DIR.exists():
    raise RuntimeError(
        f"Address representation directory missing:\n"
        f"{ADDRESS_DIR}"
    )


# ------------------------------------------------------------
# 1. TRANSLITERATION ENGINE
# ------------------------------------------------------------

try:
    from anyascii import anyascii

    TRANSLITERATION_ENGINE = "anyascii"

    print(
        "\nTransliteration engine: anyascii"
    )

except Exception as exc:

    TRANSLITERATION_ENGINE = (
        "unicodedata_nfkd_fallback"
    )

    print(
        "\n[WARN] anyascii unavailable."
    )
    print(
        "Using Unicode NFKD fallback."
    )
    print(
        f"Reason: {exc}"
    )

    def anyascii(value):
        if value is None:
            return ""

        text = str(value)

        normalized = unicodedata.normalize(
            "NFKD",
            text,
        )

        stripped = "".join(
            char
            for char in normalized
            if not unicodedata.combining(char)
        )

        return (
            stripped
            .encode(
                "ascii",
                errors="ignore",
            )
            .decode(
                "ascii",
                errors="ignore",
            )
        )


# ------------------------------------------------------------
# 2. TRANSLITERATED CANONICALIZATION
# ------------------------------------------------------------

def canonicalize_transliterated(
    value,
):
    """
    Auxiliary ASCII representation.

    Canonical Unicode strings are never modified.

    Operations:
        lowercase
        & -> and
        non-alphanumeric ASCII symbols -> spaces
        whitespace collapse
        trim
    """

    if value is None:
        return ""

    text = str(value)

    text = text.lower()

    text = text.replace(
        "&",
        " and ",
    )

    chars = []

    for char in text:

        if (
            char.isalnum()
            or char.isspace()
        ):
            chars.append(char)

        else:
            chars.append(" ")

    text = "".join(chars)

    return " ".join(
        text.split()
    )


# ------------------------------------------------------------
# 3. TRANSLITERATION HELPER
# ------------------------------------------------------------

def transliterate_column(
    df,
    source_column,
    target_column,
    non_ascii_flag_column,
):
    """
    Preserve ASCII rows exactly.
    Transliterate only rows marked non-ASCII.
    """

    base = (
        df
        .select(
            [
                "entity_id",
                source_column,
            ]
        )
        .with_columns(
            pl.col(source_column)
            .alias(target_column)
        )
    )

    changed_rows = (
        df
        .filter(
            pl.col(
                non_ascii_flag_column
            ) == 1
        )
        .select(
            [
                "entity_id",
                source_column,
            ]
        )
        .with_columns(
            pl.col(source_column)
            .map_elements(
                anyascii,
                return_dtype=pl.Utf8,
                skip_nulls=False,
            )
            .map_elements(
                canonicalize_transliterated,
                return_dtype=pl.Utf8,
                skip_nulls=False,
            )
            .alias(target_column)
        )
        .select(
            [
                "entity_id",
                target_column,
            ]
        )
    )

    if changed_rows.height == 0:
        return base

    # Replace only the affected entity rows.
    out = (
        base
        .join(
            changed_rows,
            on="entity_id",
            how="left",
            suffix="__translit",
        )
        .with_columns(
            pl.coalesce(
                [
                    pl.col(
                        f"{target_column}__translit"
                    ),
                    pl.col(target_column),
                ]
            )
            .alias(target_column)
        )
        .drop(
            f"{target_column}__translit"
        )
    )

    return out


# ------------------------------------------------------------
# 4. BUILD ONE SPLIT / SOURCE
# ------------------------------------------------------------

def build_unicode_table(
    split,
    source,
):

    print("\n" + "=" * 72)
    print(
        f"BUILDING {split.upper()} / "
        f"{source.upper()}"
    )
    print("=" * 72)

    name_path = (
        NAME_DIR
        / f"{split}_{source}"
        "_name_representation.parquet"
    )

    address_path = (
        ADDRESS_DIR
        / f"{split}_{source}"
        "_address_representation.parquet"
    )

    if not name_path.exists():
        raise RuntimeError(
            f"Name representation missing:\n"
            f"{name_path}"
        )

    if not address_path.exists():
        raise RuntimeError(
            f"Address representation missing:\n"
            f"{address_path}"
        )

    print(
        f"Name input   : "
        f"{name_path.name}"
    )

    print(
        f"Address input: "
        f"{address_path.name}"
    )

    # --------------------------------------------------------
    # LOAD ONLY REQUIRED INPUT COLUMNS
    #
    # DO NOT depend on name_length/address_length/etc.
    # --------------------------------------------------------

    name_df = (
        pl.scan_parquet(
            str(name_path)
        )
        .select(
            [
                "entity_id",
                "name_norm",
                "name_non_ascii",
            ]
        )
        .collect()
    )

    address_df = (
        pl.scan_parquet(
            str(address_path)
        )
        .select(
            [
                "entity_id",
                "address_norm",
                "address_non_ascii",
            ]
        )
        .collect()
    )

    print(
        f"Name rows   : "
        f"{name_df.height:,}"
    )

    print(
        f"Address rows: "
        f"{address_df.height:,}"
    )

    # --------------------------------------------------------
    # ENTITY SAFETY
    # --------------------------------------------------------

    name_duplicates = (
        name_df
        .group_by("entity_id")
        .len()
        .filter(
            pl.col("len") > 1
        )
        .height
    )

    address_duplicates = (
        address_df
        .group_by("entity_id")
        .len()
        .filter(
            pl.col("len") > 1
        )
        .height
    )

    if name_duplicates != 0:
        raise RuntimeError(
            f"{split}/{source}: "
            f"name table contains "
            f"{name_duplicates:,} duplicate entity IDs."
        )

    if address_duplicates != 0:
        raise RuntimeError(
            f"{split}/{source}: "
            f"address table contains "
            f"{address_duplicates:,} duplicate entity IDs."
        )

    # --------------------------------------------------------
    # TRANSLITERATE NAME
    # --------------------------------------------------------

    print(
        "\nTransliterating name representations..."
    )

    name_translit = transliterate_column(
        name_df,
        source_column="name_norm",
        target_column="name_ascii_translit",
        non_ascii_flag_column="name_non_ascii",
    )

    # --------------------------------------------------------
    # TRANSLITERATE ADDRESS
    # --------------------------------------------------------

    print(
        "Transliterating address representations..."
    )

    address_translit = transliterate_column(
        address_df,
        source_column="address_norm",
        target_column="address_ascii_translit",
        non_ascii_flag_column="address_non_ascii",
    )

    # --------------------------------------------------------
    # COMBINE
    # --------------------------------------------------------

    out = (
        name_translit
        .join(
            address_translit,
            on="entity_id",
            how="inner",
        )
    )

    if out.height != name_df.height:

        raise RuntimeError(
            f"{split}/{source}: "
            f"entity join changed row count. "
            f"Expected {name_df.height:,}; "
            f"got {out.height:,}."
        )

    # --------------------------------------------------------
    # DERIVE ORIGINAL LENGTHS LOCALLY
    # --------------------------------------------------------

    out = out.with_columns(

        pl.col("name_norm")
        .str.len_chars()
        .cast(pl.Int32)
        .alias(
            "_name_unicode_length"
        ),

        pl.col("address_norm")
        .str.len_chars()
        .cast(pl.Int32)
        .alias(
            "_address_unicode_length"
        ),

        pl.col("name_ascii_translit")
        .str.len_chars()
        .cast(pl.Int32)
        .alias(
            "name_translit_length"
        ),

        pl.col("address_ascii_translit")
        .str.len_chars()
        .cast(pl.Int32)
        .alias(
            "address_translit_length"
        ),
    )

    # --------------------------------------------------------
    # TOKEN / DIGIT FEATURES
    # --------------------------------------------------------

    out = out.with_columns(

        pl.when(
            pl.col(
                "name_ascii_translit"
            ) == ""
        )
        .then(0)
        .otherwise(
            pl.col(
                "name_ascii_translit"
            )
            .str.split(" ")
            .list.len()
        )
        .cast(pl.Int16)
        .alias(
            "name_translit_token_count"
        ),

        pl.when(
            pl.col(
                "address_ascii_translit"
            ) == ""
        )
        .then(0)
        .otherwise(
            pl.col(
                "address_ascii_translit"
            )
            .str.split(" ")
            .list.len()
        )
        .cast(pl.Int16)
        .alias(
            "address_translit_token_count"
        ),

        pl.col(
            "name_ascii_translit"
        )
        .str.count_matches(
            r"\d"
        )
        .cast(pl.Int32)
        .alias(
            "name_translit_digit_count"
        ),

        pl.col(
            "address_ascii_translit"
        )
        .str.count_matches(
            r"\d"
        )
        .cast(pl.Int32)
        .alias(
            "address_translit_digit_count"
        ),
    )

    # --------------------------------------------------------
    # UNICODE CHARACTER INFORMATION
    # --------------------------------------------------------

    out = out.with_columns(

        (
            pl.col("name_norm")
            .str.len_bytes()
            -
            pl.col("name_norm")
            .str.len_chars()
        )
        .cast(pl.Int32)
        .alias(
            "name_unicode_byte_excess"
        ),

        (
            pl.col("address_norm")
            .str.len_bytes()
            -
            pl.col("address_norm")
            .str.len_chars()
        )
        .cast(pl.Int32)
        .alias(
            "address_unicode_byte_excess"
        ),

        (
            pl.col(
                "name_ascii_translit"
            )
            !=
            pl.col("name_norm")
        )
        .cast(pl.Int8)
        .alias(
            "name_translit_changed"
        ),

        (
            pl.col(
                "address_ascii_translit"
            )
            !=
            pl.col("address_norm")
        )
        .cast(pl.Int8)
        .alias(
            "address_translit_changed"
        ),
    )

    # --------------------------------------------------------
    # LENGTH DELTAS / RATIOS
    #
    # Derive everything locally.
    # --------------------------------------------------------

    out = out.with_columns(

        (
            pl.col("name_translit_length")
            -
            pl.col("_name_unicode_length")
        )
        .cast(pl.Int32)
        .alias(
            "name_translit_length_delta"
        ),

        (
            pl.col("address_translit_length")
            -
            pl.col(
                "_address_unicode_length"
            )
        )
        .cast(pl.Int32)
        .alias(
            "address_translit_length_delta"
        ),

        (
            pl.col("name_translit_token_count")
            -
            pl.when(
                pl.col("name_norm") == ""
            )
            .then(0)
            .otherwise(
                pl.col("name_norm")
                .str.split(" ")
                .list.len()
            )
        )
        .cast(pl.Int16)
        .alias(
            "name_translit_token_delta"
        ),

        (
            pl.col(
                "address_translit_token_count"
            )
            -
            pl.when(
                pl.col("address_norm") == ""
            )
            .then(0)
            .otherwise(
                pl.col("address_norm")
                .str.split(" ")
                .list.len()
            )
        )
        .cast(pl.Int16)
        .alias(
            "address_translit_token_delta"
        ),

        (
            pl.col("name_translit_length")
            /
            pl.col("_name_unicode_length")
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "name_translit_length_ratio"
        ),

        (
            pl.col("address_translit_length")
            /
            pl.col(
                "_address_unicode_length"
            )
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "address_translit_length_ratio"
        ),
    )

    # --------------------------------------------------------
    # RESIDUAL NON-ASCII
    # --------------------------------------------------------

    out = out.with_columns(

        (
            pl.col(
                "name_ascii_translit"
            )
            .str.len_bytes()
            >
            pl.col(
                "name_ascii_translit"
            )
            .str.len_chars()
        )
        .cast(pl.Int8)
        .alias(
            "name_translit_residual_non_ascii"
        ),

        (
            pl.col(
                "address_ascii_translit"
            )
            .str.len_bytes()
            >
            pl.col(
                "address_ascii_translit"
            )
            .str.len_chars()
        )
        .cast(pl.Int8)
        .alias(
            "address_translit_residual_non_ascii"
        ),

        (
            pl.col(
                "name_ascii_translit"
            )
            .str.contains(r"\d")
        )
        .cast(pl.Int8)
        .alias(
            "name_translit_contains_digit"
        ),

        (
            pl.col(
                "address_ascii_translit"
            )
            .str.contains(r"\d")
        )
        .cast(pl.Int8)
        .alias(
            "address_translit_contains_digit"
        ),
    )

    # --------------------------------------------------------
    # FINAL OUTPUT
    # --------------------------------------------------------

    final_columns = [

        "entity_id",

        "name_ascii_translit",
        "name_translit_changed",
        "name_translit_length",
        "name_translit_token_count",
        "name_translit_digit_count",
        "name_translit_contains_digit",
        "name_translit_length_delta",
        "name_translit_token_delta",
        "name_translit_length_ratio",

        "address_ascii_translit",
        "address_translit_changed",
        "address_translit_length",
        "address_translit_token_count",
        "address_translit_digit_count",
        "address_translit_contains_digit",
        "address_translit_length_delta",
        "address_translit_token_delta",
        "address_translit_length_ratio",

        "name_unicode_byte_excess",
        "address_unicode_byte_excess",

        "name_translit_residual_non_ascii",
        "address_translit_residual_non_ascii",
    ]

    out = out.select(
        final_columns
    )

    # --------------------------------------------------------
    # FINAL SAFETY
    # --------------------------------------------------------

    duplicate_count = (
        out
        .group_by("entity_id")
        .len()
        .filter(
            pl.col("len") > 1
        )
        .height
    )

    if duplicate_count != 0:

        raise RuntimeError(
            f"{split}/{source}: "
            f"{duplicate_count:,} duplicate entity IDs "
            f"in final Unicode table."
        )

    empty_ids = (
        out
        .filter(
            pl.col("entity_id") == ""
        )
        .height
    )

    if empty_ids != 0:

        raise RuntimeError(
            f"{split}/{source}: "
            f"{empty_ids:,} empty entity IDs."
        )

    # --------------------------------------------------------
    # WRITE
    # --------------------------------------------------------

    output_path = (
        OUT_DIR
        / f"{split}_{source}"
        "_unicode_transliteration.parquet"
    )

    out.write_parquet(
        output_path,
        compression="zstd",
    )

    print(
        f"Saved: {output_path}"
    )

    print(
        f"Rows: {out.height:,}"
    )

    print(
        f"Columns: {len(out.columns)}"
    )

    # --------------------------------------------------------
    # DIAGNOSTICS
    # --------------------------------------------------------

    diagnostics = (
        out
        .select(
            [
                pl.len()
                .alias("rows"),

                pl.col(
                    "name_translit_changed"
                )
                .sum()
                .alias(
                    "name_translit_changed_rows"
                ),

                pl.col(
                    "address_translit_changed"
                )
                .sum()
                .alias(
                    "address_translit_changed_rows"
                ),

                pl.col(
                    "name_translit_residual_non_ascii"
                )
                .sum()
                .alias(
                    "name_translit_residual_non_ascii_rows"
                ),

                pl.col(
                    "address_translit_residual_non_ascii"
                )
                .sum()
                .alias(
                    "address_translit_residual_non_ascii_rows"
                ),

                pl.col(
                    "name_translit_length_ratio"
                )
                .mean()
                .alias(
                    "mean_name_translit_length_ratio"
                ),

                pl.col(
                    "address_translit_length_ratio"
                )
                .mean()
                .alias(
                    "mean_address_translit_length_ratio"
                ),
            ]
        )
        .row(
            0,
            named=True,
        )
    )

    print(
        "\nDiagnostics:"
    )

    for key, value in diagnostics.items():

        print(
            f"  {key}: {value}"
        )

    # --------------------------------------------------------
    # MEMORY
    # --------------------------------------------------------

    del name_df
    del address_df
    del name_translit
    del address_translit
    del out

    gc.collect()

    return {
        "path": str(output_path),
        "rows": int(
            diagnostics["rows"]
        ),
        "columns": len(final_columns),
        "diagnostics": {
            key: (
                float(value)
                if isinstance(
                    value,
                    float,
                )
                else int(value)
                if isinstance(
                    value,
                    int,
                )
                else value
            )
            for key, value
            in diagnostics.items()
        },
    }


# ------------------------------------------------------------
# 5. BUILD ALL SIX TABLES
# ------------------------------------------------------------

print("\n" + "-" * 80)
print(
    "BUILDING ALL UNICODE / "
    "TRANSLITERATION TABLES"
)
print("-" * 80)

results = {}

for split in [
    "train",
    "test",
]:

    for source in [
        "source1",
        "source2",
        "source3",
    ]:

        results[
            f"{split}_{source}"
        ] = build_unicode_table(
            split=split,
            source=source,
        )


# ------------------------------------------------------------
# 6. MANIFEST
# ------------------------------------------------------------

feature_names = [

    "name_ascii_translit",
    "name_translit_changed",
    "name_translit_length",
    "name_translit_token_count",
    "name_translit_digit_count",
    "name_translit_contains_digit",
    "name_translit_length_delta",
    "name_translit_token_delta",
    "name_translit_length_ratio",

    "address_ascii_translit",
    "address_translit_changed",
    "address_translit_length",
    "address_translit_token_count",
    "address_translit_digit_count",
    "address_translit_contains_digit",
    "address_translit_length_delta",
    "address_translit_token_delta",
    "address_translit_length_ratio",

    "name_unicode_byte_excess",
    "address_unicode_byte_excess",

    "name_translit_residual_non_ascii",
    "address_translit_residual_non_ascii",
]

manifest = {

    "cell": "71D_v2",

    "purpose":
        "Unicode and transliteration auxiliary "
        "representations",

    "model_training": False,

    "ground_truth_used": False,

    "predictions_generated": False,

    "canonical_representation_preserved": True,

    "transliteration_engine":
        TRANSLITERATION_ENGINE,

    "design": {
        "unicode_representation_unchanged": True,
        "transliteration_is_auxiliary": True,
        "only_non_ascii_rows_transliterated": True,
        "lengths_derived_locally": True,
    },

    "implementation_fixes": [
        "name length derived from name_norm",
        "address length derived from address_norm",
        "no dependency on omitted length columns",
    ],

    "feature_names":
        feature_names,

    "outputs":
        results,
}

manifest_path = (
    OUT_DIR
    / "manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 7. FINAL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 71D v2 COMPLETE")
print("=" * 80)

for key, info in results.items():

    print(
        f"{key:18s} "
        f"{info['rows']:,} rows | "
        f"{info['columns']} columns"
    )

print(
    f"\nManifest:\n"
    f"{manifest_path}"
)

print("\n" + "-" * 80)
print("GUARDRAILS")
print("-" * 80)

print("✅ No model trained")
print("✅ No model loaded")
print("✅ No predictions generated")
print("✅ No ground truth used")
print("✅ Canonical Unicode strings preserved")
print("✅ Transliteration stored separately")
print("✅ Train/test kept separate")
print("✅ Entity IDs checked")
print("✅ Only non-ASCII rows transliterated")
print(
    f"✅ Transliteration engine: "
    f"{TRANSLITERATION_ENGINE}"
)
print("✅ Required lengths derived locally")

print("\nFEATURE ENGINEERING CONTINUES.")

AMLC 2026 — CELL 71D v2
FEATURE ENGINEERING ONLY — UNICODE / TRANSLITERATION
ROOT       : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1
NAME DIR   : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/name_representation_v2
ADDRESS DIR: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/address_representation_v2
OUTPUT     : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/unicode_transliteration_v2

Transliteration engine: anyascii

--------------------------------------------------------------------------------
BUILDING ALL UNICODE / TRANSLITERATION TABLES
--------------------------------------------------------------------------------

BUILDING TRAIN / SOURCE1
Name input   : train_source1_name_representation.parquet
Address input: train_source1_address_representation.parquet
Name rows   : 2,206,821
Address rows: 2,206,821

Transliterating name representations...
Transliterating address representations...
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features

In [56]:
# ============================================================
# AMLC 2026 — CELL 71E
# FEATURE ENGINEERING ONLY
#
# PAIRWISE NAME FEATURES
#
# NO MODEL
# NO TRAINING
# NO GROUND TRUTH
# NO PREDICTIONS
#
# INPUT:
#   durable training candidate pair sample
#   71B name representations
#   71D transliteration representations
#
# OUTPUT:
#   features/pair_name_v1/
#
# Generates pair-level NAME features only.
# ============================================================

from pathlib import Path
import json
import gc
import re
import shutil
import subprocess

import numpy as np
import polars as pl

from rapidfuzz import fuzz, process


print("=" * 80)
print("AMLC 2026 — CELL 71E")
print("FEATURE ENGINEERING ONLY — PAIRWISE NAME FEATURES")
print("=" * 80)


# ============================================================
# 0. PATHS
# ============================================================

ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

PAIR_SAMPLE_PATH = (
    ROOT
    / "state"
    / "train_pair_sample_v1"
    / "baseline_candidate_pairs.parquet"
)

NAME_DIR = (
    ROOT
    / "features"
    / "name_representation_v2"
)

UNICODE_DIR = (
    ROOT
    / "features"
    / "unicode_transliteration_v2"
)

OUT_DIR = (
    ROOT
    / "features"
    / "pair_name_v1"
)

PART_DIR = (
    OUT_DIR
    / "parts"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PART_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(f"ROOT          : {ROOT}")
print(f"PAIR SAMPLE   : {PAIR_SAMPLE_PATH}")
print(f"NAME FEATURES : {NAME_DIR}")
print(f"UNICODE       : {UNICODE_DIR}")
print(f"OUTPUT        : {OUT_DIR}")


# ============================================================
# 1. INPUT CHECKS
# ============================================================

required_paths = [
    ROOT,
    PAIR_SAMPLE_PATH,
    NAME_DIR,
    UNICODE_DIR,
]

for path in required_paths:

    if not path.exists():

        raise RuntimeError(
            f"Required artifact does not exist:\n{path}"
        )


# ============================================================
# 2. DISK SAFETY
# ============================================================

print("\n" + "-" * 80)
print("DISK SAFETY CHECK")
print("-" * 80)

disk_result = subprocess.run(
    [
        "df",
        "-B1",
        "/kaggle/working",
    ],
    capture_output=True,
    text=True,
    check=False,
)

print(
    disk_result.stdout
)

lines = [
    line
    for line in disk_result.stdout.splitlines()
    if line.strip()
]

if len(lines) >= 2:

    parts = lines[-1].split()

    try:
        available_bytes = int(parts[3])
    except Exception:
        available_bytes = -1

else:

    available_bytes = -1


available_gb = (
    available_bytes
    / (1024 ** 3)
    if available_bytes >= 0
    else -1
)

print(
    f"Available disk: "
    f"{available_gb:.2f} GB"
)


# We expect pairwise name features to need breathing room.
# Do not start a massive operation if the filesystem is
# already nearly full.
MIN_FREE_GB = 3.0

if (
    available_gb >= 0
    and available_gb < MIN_FREE_GB
):

    raise RuntimeError(
        f"Only {available_gb:.2f} GB free. "
        f"Need at least {MIN_FREE_GB:.1f} GB "
        "before building pairwise features."
    )


# ============================================================
# 3. PAIR SAMPLE SCHEMA
# ============================================================

print("\n" + "-" * 80)
print("PAIR SAMPLE SCHEMA")
print("-" * 80)

pair_schema = (
    pl.scan_parquet(
        str(PAIR_SAMPLE_PATH)
    )
    .collect_schema()
    .names()
)

print(
    pair_schema
)


# ============================================================
# 4. COLUMN RESOLUTION
# ============================================================

def resolve_column(
    columns,
    candidates,
):
    mapping = {
        c.strip().lower(): c
        for c in columns
    }

    for candidate in candidates:

        hit = mapping.get(
            candidate.strip().lower()
        )

        if hit is not None:
            return hit

    return None


s1_id_col = resolve_column(
    pair_schema,
    [
        "s1_entity_id",
        "source1_entity_id",
    ],
)

candidate_id_col = resolve_column(
    pair_schema,
    [
        "candidate_entity_id",
        "matched_entity_id",
    ],
)

source_col = resolve_column(
    pair_schema,
    [
        "source",
        "candidate_source",
    ],
)


if s1_id_col is None:

    raise RuntimeError(
        "Could not resolve source1 entity ID "
        f"from pair sample columns:\n{pair_schema}"
    )

if candidate_id_col is None:

    raise RuntimeError(
        "Could not resolve candidate entity ID "
        f"from pair sample columns:\n{pair_schema}"
    )

if source_col is None:

    raise RuntimeError(
        "Could not resolve candidate source "
        f"from pair sample columns:\n{pair_schema}"
    )


print(
    f"S1 ID column        : {s1_id_col}"
)

print(
    f"Candidate ID column : {candidate_id_col}"
)

print(
    f"Source column       : {source_col}"
)


# ============================================================
# 5. LOAD PAIR INDEX
# ============================================================

print("\n" + "-" * 80)
print("LOADING TRAINING PAIR INDEX")
print("-" * 80)

pairs = (
    pl.scan_parquet(
        str(PAIR_SAMPLE_PATH)
    )
    .select(
        [
            pl.col(s1_id_col)
            .cast(
                pl.Utf8,
                strict=False,
            )
            .str.strip_chars()
            .alias("s1_entity_id"),

            pl.col(candidate_id_col)
            .cast(
                pl.Utf8,
                strict=False,
            )
            .str.strip_chars()
            .alias(
                "candidate_entity_id"
            ),

            pl.col(source_col)
            .alias("source_raw"),
        ]
    )
    .collect()
)

print(
    f"Pair rows: {pairs.height:,}"
)


# ============================================================
# 6. NORMALIZE SOURCE LABELS
# ============================================================

def normalize_source_value(
    value
):
    """
    Explicitly handles common source encodings
    seen in candidate-generation pipelines.

    We DO NOT silently guess arbitrary values.
    """

    if value is None:
        return None

    text = str(value).strip().lower()

    text = text.replace(
        " ",
        "",
    )

    aliases_s2 = {
        "2",
        "2.0",
        "s2",
        "source2",
        "source_2",
        "src2",
        "source-2",
    }

    aliases_s3 = {
        "3",
        "3.0",
        "s3",
        "source3",
        "source_3",
        "src3",
        "source-3",
    }

    if text in aliases_s2:
        return "source2"

    if text in aliases_s3:
        return "source3"

    return None


unique_sources = (
    pairs
    .select("source_raw")
    .unique()
)

raw_source_values = (
    unique_sources
    .get_column("source_raw")
    .to_list()
)

print(
    "\nRaw source values:"
)

for value in raw_source_values:
    print(
        f"  {repr(value)}"
    )


source_mapping = {}

unknown_values = []

for value in raw_source_values:

    normalized = normalize_source_value(
        value
    )

    if normalized is None:
        unknown_values.append(
            value
        )
    else:
        source_mapping[str(value)] = (
            normalized
        )


if unknown_values:

    raise RuntimeError(
        "Unrecognized source values in pair sample:\n"
        f"{unknown_values}\n\n"
        "No feature generation was started."
    )


pairs = pairs.with_columns(

    pl.col("source_raw")
    .cast(
        pl.Utf8,
        strict=False,
    )
    .map_elements(
        normalize_source_value,
        return_dtype=pl.Utf8,
        skip_nulls=False,
    )
    .alias("source")
)

print(
    "\nNormalized source counts:"
)

print(
    pairs
    .group_by("source")
    .len()
)


# ============================================================
# 7. NAME FEATURE INPUT COLUMNS
# ============================================================

NAME_COLUMNS = [
    "entity_id",
    "business_name",
    "name_norm",

    "name_first_token",
    "name_last_token",
    "name_first2_tokens",
    "name_last2_tokens",

    "name_initials",
    "name_first_initial",
    "name_last_initial",

    "name_first_last_signature",
    "name_initial_signature",

    "name_token_count",
    "name_unique_token_count",

    "name_numeric_signature",
]

UNICODE_NAME_COLUMNS = [
    "entity_id",
    "name_ascii_translit",
]


# ============================================================
# 8. FEATURE HELPERS
# ============================================================

FUZZ_SCORERS = {
    "ratio": fuzz.ratio,
    "qratio": fuzz.QRatio,
    "wr": fuzz.WRatio,
    "partial_ratio": fuzz.partial_ratio,
    "token_sort_ratio": fuzz.token_sort_ratio,
    "token_set_ratio": fuzz.token_set_ratio,
}


def rapid_pair_score(
    left,
    right,
    scorer,
):
    """
    RapidFuzz cpdist:
      - workers=-1
      - float32
      - normalized to [0,1]
    """

    result = process.cpdist(
        left,
        right,
        scorer=scorer,
        workers=-1,
        dtype=np.float32,
    )

    return (
        np.asarray(
            result,
            dtype=np.float32,
        )
        / np.float32(100.0)
    )


def safe_token_sets(
    value
):
    if value is None:
        return set()

    text = str(value)

    if not text:
        return set()

    return set(
        token
        for token in text.split()
        if token
    )


def token_overlap_features(
    left,
    right,
):
    """
    Return:
        Jaccard
        Dice
        left containment
        right containment
    """

    a = safe_token_sets(left)
    b = safe_token_sets(right)

    if not a and not b:

        return (
            1.0,
            1.0,
            1.0,
            1.0,
        )

    if not a or not b:

        return (
            0.0,
            0.0,
            0.0,
            0.0,
        )

    inter = len(
        a.intersection(b)
    )

    union = len(
        a.union(b)
    )

    jaccard = (
        inter / union
        if union
        else 0.0
    )

    dice = (
        2.0 * inter
        /
        (len(a) + len(b))
        if (
            len(a) + len(b)
        )
        else 0.0
    )

    left_containment = (
        inter / len(a)
        if a
        else 0.0
    )

    right_containment = (
        inter / len(b)
        if b
        else 0.0
    )

    return (
        jaccard,
        dice,
        left_containment,
        right_containment,
    )


def character_ngram_jaccard(
    left,
    right,
    n,
):
    """
    Character n-gram Jaccard.
    """

    a = (
        str(left)
        if left is not None
        else ""
    )

    b = (
        str(right)
        if right is not None
        else ""
    )

    if a == "" and b == "":
        return 1.0

    if a == "" or b == "":
        return 0.0

    if len(a) < n:
        grams_a = {a}
    else:
        grams_a = {
            a[i:i+n]
            for i in range(
                len(a) - n + 1
            )
        }

    if len(b) < n:
        grams_b = {b}
    else:
        grams_b = {
            b[i:i+n]
            for i in range(
                len(b) - n + 1
            )
        }

    union = grams_a.union(
        grams_b
    )

    if not union:
        return 1.0

    return (
        len(
            grams_a.intersection(
                grams_b
            )
        )
        / len(union)
    )


def prefix_suffix_features(
    left,
    right,
):
    a = (
        str(left)
        if left is not None
        else ""
    )

    b = (
        str(right)
        if right is not None
        else ""
    )

    if not a and not b:
        return (
            0.0,
            0.0,
            1.0,
            1.0,
        )

    max_common = min(
        len(a),
        len(b),
    )

    prefix = 0

    while (
        prefix < max_common
        and a[prefix] == b[prefix]
    ):
        prefix += 1

    suffix = 0

    while (
        suffix < max_common
        and a[
            len(a) - suffix - 1
        ]
        ==
        b[
            len(b) - suffix - 1
        ]
    ):
        suffix += 1

    prefix_ratio = (
        prefix / max_common
        if max_common
        else 0.0
    )

    suffix_ratio = (
        suffix / max_common
        if max_common
        else 0.0
    )

    prefix_relative = (
        prefix
        /
        max(
            len(a),
            1,
        )
    )

    suffix_relative = (
        suffix
        /
        max(
            len(a),
            1,
        )
    )

    return (
        float(prefix_ratio),
        float(suffix_ratio),
        float(prefix_relative),
        float(suffix_relative),
    )


# ============================================================
# 9. COMPUTE ONE PAIR CHUNK
# ============================================================

def compute_name_features(
    chunk,
):

    # --------------------------------------------------------
    # Exact / structural features
    # --------------------------------------------------------

    chunk = chunk.with_columns(

        (
            pl.col("s1_name_norm")
            ==
            pl.col("candidate_name_norm")
        )
        .cast(pl.Int8)
        .alias(
            "name_exact_norm"
        ),

        (
            pl.col("s1_business_name")
            ==
            pl.col("candidate_business_name")
        )
        .cast(pl.Int8)
        .alias(
            "name_raw_exact"
        ),

        (
            pl.col(
                "s1_name_ascii_translit"
            )
            ==
            pl.col(
                "candidate_name_ascii_translit"
            )
        )
        .cast(pl.Int8)
        .alias(
            "name_exact_translit"
        ),

        (
            pl.col(
                "s1_first_token"
            )
            ==
            pl.col(
                "candidate_first_token"
            )
        )
        .cast(pl.Int8)
        .alias(
            "name_first_token_eq"
        ),

        (
            pl.col(
                "s1_last_token"
            )
            ==
            pl.col(
                "candidate_last_token"
            )
        )
        .cast(pl.Int8)
        .alias(
            "name_last_token_eq"
        ),

        (
            pl.col(
                "s1_first2_tokens"
            )
            ==
            pl.col(
                "candidate_first2_tokens"
            )
        )
        .cast(pl.Int8)
        .alias(
            "name_first2_eq"
        ),

        (
            pl.col(
                "s1_last2_tokens"
            )
            ==
            pl.col(
                "candidate_last2_tokens"
            )
        )
        .cast(pl.Int8)
        .alias(
            "name_last2_eq"
        ),

        (
            pl.col(
                "s1_initials"
            )
            ==
            pl.col(
                "candidate_initials"
            )
        )
        .cast(pl.Int8)
        .alias(
            "name_initials_eq"
        ),

        (
            pl.col(
                "s1_first_last_signature"
            )
            ==
            pl.col(
                "candidate_first_last_signature"
            )
        )
        .cast(pl.Int8)
        .alias(
            "name_first_last_eq"
        ),

        (
            pl.col(
                "s1_initial_signature"
            )
            ==
            pl.col(
                "candidate_initial_signature"
            )
        )
        .cast(pl.Int8)
        .alias(
            "name_initial_signature_eq"
        ),

        (
            pl.col(
                "s1_numeric_signature"
            )
            ==
            pl.col(
                "candidate_numeric_signature"
            )
        )
        .cast(pl.Int8)
        .alias(
            "name_numeric_signature_eq"
        ),
    )

    # --------------------------------------------------------
    # Length / token counts
    # --------------------------------------------------------

    chunk = chunk.with_columns(

        (
            pl.col("s1_name_length")
            -
            pl.col("candidate_name_length")
        )
        .abs()
        .cast(pl.Int16)
        .alias(
            "name_length_diff"
        ),

        (
            pl.col("s1_name_length")
            /
            pl.col(
                "candidate_name_length"
            )
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "name_length_ratio_s1_candidate"
        ),

        (
            pl.col(
                "candidate_name_length"
            )
            /
            pl.col("s1_name_length")
            .cast(pl.Float32)
            .clip(
                lower_bound=1.0
            )
        )
        .alias(
            "name_length_ratio_candidate_s1"
        ),

        (
            pl.col("s1_token_count")
            -
            pl.col("candidate_token_count")
        )
        .abs()
        .cast(pl.Int16)
        .alias(
            "name_token_count_diff"
        ),

        (
            pl.col(
                "s1_unique_token_count"
            )
            -
            pl.col(
                "candidate_unique_token_count"
            )
        )
        .abs()
        .cast(pl.Int16)
        .alias(
            "name_unique_token_count_diff"
        ),
    )

    # --------------------------------------------------------
    # RapidFuzz on canonical Unicode representation
    # --------------------------------------------------------

    left_name = (
        chunk
        .get_column(
            "s1_name_norm"
        )
        .fill_null("")
        .to_list()
    )

    right_name = (
        chunk
        .get_column(
            "candidate_name_norm"
        )
        .fill_null("")
        .to_list()
    )

    for metric_name, scorer in FUZZ_SCORERS.items():

        score = rapid_pair_score(
            left_name,
            right_name,
            scorer,
        )

        chunk = chunk.with_columns(
            pl.Series(
                f"name_{metric_name}",
                score,
            )
        )

    # --------------------------------------------------------
    # RapidFuzz on transliterated representation
    # --------------------------------------------------------

    left_translit = (
        chunk
        .get_column(
            "s1_name_ascii_translit"
        )
        .fill_null("")
        .to_list()
    )

    right_translit = (
        chunk
        .get_column(
            "candidate_name_ascii_translit"
        )
        .fill_null("")
        .to_list()
    )

    translit_scorers = {
        "ratio": fuzz.ratio,
        "token_sort_ratio":
            fuzz.token_sort_ratio,
        "token_set_ratio":
            fuzz.token_set_ratio,
        "partial_ratio":
            fuzz.partial_ratio,
    }

    for metric_name, scorer in translit_scorers.items():

        score = rapid_pair_score(
            left_translit,
            right_translit,
            scorer,
        )

        chunk = chunk.with_columns(
            pl.Series(
                f"name_translit_{metric_name}",
                score,
            )
        )

    # --------------------------------------------------------
    # Token overlap
    # --------------------------------------------------------

    overlap_struct = (
        pl.struct(
            [
                "s1_name_norm",
                "candidate_name_norm",
            ]
        )
        .map_elements(
            lambda row: {
                "name_token_jaccard":
                    token_overlap_features(
                        row[
                            "s1_name_norm"
                        ],
                        row[
                            "candidate_name_norm"
                        ],
                    )[0],

                "name_token_dice":
                    token_overlap_features(
                        row[
                            "s1_name_norm"
                        ],
                        row[
                            "candidate_name_norm"
                        ],
                    )[1],

                "name_token_containment_s1":
                    token_overlap_features(
                        row[
                            "s1_name_norm"
                        ],
                        row[
                            "candidate_name_norm"
                        ],
                    )[2],

                "name_token_containment_candidate":
                    token_overlap_features(
                        row[
                            "s1_name_norm"
                        ],
                        row[
                            "candidate_name_norm"
                        ],
                    )[3],
            },
            return_dtype=pl.Struct(
                {
                    "name_token_jaccard":
                        pl.Float32,

                    "name_token_dice":
                        pl.Float32,

                    "name_token_containment_s1":
                        pl.Float32,

                    "name_token_containment_candidate":
                        pl.Float32,
                }
            ),
        )
    )

    chunk = chunk.with_columns(
        overlap_struct
    ).unnest(
        "literal"
    ) if "literal" in chunk.columns else chunk

    # The expression above can have different temporary names
    # across Polars versions. Recompute safely if required.

    overlap_columns = {
        "name_token_jaccard",
        "name_token_dice",
        "name_token_containment_s1",
        "name_token_containment_candidate",
    }

    missing_overlap = (
        overlap_columns
        -
        set(chunk.columns)
    )

    if missing_overlap:

        overlap_struct_expr = (
            pl.struct(
                [
                    "s1_name_norm",
                    "candidate_name_norm",
                ]
            )
            .map_elements(
                lambda row: (
                    token_overlap_features(
                        row["s1_name_norm"],
                        row[
                            "candidate_name_norm"
                        ],
                    )
                ),
                return_dtype=pl.List(
                    pl.Float32
                ),
            )
        )

        chunk = chunk.with_columns(
            overlap_struct_expr
            .alias(
                "_name_overlap"
            )
        )

        chunk = chunk.with_columns(

            pl.col("_name_overlap")
            .list.get(0)
            .alias(
                "name_token_jaccard"
            ),

            pl.col("_name_overlap")
            .list.get(1)
            .alias(
                "name_token_dice"
            ),

            pl.col("_name_overlap")
            .list.get(2)
            .alias(
                "name_token_containment_s1"
            ),

            pl.col("_name_overlap")
            .list.get(3)
            .alias(
                "name_token_containment_candidate"
            ),
        ).drop(
            "_name_overlap"
        )

    # --------------------------------------------------------
    # Character n-gram Jaccard
    # --------------------------------------------------------

    for n in [2, 3]:

        gram_expr = (
            pl.struct(
                [
                    "s1_name_norm",
                    "candidate_name_norm",
                ]
            )
            .map_elements(
                lambda row, n=n:
                    character_ngram_jaccard(
                        row[
                            "s1_name_norm"
                        ],
                        row[
                            "candidate_name_norm"
                        ],
                        n,
                    ),
                return_dtype=pl.Float32,
            )
        )

        chunk = chunk.with_columns(
            gram_expr.alias(
                f"name_char_{n}gram_jaccard"
            )
        )

    # --------------------------------------------------------
    # Prefix / suffix
    # --------------------------------------------------------

    prefix_expr = (
        pl.struct(
            [
                "s1_name_norm",
                "candidate_name_norm",
            ]
        )
        .map_elements(
            lambda row: (
                prefix_suffix_features(
                    row["s1_name_norm"],
                    row[
                        "candidate_name_norm"
                    ],
                )
            ),
            return_dtype=pl.List(
                pl.Float32
            ),
        )
    )

    chunk = chunk.with_columns(
        prefix_expr.alias(
            "_name_prefix_suffix"
        )
    )

    chunk = chunk.with_columns(

        pl.col(
            "_name_prefix_suffix"
        )
        .list.get(0)
        .alias(
            "name_prefix_ratio"
        ),

        pl.col(
            "_name_prefix_suffix"
        )
        .list.get(1)
        .alias(
            "name_suffix_ratio"
        ),

        pl.col(
            "_name_prefix_suffix"
        )
        .list.get(2)
        .alias(
            "name_prefix_relative"
        ),

        pl.col(
            "_name_prefix_suffix"
        )
        .list.get(3)
        .alias(
            "name_suffix_relative"
        ),
    ).drop(
        "_name_prefix_suffix"
    )

    # --------------------------------------------------------
    # NUMERIC DIGIT-ONLY SIMILARITY
    # --------------------------------------------------------

    left_numeric = (
        chunk
        .get_column(
            "s1_numeric_signature"
        )
        .fill_null("")
        .to_list()
    )

    right_numeric = (
        chunk
        .get_column(
            "candidate_numeric_signature"
        )
        .fill_null("")
        .to_list()
    )

    numeric_score = rapid_pair_score(
        left_numeric,
        right_numeric,
        fuzz.ratio,
    )

    chunk = chunk.with_columns(
        pl.Series(
            "name_numeric_ratio",
            numeric_score,
        )
    )

    # --------------------------------------------------------
    # FINAL FEATURE-ONLY TABLE
    # --------------------------------------------------------

    feature_columns = [

        "s1_entity_id",
        "candidate_entity_id",
        "source",

        "name_exact_norm",
        "name_raw_exact",
        "name_exact_translit",

        "name_first_token_eq",
        "name_last_token_eq",
        "name_first2_eq",
        "name_last2_eq",
        "name_initials_eq",
        "name_first_last_eq",
        "name_initial_signature_eq",
        "name_numeric_signature_eq",

        "name_length_diff",
        "name_length_ratio_s1_candidate",
        "name_length_ratio_candidate_s1",

        "name_token_count_diff",
        "name_unique_token_count_diff",

        "name_ratio",
        "name_qratio",
        "name_wr",
        "name_partial_ratio",
        "name_token_sort_ratio",
        "name_token_set_ratio",

        "name_translit_ratio",
        "name_translit_token_sort_ratio",
        "name_translit_token_set_ratio",
        "name_translit_partial_ratio",

        "name_token_jaccard",
        "name_token_dice",
        "name_token_containment_s1",
        "name_token_containment_candidate",

        "name_char_2gram_jaccard",
        "name_char_3gram_jaccard",

        "name_prefix_ratio",
        "name_suffix_ratio",
        "name_prefix_relative",
        "name_suffix_relative",

        "name_numeric_ratio",
    ]

    return chunk.select(
        feature_columns
    )


# ============================================================
# 10. LOAD ENTITY FEATURE TABLES
# ============================================================

def load_name_entity_features(
    split,
    source,
):

    name_path = (
        NAME_DIR
        / f"{split}_{source}"
        "_name_representation.parquet"
    )

    translit_path = (
        UNICODE_DIR
        / f"{split}_{source}"
        "_unicode_transliteration.parquet"
    )

    if not name_path.exists():

        raise RuntimeError(
            f"Missing name representation:\n"
            f"{name_path}"
        )

    if not translit_path.exists():

        raise RuntimeError(
            f"Missing transliteration representation:\n"
            f"{translit_path}"
        )

    name_cols = [
        "entity_id",
        "business_name",
        "name_norm",

        "name_first_token",
        "name_last_token",
        "name_first2_tokens",
        "name_last2_tokens",

        "name_initials",
        "name_first_initial",
        "name_last_initial",

        "name_first_last_signature",
        "name_initial_signature",

        "name_token_count",
        "name_unique_token_count",

        "name_numeric_signature",

    ]

    translit_cols = [
        "entity_id",
        "name_ascii_translit",
    ]

    name_df = (
        pl.scan_parquet(
            str(name_path)
        )
        .select(
            name_cols
        )
        .collect()
    )

    translit_df = (
        pl.scan_parquet(
            str(translit_path)
        )
        .select(
            translit_cols
        )
        .collect()
    )

    # --------------------------------------------------------
    # Derive name length locally.
    # --------------------------------------------------------

    name_df = name_df.with_columns(

        pl.col("name_norm")
        .str.len_chars()
        .cast(pl.Int32)
        .alias(
            "name_length"
        )
    )

    # --------------------------------------------------------
    # Join transliteration.
    # --------------------------------------------------------

    entity_df = (
        name_df
        .join(
            translit_df,
            on="entity_id",
            how="inner",
        )
    )

    if entity_df.height != name_df.height:

        raise RuntimeError(
            f"{split}/{source}: "
            "transliteration join changed "
            "entity count."
        )

    return entity_df


# ============================================================
# 11. LOAD SOURCE1 ONCE
# ============================================================

print("\n" + "-" * 80)
print("LOADING SOURCE1 NAME FEATURES")
print("-" * 80)

s1_features = load_name_entity_features(
    "train",
    "source1",
)

print(
    f"S1 feature rows: "
    f"{s1_features.height:,}"
)


# ============================================================
# 12. PROCESS SOURCE2 / SOURCE3
# ============================================================

CHUNK_SIZE = 100_000

all_part_manifests = []


for candidate_source in [
    "source2",
    "source3",
]:

    print("\n" + "=" * 80)
    print(
        f"PROCESSING NAME PAIRS: "
        f"SOURCE1 × {candidate_source.upper()}"
    )
    print("=" * 80)

    pair_source = (
        pairs
        .filter(
            pl.col("source")
            ==
            candidate_source
        )
        .select(
            [
                "s1_entity_id",
                "candidate_entity_id",
                "source",
            ]
        )
    )

    print(
        f"Pairs for {candidate_source}: "
        f"{pair_source.height:,}"
    )

    if pair_source.height == 0:

        raise RuntimeError(
            f"No candidate pairs for "
            f"{candidate_source}."
        )

    print(
        f"Loading {candidate_source} "
        "entity features..."
    )

    candidate_features = (
        load_name_entity_features(
            "train",
            candidate_source,
        )
    )

    print(
        f"{candidate_source} feature rows: "
        f"{candidate_features.height:,}"
    )

    # --------------------------------------------------------
    # PREPARE S1 JOIN
    # --------------------------------------------------------

    left = s1_features

    # --------------------------------------------------------
    # JOIN ENTITY FEATURES TO PAIRS
    # --------------------------------------------------------

    print(
        "Joining entity representations "
        "to candidate pairs..."
    )

    joined = (
        pair_source
        .join(
            left,
            left_on="s1_entity_id",
            right_on="entity_id",
            how="inner",
        )
        .rename(
            {
                "business_name":
                    "s1_business_name",

                "name_norm":
                    "s1_name_norm",

                "name_first_token":
                    "s1_first_token",

                "name_last_token":
                    "s1_last_token",

                "name_first2_tokens":
                    "s1_first2_tokens",

                "name_last2_tokens":
                    "s1_last2_tokens",

                "name_initials":
                    "s1_initials",

                "name_first_initial":
                    "s1_first_initial",

                "name_last_initial":
                    "s1_last_initial",

                "name_first_last_signature":
                    "s1_first_last_signature",

                "name_initial_signature":
                    "s1_initial_signature",

                "name_token_count":
                    "s1_token_count",

                "name_unique_token_count":
                    "s1_unique_token_count",

                "name_numeric_signature":
                    "s1_numeric_signature",

                "name_length":
                    "s1_name_length",

                "name_ascii_translit":
                    "s1_name_ascii_translit",
            }
        )
    )

    joined = (
        joined
        .join(
            candidate_features,
            left_on="candidate_entity_id",
            right_on="entity_id",
            how="inner",
            suffix="_candidate",
        )
        .rename(
            {
                "business_name":
                    "candidate_business_name",

                "name_norm":
                    "candidate_name_norm",

                "name_first_token":
                    "candidate_first_token",

                "name_last_token":
                    "candidate_last_token",

                "name_first2_tokens":
                    "candidate_first2_tokens",

                "name_last2_tokens":
                    "candidate_last2_tokens",

                "name_initials":
                    "candidate_initials",

                "name_first_initial":
                    "candidate_first_initial",

                "name_last_initial":
                    "candidate_last_initial",

                "name_first_last_signature":
                    "candidate_first_last_signature",

                "name_initial_signature":
                    "candidate_initial_signature",

                "name_token_count":
                    "candidate_token_count",

                "name_unique_token_count":
                    "candidate_unique_token_count",

                "name_numeric_signature":
                    "candidate_numeric_signature",

                "name_length":
                    "candidate_name_length",

                "name_ascii_translit":
                    "candidate_name_ascii_translit",
            }
        )
    )

    expected_rows = (
        pair_source.height
    )

    if joined.height != expected_rows:

        print(
            f"[WARN] Join changed row count: "
            f"{expected_rows:,} -> "
            f"{joined.height:,}"
        )

        # This is not immediately fatal because the feature
        # table should make the loss visible. But a missing
        # entity representation means a pair cannot be scored.
        #
        # We fail only if the loss is unexpectedly large.
        loss_fraction = (
            expected_rows
            - joined.height
        ) / max(
            expected_rows,
            1,
        )

        print(
            f"Join loss fraction: "
            f"{loss_fraction:.8%}"
        )

        if loss_fraction > 0.001:

            raise RuntimeError(
                f"{candidate_source}: "
                f"more than 0.1% of candidate pairs "
                "lost during entity-feature joins."
            )

    print(
        f"Joined rows: "
        f"{joined.height:,}"
    )

    # --------------------------------------------------------
    # CHUNKED FEATURE COMPUTATION
    # --------------------------------------------------------

    source_part_dir = (
        PART_DIR
        / candidate_source
    )

    source_part_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    source_manifest = []

    total_rows = joined.height

    part_number = 0

    for start in range(
        0,
        total_rows,
        CHUNK_SIZE,
    ):

        end = min(
            start + CHUNK_SIZE,
            total_rows,
        )

        print(
            f"\n[{candidate_source}] "
            f"chunk {part_number:04d} "
            f"rows {start:,}:{end:,}"
        )

        chunk = joined.slice(
            start,
            end - start,
        )

        feature_chunk = (
            compute_name_features(
                chunk
            )
        )

        output_path = (
            source_part_dir
            / f"part_{part_number:04d}.parquet"
        )

        feature_chunk.write_parquet(
            output_path,
            compression="zstd",
        )

        source_manifest.append(
            {
                "part":
                    part_number,

                "path":
                    str(output_path),

                "rows":
                    feature_chunk.height,

                "columns":
                    len(feature_chunk.columns),
            }
        )

        print(
            f"Saved: {output_path}"
        )

        print(
            f"Rows: "
            f"{feature_chunk.height:,}"
        )

        del chunk
        del feature_chunk

        gc.collect()

        part_number += 1

    # --------------------------------------------------------
    # SOURCE MANIFEST
    # --------------------------------------------------------

    source_manifest_path = (
        source_part_dir
        / "manifest.json"
    )

    source_manifest_payload = {
        "source":
            candidate_source,

        "rows":
            total_rows,

        "chunk_size":
            CHUNK_SIZE,

        "parts":
            source_manifest,
    }

    with open(
        source_manifest_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            source_manifest_payload,
            f,
            indent=2,
        )

    all_part_manifests.append(
        source_manifest_payload
    )

    print(
        f"\nSource manifest: "
        f"{source_manifest_path}"
    )

    del pair_source
    del candidate_features
    del joined

    gc.collect()


# ============================================================
# 13. GLOBAL MANIFEST
# ============================================================

feature_names = [

    "name_exact_norm",
    "name_raw_exact",
    "name_exact_translit",

    "name_first_token_eq",
    "name_last_token_eq",
    "name_first2_eq",
    "name_last2_eq",
    "name_initials_eq",
    "name_first_last_eq",
    "name_initial_signature_eq",
    "name_numeric_signature_eq",

    "name_length_diff",
    "name_length_ratio_s1_candidate",
    "name_length_ratio_candidate_s1",

    "name_token_count_diff",
    "name_unique_token_count_diff",

    "name_ratio",
    "name_qratio",
    "name_wr",
    "name_partial_ratio",
    "name_token_sort_ratio",
    "name_token_set_ratio",

    "name_translit_ratio",
    "name_translit_token_sort_ratio",
    "name_translit_token_set_ratio",
    "name_translit_partial_ratio",

    "name_token_jaccard",
    "name_token_dice",
    "name_token_containment_s1",
    "name_token_containment_candidate",

    "name_char_2gram_jaccard",
    "name_char_3gram_jaccard",

    "name_prefix_ratio",
    "name_suffix_ratio",
    "name_prefix_relative",
    "name_suffix_relative",

    "name_numeric_ratio",
]

global_manifest = {

    "cell":
        "71E",

    "purpose":
        "pairwise_name_feature_engineering",

    "model_training":
        False,

    "ground_truth_used":
        False,

    "predictions_generated":
        False,

    "pair_sample":
        str(PAIR_SAMPLE_PATH),

    "chunk_size":
        CHUNK_SIZE,

    "feature_count":
        len(feature_names),

    "feature_names":
        feature_names,

    "source_manifests":
        all_part_manifests,

    "rapidfuzz_scaling":
        "all fuzzy ratios converted from 0-100 to 0-1",

    "design_notes": [
        "canonical Unicode name comparison retained",
        "raw-name exact equality retained",
        "transliterated name comparison retained",
        "token overlap computed independently",
        "character n-gram Jaccard computed independently",
        "numeric signature comparison retained",
        "no ground-truth-derived features",
        "no model-derived features",
    ],
}


manifest_path = (
    OUT_DIR
    / "manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        global_manifest,
        f,
        indent=2,
    )


# ============================================================
# 14. FINAL DISK REPORT
# ============================================================

print("\n" + "=" * 80)
print("CELL 71E COMPLETE")
print("=" * 80)

print(
    f"Feature families: "
    f"{len(feature_names)}"
)

print(
    f"Manifest:\n"
    f"{manifest_path}"
)

print("\nOUTPUT PARTS:")

for source_manifest in all_part_manifests:

    total = sum(
        part["rows"]
        for part in source_manifest["parts"]
    )

    print(
        f"  {source_manifest['source']}: "
        f"{total:,} rows | "
        f"{len(source_manifest['parts'])} parts"
    )

print("\n" + "-" * 80)
print("FINAL GUARDRAILS")
print("-" * 80)

print("✅ No model trained")
print("✅ No model loaded")
print("✅ No predictions generated")
print("✅ No ground truth used")
print("✅ Canonical Unicode name features included")
print("✅ Transliteration features included")
print("✅ RapidFuzz ratios scaled to [0,1]")
print("✅ Numeric-name agreement included")
print("✅ Token overlap included")
print("✅ Character n-gram similarity included")
print("✅ Prefix/suffix structure included")
print("✅ Output written incrementally")
print("✅ No giant final dataframe retained in RAM")

print("\nFEATURE ENGINEERING CONTINUES.")

AMLC 2026 — CELL 71E
FEATURE ENGINEERING ONLY — PAIRWISE NAME FEATURES
ROOT          : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1
PAIR SAMPLE   : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/state/train_pair_sample_v1/baseline_candidate_pairs.parquet
NAME FEATURES : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/name_representation_v2
UNICODE       : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/unicode_transliteration_v2
OUTPUT        : /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/features/pair_name_v1

--------------------------------------------------------------------------------
DISK SAFETY CHECK
--------------------------------------------------------------------------------
Filesystem       1B-blocks        Used  Available Use% Mounted on
/dev/loop2     20957446144 16047726592 4892942336  77% /kaggle/working

Available disk: 4.56 GB

--------------------------------------------------------------------------------
PAIR SAMPLE SCHEMA
--------------------

In [57]:
# ============================================================
# AMLC 2026 — CELL 71F
# MASTER RECOVERY CHECKPOINT
#
# CRITICAL:
#   This creates ONE master ZIP containing the entire
#   /kaggle/working/AMLC2026 tree.
#
#   It uses ZIP STORE (-0) + MOVE (-m):
#
#       file -> ZIP -> verify entry -> delete original
#
#   This is specifically designed for the ~20 GB Kaggle
#   filesystem where we do NOT have enough free disk to create
#   a second full copy.
#
# BEFORE RUNNING:
#   SAVE A KAGGLE NOTEBOOK VERSION FIRST.
#
# RESULT:
#   /kaggle/working/AMLC2026_MASTER_CHECKPOINT_71E.zip
#
# NO MODEL TRAINING.
# NO GT.
# NO PREDICTIONS.
# ============================================================

from pathlib import Path
import hashlib
import json
import os
import subprocess
import time
import zipfile

print("=" * 80)
print("AMLC 2026 — CELL 71F")
print("MASTER RECOVERY CHECKPOINT")
print("=" * 80)


# ============================================================
# 0. PATHS
# ============================================================

AML_ROOT = Path(
    "/kaggle/working/AMLC2026"
)

ZIP_PATH = Path(
    "/kaggle/working/"
    "AMLC2026_MASTER_CHECKPOINT_71E.zip"
)

MANIFEST_NAME = (
    "AMLC2026_CHECKPOINT_MANIFEST.json"
)


if not AML_ROOT.exists():
    raise RuntimeError(
        f"AMLC root does not exist:\n{AML_ROOT}"
    )


# ============================================================
# 1. DISK STATUS
# ============================================================

print("\n" + "-" * 80)
print("CURRENT DISK")
print("-" * 80)

subprocess.run(
    [
        "df",
        "-h",
        "/kaggle/working",
    ],
    check=False,
)


# ============================================================
# 2. INVENTORY EVERYTHING
# ============================================================

print("\n" + "-" * 80)
print("BUILDING COMPLETE FILE INVENTORY")
print("-" * 80)

files = []

total_bytes = 0

for path in AML_ROOT.rglob("*"):

    if not path.is_file():
        continue

    # Never include an old master ZIP in a new master ZIP.
    if path.resolve() == ZIP_PATH.resolve():
        continue

    try:
        size = path.stat().st_size
    except OSError as exc:
        raise RuntimeError(
            f"Could not stat:\n{path}\n{exc}"
        )

    files.append(
        (
            path,
            size,
        )
    )

    total_bytes += size


files.sort(
    key=lambda item: str(item[0])
)

total_gb = (
    total_bytes
    / (1024 ** 3)
)

print(
    f"Files to archive : {len(files):,}"
)

print(
    f"Total source size: {total_gb:.3f} GB"
)


# ============================================================
# 3. ZERO-FILE SAFETY
# ============================================================

if not files:

    raise RuntimeError(
        "No files found under AMLC2026. "
        "Refusing to create an empty checkpoint."
    )


# ============================================================
# 4. SHOW MAJOR DIRECTORIES
# ============================================================

print("\nTop-level directories:")

top_level = {}

for path, size in files:

    try:
        relative = path.relative_to(
            AML_ROOT
        )
    except ValueError:
        continue

    first = relative.parts[0]

    top_level[first] = (
        top_level.get(first, 0)
        + size
    )

for name, size in sorted(
    top_level.items(),
    key=lambda x: -x[1],
):

    print(
        f"  {size / (1024 ** 3):8.3f} GB  "
        f"{name}"
    )


# ============================================================
# 5. REMOVE PREVIOUS FAILED ZIP ONLY
# ============================================================

if ZIP_PATH.exists():

    old_size = (
        ZIP_PATH.stat().st_size
        / (1024 ** 3)
    )

    print(
        f"\nExisting checkpoint ZIP found: "
        f"{old_size:.3f} GB"
    )

    print(
        "Removing old checkpoint ZIP."
    )

    ZIP_PATH.unlink()


# ============================================================
# 6. CREATE ZIP
#
# We deliberately use ZIP_STORED.
#
# Why?
#
#   - Parquet is already compressed.
#   - NPY is not worth spending huge CPU time compressing here.
#   - Store mode is much faster.
#   - The -m equivalent is implemented manually:
#         archive file
#         close archive entry
#         delete source
#
# This keeps disk usage approximately constant.
# ============================================================

print("\n" + "-" * 80)
print("CREATING MASTER CHECKPOINT")
print("-" * 80)

start_time = time.time()

manifest_entries = []

processed_files = 0

processed_bytes = 0


# ------------------------------------------------------------
# SHA256 helper
# ------------------------------------------------------------

def copy_into_zip_and_hash(
    zip_file,
    source_path,
    archive_name,
):
    """
    Stream source -> ZIP while calculating SHA256.

    No full file is loaded into RAM.
    """

    hasher = hashlib.sha256()

    with open(
        source_path,
        "rb",
    ) as src:

        with zip_file.open(
            archive_name,
            mode="w",
        ) as dst:

            while True:

                chunk = src.read(
                    8 * 1024 * 1024
                )

                if not chunk:
                    break

                hasher.update(
                    chunk
                )

                dst.write(
                    chunk
                )

    return hasher.hexdigest()


# ============================================================
# 7. WRITE ZIP
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_STORED,
    allowZip64=True,
) as zf:

    for index, (
        source_path,
        source_size,
    ) in enumerate(
        files,
        start=1,
    ):

        relative_path = source_path.relative_to(
            AML_ROOT
        )

        archive_name = str(
            Path("AMLC2026")
            / relative_path
        ).replace(
            "\\",
            "/",
        )

        print(
            f"[{index:5d}/{len(files):5d}] "
            f"{source_size / (1024 ** 2):8.1f} MB  "
            f"{relative_path}"
        )

        # ----------------------------------------------------
        # Add file.
        # ----------------------------------------------------

        sha256 = copy_into_zip_and_hash(
            zf,
            source_path,
            archive_name,
        )

        # ----------------------------------------------------
        # Record manifest metadata.
        # ----------------------------------------------------

        manifest_entries.append(
            {
                "archive_path":
                    archive_name,

                "source_relative_path":
                    str(relative_path),

                "size_bytes":
                    int(source_size),

                "sha256":
                    sha256,
            }
        )

        processed_files += 1
        processed_bytes += source_size

        # ----------------------------------------------------
        # ONLY NOW delete source.
        # ----------------------------------------------------

        try:
            source_path.unlink()

        except OSError as exc:

            raise RuntimeError(
                "Archive entry was written, "
                "but source deletion failed.\n"
                f"Source: {source_path}\n"
                f"Error : {exc}"
            )

        # ----------------------------------------------------
        # Progress every 25 files.
        # ----------------------------------------------------

        if (
            index == 1
            or index % 25 == 0
            or index == len(files)
        ):

            elapsed = (
                time.time()
                - start_time
            )

            rate_mb_s = (
                processed_bytes
                / (1024 ** 2)
                / max(
                    elapsed,
                    1e-6,
                )
            )

            print(
                f"    → archived "
                f"{processed_bytes / (1024 ** 3):.3f} GB "
                f"| "
                f"{rate_mb_s:.1f} MB/s"
            )


# ============================================================
# 8. WRITE MANIFEST INTO ZIP
#
# We need to reopen in append mode because the master archive
# is already finalized.
# ============================================================

manifest_payload = {

    "checkpoint":
        "AMLC2026_MASTER_CHECKPOINT_71E",

    "created_from":
        str(AML_ROOT),

    "created_at_epoch":
        time.time(),

    "archive":
        str(ZIP_PATH),

    "archive_format":
        "ZIP64",

    "compression":
        "stored",

    "source_file_count":
        len(files),

    "processed_file_count":
        processed_files,

    "source_total_bytes":
        int(total_bytes),

    "processed_total_bytes":
        int(processed_bytes),

    "feature_engineering_status": {
        "cell_71A":
            "completed",

        "cell_71B":
            "completed",

        "cell_71C":
            "completed",

        "cell_71D":
            "completed",

        "cell_71E":
            "completed",
    },

    "model_training":
        False,

    "ground_truth_used":
        False,

    "predictions_generated":
        False,

    "files":
        manifest_entries,
}


manifest_bytes = json.dumps(
    manifest_payload,
    indent=2,
).encode(
    "utf-8"
)


with zipfile.ZipFile(
    ZIP_PATH,
    mode="a",
    compression=zipfile.ZIP_STORED,
    allowZip64=True,
) as zf:

    zf.writestr(
        f"AMLC2026/{MANIFEST_NAME}",
        manifest_bytes,
    )


# ============================================================
# 9. REMOVE EMPTY DIRECTORIES
# ============================================================

print("\n" + "-" * 80)
print("CLEANING EMPTY DIRECTORIES")
print("-" * 80)

directories = sorted(
    [
        p
        for p in AML_ROOT.rglob("*")
        if p.is_dir()
    ],
    key=lambda p: len(
        p.parts
    ),
    reverse=True,
)

for directory in directories:

    try:
        directory.rmdir()

    except OSError:
        pass


# ============================================================
# 10. VERIFY ZIP STRUCTURE
# ============================================================

print("\n" + "-" * 80)
print("VERIFYING MASTER ZIP")
print("-" * 80)

if not ZIP_PATH.exists():

    raise RuntimeError(
        "Master checkpoint ZIP was not created."
    )


zip_size = (
    ZIP_PATH.stat().st_size
    / (1024 ** 3)
)

print(
    f"ZIP size: "
    f"{zip_size:.3f} GB"
)

with zipfile.ZipFile(
    ZIP_PATH,
    mode="r",
    allowZip64=True,
) as zf:

    bad_file = zf.testzip()

    if bad_file is not None:

        raise RuntimeError(
            "ZIP integrity test failed at:\n"
            f"{bad_file}"
        )

    names = zf.namelist()

    manifest_name = (
        f"AMLC2026/{MANIFEST_NAME}"
    )

    if manifest_name not in names:

        raise RuntimeError(
            "Checkpoint manifest is missing "
            "from ZIP."
        )

    manifest_inside = json.loads(
        zf.read(
            manifest_name
        )
    )

    print(
        f"ZIP entries: "
        f"{len(names):,}"
    )

    print(
        f"Manifest files: "
        f"{manifest_inside['processed_file_count']:,}"
    )


# ============================================================
# 11. FINAL DISK STATUS
# ============================================================

print("\n" + "=" * 80)
print("MASTER CHECKPOINT COMPLETE")
print("=" * 80)

print(
    f"MASTER ZIP:\n{ZIP_PATH}"
)

print(
    f"\nSource files archived : "
    f"{processed_files:,}"
)

print(
    f"Source bytes archived : "
    f"{processed_bytes / (1024 ** 3):.3f} GB"
)

print(
    f"ZIP size              : "
    f"{zip_size:.3f} GB"
)

print(
    f"Compression ratio     : "
    f"{zip_size / max(total_gb, 1e-9):.4f}"
)

print(
    "\nFILESYSTEM:"
)

subprocess.run(
    [
        "df",
        "-h",
        "/kaggle/working",
    ],
    check=False,
)

print("\nARCHIVE CHECK:")

subprocess.run(
    [
        "unzip",
        "-t",
        str(ZIP_PATH),
    ],
    check=False,
)

print("\n" + "-" * 80)
print("WHAT THIS CHECKPOINT CONTAINS")
print("-" * 80)

print(
    "✅ Entire /kaggle/working/AMLC2026 tree"
)

print(
    "✅ Current 71A rarity features"
)

print(
    "✅ Current 71B name representations"
)

print(
    "✅ Current 71C address representations"
)

print(
    "✅ Current 71D Unicode/transliteration"
)

print(
    "✅ Current 71E pairwise name features"
)

print(
    "✅ Semantic feature lake"
)

print(
    "✅ Embeddings"
)

print(
    "✅ Candidate-pair infrastructure"
)

print(
    "✅ State/checkpoint artifacts still present"
)

print(
    "✅ SHA256 manifest inside ZIP"
)

print(
    "\nNO MODEL TRAINED."
)

AMLC 2026 — CELL 71F
MASTER RECOVERY CHECKPOINT

--------------------------------------------------------------------------------
CURRENT DISK
--------------------------------------------------------------------------------
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   16G  4.5G  78% /kaggle/working

--------------------------------------------------------------------------------
BUILDING COMPLETE FILE INVENTORY
--------------------------------------------------------------------------------
Files to archive : 621
Total source size: 15.084 GB

Top-level directories:
    15.084 GB  FINAL_FEATURE_LAKE_V1

--------------------------------------------------------------------------------
CREATING MASTER CHECKPOINT
--------------------------------------------------------------------------------
[    1/  621]      0.0 MB  FINAL_FEATURE_LAKE_V1/embeddings/train/s1/full_record/COMPLETE
    → archived 0.000 GB | 0.1 MB/s
[    2/  621]     73.2 MB  FINAL_FEATURE_LAKE_V1/

OSError: [Errno 28] No space left on device